# نسخهٔ اصلاح‌شدهٔ اجرایی V1.7 — آموزش واقعی در این بسته اجرا نشده است

این بازنگری، اصلاح اجرایی/اعتبارسنجی است؛ بودجه HPO، مدل‌ها، تقسیم داده، آستانه‌ها،
تعداد bootstrap، روش p-value و خانواده‌های Holm نسبت به V1.6 تغییر نکرده‌اند.
روش p-value همچنان تقریب bootstrap-tail است، نه آزمون جایگشتی دقیق.
OOF همچنان برای توسعه استفاده می‌شود و معادل ارزیابی دست‌نخوردهٔ بیرونی نیست.

خروجی و شناسهٔ ذخیره‌سازی پیش‌فرض جدید `V17` است. خروجی نسخهٔ قبلی را کپی یا بازنویسی نکنید.
با `UNLOCK_REMAINING_MODELS = False` آغاز کنید. پایان تست CPU به معنی تأیید GPU نیست.
برای گزارش‌سازی مجدد بدون آموزش، ابزار `rebuild_master_statistics.py` داخل ZIP را ببینید.

## اصلاحات این نسخه
- ذخیرهٔ دقیق احتمال‌ها و کنترل رفت‌وبرگشت CSV.
- اعتبارسنجی تاریخچه، ترتیب مراحل، علت توقف زودهنگام و شمار snapshotهای SWA.
- آزمون مقدماتی head و finetune با کنترل آرایه‌های NaN/Inf و ثبت شکست.
- رسید نهایی تنها پس از تحلیل آماری و ثبت هش فایل‌های آن.
- پیش‌محاسبهٔ گروه‌های بیمار بدون تغییر ترتیب نمونه‌های تصادفی bootstrap.

---

# PNEUMONIA MASTER CAMPAIGN — M01…M12 — RUN-SAFE V1.7
## M07 Confirmation Gate First → Manual Unlock → Full Canonical Ablation Matrix

این Notebook **همه مدل‌های M01 تا M12 را داخل خودش دارد**.

ترتیب اجرای واقعی:

1. **M07@224 ابتدا اجرا می‌شود و سپس M07@320 و M07@384 در همان Gate کامل می‌شوند**
   - Multi-fold Confirmation Random Search
   - Recipe Freeze
   - Fresh 5-Fold
   - OOF
   - Locked Test: expected 614 images / 279 filename-derived patient groups (real counts still require runtime verification)
   - F2 report-only
   - Fixed 20/80 report-only
   - Bootstrap / calibration / risk-coverage / decision curve
   - Loss diagnostics
   - XAI / EdgeBlock / CBAM / Casebook
   - External validation
   - M07 Gate Report

2. **توقف**
   - `UNLOCK_REMAINING_MODELS = False`

3. فقط پس از بررسی دستی M07:
   - Flag را `True` می‌کنیم
   - همان Notebook دوباره Run All می‌شود
   - M07 state قبلی restore می‌شود
   - Phase-2 اجرا می‌شود

4. Phase-2:
   - M01, M02, M03, M04, M05, M06, M08, M09, M10, M11, M12
   - M07 در هر سه وضوح قبلاً در Gate کامل شده و در Phase-2 دوباره آموزش داده نمی‌شود
   - Resolution matrix = 224 / 320 / 384

---

## Canonical model identities

| Model | Definition |
|---|---|
| M01 | ConvNeXt-Tiny + BCE |
| M02 | ConvNeXt-Tiny + weighted BCE |
| M03 | ConvNeXt-Tiny + static focal |
| M04 | ConvNeXt-Tiny + FLSD-53 |
| M05 | M04 + CBAM |
| M06 | M04 + EdgeBlock |
| M07 | M04 + EdgeBlock + CBAM |
| M08 | M07 + MixUp |
| M09 | M07 + CutMix |
| M10 | M07 + stochastic MixUp/CutMix |
| M11 | M10 + EMA |
| M12 | M10 + SWA |

These identities are preserved from the user's final canonical source package.

---

## Shared scientific contract for ALL models

- Same frozen clean split
- Same split fingerprint
- Same patient isolation
- Same exact-SHA isolation
- Same preprocessing
- Same final 5 patient-locked folds
- Same threshold-selection policy
- Same precision/recall-first checkpoint policy
- Same OOF / Locked-Test reporting policy
- Same F2 policy: **report-only**
- Same fixed 20/80 policy: **report-only**
- Shared core OOF / Locked-Test / calibration reporting; Phase-2 has a reduced XAI/internal/external scope disclosed in its final report.
- No Test/External tuning

### Important fairness rule

The **confirmed M07 common training recipe** is reused as the shared training recipe for the remaining canonical ablations.

Shared:
- batch size
- head LR
- fine-tune LR
- weight decay
- dropout
- patience
- LR schedule
- min LR
- warmups

Architecture-specific values apply only where the model identity needs them:
- Edge settings only for Edge models
- CBAM settings only for CBAM models
- MixUp/CutMix fixed identity
- EMA fixed identity
- SWA fixed identity

---

## Focal correction policy

- M04–M12 use FLSD-53 with `alpha=0.50` when balanced batches are active.
- M03 retains its canonical static-focal identity and its original prevalence-derived positive alpha.
- M02 uses natural sampling + weighted BCE.
- All other models use exact balanced batches.

---

## Governance

Default:

`UNLOCK_REMAINING_MODELS = False`

So the first Run stops after M07.

Phase-2 code is already present; it is merely locked.

**No automatic M01–M12 continuation is allowed before manual review of M07.**

---

## V1.2 — Self-healing Frozen Split

The Master no longer requires a private saved Notebook Output for the split. If the exact frozen package is absent, it deterministically rebuilds the original clean split from the public canonical Kermany/Mooney dataset, verifies every frozen manifest SHA256 and the final frozen fingerprint, and only then allows HPO/training to continue.


---

## M07 RUN-SAFE MASTER V1.3

All M01–M12 code remains in this notebook, but Phase-2 is intentionally locked.

Before Confirmation HPO:
- remote persistence write must pass
- exact Frozen Split discovery/self-heal must match frozen hashes/fingerprint
- five patient-locked Fold checks must pass
- real TensorFlow M07 build/forward/FLSD/train-on-batch smoke test must pass

Locked-Test and External confidence intervals use patient-cluster bootstrap.

Do not unlock Phase-2 in this build before reviewing M07 evidence.


---

## V1.6 — edited version and evidence scope

نسخهٔ حاضر برای رفع خطاهای گزارش، اعتبارسنجی هویت آزمایش، بازیابی نتایج، هش واقعی تصاویر و یکسان‌کردن bootstrap بیمارمحور ویرایش شده است.

- خروجی و مجموعه‌دادهٔ ذخیرهٔ پیش‌فرض V1.6 از نسخهٔ قبل جدا هستند؛ cache قدیمی بدون رسید معتبر پذیرفته نمی‌شود.
- `UNLOCK_REMAINING_MODELS=False` و `ENABLE_HIGH_RESOLUTIONS=False` باقی مانده‌اند. فلگ دوم اکنون تنظیمات وضوح را تعیین می‌کند.
- helperهای XAI مستقل از اجرای تصویرسازی تعریف می‌شوند؛ `RUN_MODEL_INTERNAL_VISUALS` و `RUN_TEST_CASEBOOK` امکان کنترل جداگانه را فراهم می‌کنند.
- Grad-CAM++ به‌صراحت «تقریبی» معرفی می‌شود؛ IG دادهٔ علامت‌دار و شاخص completeness ذخیره می‌کند.
- دادهٔ خارجی باید patient_id واقعی منبع داشته باشد؛ شناسهٔ تصویر جایگزین شناسهٔ بیمار نمی‌شود.
- آزمون‌های محلی CPU/AST موفقیت آموزش یا اجرای GPU را اثبات نمی‌کنند. فایل `VALIDATION_RESULTS.json` همراه بسته، شواهد محلی را مشخص می‌کند؛ `RUNTIME_PREFLIGHT_V14.json` فقط در اجرای واقعی سلول preflight ساخته می‌شود.
- OOF همچنان برآورد توسعه است. مدل‌ها و سیاست نمونه‌برداری اصلی حفظ شده‌اند؛ مقایسه‌های چندعاملی، نبود چند seed و نقش NIH به‌عنوان sentinel محدودیت‌های علمی اعلام‌شده‌اند.


---

## V1.6 — M07 Multi-Resolution Gate + Paired Statistics

- Confirmation HPO remains only at 224.
- The confirmed M07 recipe is frozen and M07 then completes full 5-fold training/evaluation at 224, 320 and 384 inside the Gate.
- 320/384 preserve the confirmed effective optimizer batch via microbatch + gradient accumulation; if the installed Keras cannot support exact accumulation, the run fails closed rather than silently changing batch semantics.
- M07 is permanently excluded from Phase-2. Phase-2 contains only M01–M06 and M08–M12 at 224/320/384.
- Resolution comparisons and later model comparisons use paired patient-cluster bootstrap.
- Holm correction is reported for the predeclared primary Balanced-Accuracy family and a stricter all-reported-metrics family.
- Locked Test statistics remain confirmatory/report-only and never tune resolution, model, recipe or threshold.


---

## V1.6 corrective revision

Fixes:
- R224/R320/R384 Locked-Test primary-prediction column mismatch
- undefined Phase-2 Gate source variables
- statistically unusable 216-test Holm family
- custom external cohort missing at R320/R384
- gradient-accumulation epoch schedule now preserves optimizer-update counts
  and forbids partial accumulation groups


In [ ]:
# ============================================================
# 0) CONFIG — M07 MULTI-RES GATE + MASTER V1.7
# ============================================================
from pathlib import Path
import os, re, sys, json, math, time, gc, hashlib, zipfile, shutil, subprocess
from typing import Any

import numpy as np
import pandas as pd

WORK = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)

SEED = 42
IMAGE_SIZE = 224

MODEL_ID = "M07"
MODEL_DESCRIPTION = (
    "M07 Final Gate — ConvNeXt-Tiny + EdgeBlock + CBAM + FLSD-53"
)

EXPECTED_SPLIT_FINGERPRINT = (
    "896491de87f9dc2a1d7d63548b7c5c22206da11f27a37efece8efc8e1557c8a9"
)

DUAL_THRESHOLD_LOW = 0.20
DUAL_THRESHOLD_HIGH = 0.80

RUN_VALIDATION_XAI = True
RUN_MODEL_INTERNAL_VISUALS = True
RUN_TEST_CASEBOOK = True
XAI_SAMPLES = 8
XAI_METHODS = (
    "gradcam",
    "gradcampp",
    "integrated_gradients",
    "occlusion",
)
XAI_OCCLUSION_PATCH = 32
XAI_OCCLUSION_STRIDE = 24

RUN_NIH_SENTINEL = True
EXTERNAL_BOOTSTRAPS = 2000
LOCKED_TEST_BOOTSTRAPS = 2000

CUSTOM_EXTERNAL_MANIFEST = os.environ.get(
    "M07_EXTERNAL_MANIFEST",
    "",
).strip()

# Confirmation HPO
HPO_CANDIDATE_SEED = 2026
HPO_TRAIN_SEED_BASE = 42000
HPO_CANDIDATES = 8
HPO_SCREEN_FOLDS = (1, 2)
HPO_CONFIRM_FOLD = 3
HPO_TOP_K = 3
HPO_HEAD_EPOCHS = 2
HPO_FINETUNE_EPOCHS = 6

FINAL_HEAD_EPOCHS = 3
FINAL_FINETUNE_EPOCHS = 25

PRIOR_SUCCESSFUL_RECIPE = {
    "batch_size": 16,
    "head_lr": 0.00031320688808763263,
    "finetune_lr": 5.443527706210183e-05,
    "weight_decay": 6.41370410511974e-06,
    "dropout": 0.40,
    "patience": 4,
    "lr_schedule": "cosine",
    "min_lr": 1e-07,
    "head_warmup_epochs": 0,
    "finetune_warmup_epochs": 2,
    "edge_filters": 8,
    "edge_max_gate": 0.15,
    "cbam_reduction": 8,
}
PRIOR_SUCCESSFUL_RECIPE_FINGERPRINT = (
    "c2f519845f38c1ff53d17c851f27420aa7bd7f96ccb02511d2a491115652b9f5"
)

# M07 identity
EDGE_FILTERS = 8
EDGE_MAX_GATE = 0.15
CBAM_REDUCTION = 8
CBAM_SPATIAL_KERNEL = 7

FLSD_THRESHOLD = 0.20
FLSD_HARD_GAMMA = 5.0
FLSD_EASY_GAMMA = 3.0
FLSD_ALPHA = 0.50

CASEBOOK_PER_ERROR_CLASS = 5
MODEL_INTERNAL_VISUAL_SAMPLES = 6

# Persistence
PERSIST_STRICT = True

PERSIST_DATASET_SLUG = os.environ.get(
    "M07_PERSIST_DATASET_SLUG",
    "m07-multires-gate-run-safe-v17",
).strip()

PERSIST_DATASET_HANDLE = os.environ.get(
    "M07_PERSIST_DATASET_HANDLE",
    "",
).strip() or None

OUTPUT_ROOT = WORK / "M07_GATE_R224_RUN_SAFE_V17"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# MASTER / Phase-2 — intentionally LOCKED for current run
UNLOCK_REMAINING_MODELS = False
# Re-run the last cell only with this True to rebuild statistics from local sealed reports.
# This forbids restore, persistence uploads, model.fit and inference in that cell.
MASTER_REPORT_ONLY = False

PHASE2_MODEL_ORDER = (
    "M01", "M02", "M03", "M04", "M05", "M06",
    "M08", "M09", "M10", "M11", "M12",
)

# M07 is completed in the Gate at all three canonical resolutions.
M07_GATE_RESOLUTIONS = (224, 320, 384)
M07_GATE_ADDITIONAL_RESOLUTIONS = (320, 384)
M07_GATE_MULTIRES_ROOT = WORK / "M07_GATE_MULTIRES_V17"
M07_GATE_MULTIRES_ROOT.mkdir(parents=True, exist_ok=True)

# After manual unlock, Phase-2 runs ONLY the remaining models at all resolutions.
ENABLE_HIGH_RESOLUTIONS = True
PHASE2_RESOLUTIONS = (224, 320, 384)
INCLUDE_M07_OTHER_RESOLUTIONS = False

PHASE2_FINAL_HEAD_EPOCHS = 3
PHASE2_FINAL_FINETUNE_EPOCHS = 25

PHASE2_STATIC_FOCAL_GAMMA = 2.0
PHASE2_MIX_ALPHA = 0.20
PHASE2_EMA_MOMENTUM = 0.99
PHASE2_SWA_START = 8

PHASE2_BOOTSTRAPS = 2000
PHASE2_RUN_XAI = True
PHASE2_RUN_EXTERNAL = True
PHASE2_XAI_SAMPLES = 4
PHASE2_CASEBOOK_PER_CLASS = 5

# Microbatch ceilings preserve the confirmed effective optimizer batch via
# gradient accumulation for 320/384.
PHASE2_MICROBATCH_MAX_BY_RESOLUTION = {
    224: None,
    320: 8,
    384: 4,
}

PHASE2_PERSIST_OWNER = os.environ.get(
    "PHASE2_PERSIST_OWNER",
    "",
).strip() or None

PHASE2_CAMPAIGN_ROOT = WORK / "PNEUMONIA_MASTER_PHASE2_V17"
PHASE2_CAMPAIGN_ROOT.mkdir(parents=True, exist_ok=True)

# Predeclared paired statistical comparison policy.
PAIRED_BOOTSTRAPS = 5000
PAIRED_BOOTSTRAP_SEED = 260915
PAIRED_PRIMARY_METRIC = "balanced_accuracy"
PAIRED_SECONDARY_METRICS = (
    "macro_f1", "mcc", "auroc", "recall_normal", "recall_pneumonia",
)
PAIRED_ALPHA = 0.05
PAIRED_MULTIPLICITY_POLICY = (
    "CONFIRMATORY_PRIMARY_BALACC_GLOBAL_HOLM_PER_DATASET; "
    "SECONDARY_HOLM_SEPARATELY_WITHIN_EACH_METRIC_FAMILY_PER_DATASET; "
    "NO_SINGLE_216_TEST_ALL_METRICS_FAMILY"
)

# Direction is candidate - reference; positive delta favors candidate.
PREDECLARED_MODEL_COMPARISONS = (
    ("M01", "M02", "BCE_vs_weighted_BCE_sampling_bundle"),
    ("M03", "M04", "static_focal_vs_FLSD53"),
    ("M04", "M05", "CBAM_added_to_M04"),
    ("M04", "M06", "EdgeBlock_added_to_M04"),
    ("M04", "M07", "EdgeBlock_plus_CBAM_added_to_M04"),
    ("M05", "M07", "EdgeBlock_added_to_CBAM_branch"),
    ("M06", "M07", "CBAM_added_to_Edge_branch"),
    ("M07", "M08", "MixUp_added_to_M07"),
    ("M07", "M09", "CutMix_added_to_M07"),
    ("M07", "M10", "stochastic_MixUp_CutMix_added_to_M07"),
    ("M10", "M11", "EMA_added_to_M10"),
    ("M10", "M12", "SWA_added_to_M10"),
)

print("MODEL:", MODEL_ID)
print(MODEL_DESCRIPTION)
print("WORK:", WORK)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print(
    "Confirmation HPO:",
    HPO_CANDIDATES,
    "| screen:",
    HPO_SCREEN_FOLDS,
    "| confirm:",
    HPO_CONFIRM_FOLD,
)
print("Phase-2 locked:", not UNLOCK_REMAINING_MODELS)

# Experiment identity and artifact integrity helpers — V1.7
# Embedded verbatim at the end of notebook CONFIG by build_revised.py.
RUN_SAFE_VERSION = "1.7"
EXPERIMENT_CODE_SHA256 = "d099b6e51c243b3c489f0a280ca247971e84b1cf1c504cba2a51028c93fba27f"


def _canonical_value(value):
    if isinstance(value, dict):
        return {str(k): _canonical_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_canonical_value(v) for v in value]
    if isinstance(value, np.ndarray):
        return _canonical_value(value.tolist())
    if isinstance(value, np.generic):
        return _canonical_value(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        raise ValueError("Non-finite value in experiment identity/receipt")
    return value


def contract_fingerprint(contract):
    encoded = json.dumps(_canonical_value(contract), sort_keys=True,
                         separators=(",", ":"), ensure_ascii=False, allow_nan=False)
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()


def runtime_environment():
    from importlib.metadata import version, PackageNotFoundError
    versions = {"python": sys.version.split()[0]}
    for name in ("tensorflow", "keras", "numpy", "pandas", "scikit-learn",
                 "kagglehub", "pillow", "h5py"):
        try:
            versions[name] = version(name)
        except PackageNotFoundError:
            versions[name] = "NOT_INSTALLED"
    return versions


def make_run_contract(stage, *, model_id, resolution, params, fold_id=None,
                      seed=None, head_epochs=None, finetune_epochs=None, extra=None):
    return _canonical_value({
        "schema": "pneumonia.experiment.v1.7",
        "code_sha256": EXPERIMENT_CODE_SHA256,
        "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
        "stage": str(stage), "model_id": str(model_id),
        "resolution": int(resolution), "params": params,
        "fold_id": fold_id, "seed": seed, "head_epochs": head_epochs,
        "finetune_epochs": finetune_epochs, "extra": extra or {},
        "runtime": runtime_environment(),
    })


def run_file_sha256(path):
    h = hashlib.sha256()
    with open(Path(path), "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def seal_receipt(payload, contract, artifacts=None):
    result = _canonical_value(dict(payload))
    result.pop("receipt_sha256", None)
    result["run_contract"] = _canonical_value(contract)
    result["run_fingerprint"] = contract_fingerprint(contract)
    result["artifact_sha256"] = {
        str(name): run_file_sha256(path)
        for name, path in (artifacts or {}).items()
    }
    result["receipt_sha256"] = contract_fingerprint(result)
    return result


def validate_receipt(payload, contract, artifacts=None, allowed_statuses=None):
    if not isinstance(payload, dict):
        raise ValueError("Invalid receipt object")
    receipt_body = {k: v for k, v in payload.items() if k != "receipt_sha256"}
    if payload.get("receipt_sha256") != contract_fingerprint(receipt_body):
        raise ValueError("Corrupt or unsealed receipt payload")
    expected = contract_fingerprint(contract)
    if (payload.get("run_fingerprint") != expected
            or payload.get("run_contract") != _canonical_value(contract)):
        raise ValueError("Stale or unbound experiment receipt; do not reuse legacy results")
    if contract_fingerprint(payload["run_contract"]) != payload["run_fingerprint"]:
        raise ValueError("Corrupt experiment receipt identity")
    if allowed_statuses is not None and payload.get("status") not in set(allowed_statuses):
        raise ValueError("Receipt is not in an accepted completion state")
    hashes = payload.get("artifact_sha256")
    if not isinstance(hashes, dict):
        raise ValueError("Missing artifact hash manifest")
    for name, path in (artifacts or {}).items():
        path = Path(path)
        if not path.is_file() or hashes.get(str(name)) != run_file_sha256(path):
            raise ValueError(f"Missing or changed artifact: {name}")
    return payload


def atomic_write_json(path, payload):
    import tempfile
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as stream:
            json.dump(_canonical_value(payload), stream, indent=2,
                      ensure_ascii=False, allow_nan=False)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(name, path)
    finally:
        if os.path.exists(name):
            os.unlink(name)


def dataframe_identity(frame):
    cols = [c for c in ("relative_path", "patient_id", "label", "model_label", "sha256")
            if c in frame.columns]
    if not cols:
        raise ValueError("No row identity columns")
    identity = frame[cols].copy()
    if identity.isna().any().any():
        raise ValueError("Missing row identity value")
    for c in cols:
        if c == "label":
            values = pd.to_numeric(identity[c], errors="raise")
            if not values.isin([0, 1]).all():
                raise ValueError("Labels must be 0 or 1")
            identity[c] = values.astype(int)
        else:
            identity[c] = identity[c].astype(str)
    return contract_fingerprint({"columns": cols, "rows": identity.to_dict("records")})


def validate_prediction_frame(pred, manifest, probability_column="probability_pneumonia"):
    columns = ["relative_path", "patient_id", "label", "sha256"]
    if "model_label" in manifest.columns:
        columns.append("model_label")
    if any(c not in pred.columns or c not in manifest.columns for c in columns):
        raise ValueError("Prediction row identities are missing")
    if len(pred) != len(manifest) or len(pred) == 0:
        raise ValueError("Prediction row count does not match manifest")
    if dataframe_identity(pred[columns]) != dataframe_identity(manifest[columns]):
        raise ValueError("Prediction row identities/order differ from current manifest")
    if probability_column not in pred.columns:
        raise ValueError("Missing prediction probability")
    probability = pd.to_numeric(pred[probability_column], errors="raise").to_numpy(float)
    if not np.isfinite(probability).all() or np.any((probability < 0) | (probability > 1)):
        raise ValueError("Predictions must be finite probabilities in [0,1]")
    return pred


def safe_extract_zip(zip_path, destination):
    import stat
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for entry in archive.infolist():
            target = (destination / entry.filename).resolve()
            if (not target.is_relative_to(destination)
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError("Unsafe archive member")
        archive.extractall(destination)


In [ ]:

# ============================================================
# 1) RUNTIME / GPU / DEPENDENCIES
# ============================================================
import tensorflow as tf

try:
    import sklearn
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    import sklearn

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    fbeta_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("GPU not detected. In Kaggle enable Settings > Accelerator > GPU.")

# Reproducibility
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic ops: ON")
except Exception as exc:
    print("Deterministic-op warning:", exc)

# Mixed precision exactly where useful on Kaggle GPUs.
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

# Record the actual environment; this is not a predeclared tested version lock.
atomic_write_json(OUTPUT_ROOT / "RUNTIME_ENVIRONMENT_V17.json", runtime_environment())


In [ ]:
# ============================================================
# 1B) PERSISTENCE GATE — MUST PASS BEFORE TRAINING
# ============================================================
import importlib.util

if importlib.util.find_spec("kagglehub") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "kagglehub"]
    )

import kagglehub

def _resolve_kaggle_owner() -> str:
    explicit = os.environ.get(
        "KAGGLE_PERSIST_OWNER",
        "",
    ).strip()
    if explicit:
        return explicit

    try:
        info = kagglehub.whoami()

        if isinstance(info, dict):
            username = str(
                info.get("username", "")
            ).strip()
            if username:
                return username

        if isinstance(info, str) and info.strip():
            return info.strip()

    except Exception as exc:
        print("whoami fallback:", type(exc).__name__)

    try:
        from kagglehub.config import get_kaggle_credentials

        creds = get_kaggle_credentials()
        username = str(
            getattr(creds, "username", "") or ""
        ).strip()

        if username:
            return username

    except Exception as exc:
        print("credentials fallback:", type(exc).__name__)

    env_username = os.environ.get(
        "KAGGLE_USERNAME",
        "",
    ).strip()

    if env_username:
        return env_username

    if PERSIST_DATASET_HANDLE:
        owner = PERSIST_DATASET_HANDLE.split("/", 1)[0].strip()
        if owner:
            return owner

    raise RuntimeError(
        "Cannot resolve current Kaggle username. "
        "Set KAGGLE_PERSIST_OWNER or M07_PERSIST_DATASET_HANDLE."
    )


KAGGLE_PERSIST_OWNER = _resolve_kaggle_owner()

if not PERSIST_DATASET_HANDLE:
    PERSIST_DATASET_HANDLE = (
        f"{KAGGLE_PERSIST_OWNER}/{PERSIST_DATASET_SLUG}"
    )

if not PHASE2_PERSIST_OWNER:
    PHASE2_PERSIST_OWNER = KAGGLE_PERSIST_OWNER

print("✅ Kaggle owner:", KAGGLE_PERSIST_OWNER)
print("M07 persistence:", PERSIST_DATASET_HANDLE)

PERSIST_ROOT = WORK / "M07_PERSIST_STATE_RUN_SAFE_V1_7"
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

LIGHT_ROOT = PERSIST_ROOT / "LIGHT_STATE"
LIGHT_ROOT.mkdir(parents=True, exist_ok=True)

LIGHT_ZIP = PERSIST_ROOT / "M07_LIGHT_STATE.zip"


PERSIST_RESTORE_VERIFIED = False
PERSIST_MANIFEST_NAME = "PERSISTENCE_MANIFEST_V1_4.json"


def _persistence_campaign_contract():
    # Available before manifests are materialized. Individual experiment receipts
    # additionally bind the actual train/validation row identities.
    return make_run_contract(
        "M07_PERSISTENCE_CAMPAIGN", model_id="M07", resolution=IMAGE_SIZE,
        seed=SEED, head_epochs=FINAL_HEAD_EPOCHS,
        finetune_epochs=FINAL_FINETUNE_EPOCHS,
        params={
            "hpo_candidate_seed": HPO_CANDIDATE_SEED,
            "hpo_train_seed_base": HPO_TRAIN_SEED_BASE,
            "hpo_candidates": HPO_CANDIDATES,
            "hpo_screen_folds": list(HPO_SCREEN_FOLDS),
            "hpo_confirm_fold": HPO_CONFIRM_FOLD,
            "hpo_top_k": HPO_TOP_K,
            "hpo_head_epochs": HPO_HEAD_EPOCHS,
            "hpo_finetune_epochs": HPO_FINETUNE_EPOCHS,
            "prior_recipe": PRIOR_SUCCESSFUL_RECIPE,
            "flsd": [FLSD_ALPHA, FLSD_THRESHOLD, FLSD_HARD_GAMMA, FLSD_EASY_GAMMA],
            "cbam_spatial_kernel": CBAM_SPATIAL_KERNEL,
        },
        extra={"dataset_handle": PERSIST_DATASET_HANDLE},
    )


def _persistence_artifacts(root):
    return {
        p.relative_to(root).as_posix(): p
        for p in sorted(Path(root).rglob("*"))
        if p.is_file() and p.name != PERSIST_MANIFEST_NAME
    }


def _verify_persistence_snapshot(root):
    manifest = Path(root) / PERSIST_MANIFEST_NAME
    if not manifest.is_file():
        raise ValueError("PERSISTENCE_IDENTITY_MISSING: refusing an unsealed/legacy snapshot")
    payload = json.loads(manifest.read_text(encoding="utf-8"))
    artifacts = _persistence_artifacts(Path(root))
    if set(payload.get("artifact_sha256", {})) != set(artifacts):
        raise ValueError("PERSISTENCE_ARTIFACT_SET_MISMATCH")
    return validate_receipt(
        payload, _persistence_campaign_contract(), artifacts=artifacts,
        allowed_statuses={"SEALED"},
    )


def _persist_upload(note: str, strict: bool = True):
    if not PERSIST_RESTORE_VERIFIED:
        raise RuntimeError("PERSISTENCE_RESTORE_NOT_VERIFIED: upload is blocked")
    # Seal every archive and lightweight evidence file before publishing a version.
    atomic_write_json(
        PERSIST_ROOT / PERSIST_MANIFEST_NAME,
        seal_receipt(
            {"schema": "m07.persistence.manifest.v1.6", "status": "SEALED"},
            _persistence_campaign_contract(), artifacts=_persistence_artifacts(PERSIST_ROOT),
        ),
    )
    try:
        kagglehub.dataset_upload(
            PERSIST_DATASET_HANDLE, str(PERSIST_ROOT), version_notes=str(note),
        )
        print(f"✅ Persistent upload OK: {note}")
        return True
    except Exception as exc:
        print("❌ Persistent upload failed:", type(exc).__name__)
        if strict:
            raise RuntimeError(
                "PERSISTENCE_GATE_FAILED before expensive training. "
                f"Handle={PERSIST_DATASET_HANDLE}"
            ) from exc
        return False


def _write_persistence_zip(zip_path, entries):
    import tempfile
    zip_path = Path(zip_path)
    with tempfile.NamedTemporaryFile(dir=zip_path.parent, prefix=zip_path.name + ".", suffix=".tmp", delete=False) as tmp:
        temporary = Path(tmp.name)
    try:
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as z:
            for source, arcname in entries:
                z.write(source, arcname=str(arcname))
        os.replace(temporary, zip_path)
    finally:
        if temporary.exists():
            temporary.unlink()


def _refresh_light_zip():
    _write_persistence_zip(LIGHT_ZIP, [
        (p, p.relative_to(LIGHT_ROOT).as_posix())
        for p in sorted(LIGHT_ROOT.rglob("*")) if p.is_file()
    ])


def _restore_m07_archive_checked(zip_path):
    import tempfile
    # Validate/extract in isolation so traversal, malformed archives, or local
    # evidence conflicts cannot overwrite the current output tree.
    with tempfile.TemporaryDirectory(prefix="m07_restore_", dir=WORK) as temporary:
        staging = Path(temporary)
        safe_extract_zip(zip_path, staging)
        entries = [(source, OUTPUT_ROOT / source.relative_to(staging))
                   for source in staging.rglob("*") if source.is_file()]
        for source, target in entries:
            if target.exists() and (not target.is_file()
                                    or run_file_sha256(target) != run_file_sha256(source)):
                raise ValueError(f"LOCAL_ARCHIVE_EVIDENCE_CONFLICT: {target.name}")
        for source, target in entries:
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():
                shutil.copy2(source, target)


def _restore_light_state(root: Path) -> int:
    archives = list(Path(root).rglob("M07_LIGHT_STATE.zip"))
    if len(archives) > 1:
        raise ValueError("Ambiguous lightweight persistence archives")
    if not archives:
        return 0
    _restore_m07_archive_checked(archives[0])
    print("✅ Restored lightweight M07 state")
    return 1


def _is_explicit_http_not_found(exc):
    # Never infer absence from an error string, authentication, connection failure,
    # or generic API exception. Only an explicit HTTP 404 admits a new campaign.
    status = getattr(exc, "status_code", None)
    response = getattr(exc, "response", None)
    if response is not None:
        status = getattr(response, "status_code", status)
    return type(status) is int and status == 404


def _try_restore_persisted_state():
    global PERSIST_RESTORE_VERIFIED
    PERSIST_RESTORE_VERIFIED = False
    restore_dir = WORK / "_M07_REMOTE_STATE_RUN_SAFE_V1_7"
    if restore_dir.exists():
        shutil.rmtree(restore_dir)
    restore_dir.mkdir(parents=True, exist_ok=True)
    try:
        downloaded = kagglehub.dataset_download(
            PERSIST_DATASET_HANDLE, output_dir=str(restore_dir), force_download=True,
        )
    except Exception as exc:
        if not _is_explicit_http_not_found(exc):
            raise RuntimeError(
                "PERSISTENCE_RESTORE_FAILED: absence was not confirmed; "
                "training and all uploads remain blocked. Retry after resolving access/network."
            ) from exc
        # A local prior campaign must not be republished after remote deletion.
        if (PERSIST_ROOT / PERSIST_MANIFEST_NAME).exists():
            raise RuntimeError("REMOTE_CAMPAIGN_MISSING_WITH_LOCAL_STATE: resolve explicitly") from exc
        PERSIST_RESTORE_VERIFIED = True
        print("Confirmed HTTP 404: starting a new v1.6 M07 campaign.")
        return {"folds": 0, "light_state": 0, "restore_status": "CONFIRMED_NOT_FOUND"}

    if not isinstance(downloaded, (str, Path)) or not Path(downloaded).is_dir():
        raise RuntimeError("PERSISTENCE_DOWNLOAD_PATH_INVALID: Kaggle returned no readable snapshot directory")
    downloaded_root = Path(downloaded).resolve()
    _verify_persistence_snapshot(downloaded_root)
    # Conflicting local evidence is never silently replaced by downloaded state.
    for source in downloaded_root.rglob("*"):
        if source.is_file():
            target = PERSIST_ROOT / source.relative_to(downloaded_root)
            if target.is_file() and run_file_sha256(target) != run_file_sha256(source):
                raise ValueError(f"LOCAL_REMOTE_PERSISTENCE_CONFLICT: {target.name}")
    for source in downloaded_root.rglob("*"):
        if source.is_file():
            target = PERSIST_ROOT / source.relative_to(downloaded_root)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)

    folds = 0
    for zpath in sorted(PERSIST_ROOT.glob("M07_FOLD_*_RECOVERY.zip")):
        _restore_m07_archive_checked(zpath)
        folds += 1
    light = _restore_light_state(PERSIST_ROOT)
    PERSIST_RESTORE_VERIFIED = True
    summary = {"folds": folds, "light_state": light, "restore_status": "VERIFIED"}
    print("✅ Restore summary:", summary)
    return summary


def _snapshot_fold_to_persist(
    fold: int,
    recipe_path: Path,
):
    fold_dir = (
        OUTPUT_ROOT
        / "FINAL_5FOLD"
        / f"fold_{fold}"
    )

    # A completed marker alone is insufficient: verify its identity and every artifact.
    _read_completed_m07_fold(
        fold, fold_dir, load_fold_manifest(fold, "train"), load_fold_manifest(fold, "val")
    )

    zpath = (
        PERSIST_ROOT
        / f"M07_FOLD_{fold}_RECOVERY.zip"
    )

    entries = [
        (p, (Path("FINAL_5FOLD") / f"fold_{fold}" / p.relative_to(fold_dir)).as_posix())
        for p in sorted(fold_dir.rglob("*")) if p.is_file()
    ]
    if recipe_path.is_file():
        entries.append((recipe_path, recipe_path.name))
    _write_persistence_zip(zpath, entries)

    completed = sorted([
        int(m.group(1))
        for p in PERSIST_ROOT.glob(
            "M07_FOLD_*_RECOVERY.zip"
        )
        if (
            m := re.search(
                r"M07_FOLD_(\d+)_RECOVERY\.zip$",
                p.name,
            )
        )
    ])

    (
        PERSIST_ROOT
        / "M07_CAMPAIGN_STATE.json"
    ).write_text(
        json.dumps(
            {
                "schema": "m07.persistence.state.run_safe.v1.6",
                "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
                "completed_folds": completed,
                "dataset_handle": PERSIST_DATASET_HANDLE,
                "last_update_utc": pd.Timestamp.utcnow().isoformat(),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    _persist_upload(
        note=(
            f"M07 completed fold {fold}; "
            f"folds={completed}"
        ),
        strict=PERSIST_STRICT,
    )


def _persist_lightweight_files(
    file_paths,
    note: str,
):
    for src in file_paths:
        src = Path(src)

        if not src.is_file():
            continue

        try:
            rel = src.relative_to(OUTPUT_ROOT)
        except ValueError:
            rel = Path(src.name)

        target = LIGHT_ROOT / rel

        target.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(src, target)

    _refresh_light_zip()

    _persist_upload(
        note=note,
        strict=PERSIST_STRICT,
    )


RESTORE_SUMMARY = _try_restore_persisted_state()

(
    PERSIST_ROOT
    / "PERSISTENCE_HEALTHCHECK.json"
).write_text(
    json.dumps(
        {
            "schema": "m07.persistence.health.run_safe.v1.6",
            "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
            "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
            "dataset_handle": PERSIST_DATASET_HANDLE,
        },
        indent=2,
    ),
    encoding="utf-8",
)

_persist_upload(
    note="M07 run-safe persistence healthcheck before training",
    strict=PERSIST_STRICT,
)

print("✅ PERSISTENCE GATE PASSED")

In [ ]:
# ============================================================
# 2) FROZEN SPLIT — DISCOVER OR SELF-HEAL REBUILD
#
# Order:
# 1) Use an already attached exact frozen split if available.
# 2) Use an attached frozen ZIP if available.
# 3) Otherwise rebuild the SAME frozen split from the public
#    canonical Kermany/Mooney raw dataset.
# 4) Verify exact manifest SHA256 values + final fingerprint.
#
# No private `kaggle kernels output` dependency.
# No user intervention is required if the raw public dataset
# is available/attachable/downloadable.
#
# Near-duplicate handling is NOT re-tuned here. We recreate the
# already frozen policy: audit-only, no automatic near removal,
# because the prior exhaustive pHash→SSIM audit found zero
# strong SSIM>=0.95 pairs.
# ============================================================

import importlib.util
from sklearn.model_selection import (
    train_test_split,
    StratifiedGroupKFold,
)

FROZEN_EXPECTED_MANIFEST_HASHES = {
    "08_clean_master_manifest.csv":
        "ab5b4cbf111109868fdc2b89d267eb849316904d783b9dcabf2563e895b5d5b8",
    "09_clean_partition_master.csv":
        "cf76d06ecc2a5b1eca2cbb873ce6d2841a9e8e3c8b28862c0cd2343de1eb3bee",
    "10_development_manifest.csv":
        "8d6daa8d93f2bfdecd3bd18f47d2b0b4a0119360d9656a23767f4a1dece14711",
    "11_locked_test_manifest.csv":
        "0da162edb47a85314e62d6cd987363262486312a0999e868c50e709879b4c744",
    "12_development_patients.csv":
        "b8042210c5e067e57be8fe9bf8267580a7ef48983cca5a610edc315755fa0e5e",
    "13_locked_test_patients.csv":
        "7e20d8a50b99a8eae5b150a028086dc97f047cdd753d71f3198dcfe02294a91d",
    "14_development_with_fold_assignments.csv":
        "f1ad389c3c6fc145461a81a4c60a31c1cb0d18c0a6e4f53dc504403f3b99ef02",
    "15_fold_statistics.csv":
        "e86680053ec52672df05a77bf9b54545a9974e625fa0dd2f4e82dfa4957ad513",
    "fold_1_train.csv":
        "916219bc195ab4cb6c2f0f9be0f25d660abdd5c57443f10a1f17f6714c511c4d",
    "fold_2_train.csv":
        "e2df3ccab6c20bd6c6839d10a6994ff91170dace62d616d3906acf6d1120432c",
    "fold_3_train.csv":
        "3761fe59d7ab63267fbeaeb2dd955f3521e9427924ff680627f66e501b7771ca",
    "fold_4_train.csv":
        "5d08f5e0398326ce71433ebe0e639254c13705d032446321e62d23a7b11bab1d",
    "fold_5_train.csv":
        "218f081841429cfb2b0d3d4e68de44d431449e7b91cd5cc035fe2d40e8ea4e83",
    "fold_1_val.csv":
        "dff214ee67a1a45d1c657b1b126c88a378c60e353ee4ca0fc04d4ed40050b101",
    "fold_2_val.csv":
        "daca7171c0b894f95282fa46d999a818f39d4392f079bc7bfa10c5800bf3e60e",
    "fold_3_val.csv":
        "ca398675345c9a3f0565d994fac0f3cd3ed2233f4a19e52c8eae0adc11c1c3b4",
    "fold_4_val.csv":
        "36fe0a399ce5dbd44560e725b9ce1b6296bda6619b5ba9fef86a65b1a12cb127",
    "fold_5_val.csv":
        "f4fdd4795fe6bd5cdb7cb443db9033d11cff73c3b77f99980001b2721796e2ba",
}

FROZEN_DATASET_SLUG = "paultimothymooney/chest-xray-pneumonia"
FROZEN_SEED = 42
FROZEN_TEST_SIZE = 0.10
FROZEN_N_FOLDS = 5

FROZEN_EXPECTED_TOTAL = 5856
FROZEN_EXPECTED_NORMAL = 1583
FROZEN_EXPECTED_PNEUMONIA = 4273

# Important for exact reproduction of the original manifest hashes.
# The original frozen package was produced while the dataset was
# mounted at this canonical Kaggle path. The `absolute_path` column
# is provenance only; training later reconstructs live file paths
# from `relative_path`.
FROZEN_CANONICAL_ABSOLUTE_ROOT = Path(
    "/kaggle/input/datasets/paultimothymooney/"
    "chest-xray-pneumonia/chest_xray"
)


def file_sha256(path: Path, chunk=1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _frozen_valid_policy(path: Path) -> bool:
    try:
        payload = json.loads(
            path.read_text(encoding="utf-8")
        )
    except Exception:
        return False

    return (
        payload.get("split_package_fingerprint_sha256")
        == EXPECTED_SPLIT_FINGERPRINT
    )


def _frozen_verify_root(root: Path) -> bool:
    policy_path = (
        root
        / "17_FROZEN_SPLIT_POLICY_AND_HASHES.json"
    )
    if not policy_path.is_file():
        return False

    if not _frozen_valid_policy(policy_path):
        return False

    policy = json.loads(
        policy_path.read_text(encoding="utf-8")
    )

    for name, expected_hash in (
        FROZEN_EXPECTED_MANIFEST_HASHES.items()
    ):
        p = root / name

        if not p.is_file():
            return False

        actual = file_sha256(p)
        if actual != expected_hash:
            print(
                "Frozen-root hash rejection:",
                name,
                actual,
                "!=",
                expected_hash,
            )
            return False

        policy_expected = (
            policy
            .get("manifest_hashes", {})
            .get(name)
        )
        if (
            policy_expected
            and policy_expected != actual
        ):
            return False

    return True


def _frozen_find_attached_root():
    for base in (
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ):
        if not base.exists():
            continue

        for hit in base.rglob(
            "17_FROZEN_SPLIT_POLICY_AND_HASHES.json"
        ):
            candidate = hit.parent

            if _frozen_verify_root(candidate):
                print(
                    "✅ Exact frozen split already attached:",
                    candidate,
                )
                return candidate

    return None


def _frozen_extract_attached_zip():
    zip_paths = []

    for base in (
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ):
        if not base.exists():
            continue

        zip_paths.extend(
            base.rglob(
                "KERMANY_CLEAN_SPLIT_V1_FROZEN.zip"
            )
        )

    for idx, zpath in enumerate(zip_paths):
        extract_root = (
            WORK
            / f"_selfheal_frozen_zip_{idx:02d}"
        )

        if extract_root.exists():
            shutil.rmtree(extract_root)

        extract_root.mkdir(
            parents=True,
            exist_ok=True,
        )

        try:
            with zipfile.ZipFile(
                zpath,
                "r",
            ) as z:
                z.extractall(extract_root)
        except Exception as exc:
            print(
                "Skipping unreadable frozen ZIP:",
                zpath,
                repr(exc),
            )
            continue

        for hit in extract_root.rglob(
            "17_FROZEN_SPLIT_POLICY_AND_HASHES.json"
        ):
            candidate = hit.parent

            if _frozen_verify_root(candidate):
                print(
                    "✅ Exact frozen split restored from attached ZIP:",
                    zpath,
                )
                return candidate

    return None


def _frozen_is_raw_root(p: Path) -> bool:
    return all(
        (p / split / label).is_dir()
        for split in ("train", "val", "test")
        for label in ("NORMAL", "PNEUMONIA")
    )


def _frozen_locate_raw_dataset() -> Path:
    # 1) Attached Kaggle input.
    base = Path("/kaggle/input")

    if base.exists():
        likely = (
            base
            / "datasets"
            / "paultimothymooney"
            / "chest-xray-pneumonia"
            / "chest_xray"
        )

        if _frozen_is_raw_root(likely):
            print(
                "✅ Raw Kermany dataset attached:",
                likely,
            )
            return likely

        for p in base.rglob("chest_xray"):
            if (
                p.is_dir()
                and _frozen_is_raw_root(p)
            ):
                print(
                    "✅ Raw Kermany dataset discovered:",
                    p,
                )
                return p

    # 2) Public KaggleHub fallback.
    if importlib.util.find_spec(
        "kagglehub"
    ) is None:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "kagglehub",
        ])

    import kagglehub

    print(
        "Frozen split absent. Downloading public raw dataset "
        "for deterministic self-heal..."
    )

    dl = Path(
        kagglehub.dataset_download(
            FROZEN_DATASET_SLUG
        )
    )

    for p in (
        dl,
        dl / "chest_xray",
        dl / "chest_xray" / "chest_xray",
    ):
        if _frozen_is_raw_root(p):
            return p

    for p in dl.rglob("chest_xray"):
        if _frozen_is_raw_root(p):
            return p

    raise FileNotFoundError(
        "Canonical Kermany/Mooney raw chest_xray root "
        "could not be attached or downloaded."
    )


def _frozen_subtype(filename, label):
    if label != "PNEUMONIA":
        return ""

    name = filename.lower()

    if "_bacteria_" in name:
        return "BACTERIA"

    if "_virus_" in name:
        return "VIRUS"

    return "PNEUMONIA_UNSPECIFIED"


def _frozen_patient_id(filename, label):
    stem = Path(filename).stem

    if label == "PNEUMONIA":
        m = re.match(
            r"(?i)^person(\d+)(?:_|$)",
            stem,
        )
        if m:
            return (
                f"PNEU_person{int(m.group(1))}"
            )

    if label == "NORMAL":
        m = re.search(
            r"(?i)(?:NORMAL2-)?IM-(\d+)",
            stem,
        )
        if m:
            return (
                f"NORMAL_IM-{int(m.group(1)):04d}"
            )

    return None


def _frozen_rebuild_exact_package() -> Path:
    raw_root = _frozen_locate_raw_dataset()

    out = (
        WORK
        / "kermany_clean_split_v1_self_healed"
    )

    if out.exists():
        shutil.rmtree(out)

    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("\n" + "=" * 110)
    print("SELF-HEAL: REBUILDING FROZEN CLEAN SPLIT")
    print("=" * 110)
    print("Live raw root:", raw_root)
    print(
        "Manifest provenance absolute_path root:",
        FROZEN_CANONICAL_ABSOLUTE_ROOT,
    )

    # --------------------------------------------------------
    # A) Canonical inventory.
    # --------------------------------------------------------
    img_exts = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
    }

    rows = []

    for legacy_split in (
        "train",
        "val",
        "test",
    ):
        for model_label in (
            "NORMAL",
            "PNEUMONIA",
        ):
            folder = (
                raw_root
                / legacy_split
                / model_label
            )

            for p in sorted(
                folder.iterdir()
            ):
                if (
                    not p.is_file()
                    or p.suffix.lower()
                    not in img_exts
                    or p.name.startswith("._")
                ):
                    continue

                pid = _frozen_patient_id(
                    p.name,
                    model_label,
                )

                rel = p.relative_to(
                    raw_root
                )

                rows.append({
                    "filename": p.name,
                    "relative_path": str(rel),
                    # Reproduce the original frozen provenance column
                    # independent of the current live mount path.
                    "absolute_path": str(
                        FROZEN_CANONICAL_ABSOLUTE_ROOT
                        / rel
                    ),
                    "legacy_split": legacy_split,
                    "model_label": model_label,
                    "pneumonia_subtype": _frozen_subtype(
                        p.name,
                        model_label,
                    ),
                    "patient_id": pid,
                    "file_size_bytes": p.stat().st_size,
                    "sha256": file_sha256(p),
                    "image_ok": True,
                })

    inv = pd.DataFrame(rows)

    assert len(inv) == FROZEN_EXPECTED_TOTAL
    assert (
        inv["model_label"]
        == "NORMAL"
    ).sum() == FROZEN_EXPECTED_NORMAL
    assert (
        inv["model_label"]
        == "PNEUMONIA"
    ).sum() == FROZEN_EXPECTED_PNEUMONIA
    assert inv["patient_id"].notna().all()

    inv.to_csv(
        out / "00_raw_canonical_inventory.csv",
        index=False,
    )

    # --------------------------------------------------------
    # B) Pre-cleaning evidence.
    # --------------------------------------------------------
    (
        inv.groupby(
            "model_label"
        )["patient_id"]
        .nunique()
        .rename(
            "unique_patients"
        )
        .reset_index()
        .to_csv(
            out
            / "01_true_patient_counts_by_class.csv",
            index=False,
        )
    )

    normal_images = int(
        (
            inv["model_label"]
            == "NORMAL"
        ).sum()
    )
    pneumonia_patients = int(
        inv.loc[
            inv[
                "model_label"
            ]
            == "PNEUMONIA",
            "patient_id",
        ].nunique()
    )

    paper_repro = {
        "true_unique_patients_both_classes": int(
            inv[
                "patient_id"
            ].nunique()
        ),
        "normal_unique_patients_from_filename": int(
            inv.loc[
                inv[
                    "model_label"
                ]
                == "NORMAL",
                "patient_id",
            ].nunique()
        ),
        "pneumonia_unique_patients_from_filename": pneumonia_patients,
        "normal_images": normal_images,
        "paper_reported_patient_count": 3257,
        "pneumonia_patients_plus_each_normal_image": (
            pneumonia_patients
            + normal_images
        ),
        "matches_paper_reported_count": (
            pneumonia_patients
            + normal_images
            == 3257
        ),
    }

    (
        out
        / "02_paper_patient_count_reproduction.json"
    ).write_text(
        json.dumps(
            paper_repro,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    legacy_train_ids = set(
        inv.loc[
            inv[
                "legacy_split"
            ]
            == "train",
            "patient_id",
        ]
    )
    legacy_test_ids = set(
        inv.loc[
            inv[
                "legacy_split"
            ]
            == "test",
            "patient_id",
        ]
    )
    legacy_common = sorted(
        legacy_train_ids
        & legacy_test_ids
    )

    legacy_overlap_images = inv[
        inv[
            "patient_id"
        ].isin(
            legacy_common
        )
        & inv[
            "legacy_split"
        ].isin(
            [
                "train",
                "test",
            ]
        )
    ].copy()

    legacy_overlap_images.to_csv(
        out
        / "03_legacy_train_test_patient_overlap_images.csv",
        index=False,
    )

    legacy_summary_rows = []

    for pid in legacy_common:
        g = legacy_overlap_images[
            legacy_overlap_images[
                "patient_id"
            ]
            == pid
        ]
        tr = g[
            g[
                "legacy_split"
            ]
            == "train"
        ]
        te = g[
            g[
                "legacy_split"
            ]
            == "test"
        ]

        legacy_summary_rows.append({
            "patient_id": pid,
            "model_label": g[
                "model_label"
            ].iloc[0],
            "n_train_images": len(tr),
            "n_test_images": len(te),
            "train_filenames": " | ".join(
                tr[
                    "filename"
                ].tolist()
            ),
            "test_filenames": " | ".join(
                te[
                    "filename"
                ].tolist()
            ),
        })

    pd.DataFrame(
        legacy_summary_rows
    ).to_csv(
        out
        / "04_legacy_train_test_patient_overlap_summary.csv",
        index=False,
    )

    # --------------------------------------------------------
    # C) Exact dedup — exact original frozen rules.
    # --------------------------------------------------------
    cleaning = inv.copy()

    cleaning[
        "cleaning_action"
    ] = "KEEP"
    cleaning[
        "cleaning_reason"
    ] = (
        "UNIQUE_OR_CANONICAL_REPRESENTATIVE"
    )
    cleaning[
        "duplicate_group_id"
    ] = ""
    cleaning[
        "representative_relative_path"
    ] = ""

    duplicate_group_rows = []
    group_num = 0

    for sha, g in cleaning.groupby(
        "sha256",
        sort=True,
    ):
        if len(g) <= 1:
            continue

        group_num += 1
        gid = (
            f"EXACT_{group_num:05d}"
        )

        idxs = list(
            g.index
        )
        pids = sorted(
            g[
                "patient_id"
            ].unique()
        )
        labels = sorted(
            g[
                "model_label"
            ].unique()
        )

        cleaning.loc[
            idxs,
            "duplicate_group_id",
        ] = gid

        if len(labels) > 1:
            cleaning.loc[
                idxs,
                "cleaning_action",
            ] = "QUARANTINE"
            cleaning.loc[
                idxs,
                "cleaning_reason",
            ] = (
                "EXACT_DUPLICATE_LABEL_CONFLICT"
            )
            representative = ""

        elif len(pids) > 1:
            cleaning.loc[
                idxs,
                "cleaning_action",
            ] = "QUARANTINE"
            cleaning.loc[
                idxs,
                "cleaning_reason",
            ] = (
                "EXACT_DUPLICATE_PATIENT_ID_CONFLICT"
            )
            representative = ""

        else:
            representative = sorted(
                g[
                    "relative_path"
                ].tolist()
            )[0]

            for idx in idxs:
                if (
                    cleaning.at[
                        idx,
                        "relative_path",
                    ]
                    == representative
                ):
                    cleaning.at[
                        idx,
                        "cleaning_action",
                    ] = "KEEP"
                    cleaning.at[
                        idx,
                        "cleaning_reason",
                    ] = (
                        "EXACT_DUPLICATE_CANONICAL_REPRESENTATIVE"
                    )
                else:
                    cleaning.at[
                        idx,
                        "cleaning_action",
                    ] = "EXCLUDE"
                    cleaning.at[
                        idx,
                        "cleaning_reason",
                    ] = (
                        "EXACT_DUPLICATE_REDUNDANT"
                    )

            cleaning.loc[
                idxs,
                "representative_relative_path",
            ] = representative

        duplicate_group_rows.append({
            "duplicate_group_id": gid,
            "sha256": sha,
            "n_files": len(g),
            "patient_ids": " | ".join(
                pids
            ),
            "labels": " | ".join(
                labels
            ),
            "legacy_splits": " | ".join(
                sorted(
                    g[
                        "legacy_split"
                    ].unique()
                )
            ),
            "filenames": " | ".join(
                sorted(
                    g[
                        "filename"
                    ].tolist()
                )
            ),
            "rule": (
                "QUARANTINE_LABEL_CONFLICT"
                if len(labels) > 1
                else
                "QUARANTINE_PATIENT_ID_CONFLICT"
                if len(pids) > 1
                else
                "KEEP_ONE_DETERMINISTIC"
            ),
            "representative_relative_path": representative,
        })

    cleaning.to_csv(
        out
        / "05_cleaning_decisions_all_images.csv",
        index=False,
    )

    pd.DataFrame(
        duplicate_group_rows
    ).to_csv(
        out
        / "06_exact_duplicate_group_decisions.csv",
        index=False,
    )

    cleaning[
        cleaning[
            "cleaning_action"
        ]
        != "KEEP"
    ].copy().to_csv(
        out
        / "07_excluded_or_quarantined_images.csv",
        index=False,
    )

    clean_master = (
        cleaning[
            cleaning[
                "cleaning_action"
            ]
            == "KEEP"
        ]
        .copy()
        .sort_values(
            [
                "patient_id",
                "model_label",
                "relative_path",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    clean_master.to_csv(
        out
        / "08_clean_master_manifest.csv",
        index=False,
    )

    # Original safety assertions.
    assert (
        clean_master
        .groupby(
            "patient_id"
        )[
            "model_label"
        ]
        .nunique()
        .max()
        == 1
    )
    assert (
        clean_master[
            "sha256"
        ]
        .value_counts()
        .max()
        == 1
    )
    assert clean_master[
        "relative_path"
    ].is_unique
    assert clean_master[
        "patient_id"
    ].notna().all()

    # --------------------------------------------------------
    # D) Exact patient-wise 90/10 split.
    # --------------------------------------------------------
    patient_table = (
        clean_master.groupby(
            "patient_id"
        )
        .agg(
            model_label=(
                "model_label",
                "first",
            ),
            n_images=(
                "relative_path",
                "size",
            ),
        )
        .reset_index()
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    dev_patients, test_patients = train_test_split(
        patient_table,
        test_size=FROZEN_TEST_SIZE,
        random_state=FROZEN_SEED,
        shuffle=True,
        stratify=patient_table[
            "model_label"
        ],
    )

    dev_ids = set(
        dev_patients[
            "patient_id"
        ]
    )
    test_ids = set(
        test_patients[
            "patient_id"
        ]
    )

    assert dev_ids.isdisjoint(
        test_ids
    )
    assert (
        dev_ids
        | test_ids
        == set(
            patient_table[
                "patient_id"
            ]
        )
    )

    split_master = clean_master.copy()

    split_master[
        "final_partition"
    ] = np.where(
        split_master[
            "patient_id"
        ].isin(
            test_ids
        ),
        "TEST_LOCKED",
        "DEVELOPMENT",
    )

    development = split_master[
        split_master[
            "final_partition"
        ]
        == "DEVELOPMENT"
    ].copy()

    locked_test = split_master[
        split_master[
            "final_partition"
        ]
        == "TEST_LOCKED"
    ].copy()

    split_master.to_csv(
        out
        / "09_clean_partition_master.csv",
        index=False,
    )
    development.to_csv(
        out
        / "10_development_manifest.csv",
        index=False,
    )
    locked_test.to_csv(
        out
        / "11_locked_test_manifest.csv",
        index=False,
    )

    dev_patients.sort_values(
        "patient_id"
    ).to_csv(
        out
        / "12_development_patients.csv",
        index=False,
    )
    test_patients.sort_values(
        "patient_id"
    ).to_csv(
        out
        / "13_locked_test_patients.csv",
        index=False,
    )

    # Hard expected counts from the already frozen package.
    assert len(clean_master) == 5823
    assert (
        clean_master[
            "patient_id"
        ].nunique()
        == 2789
    )
    assert len(development) == 5209
    assert (
        development[
            "patient_id"
        ].nunique()
        == 2510
    )
    assert len(locked_test) == 614
    assert (
        locked_test[
            "patient_id"
        ].nunique()
        == 279
    )

    # --------------------------------------------------------
    # E) Exact 5-fold patient-locked validation.
    # --------------------------------------------------------
    sgkf = StratifiedGroupKFold(
        n_splits=FROZEN_N_FOLDS,
        shuffle=True,
        random_state=FROZEN_SEED,
    )

    dev = (
        development.sort_values(
            [
                "patient_id",
                "relative_path",
            ]
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    dev[
        "fold_id"
    ] = -1

    X_dummy = np.zeros(
        (
            len(dev),
            1,
        )
    )
    y = dev[
        "model_label"
    ].to_numpy()
    groups = dev[
        "patient_id"
    ].to_numpy()

    for fold_id, (
        _,
        val_idx,
    ) in enumerate(
        sgkf.split(
            X_dummy,
            y,
            groups,
        ),
        start=1,
    ):
        dev.loc[
            val_idx,
            "fold_id",
        ] = fold_id

    assert (
        dev[
            "fold_id"
        ]
        >= 1
    ).all()
    assert set(
        dev[
            "fold_id"
        ].unique()
    ) == set(
        range(
            1,
            FROZEN_N_FOLDS + 1,
        )
    )

    dev.to_csv(
        out
        / "14_development_with_fold_assignments.csv",
        index=False,
    )

    fold_stats = []

    for fold in range(
        1,
        FROZEN_N_FOLDS + 1,
    ):
        val = dev[
            dev[
                "fold_id"
            ]
            == fold
        ].copy()

        train = dev[
            dev[
                "fold_id"
            ]
            != fold
        ].copy()

        train_ids_fold = set(
            train[
                "patient_id"
            ]
        )
        val_ids_fold = set(
            val[
                "patient_id"
            ]
        )
        locked_ids = set(
            locked_test[
                "patient_id"
            ]
        )

        assert train_ids_fold.isdisjoint(
            val_ids_fold
        )
        assert train_ids_fold.isdisjoint(
            locked_ids
        )
        assert val_ids_fold.isdisjoint(
            locked_ids
        )

        assert set(
            train[
                "sha256"
            ]
        ).isdisjoint(
            set(
                val[
                    "sha256"
                ]
            )
        )
        assert set(
            train[
                "sha256"
            ]
        ).isdisjoint(
            set(
                locked_test[
                    "sha256"
                ]
            )
        )
        assert set(
            val[
                "sha256"
            ]
        ).isdisjoint(
            set(
                locked_test[
                    "sha256"
                ]
            )
        )

        train.to_csv(
            out
            / f"fold_{fold}_train.csv",
            index=False,
        )
        val.to_csv(
            out
            / f"fold_{fold}_val.csv",
            index=False,
        )

        fold_stats.append({
            "fold_id": fold,
            "train_images": len(
                train
            ),
            "val_images": len(
                val
            ),
            "train_patients": len(
                train_ids_fold
            ),
            "val_patients": len(
                val_ids_fold
            ),
            "train_NORMAL": int(
                (
                    train[
                        "model_label"
                    ]
                    == "NORMAL"
                ).sum()
            ),
            "train_PNEUMONIA": int(
                (
                    train[
                        "model_label"
                    ]
                    == "PNEUMONIA"
                ).sum()
            ),
            "val_NORMAL": int(
                (
                    val[
                        "model_label"
                    ]
                    == "NORMAL"
                ).sum()
            ),
            "val_PNEUMONIA": int(
                (
                    val[
                        "model_label"
                    ]
                    == "PNEUMONIA"
                ).sum()
            ),
            "train_val_patient_overlap": len(
                train_ids_fold
                & val_ids_fold
            ),
            "val_test_patient_overlap": len(
                val_ids_fold
                & locked_ids
            ),
            "train_test_patient_overlap": len(
                train_ids_fold
                & locked_ids
            ),
        })

    fold_stats = pd.DataFrame(
        fold_stats
    )

    fold_stats.to_csv(
        out
        / "15_fold_statistics.csv",
        index=False,
    )

    # --------------------------------------------------------
    # F) Leakage assertions.
    # --------------------------------------------------------
    assertions = {
        "clean_master_exact_sha_duplicates": int(
            clean_master[
                "sha256"
            ].duplicated().sum()
        ),
        "development_test_patient_overlap": len(
            set(
                development[
                    "patient_id"
                ]
            )
            & set(
                locked_test[
                    "patient_id"
                ]
            )
        ),
        "development_test_sha_overlap": len(
            set(
                development[
                    "sha256"
                ]
            )
            & set(
                locked_test[
                    "sha256"
                ]
            )
        ),
        "all_fold_train_val_patient_overlap_zero": bool(
            (
                fold_stats[
                    "train_val_patient_overlap"
                ]
                == 0
            ).all()
        ),
        "all_fold_val_test_patient_overlap_zero": bool(
            (
                fold_stats[
                    "val_test_patient_overlap"
                ]
                == 0
            ).all()
        ),
        "all_fold_train_test_patient_overlap_zero": bool(
            (
                fold_stats[
                    "train_test_patient_overlap"
                ]
                == 0
            ).all()
        ),
    }

    (
        out
        / "16_leakage_assertion_report.json"
    ).write_text(
        json.dumps(
            assertions,
            indent=2,
        ),
        encoding="utf-8",
    )

    assert (
        assertions[
            "clean_master_exact_sha_duplicates"
        ]
        == 0
    )
    assert (
        assertions[
            "development_test_patient_overlap"
        ]
        == 0
    )
    assert (
        assertions[
            "development_test_sha_overlap"
        ]
        == 0
    )
    assert assertions[
        "all_fold_train_val_patient_overlap_zero"
    ]
    assert assertions[
        "all_fold_val_test_patient_overlap_zero"
    ]
    assert assertions[
        "all_fold_train_test_patient_overlap_zero"
    ]

    # --------------------------------------------------------
    # G) Recreate original manifest hashes + frozen fingerprint.
    # --------------------------------------------------------
    manifest_names = [
        "08_clean_master_manifest.csv",
        "09_clean_partition_master.csv",
        "10_development_manifest.csv",
        "11_locked_test_manifest.csv",
        "12_development_patients.csv",
        "13_locked_test_patients.csv",
        "14_development_with_fold_assignments.csv",
        "15_fold_statistics.csv",
    ] + [
        f"fold_{i}_train.csv"
        for i in range(
            1,
            FROZEN_N_FOLDS + 1,
        )
    ] + [
        f"fold_{i}_val.csv"
        for i in range(
            1,
            FROZEN_N_FOLDS + 1,
        )
    ]

    hashes = {
        name: file_sha256(
            out / name
        )
        for name in manifest_names
    }

    # Verify EVERY known frozen hash independently.
    hash_mismatches = {
        name: {
            "expected": expected,
            "actual": hashes.get(
                name
            ),
        }
        for name, expected
        in FROZEN_EXPECTED_MANIFEST_HASHES.items()
        if hashes.get(
            name
        ) != expected
    }

    if hash_mismatches:
        (
            out
            / "SELF_HEAL_HASH_MISMATCH.json"
        ).write_text(
            json.dumps(
                hash_mismatches,
                indent=2,
            ),
            encoding="utf-8",
        )

        raise RuntimeError(
            "SELF-HEAL REBUILD DID NOT MATCH THE FROZEN MANIFEST HASHES. "
            "Training is blocked. "
            f"Mismatch report: {out / 'SELF_HEAL_HASH_MISMATCH.json'}"
        )

    policy = {
        "schema": "kermany.clean.split.v1",
        "dataset_slug": FROZEN_DATASET_SLUG,
        "seed": FROZEN_SEED,
        "test_size": FROZEN_TEST_SIZE,
        "n_folds": FROZEN_N_FOLDS,
        "patient_id_policy": {
            "PNEUMONIA": "personXXXX from filename",
            "NORMAL": "IM-XXXX from filename",
        },
        "class_policy": {
            "BACTERIA": "PNEUMONIA",
            "VIRUS": "PNEUMONIA",
            "NORMAL": "NORMAL",
        },
        "exact_duplicate_policy": {
            "same_patient_same_label":
                "keep one lexicographically smallest relative_path",
            "different_patient_ids":
                "quarantine entire exact-duplicate group",
            "different_labels":
                "quarantine entire exact-duplicate group",
        },
        "near_duplicate_policy":
            "audit only; no automatic removal because no strong SSIM>=0.95 evidence",
        "split_policy":
            "patient-wise stratified 90/10, random_state=42",
        "validation_policy":
            "5-fold StratifiedGroupKFold, group=patient_id, shuffle=True, random_state=42",
        "manifest_hashes": hashes,
    }

    policy_blob = json.dumps(
        policy,
        sort_keys=True,
        ensure_ascii=False,
    ).encode(
        "utf-8"
    )

    policy[
        "split_package_fingerprint_sha256"
    ] = hashlib.sha256(
        policy_blob
    ).hexdigest()

    (
        out
        / "17_FROZEN_SPLIT_POLICY_AND_HASHES.json"
    ).write_text(
        json.dumps(
            policy,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    if (
        policy[
            "split_package_fingerprint_sha256"
        ]
        != EXPECTED_SPLIT_FINGERPRINT
    ):
        raise RuntimeError(
            "Self-healed split manifest hashes matched, "
            "but final policy fingerprint did not. "
            "Training is blocked."
        )

    report = {
        "schema": "master.self_heal.split.v1",
        "status": "EXACT_FROZEN_SPLIT_REBUILT",
        "raw_root": str(
            raw_root
        ),
        "fingerprint": policy[
            "split_package_fingerprint_sha256"
        ],
        "clean_images": int(
            len(
                clean_master
            )
        ),
        "clean_patients": int(
            clean_master[
                "patient_id"
            ].nunique()
        ),
        "development_images": int(
            len(
                development
            )
        ),
        "development_patients": int(
            development[
                "patient_id"
            ].nunique()
        ),
        "locked_test_images": int(
            len(
                locked_test
            )
        ),
        "locked_test_patients": int(
            locked_test[
                "patient_id"
            ].nunique()
        ),
        "manifest_hashes_all_matched": True,
        "private_kernel_dependency": False,
    }

    (
        out
        / "SELF_HEAL_SPLIT_REPORT.json"
    ).write_text(
        json.dumps(
            report,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("\n" + "=" * 110)
    print("✅ SELF-HEAL FROZEN SPLIT REBUILD PASSED")
    print("=" * 110)
    print(
        "Fingerprint:",
        policy[
            "split_package_fingerprint_sha256"
        ],
    )
    print(
        "Clean:",
        len(
            clean_master
        ),
        "images /",
        clean_master[
            "patient_id"
        ].nunique(),
        "patients",
    )
    print(
        "Development:",
        len(
            development
        ),
        "/",
        development[
            "patient_id"
        ].nunique(),
    )
    print(
        "Locked Test:",
        len(
            locked_test
        ),
        "/",
        locked_test[
            "patient_id"
        ].nunique(),
    )
    print(
        "✅ All 18 frozen manifest hashes matched exactly."
    )

    return out


def locate_split_root() -> Path:
    # 1) Exact attached frozen split.
    attached = _frozen_find_attached_root()

    if attached is not None:
        return attached

    # 2) Exact attached frozen ZIP.
    restored_zip = _frozen_extract_attached_zip()

    if restored_zip is not None:
        return restored_zip

    # 3) SELF-HEAL from the raw public data.
    print(
        "\n⚠️ Frozen split package not attached."
    )
    print(
        "→ Self-healing from the canonical raw Kermany dataset."
    )

    rebuilt = _frozen_rebuild_exact_package()

    if not _frozen_verify_root(
        rebuilt
    ):
        raise RuntimeError(
            "Self-healed frozen split did not pass final verification."
        )

    return rebuilt


SPLIT_ROOT = locate_split_root()

POLICY_PATH = (
    SPLIT_ROOT
    / "17_FROZEN_SPLIT_POLICY_AND_HASHES.json"
)

split_policy = json.loads(
    POLICY_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    split_policy[
        "split_package_fingerprint_sha256"
    ]
    == EXPECTED_SPLIT_FINGERPRINT
)

# Verify all exact manifest hashes again at the consumption boundary.
for name, expected_hash in (
    FROZEN_EXPECTED_MANIFEST_HASHES.items()
):
    p = SPLIT_ROOT / name

    assert p.is_file(), (
        f"Missing frozen manifest: {name}"
    )

    actual_hash = file_sha256(
        p
    )

    assert (
        actual_hash
        == expected_hash
    ), (
        f"Frozen manifest SHA mismatch: {name}\n"
        f"expected={expected_hash}\n"
        f"actual={actual_hash}"
    )

print("\nSPLIT_ROOT:", SPLIT_ROOT)
print("✅ Frozen split fingerprint verified")
print("✅ All frozen manifest SHA256 values verified")
print("✅ Private kernel dependency: NONE")
print("✅ Split dependency resolved; HPO may continue")

In [ ]:
# ============================================================
# 3) LOCATE RAW KAGGLE DATASET + LOAD FROZEN MANIFESTS
# ============================================================
DATASET_SLUG = "paultimothymooney/chest-xray-pneumonia"

def is_dataset_root(p: Path) -> bool:
    return all(
        (p/s/l).is_dir()
        for s in ("train", "val", "test")
        for l in ("NORMAL", "PNEUMONIA")
    )

def locate_data_root() -> Path:
    base = Path("/kaggle/input")
    if base.exists():
        likely = (
            base / "datasets" / "paultimothymooney"
            / "chest-xray-pneumonia" / "chest_xray"
        )
        if is_dataset_root(likely):
            return likely
        for p in base.rglob("chest_xray"):
            if p.is_dir() and is_dataset_root(p):
                return p

    import importlib.util
    if importlib.util.find_spec("kagglehub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub
    dl = Path(kagglehub.dataset_download(DATASET_SLUG))
    for p in [dl, dl/"chest_xray", dl/"chest_xray"/"chest_xray"]:
        if is_dataset_root(p):
            return p
    for p in dl.rglob("chest_xray"):
        if is_dataset_root(p):
            return p
    raise FileNotFoundError("Canonical Kermany/Mooney chest_xray root not found.")

DATA_ROOT = locate_data_root()
print("DATA_ROOT:", DATA_ROOT)

def prepare_manifest(frame: pd.DataFrame) -> pd.DataFrame:
    required = {"relative_path", "patient_id", "model_label", "sha256"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Manifest missing columns: {sorted(missing)}")

    out = frame.copy()
    if out.empty or out[list(required)].isna().any().any():
        raise ValueError("Manifest is empty or contains missing identity fields.")
    if not out["model_label"].isin(["NORMAL", "PNEUMONIA"]).all():
        raise ValueError("Manifest contains unsupported model labels.")
    if out["relative_path"].astype(str).duplicated().any():
        raise ValueError("Manifest contains duplicated image paths.")
    if out["sha256"].astype(str).duplicated().any():
        raise ValueError("Manifest contains duplicated image content identities.")
    if not out["sha256"].astype(str).str.fullmatch(r"[0-9a-f]{64}").all():
        raise ValueError("Manifest SHA256 values must be lowercase hexadecimal hashes.")
    if out["patient_id"].astype(str).str.strip().eq("").any():
        raise ValueError("Manifest contains empty patient grouping IDs.")
    out["label"] = (
        out["model_label"]
        .map({"NORMAL": 0, "PNEUMONIA": 1})
        .astype(int)
    )
    dataset_root = DATA_ROOT.resolve()
    actual_paths = []
    for row in out.itertuples(index=False):
        relative = Path(str(row.relative_path))
        path = (dataset_root / relative).resolve()
        if relative.is_absolute() or not path.is_relative_to(dataset_root):
            raise ValueError(f"Image path escapes dataset root: {relative}")
        if not path.is_file():
            raise FileNotFoundError(f"Manifest image is missing: {path}")
        # Always hash live bytes at consumption. A CSV hash alone cannot prove
        # that an attached or restored image still matches the frozen split.
        stat_before = path.stat()
        observed_sha = run_file_sha256(path)
        stat_after = path.stat()
        identity_before = (stat_before.st_ino, stat_before.st_size,
                           stat_before.st_mtime_ns, stat_before.st_ctime_ns)
        identity_after = (stat_after.st_ino, stat_after.st_size,
                          stat_after.st_mtime_ns, stat_after.st_ctime_ns)
        if identity_before != identity_after:
            raise ValueError(f"Image changed while verifying bytes: {relative}")
        if observed_sha != str(row.sha256):
            raise ValueError(f"Live image SHA256 differs from frozen manifest: {relative}")
        actual_paths.append(str(path))
    out["filepath"] = actual_paths
    return out.reset_index(drop=True)

def load_fold_manifest(fold: int, role: str) -> pd.DataFrame:
    assert role in {"train", "val"}
    return prepare_manifest(
        pd.read_csv(SPLIT_ROOT / f"fold_{fold}_{role}.csv")
    )

def load_locked_test_manifest() -> pd.DataFrame:
    return prepare_manifest(
        pd.read_csv(SPLIT_ROOT / "11_locked_test_manifest.csv")
    )

development_manifest = prepare_manifest(
    pd.read_csv(SPLIT_ROOT / "14_development_with_fold_assignments.csv")
)

for fold in range(1, 6):
    tr = load_fold_manifest(fold, "train")
    va = load_fold_manifest(fold, "val")

    assert set(tr["patient_id"]).isdisjoint(set(va["patient_id"]))
    assert set(tr["sha256"]).isdisjoint(set(va["sha256"]))

    print(
        f"Fold {fold}: train={len(tr)}/{tr.patient_id.nunique()} patients | "
        f"val={len(va)}/{va.patient_id.nunique()} patients | overlap=0"
    )

print("✅ Five patient-locked folds verified")
print("⚠️ Locked Test exists but is NOT loaded yet")


In [ ]:

# ============================================================
# 4) BALANCED METRICS + THRESHOLD SELECTION
#    Based on the corrected metric policy from the prior ZIP.
# ============================================================
EPS = 1e-7

def calculate_metrics(y_true, probability, threshold):
    y = np.asarray(y_true, dtype=int)
    p = np.clip(np.asarray(probability, dtype=float), EPS, 1.0-EPS)
    pred = (p >= float(threshold)).astype(int)

    cm = confusion_matrix(y, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall_n = tn / max(tn + fp, 1)
    recall_p = tp / max(tp + fn, 1)
    precision_n = precision_score(y, pred, pos_label=0, zero_division=0)
    precision_p = precision_score(y, pred, pos_label=1, zero_division=0)
    f1_n = f1_score(y, pred, pos_label=0, zero_division=0)
    f1_p = f1_score(y, pred, pos_label=1, zero_division=0)

    bal_acc = balanced_accuracy_score(y, pred)
    mcc = matthews_corrcoef(y, pred)
    gmean = math.sqrt(max(recall_n * recall_p, 0.0))
    min_recall = min(recall_n, recall_p)
    min_precision = min(precision_n, precision_p)
    min_dual = min(recall_n, recall_p, precision_n, precision_p)
    macro_precision = (precision_n + precision_p) / 2
    macro_f1 = (f1_n + f1_p) / 2
    recall_gap = abs(recall_n - recall_p)
    precision_gap = abs(precision_n - precision_p)

    auroc = roc_auc_score(y, p)
    auprc_p = average_precision_score(y, p)
    auprc_n = average_precision_score(1-y, 1-p)
    macro_auprc = (auprc_n + auprc_p) / 2
    brier = brier_score_loss(y, p)
    mcc01 = (mcc + 1) / 2

    dual_score = (
        0.40 * min_dual
        + 0.15 * macro_precision
        + 0.15 * min_recall
        + 0.10 * mcc01
        + 0.10 * bal_acc
        + 0.05 * macro_f1
        + 0.05 * macro_auprc
        - 0.025 * recall_gap
        - 0.025 * precision_gap
        - 0.05 * brier
    )

    return {
        "n": int(len(y)),
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(bal_acc),
        "precision_normal": float(precision_n),
        "recall_normal": float(recall_n),
        "f1_normal": float(f1_n),
        "precision_pneumonia": float(precision_p),
        "recall_pneumonia": float(recall_p),
        "f1_pneumonia": float(f1_p),
        "f2": float(fbeta_score(y, pred, beta=2, pos_label=1, zero_division=0)),
        "macro_precision": float(macro_precision),
        "macro_f1": float(macro_f1),
        "min_class_recall": float(min_recall),
        "min_class_precision": float(min_precision),
        "min_dual_precision_recall": float(min_dual),
        "dual_precision_recall_score": float(dual_score),
        "mcc": float(mcc),
        "gmean": float(gmean),
        "auroc": float(auroc),
        "auprc_pneumonia": float(auprc_p),
        "auprc_normal": float(auprc_n),
        "macro_auprc": float(macro_auprc),
        "brier": float(brier),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "confusion_matrix": cm.tolist(),
    }

def threshold_rank(metrics):
    return (
        metrics["min_dual_precision_recall"],
        metrics["dual_precision_recall_score"],
        metrics["min_class_recall"],
        metrics["min_class_precision"],
        metrics["mcc"],
        metrics["balanced_accuracy"],
        metrics["macro_f1"],
        metrics["macro_auprc"],
        -abs(metrics["threshold"] - 0.5),
    )

def select_balanced_threshold(y_true, probability):
    """Same threshold policy using sorted confusion counts, not repeated sklearn fits.

    Exact candidate locations and lexicographic objectives are preserved.
    Threshold-independent AUROC/AP/Brier are computed once; the final report
    still comes from calculate_metrics, the canonical sklearn implementation.
    """
    y = np.asarray(y_true, dtype=int)
    raw = np.asarray(probability, dtype=float)
    if y.ndim != 1 or raw.ndim != 1 or len(y) != len(raw) or len(y) == 0:
        raise ValueError("Threshold selection requires equally sized nonempty vectors")
    if set(np.unique(y)) != {0, 1} or not np.isfinite(raw).all():
        raise ValueError("Threshold selection requires both labels and finite scores")
    p = np.clip(raw, EPS, 1.0 - EPS)
    unique = np.unique(p)
    mid = (unique[:-1] + unique[1:]) / 2 if len(unique) > 1 else np.array([])
    candidates = np.unique(np.concatenate(([EPS, 0.5, 1.0-EPS, 1.0], unique, mid)))
    order = np.argsort(p, kind="stable")
    sorted_p = p[order]
    pos_prefix = np.concatenate(([0], np.cumsum(y[order], dtype=np.int64)))
    below = np.searchsorted(sorted_p, candidates, side="left")
    fn = pos_prefix[below].astype(float)
    tn = below.astype(float) - fn
    tp = float(np.sum(y)) - fn
    fp = float(len(y) - np.sum(y)) - tn

    def divide(a, b):
        return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)

    recall_n = divide(tn, tn + fp)
    recall_p = divide(tp, tp + fn)
    precision_n = divide(tn, tn + fn)
    precision_p = divide(tp, tp + fp)
    f1_n = divide(2.0 * tn, 2.0 * tn + fp + fn)
    f1_p = divide(2.0 * tp, 2.0 * tp + fp + fn)
    mcc = divide(tp * tn - fp * fn,
                 np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    bal_acc = (recall_n + recall_p) / 2.0
    min_recall = np.minimum(recall_n, recall_p)
    min_precision = np.minimum(precision_n, precision_p)
    min_dual = np.minimum(min_recall, min_precision)
    macro_precision = (precision_n + precision_p) / 2.0
    macro_f1 = (f1_n + f1_p) / 2.0
    macro_auprc = (average_precision_score(1-y, 1-p)
                   + average_precision_score(y, p)) / 2.0
    brier = brier_score_loss(y, p)
    dual_score = (0.40 * min_dual + 0.15 * macro_precision + 0.15 * min_recall
                  + 0.10 * ((mcc + 1) / 2) + 0.10 * bal_acc + 0.05 * macro_f1
                  + 0.05 * macro_auprc - 0.025 * np.abs(recall_n - recall_p)
                  - 0.025 * np.abs(precision_n - precision_p) - 0.05 * brier)
    # np.lexsort uses the last key as the highest priority.
    ranks = np.lexsort((-np.abs(candidates - 0.5),
                        np.full_like(candidates, macro_auprc), macro_f1,
                        bal_acc, mcc, min_precision, min_recall, dual_score, min_dual))
    selected = float(candidates[int(ranks[-1])])
    return calculate_metrics(y, p, selected)


print("Metric policy ready.")


def calculate_binary_metrics(y_true, pred, score=None):
    """Metrics when the classification rule is already fixed before Test."""
    y = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)

    cm = confusion_matrix(y, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    precision_n = precision_score(y, pred, pos_label=0, zero_division=0)
    precision_p = precision_score(y, pred, pos_label=1, zero_division=0)
    recall_n = recall_score(y, pred, pos_label=0, zero_division=0)
    recall_p = recall_score(y, pred, pos_label=1, zero_division=0)
    f1_n = f1_score(y, pred, pos_label=0, zero_division=0)
    f1_p = f1_score(y, pred, pos_label=1, zero_division=0)

    result = {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "macro_precision": float(precision_score(y, pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y, pred, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "precision_normal": float(precision_n),
        "recall_normal": float(recall_n),
        "f1_normal": float(f1_n),
        "precision_pneumonia": float(precision_p),
        "recall_pneumonia": float(recall_p),
        "f1_pneumonia": float(f1_p),
        "f2": float(fbeta_score(y, pred, beta=2, pos_label=1, zero_division=0)),
        "mcc": float(matthews_corrcoef(y, pred)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "confusion_matrix": cm.tolist(),
    }

    if score is not None:
        score = np.asarray(score, dtype=float)
        result["auroc"] = float(roc_auc_score(y, score))
        result["auprc_pneumonia"] = float(average_precision_score(y, score))
    else:
        result["auroc"] = None
        result["auprc_pneumonia"] = None

    return result

print("Binary fixed-rule metrics ready.")


# ------------------------------------------------------------------
# Fixed dual-threshold 20/80 reporting policy.
# This is descriptive only; it never selects/tunes a threshold.
# ------------------------------------------------------------------
def analyze_dual_threshold(
    y_true,
    probability,
    low=DUAL_THRESHOLD_LOW,
    high=DUAL_THRESHOLD_HIGH,
):
    if not 0.0 <= float(low) < float(high) <= 1.0:
        raise ValueError("Dual thresholds must satisfy 0 <= low < high <= 1.")

    y = np.asarray(y_true, dtype=int)
    p = np.clip(np.asarray(probability, dtype=float), 0.0, 1.0)

    # -1 = uncertain, 0 = confident normal, 1 = confident pneumonia.
    decision = np.full(len(p), -1, dtype=int)
    decision[p <= float(low)] = 0
    decision[p >= float(high)] = 1

    certain = decision != -1
    uncertain = ~certain

    normal_mask = y == 0
    pneumonia_mask = y == 1

    table = np.zeros((2, 3), dtype=int)
    # columns: confident NORMAL, UNCERTAIN, confident PNEUMONIA
    table[0, 0] = int(np.sum(normal_mask & (decision == 0)))
    table[0, 1] = int(np.sum(normal_mask & uncertain))
    table[0, 2] = int(np.sum(normal_mask & (decision == 1)))
    table[1, 0] = int(np.sum(pneumonia_mask & (decision == 0)))
    table[1, 1] = int(np.sum(pneumonia_mask & uncertain))
    table[1, 2] = int(np.sum(pneumonia_mask & (decision == 1)))

    report = {
        "policy": "FIXED_DUAL_THRESHOLD_DESCRIPTIVE_ONLY",
        "low_threshold": float(low),
        "high_threshold": float(high),
        "n_total": int(len(y)),
        "n_confident": int(np.sum(certain)),
        "n_uncertain": int(np.sum(uncertain)),
        "coverage": float(np.mean(certain)),
        "uncertainty_rate": float(np.mean(uncertain)),
        "normal_class_coverage": float(
            np.mean(certain[normal_mask]) if np.any(normal_mask) else np.nan
        ),
        "pneumonia_class_coverage": float(
            np.mean(certain[pneumonia_mask]) if np.any(pneumonia_mask) else np.nan
        ),
        "confident_normal_count": int(np.sum(decision == 0)),
        "confident_pneumonia_count": int(np.sum(decision == 1)),
        "three_way_confusion_rows_true_NORMAL_PNEUMONIA_cols_NORMAL_UNCERTAIN_PNEUMONIA": (
            table.tolist()
        ),
        "threshold_tuned": False,
    }

    if np.sum(certain) > 0:
        y_c = y[certain]
        pred_c = decision[certain]
        p_c = p[certain]

        cm = confusion_matrix(y_c, pred_c, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        report["confident_subset"] = {
            "n": int(len(y_c)),
            "accuracy": float(accuracy_score(y_c, pred_c)),
            "precision_pneumonia": float(
                precision_score(y_c, pred_c, pos_label=1, zero_division=0)
            ),
            "recall_pneumonia": float(
                recall_score(y_c, pred_c, pos_label=1, zero_division=0)
            ),
            "f1_pneumonia": float(
                f1_score(y_c, pred_c, pos_label=1, zero_division=0)
            ),
            "f2": float(
                fbeta_score(y_c, pred_c, beta=2, pos_label=1, zero_division=0)
            ),
            "precision_normal": float(
                precision_score(y_c, pred_c, pos_label=0, zero_division=0)
            ),
            "recall_normal": float(
                recall_score(y_c, pred_c, pos_label=0, zero_division=0)
            ),
            "mcc": float(matthews_corrcoef(y_c, pred_c))
            if len(np.unique(y_c)) > 1 else None,
            "balanced_accuracy": float(balanced_accuracy_score(y_c, pred_c))
            if len(np.unique(y_c)) > 1 else None,
            "auroc": float(roc_auc_score(y_c, p_c))
            if len(np.unique(y_c)) > 1 else None,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }
    else:
        report["confident_subset"] = None

    return report, decision

print(
    "Fixed dual-threshold reporting ready:",
    DUAL_THRESHOLD_LOW,
    DUAL_THRESHOLD_HIGH,
)

# ============================================================
# PATIENT-CLUSTER BOOTSTRAP
# ============================================================
def patient_cluster_bootstrap_indices(
    patient_ids,
    rng,
):
    patient_ids = np.asarray(
        patient_ids,
        dtype=object,
    )

    unique_patients = np.unique(
        patient_ids
    )

    sampled = rng.choice(
        unique_patients,
        size=len(unique_patients),
        replace=True,
    )

    parts = []

    for pid in sampled:
        idx = np.flatnonzero(
            patient_ids == pid
        )

        if len(idx):
            parts.append(idx)

    if not parts:
        return np.array(
            [],
            dtype=int,
        )

    return np.concatenate(
        parts
    ).astype(int)


def bootstrap_fixed_rule_by_patient(
    y,
    pred,
    score,
    patient_ids,
    n_boot,
    seed,
):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    score = np.asarray(score, dtype=float)
    patient_ids = np.asarray(
        patient_ids,
        dtype=object,
    )

    assert (
        len(y)
        == len(pred)
        == len(score)
        == len(patient_ids)
    )

    rng = np.random.default_rng(
        int(seed)
    )

    rows = []

    for _ in range(int(n_boot)):
        idx = (
            patient_cluster_bootstrap_indices(
                patient_ids,
                rng,
            )
        )

        if len(idx) == 0:
            continue

        yy = y[idx]

        if len(np.unique(yy)) < 2:
            continue

        mm = calculate_binary_metrics(
            yy,
            pred[idx],
            score=score[idx],
        )

        rows.append({
            "balanced_accuracy": mm["balanced_accuracy"],
            "macro_precision": mm["macro_precision"],
            "macro_recall": mm["macro_recall"],
            "macro_f1": mm["macro_f1"],
            "mcc": mm["mcc"],
            "auroc": mm["auroc"],
            "precision_normal": mm["precision_normal"],
            "recall_normal": mm["recall_normal"],
            "precision_pneumonia": mm["precision_pneumonia"],
            "recall_pneumonia": mm["recall_pneumonia"],
            "f2": mm["f2"],
        })

    return pd.DataFrame(rows)

In [ ]:

# ============================================================
# 5) DATA PIPELINE — M07
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

def decode_resize(path, label):
    data = tf.io.read_file(path)
    image = tf.io.decode_image(data, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.cast(image, tf.float32)

    # Prior code contract: preserve aspect ratio, pad to square,
    # and DO NOT divide by 255 because ConvNeXt preprocessing is serialized.
    image = tf.image.resize_with_pad(
        image,
        target_height=IMAGE_SIZE,
        target_width=IMAGE_SIZE,
        method="bilinear",
        antialias=True,
    )
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, tf.cast(label, tf.float32)

def balanced_steps_per_epoch(frame, batch_size):
    counts = frame["label"].value_counts().to_dict()
    n0 = batch_size // 2
    n1 = batch_size - n0
    return max(
        1,
        int(math.ceil(counts[0] / max(n0, 1))),
        int(math.ceil(counts[1] / max(n1, 1))),
    )

def build_dataset(frame, batch_size, training, seed):
    options = tf.data.Options()
    options.experimental_deterministic = True

    if training:
        # M07 in the original design is the balanced-BCE baseline.
        normal = frame[frame["label"] == 0]
        pneumonia = frame[frame["label"] == 1]

        n0 = batch_size // 2
        n1 = batch_size - n0
        assert n0 >= 1 and n1 >= 1

        def stream(part, class_batch, class_seed):
            ds = tf.data.Dataset.from_tensor_slices((
                part["filepath"].astype(str).to_numpy(),
                part["label"].astype(np.float32).to_numpy(),
            )).with_options(options)
            ds = ds.shuffle(
                len(part), seed=class_seed, reshuffle_each_iteration=True
            ).repeat()
            ds = ds.map(
                decode_resize,
                num_parallel_calls=AUTOTUNE,
                deterministic=True,
            )
            return ds.batch(class_batch, drop_remainder=True)

        ds0 = stream(normal, n0, seed + 17)
        ds1 = stream(pneumonia, n1, seed + 31)
        ds = tf.data.Dataset.zip((ds0, ds1))

        def merge(a, b):
            x0, y0 = a
            x1, y1 = b
            x = tf.concat([x0, x1], axis=0)
            y = tf.concat([y0, y1], axis=0)
            order = tf.random.shuffle(tf.range(tf.shape(y)[0]), seed=seed)
            return tf.gather(x, order), tf.gather(y, order)

        return ds.map(merge, num_parallel_calls=1, deterministic=True).prefetch(AUTOTUNE)

    ds = tf.data.Dataset.from_tensor_slices((
        frame["filepath"].astype(str).to_numpy(),
        frame["label"].astype(np.float32).to_numpy(),
    )).with_options(options)
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE, deterministic=True)
    return ds.batch(batch_size, drop_remainder=False).prefetch(AUTOTUNE)

print("Data pipeline ready.")


In [ ]:
# ============================================================
# 6) M07 MODEL — EXACT TECHNIQUES FROM PRIOR ZIP
# ============================================================
@tf.keras.utils.register_keras_serializable(package="PneumoniaAI")
class EdgeBlock(tf.keras.layers.Layer):
    """Trainable parallel Sobel/Laplacian branch with gated RGB fusion."""

    def __init__(self, filters=16, max_gate=0.25, **kwargs):
        super().__init__(**kwargs)
        self.filters = int(filters)
        self.max_gate = float(max_gate)
        self.conv1 = tf.keras.layers.Conv2D(
            self.filters, 3, padding="same", activation="gelu", dtype="float32"
        )
        self.conv2 = tf.keras.layers.Conv2D(
            3, 1, padding="same", activation="tanh", dtype="float32"
        )

    def build(self, input_shape):
        self.raw_gate = self.add_weight(
            name="raw_gate",
            shape=(),
            initializer=tf.keras.initializers.Constant(-4.0),
            trainable=True,
            dtype="float32",
        )
        super().build(input_shape)

    @staticmethod
    def _normalize_per_image(tensor):
        minimum = tf.stop_gradient(
            tf.reduce_min(tensor, axis=[1, 2, 3], keepdims=True)
        )
        maximum = tf.stop_gradient(
            tf.reduce_max(tensor, axis=[1, 2, 3], keepdims=True)
        )
        range_value = tf.maximum(
            maximum - minimum,
            tf.constant(1e-3, tf.float32)
        )
        return tf.math.divide_no_nan(
            tensor - minimum,
            range_value
        )

    def call(self, inputs):
        original_dtype = inputs.dtype
        x = tf.cast(inputs, tf.float32)
        gray = tf.image.rgb_to_grayscale(x) / 255.0

        sobel = tf.image.sobel_edges(gray)
        sobel_y = tf.abs(sobel[..., 0])
        sobel_x = tf.abs(sobel[..., 1])

        lap_kernel = tf.constant(
            [[0.0, 1.0, 0.0],
             [1.0, -4.0, 1.0],
             [0.0, 1.0, 0.0]],
            dtype=tf.float32,
        )
        lap_kernel = tf.reshape(lap_kernel, [3, 3, 1, 1])
        lap = tf.abs(
            tf.nn.conv2d(gray, lap_kernel, strides=1, padding="SAME")
        )

        edge_input = tf.concat([
            self._normalize_per_image(sobel_x),
            self._normalize_per_image(sobel_y),
            self._normalize_per_image(lap),
        ], axis=-1)

        edge_delta = tf.cast(
            self.conv2(self.conv1(edge_input)),
            tf.float32
        ) * tf.constant(255.0, dtype=tf.float32)

        gate = tf.nn.sigmoid(
            tf.cast(self.raw_gate, tf.float32)
        )
        gate *= tf.constant(
            self.max_gate,
            dtype=tf.float32
        )

        output = tf.clip_by_value(
            x + gate * edge_delta,
            0.0,
            255.0
        )
        return tf.cast(output, original_dtype)

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "max_gate": self.max_gate,
        })
        return config


@tf.keras.utils.register_keras_serializable(package="PneumoniaAI")
class CBAM(tf.keras.layers.Layer):
    def __init__(self, reduction=16, spatial_kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction = int(reduction)
        self.spatial_kernel = int(spatial_kernel)

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden = max(channels // self.reduction, 8)

        self.fc1 = tf.keras.layers.Dense(
            hidden,
            activation="relu",
            use_bias=True,
            dtype="float32",
        )
        self.fc2 = tf.keras.layers.Dense(
            channels,
            use_bias=True,
            dtype="float32",
        )
        self.spatial = tf.keras.layers.Conv2D(
            1,
            kernel_size=self.spatial_kernel,
            padding="same",
            activation="sigmoid",
            use_bias=False,
            dtype="float32",
        )
        super().build(input_shape)

    def call(self, inputs):
        input_dtype = inputs.dtype
        x = tf.cast(inputs, tf.float32)

        avg_pool = tf.reduce_mean(x, axis=[1, 2])
        max_pool = tf.reduce_max(x, axis=[1, 2])

        channel_attention = tf.nn.sigmoid(
            self.fc2(self.fc1(avg_pool))
            + self.fc2(self.fc1(max_pool))
        )
        channel_attention = channel_attention[:, None, None, :]
        x = x * channel_attention

        spatial_avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        spatial_max = tf.reduce_max(x, axis=-1, keepdims=True)

        spatial_attention = self.spatial(
            tf.concat([spatial_avg, spatial_max], axis=-1)
        )

        return tf.cast(
            x * spatial_attention,
            input_dtype
        )

    def get_config(self):
        config = super().get_config()
        config.update({
            "reduction": self.reduction,
            "spatial_kernel": self.spatial_kernel,
        })
        return config


@tf.keras.utils.register_keras_serializable(package="PneumoniaAI")
class FLSD53BinaryFocalLoss(tf.keras.losses.Loss):
    """Mukhoti et al. FLSD-53: gamma=5 for p_t<0.2, otherwise gamma=3."""

    def __init__(
        self,
        alpha=0.5,
        threshold=0.20,
        hard_gamma=5.0,
        easy_gamma=3.0,
        name="flsd53_binary_focal_loss",
        reduction="sum_over_batch_size",
        **kwargs,
    ):
        super().__init__(name=name, reduction=reduction, **kwargs)
        self.alpha = float(alpha)
        self.threshold = float(threshold)
        self.hard_gamma = float(hard_gamma)
        self.easy_gamma = float(easy_gamma)

    def call(self, y_true, y_pred):
        y_pred = tf.cast(y_pred, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        y_true = tf.reshape(y_true, tf.shape(y_pred))

        y_pred = tf.clip_by_value(
            y_pred,
            1e-6,
            1.0 - 1e-6
        )

        p_t = (
            y_true * y_pred
            + (1.0 - y_true) * (1.0 - y_pred)
        )

        gamma_t = tf.where(
            tf.stop_gradient(p_t) < self.threshold,
            tf.cast(self.hard_gamma, tf.float32),
            tf.cast(self.easy_gamma, tf.float32),
        )

        alpha_t = (
            y_true * self.alpha
            + (1.0 - y_true) * (1.0 - self.alpha)
        )

        focal = tf.pow(
            1.0 - p_t,
            gamma_t
        )

        bce = tf.keras.backend.binary_crossentropy(
            y_true,
            y_pred
        )

        return tf.reduce_mean(
            alpha_t * focal * bce,
            axis=-1
        )

    def get_config(self):
        return {
            **super().get_config(),
            "alpha": self.alpha,
            "threshold": self.threshold,
            "hard_gamma": self.hard_gamma,
            "easy_gamma": self.easy_gamma,
        }


def augmentation_block(seed):
    return tf.keras.Sequential([
        tf.keras.layers.RandomRotation(
            factor=0.025,
            fill_mode="reflect",
            seed=seed + 101,
        ),
        tf.keras.layers.RandomTranslation(
            height_factor=0.03,
            width_factor=0.03,
            fill_mode="reflect",
            seed=seed + 102,
        ),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.05, 0.05),
            width_factor=(-0.05, 0.05),
            fill_mode="reflect",
            seed=seed + 103,
        ),
        tf.keras.layers.RandomContrast(
            factor=0.08,
            seed=seed + 104,
        ),
        tf.keras.layers.ReLU(
            max_value=255.0,
            name="clip_augmented_pixels",
        ),
    ], name="augmentation")


def build_m07(
    dropout,
    seed,
    edge_filters=None,
    edge_max_gate=None,
    cbam_reduction=None,
):
    edge_filters = EDGE_FILTERS if edge_filters is None else int(edge_filters)
    edge_max_gate = EDGE_MAX_GATE if edge_max_gate is None else float(edge_max_gate)
    cbam_reduction = CBAM_REDUCTION if cbam_reduction is None else int(cbam_reduction)

    inputs = tf.keras.Input(
        shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        dtype=tf.float32,
        name="image",
    )

    x = augmentation_block(seed)(inputs)

    x = EdgeBlock(
        filters=edge_filters,
        max_gate=edge_max_gate,
        name="edge_block",
    )(x)

    import inspect
    kwargs = dict(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        pooling=None,
    )

    signature = inspect.signature(
        tf.keras.applications.ConvNeXtTiny
    )
    if "include_preprocessing" in signature.parameters:
        kwargs["include_preprocessing"] = True

    backbone = tf.keras.applications.ConvNeXtTiny(**kwargs)

    x = backbone(x)

    x = CBAM(
        reduction=cbam_reduction,
        spatial_kernel=CBAM_SPATIAL_KERNEL,
        name="cbam",
    )(x)

    x = tf.keras.layers.Activation(
        "linear",
        name="aez_spatial_features",
    )(x)

    x = tf.keras.layers.GlobalAveragePooling2D(
        name="global_pool"
    )(x)

    x = tf.keras.layers.LayerNormalization(
        epsilon=1e-6,
        dtype="float32",
        name="head_norm",
    )(x)

    x = tf.keras.layers.Dropout(
        float(dropout),
        name="head_dropout",
    )(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="probability",
    )(x)

    return (
        tf.keras.Model(inputs, outputs, name=MODEL_ID),
        backbone,
    )


def compile_m07(model, lr, weight_decay):
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=float(lr),
        weight_decay=float(weight_decay),
        global_clipnorm=1.0,
    )

    model.compile(
        optimizer=optimizer,
        loss=FLSD53BinaryFocalLoss(
            alpha=FLSD_ALPHA,
            threshold=FLSD_THRESHOLD,
            hard_gamma=FLSD_HARD_GAMMA,
            easy_gamma=FLSD_EASY_GAMMA,
        ),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(curve="ROC", name="auc_roc"),
            tf.keras.metrics.AUC(curve="PR", name="auc_pr"),
        ],
    )

print("✅ M07 EdgeBlock + CBAM + FLSD-53 architecture ready")
print("✅ Balanced batches + FLSD alpha=0.5 correction active")

In [ ]:

# ============================================================
# 7) CALLBACKS / LR SCHEDULE / BALANCED EARLY STOPPING
# ============================================================
class WarmupCosine(tf.keras.callbacks.Callback):
    def __init__(self, initial_lr, total_epochs, warmup_epochs, min_lr):
        super().__init__()
        self.initial_lr = float(initial_lr)
        self.total_epochs = max(1, int(total_epochs))
        self.warmup_epochs = max(0, min(int(warmup_epochs), self.total_epochs))
        self.min_lr = float(min_lr)

    def value(self, local_epoch):
        e = max(0, min(int(local_epoch), self.total_epochs - 1))
        if self.warmup_epochs > 0 and e < self.warmup_epochs:
            frac = (e + 1) / self.warmup_epochs
            start = max(self.min_lr, self.initial_lr * 0.1)
            return start + frac * (self.initial_lr - start)

        decay_epochs = max(1, self.total_epochs - self.warmup_epochs)
        decay_index = max(0, e - self.warmup_epochs)
        progress = min(1.0, decay_index / max(1, decay_epochs - 1))
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        return self.min_lr + (self.initial_lr - self.min_lr) * cosine

    def on_epoch_begin(self, epoch, logs=None):
        # Keras epoch may be global after head training; use phase-local index.
        local_epoch = epoch - getattr(self, "phase_start_epoch", 0)
        lr = self.value(local_epoch)
        try:
            self.model.optimizer.learning_rate.assign(lr)
        except Exception:
            self.model.optimizer.learning_rate = lr

class BalancedCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, y_val, checkpoint_path, patience):
        super().__init__()
        self.val_ds = val_ds
        self.y_val = np.asarray(y_val, dtype=int)
        self.checkpoint_path = Path(checkpoint_path)
        self.state_path = self.checkpoint_path.with_suffix(
            self.checkpoint_path.suffix + ".state.json"
        )
        self.patience = int(patience)
        self.best_rank = None
        self.best_metrics = None
        self.best_epoch = 0
        self.wait = 0

        if self.checkpoint_path.is_file() and self.state_path.is_file():
            try:
                state = json.loads(self.state_path.read_text(encoding="utf-8"))
                self.best_rank = tuple(state["best_rank"])
                self.best_metrics = state["best_metrics"]
                self.best_epoch = int(state["best_epoch"])
                self.wait = int(state.get("wait", 0))
            except Exception:
                pass

    def _save_state(self):
        payload = {
            "best_rank": list(self.best_rank) if self.best_rank is not None else None,
            "best_metrics": self.best_metrics,
            "best_epoch": int(self.best_epoch),
            "wait": int(self.wait),
        }
        self.state_path.write_text(
            json.dumps(payload, indent=2), encoding="utf-8"
        )

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        p = self.model.predict(self.val_ds, verbose=0).reshape(-1)
        metrics = select_balanced_threshold(self.y_val, p)
        rank = threshold_rank(metrics)

        logs["val_min_dual_precision_recall_selected"] = metrics["min_dual_precision_recall"]
        logs["val_dual_precision_recall_score"] = metrics["dual_precision_recall_score"]
        logs["val_selected_threshold"] = metrics["threshold"]
        logs["val_recall_normal_selected"] = metrics["recall_normal"]
        logs["val_recall_pneumonia_selected"] = metrics["recall_pneumonia"]
        logs["val_precision_normal_selected"] = metrics["precision_normal"]
        logs["val_precision_pneumonia_selected"] = metrics["precision_pneumonia"]

        if self.best_rank is None or rank > self.best_rank:
            self.best_rank = rank
            self.best_metrics = metrics
            self.best_epoch = int(epoch) + 1
            self.wait = 0
            self.checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
            self.model.save_weights(self.checkpoint_path)
            self._save_state()
            print(
                f"[best] epoch={epoch+1} minPR={metrics['min_dual_precision_recall']:.4f} "
                f"P_N={metrics['precision_normal']:.4f} R_N={metrics['recall_normal']:.4f} "
                f"P_P={metrics['precision_pneumonia']:.4f} R_P={metrics['recall_pneumonia']:.4f}"
            )
        else:
            self.wait += 1
            self._save_state()
            if self.patience >= 0 and self.wait > self.patience:
                print(
                    f"[early-stop] best_epoch={self.best_epoch} "
                    f"best_minPR={self.best_metrics['min_dual_precision_recall']:.4f}"
                )
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.checkpoint_path.is_file():
            self.model.load_weights(self.checkpoint_path)

def make_callbacks(
    run_dir,
    val_ds,
    y_val,
    checkpoint_path,
    patience,
    schedule,
    initial_lr,
    total_epochs,
    warmup_epochs,
    min_lr,
    phase_start_epoch,
    allow_early_stop=True,
):
    callbacks = [
        tf.keras.callbacks.TerminateOnNaN(),
    ]

    monitor = BalancedCheckpoint(
        val_ds=val_ds,
        y_val=y_val,
        checkpoint_path=checkpoint_path,
        patience=patience if allow_early_stop else 10**9,
    )
    callbacks.append(monitor)

    if schedule == "cosine":
        cosine = WarmupCosine(
            initial_lr=initial_lr,
            total_epochs=total_epochs,
            warmup_epochs=warmup_epochs,
            min_lr=min_lr,
        )
        cosine.phase_start_epoch = phase_start_epoch
        callbacks.append(cosine)
    elif schedule == "plateau":
        callbacks.append(
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_min_dual_precision_recall_selected",
                mode="max",
                factor=0.5,
                patience=2,
                min_lr=min_lr,
                verbose=1,
            )
        )
    else:
        raise ValueError(schedule)

    return callbacks, monitor

print("Callbacks ready.")


In [ ]:

# ============================================================
# 8) TRAIN ONE RUN
# ============================================================
def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def history_frame(history, stage):
    if history is None:
        return pd.DataFrame()
    df = pd.DataFrame(history.history)
    if not df.empty:
        df.insert(0, "epoch", np.arange(history.epoch[0] + 1, history.epoch[-1] + 2))
        df.insert(1, "stage", stage)
    return df

def m07_run_contract(stage, train_df, val_df, params, seed, head_epochs,
                     finetune_epochs, fold_id=None, extra=None):
    identity = {
        "train_dataframe": dataframe_identity(train_df),
        "validation_dataframe": dataframe_identity(val_df),
        "flsd_alpha": FLSD_ALPHA,
        "flsd_threshold": FLSD_THRESHOLD,
        "flsd_hard_gamma": FLSD_HARD_GAMMA,
        "flsd_easy_gamma": FLSD_EASY_GAMMA,
        "edge_filters": int(params.get("edge_filters", EDGE_FILTERS)),
        "edge_max_gate": float(params.get("edge_max_gate", EDGE_MAX_GATE)),
        "cbam_reduction": int(params.get("cbam_reduction", CBAM_REDUCTION)),
        "cbam_spatial_kernel": CBAM_SPATIAL_KERNEL,
    }
    if extra:
        identity.update(extra)
    return make_run_contract(
        stage, model_id="M07", resolution=IMAGE_SIZE, params=params,
        fold_id=fold_id, seed=seed, head_epochs=head_epochs,
        finetune_epochs=finetune_epochs, extra=identity,
    )


def m07_run_artifacts(run_dir, keep_weights):
    artifacts = {
        "validation_predictions": Path(run_dir) / "validation_predictions.csv",
        "history": Path(run_dir) / "history.csv",
    }
    if keep_weights:
        artifacts["weights"] = Path(run_dir) / "best.weights.h5"
    return artifacts


def validate_m07_history(history, head_epochs, finetune_epochs):
    required = {"epoch", "stage", "loss", "val_loss"}
    if not required.issubset(history.columns) or history.empty:
        raise ValueError("TRAINING_HISTORY_MISSING: no complete training evidence")
    if set(history["stage"]) != {"head", "finetune"}:
        raise ValueError("TRAINING_HISTORY_STAGE_MISSING")
    for column in ("epoch", "loss", "val_loss"):
        values = pd.to_numeric(history[column], errors="raise").to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError(f"NONFINITE_TRAINING_HISTORY: {column}; failed fits are never completed")
    epochs = pd.to_numeric(history["epoch"]).to_numpy(dtype=float)
    if np.any(epochs != np.floor(epochs)) or history.duplicated(["stage", "epoch"]).any():
        raise ValueError("INVALID_TRAINING_EPOCH_HISTORY")
    head = history.loc[history["stage"] == "head", "epoch"].to_numpy(dtype=int)
    finetune = history.loc[history["stage"] == "finetune", "epoch"].to_numpy(dtype=int)
    if not np.array_equal(head, np.arange(1, int(head_epochs) + 1)):
        raise ValueError("HEAD_STAGE_INCOMPLETE: early stopping is disabled for the head stage")
    if (len(finetune) < 1 or len(finetune) > int(finetune_epochs)
            or not np.array_equal(finetune, np.arange(int(head_epochs) + 1,
                                                      int(head_epochs) + len(finetune) + 1))):
        raise ValueError("FINETUNE_STAGE_INCOMPLETE_OR_INVALID")
    if history["stage"].tolist() != ["head"] * len(head) + ["finetune"] * len(finetune):
        raise ValueError("TRAINING_HISTORY_STAGE_ORDER_INVALID")
    return history


def read_verified_m07_run(run_dir, train_df, val_df, params, seed,
                          head_epochs, finetune_epochs, keep_weights):
    result_path = Path(run_dir) / "run_result.json"
    result = json.loads(result_path.read_text(encoding="utf-8"))
    contract = m07_run_contract(
        "M07_TRAIN", train_df, val_df, params, seed,
        head_epochs, finetune_epochs,
    )
    artifacts = m07_run_artifacts(run_dir, keep_weights)
    validate_receipt(result, contract, artifacts=artifacts, allowed_statuses={"COMPLETED"})
    pred = pd.read_csv(artifacts["validation_predictions"], float_precision="round_trip",
                       dtype={c: str for c in ("relative_path", "patient_id", "model_label", "sha256")})
    validate_prediction_frame(pred, val_df)
    validate_m07_history(pd.read_csv(artifacts["history"]), head_epochs, finetune_epochs)
    # Check that reported metrics actually describe the sealed probabilities.
    expected_metrics = calculate_metrics(
        pred["label"].to_numpy(dtype=int),
        pred["probability_pneumonia"].to_numpy(dtype=float), result["metrics"]["threshold"],
    )
    for name, expected in expected_metrics.items():
        if not np.allclose(np.asarray(result["metrics"].get(name)), np.asarray(expected),
                           rtol=1e-8, atol=1e-10, equal_nan=False):
            raise ValueError(f"SAVED_METRIC_MISMATCH: {name}")
    return result, pred


def train_one_run(
    train_df,
    val_df,
    params,
    run_dir,
    seed,
    head_epochs,
    finetune_epochs,
    keep_weights=True,
):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    if (run_dir / "COMPLETED.json").exists() or (run_dir / "run_result.json").exists():
        raise ValueError("EXISTING_RUN_RECEIPT: validate and reuse through the caller; never overwrite")
    run_contract = m07_run_contract(
        "M07_TRAIN", train_df, val_df, params, seed, head_epochs, finetune_epochs,
    )
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)

    batch_size = int(params["batch_size"])
    train_ds = build_dataset(train_df, batch_size, True, seed)
    val_ds = build_dataset(val_df, batch_size, False, seed)
    y_val = val_df["label"].to_numpy(dtype=int)
    steps = balanced_steps_per_epoch(train_df, batch_size)

    checkpoint = run_dir / "best.weights.h5"
    state = checkpoint.with_suffix(checkpoint.suffix + ".state.json")
    # Recovery is deliberately completed-trial/fold only. An interrupted fit starts
    # its two stages again; optimizer/epoch/RNG state is not resumed.
    if checkpoint.exists():
        checkpoint.unlink()
    if state.exists():
        state.unlink()

    model, backbone = build_m07(
        float(params["dropout"]),
        seed,
        edge_filters=int(params.get("edge_filters", EDGE_FILTERS)),
        edge_max_gate=float(params.get("edge_max_gate", EDGE_MAX_GATE)),
        cbam_reduction=int(params.get("cbam_reduction", CBAM_REDUCTION)),
    )

    # ---------- Stage A: head ----------
    backbone.trainable = False
    compile_m07(
        model,
        lr=float(params["head_lr"]),
        weight_decay=float(params["weight_decay"]),
    )

    head_callbacks, head_monitor = make_callbacks(
        run_dir=run_dir,
        val_ds=val_ds,
        y_val=y_val,
        checkpoint_path=checkpoint,
        patience=999,
        schedule=params["lr_schedule"],
        initial_lr=float(params["head_lr"]),
        total_epochs=head_epochs,
        warmup_epochs=int(params["head_warmup_epochs"]),
        min_lr=float(params["min_lr"]),
        phase_start_epoch=0,
        allow_early_stop=False,
    )

    h1 = model.fit(
        train_ds,
        validation_data=val_ds,
        steps_per_epoch=steps,
        epochs=head_epochs,
        verbose=1,
        callbacks=head_callbacks,
    )

    # Preserve the best head checkpoint, but do not carry head-stage patience
    # debt into full fine-tuning.
    if state.is_file():
        state_payload = json.loads(state.read_text(encoding="utf-8"))
        state_payload["wait"] = 0
        atomic_write_json(state, state_payload)

    # ---------- Stage B: full fine-tuning ----------
    backbone.trainable = True
    compile_m07(
        model,
        lr=float(params["finetune_lr"]),
        weight_decay=float(params["weight_decay"]),
    )

    ft_callbacks, ft_monitor = make_callbacks(
        run_dir=run_dir,
        val_ds=val_ds,
        y_val=y_val,
        checkpoint_path=checkpoint,
        patience=int(params["patience"]),
        schedule=params["lr_schedule"],
        initial_lr=float(params["finetune_lr"]),
        total_epochs=finetune_epochs,
        warmup_epochs=int(params["finetune_warmup_epochs"]),
        min_lr=float(params["min_lr"]),
        phase_start_epoch=head_epochs,
        allow_early_stop=True,
    )

    h2 = model.fit(
        train_ds,
        validation_data=val_ds,
        steps_per_epoch=steps,
        initial_epoch=head_epochs,
        epochs=head_epochs + finetune_epochs,
        verbose=1,
        callbacks=ft_callbacks,
    )

    # BalancedCheckpoint restores best saved weights at train end.
    if not checkpoint.is_file():
        raise RuntimeError("TRAINING_CHECKPOINT_MISSING: cannot seal a completed run")
    model.load_weights(checkpoint)

    probability = model.predict(val_ds, verbose=0).reshape(-1)
    metrics = select_balanced_threshold(y_val, probability)

    pred_df = val_df[
        ["relative_path", "patient_id", "model_label", "label", "sha256"]
    ].copy()
    pred_df["probability_pneumonia"] = np.asarray(probability, dtype=np.float64)
    validate_prediction_frame(pred_df, val_df)
    pred_df.to_csv(run_dir / "validation_predictions.csv", index=False, float_format="%.17g")

    hist = pd.concat(
        [history_frame(h1, "head"), history_frame(h2, "finetune")],
        ignore_index=True,
    )
    validate_m07_history(hist, head_epochs, finetune_epochs)
    hist.to_csv(run_dir / "history.csv", index=False)

    result = {
        "status": "COMPLETED",
        "model_id": MODEL_ID,
        "seed": int(seed),
        "image_size": IMAGE_SIZE,
        "head_epochs_budget": int(head_epochs),
        "finetune_epochs_budget": int(finetune_epochs),
        "train_images": int(len(train_df)),
        "val_images": int(len(val_df)),
        "train_patients": int(train_df["patient_id"].nunique()),
        "val_patients": int(val_df["patient_id"].nunique()),
        "params": json_safe(params),
        "m07_fixed": {
            "edge_filters": int(params.get("edge_filters", EDGE_FILTERS)),
            "edge_max_gate": float(params.get("edge_max_gate", EDGE_MAX_GATE)),
            "cbam_reduction": int(params.get("cbam_reduction", CBAM_REDUCTION)),
            "cbam_spatial_kernel": CBAM_SPATIAL_KERNEL,
            "flsd_alpha": FLSD_ALPHA,
            "flsd_threshold": FLSD_THRESHOLD,
            "flsd_hard_gamma": FLSD_HARD_GAMMA,
            "flsd_easy_gamma": FLSD_EASY_GAMMA,
        },
        "metrics": json_safe(metrics),
        "checkpoint": str(checkpoint),
        "locked_test_used": False,
    }
    if not keep_weights and checkpoint.exists():
        checkpoint.unlink()
        if state.exists():
            state.unlink()

    result = seal_receipt(
        result, run_contract, artifacts=m07_run_artifacts(run_dir, keep_weights),
    )
    atomic_write_json(run_dir / "run_result.json", result)

    del model, backbone, train_ds, val_ds
    tf.keras.backend.clear_session()
    gc.collect()

    return result, pred_df

print("Training function ready.")


In [ ]:
# ============================================================
# 8B) M07 RUNTIME PRE-FLIGHT — V1.6 BEFORE HPO
# Exercises actual maximum configured HPO batch, unfrozen training,
# callback/checkpoint operation and deterministic weight round-trip.
# This cell records PASS only after it actually runs on the user's GPU.
# ============================================================
import tempfile

print("\nM07 RUNTIME PRE-FLIGHT V1.6")
_smoke_train = load_fold_manifest(1, "train")
_smoke_val = load_fold_manifest(1, "val")
assert len(_smoke_train) == 4160 and len(_smoke_val) == 1049
_smoke_bs = max(16, int(PRIOR_SUCCESSFUL_RECIPE["batch_size"]))
_smoke_ds = build_dataset(_smoke_train, _smoke_bs, True, SEED + 777)
_smoke_images, _smoke_labels = next(iter(_smoke_ds.take(1)))
assert tuple(_smoke_images.shape) == (_smoke_bs, IMAGE_SIZE, IMAGE_SIZE, 3)
assert set(np.asarray(_smoke_labels).astype(int).reshape(-1)) == {0, 1}

tf.keras.backend.clear_session()
gc.collect()
_smoke_model, _smoke_backbone = build_m07(
    dropout=float(PRIOR_SUCCESSFUL_RECIPE["dropout"]), seed=SEED + 777,
    edge_filters=int(PRIOR_SUCCESSFUL_RECIPE["edge_filters"]),
    edge_max_gate=float(PRIOR_SUCCESSFUL_RECIPE["edge_max_gate"]),
    cbam_reduction=int(PRIOR_SUCCESSFUL_RECIPE["cbam_reduction"]),
)
_smoke_checks = {}

def _check_smoke_values(values, stage):
    for key, value in values.items():
        numeric = np.asarray(value, dtype=float)
        if not np.isfinite(numeric).all():
            raise RuntimeError(f"Non-finite runtime preflight metric: {stage}/{key}")

for _stage, _trainable, _lr in (
    ("head_train_on_batch", False, PRIOR_SUCCESSFUL_RECIPE["head_lr"]),
    ("unfrozen_train_on_batch", True, PRIOR_SUCCESSFUL_RECIPE["finetune_lr"]),
):
    _smoke_backbone.trainable = _trainable
    compile_m07(_smoke_model, lr=float(_lr),
                weight_decay=float(PRIOR_SUCCESSFUL_RECIPE["weight_decay"]))
    _result = _smoke_model.train_on_batch(_smoke_images, _smoke_labels, return_dict=True)
    _check_smoke_values(_result, _stage)
    _smoke_checks[_stage] = "PASS"

# A balanced, bounded diagnostic batch; not scientific validation evidence.
_smoke_eval = tf.data.Dataset.from_tensor_slices((_smoke_images, _smoke_labels)).batch(_smoke_bs)
with tempfile.TemporaryDirectory(prefix="m07_preflight_", dir=OUTPUT_ROOT) as _smoke_dir:
    _smoke_checkpoint = Path(_smoke_dir) / "best.weights.h5"
    _callbacks, _monitor = make_callbacks(
        run_dir=Path(_smoke_dir), val_ds=_smoke_eval,
        y_val=np.asarray(_smoke_labels).astype(int).reshape(-1),
        checkpoint_path=_smoke_checkpoint, patience=1, schedule="cosine",
        initial_lr=float(PRIOR_SUCCESSFUL_RECIPE["finetune_lr"]),
        total_epochs=1, warmup_epochs=0, min_lr=float(PRIOR_SUCCESSFUL_RECIPE["min_lr"]),
        phase_start_epoch=0, allow_early_stop=False,
    )
    _history = _smoke_model.fit(_smoke_eval, validation_data=_smoke_eval,
                               epochs=1, verbose=0, callbacks=_callbacks)
    _check_smoke_values(_history.history, "callback_epoch")
    if not _smoke_checkpoint.is_file() or not _smoke_checkpoint.with_suffix(
            ".h5.state.json").is_file():
        raise RuntimeError("Preflight checkpoint/state round-trip failed")
    _before = np.asarray(_smoke_model(_smoke_images, training=False), dtype=float)
    _smoke_model.load_weights(_smoke_checkpoint)
    _after = np.asarray(_smoke_model(_smoke_images, training=False), dtype=float)
    if (_after.shape != (_smoke_bs, 1) or not np.isfinite(_after).all()
            or not np.allclose(_before, _after, rtol=1e-5, atol=1e-6)):
        raise RuntimeError("Preflight inference/weight reload mismatch")
    _smoke_checks["callback_checkpoint_epoch"] = "PASS"
    _smoke_checks["weights_round_trip"] = "PASS"

_smoke_contract = make_run_contract(
    "runtime_preflight", model_id=MODEL_ID, resolution=IMAGE_SIZE,
    params=PRIOR_SUCCESSFUL_RECIPE, seed=SEED + 777,
    head_epochs=1, finetune_epochs=2, extra={"batch_size": _smoke_bs},
)
_smoke_receipt = seal_receipt({
    "status": "PASS", "checks": _smoke_checks,
    "batch_size": _smoke_bs, "runtime": runtime_environment(),
    "gpu_devices": [str(d) for d in tf.config.list_physical_devices("GPU")],
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "scope": "runtime integrity only; not model performance evidence",
}, _smoke_contract)
atomic_write_json(OUTPUT_ROOT / "RUNTIME_PREFLIGHT_V14.json", _smoke_receipt)

del _smoke_model, _smoke_backbone, _smoke_ds, _smoke_eval, _smoke_images, _smoke_labels
# Callback objects hold model references, so release them before HPO.
del _callbacks, _monitor, _history
tf.keras.backend.clear_session()
gc.collect()
print("✅ M07 RUNTIME PRE-FLIGHT PASS: full batch, unfrozen step, callbacks, weight reload")


In [ ]:
# ============================================================
# 9) FINAL MULTI-FOLD CONFIRMATION HPO
#
# F2 IS REPORT-ONLY AND IS EXPLICITLY EXCLUDED FROM ALL RANKS.
# Locked Test / External are not loaded here.
# ============================================================

HPO_ROOT = OUTPUT_ROOT / "CONFIRMATION_HPO"
HPO_ROOT.mkdir(parents=True, exist_ok=True)

HPO_PERSIST_ROOT = PERSIST_ROOT / "CONFIRMATION_HPO"
HPO_PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

RECIPE_PATH = OUTPUT_ROOT / "FROZEN_M07_CONFIRMED_RECIPE.json"

def _loguniform(rng, low, high):
    return float(10 ** rng.uniform(np.log10(low), np.log10(high)))

def generate_confirmation_candidates():
    rng = np.random.default_rng(HPO_CANDIDATE_SEED)

    candidates = [
        {
            "candidate_id": 0,
            "source": "PRIOR_SUCCESSFUL_WINNER",
            **PRIOR_SUCCESSFUL_RECIPE,
        }
    ]

    seen = {
        json.dumps(
            PRIOR_SUCCESSFUL_RECIPE,
            sort_keys=True,
        )
    }

    while len(candidates) < HPO_CANDIDATES:
        schedule = str(
            rng.choice(["plateau", "cosine"])
        )

        cfg = {
            "batch_size": int(
                rng.choice([8, 12, 16])
            ),
            "head_lr": _loguniform(
                rng, 8e-5, 5e-4
            ),
            "finetune_lr": _loguniform(
                rng, 3e-6, 6e-5
            ),
            "weight_decay": _loguniform(
                rng, 1e-6, 3e-4
            ),
            "dropout": float(
                rng.choice(
                    [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
                )
            ),
            "patience": int(
                rng.choice([3, 4, 5])
            ),
            "lr_schedule": schedule,
            "min_lr": float(
                rng.choice([1e-7, 3e-7, 1e-6])
            ),
            "head_warmup_epochs": int(
                rng.choice([0, 1])
            ) if schedule == "cosine" else 0,
            "finetune_warmup_epochs": int(
                rng.choice([0, 1, 2])
            ) if schedule == "cosine" else 0,
            "edge_filters": int(
                rng.choice([8, 16, 24])
            ),
            "edge_max_gate": float(
                rng.choice([0.10, 0.15, 0.25, 0.35])
            ),
            "cbam_reduction": int(
                rng.choice([8, 16, 32])
            ),
        }

        key = json.dumps(
            cfg,
            sort_keys=True,
        )
        if key in seen:
            continue

        seen.add(key)
        candidates.append({
            "candidate_id": len(candidates),
            "source": "NEW_SEEDED_RANDOM",
            **cfg,
        })

    return candidates

def candidate_params(candidate):
    return {
        k: v
        for k, v in candidate.items()
        if k not in {"candidate_id", "source"}
    }

def hpo_result_path(candidate_id, fold):
    return (
        HPO_PERSIST_ROOT
        / f"candidate_{int(candidate_id):02d}_fold_{int(fold)}.json"
    )

def _hpo_evidence_artifacts(directory):
    directory = Path(directory)
    return {
        "run_result": directory / "run_result.json",
        "validation_predictions": directory / "validation_predictions.csv",
        "history": directory / "history.csv",
    }


def _copy_verified_hpo_evidence(source_dir, destination_dir):
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    for key, source in _hpo_evidence_artifacts(source_dir).items():
        target = _hpo_evidence_artifacts(destination_dir)[key]
        if source.resolve() == target.resolve():
            continue
        if target.exists() and run_file_sha256(target) != run_file_sha256(source):
            raise ValueError(f"HPO_EVIDENCE_CONFLICT: {key}")
        shutil.copy2(source, target)


def run_or_restore_hpo_candidate_fold(candidate, fold):
    candidate_id = int(candidate["candidate_id"])
    fold = int(fold)
    params = candidate_params(candidate)
    # The manifest loader verifies current image bytes, not just CSV metadata.
    train_df = load_fold_manifest(fold, "train")
    val_df = load_fold_manifest(fold, "val")
    training_seed = HPO_TRAIN_SEED_BASE + fold
    contract = m07_run_contract(
        "M07_HPO_CANDIDATE_FOLD", train_df, val_df, params, training_seed,
        HPO_HEAD_EPOCHS, HPO_FINETUNE_EPOCHS, fold_id=fold,
        extra={"candidate_id": candidate_id, "candidate_source": candidate["source"],
               "candidate_generation_seed": HPO_CANDIDATE_SEED},
    )
    persistent_json = hpo_result_path(candidate_id, fold)
    persistent_dir = HPO_PERSIST_ROOT / persistent_json.stem
    local_dir = HPO_ROOT / f"candidate_{candidate_id:02d}" / f"fold_{fold}"
    local_dir.mkdir(parents=True, exist_ok=True)
    local_json = local_dir / "candidate_fold_result.json"

    restored = []
    for source_path, evidence_dir in ((persistent_json, persistent_dir), (local_json, local_dir)):
        if not source_path.exists():
            continue
        # Any existing stale/incomplete marker is an error, never a silent retrain.
        payload = json.loads(source_path.read_text(encoding="utf-8"))
        validate_receipt(payload, contract, artifacts=_hpo_evidence_artifacts(evidence_dir),
                         allowed_statuses={"SUCCESS"})
        if (payload.get("schema") != "m07.confirmation_hpo.candidate_fold.v1.6"
                or payload.get("candidate_id") != candidate_id
                or payload.get("fold") != fold
                or payload.get("training_seed") != training_seed
                or payload.get("params") != params):
            raise ValueError("HPO_RECEIPT_FIELDS_MISMATCH")
        run_result, _ = read_verified_m07_run(
            evidence_dir, train_df, val_df, params, training_seed,
            HPO_HEAD_EPOCHS, HPO_FINETUNE_EPOCHS, keep_weights=False,
        )
        if payload.get("metrics") != run_result["metrics"]:
            raise ValueError("HPO_RECEIPT_METRICS_MISMATCH")
        restored.append((payload, evidence_dir))
    if restored:
        payload, evidence_dir = restored[0]
        if any(other != payload for other, _ in restored[1:]):
            raise ValueError("HPO_LOCAL_REMOTE_RECEIPT_CONFLICT")
        _copy_verified_hpo_evidence(evidence_dir, local_dir)
        _copy_verified_hpo_evidence(evidence_dir, persistent_dir)
        atomic_write_json(local_json, payload)
        atomic_write_json(persistent_json, payload)
        print(f"[HPO C{candidate_id:02d} F{fold}] ✅ identity and artifacts verified — reuse")
        return payload

    if (persistent_dir.exists() and any(persistent_dir.iterdir())
            and not (local_dir / "run_result.json").exists()):
        raise ValueError("HPO_ORPHANED_PERSISTED_EVIDENCE: no complete receipt; resolve before retraining")
    print("\n" + "=" * 110)
    print(f"CONFIRMATION HPO | Candidate {candidate_id} | Fold {fold}")
    print(json.dumps(params, indent=2))
    print("=" * 110)
    started = time.time()
    if (local_dir / "run_result.json").exists():
        result, _ = read_verified_m07_run(
            local_dir, train_df, val_df, params, training_seed,
            HPO_HEAD_EPOCHS, HPO_FINETUNE_EPOCHS, keep_weights=False,
        )
        elapsed_minutes = None
        print("Verified finished HPO fit; completing outer receipt.")
    else:
        result, _ = train_one_run(
            train_df=train_df, val_df=val_df, params=params, run_dir=local_dir,
            seed=training_seed, head_epochs=HPO_HEAD_EPOCHS,
            finetune_epochs=HPO_FINETUNE_EPOCHS, keep_weights=False,
        )
        elapsed_minutes = float((time.time() - started) / 60.0)
    payload = seal_receipt({
        "schema": "m07.confirmation_hpo.candidate_fold.v1.6",
        "status": "SUCCESS", "candidate_id": candidate_id,
        "candidate_source": candidate["source"], "fold": fold,
        "training_seed": int(training_seed),
        "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT, "params": params,
        "metrics": result["metrics"],
        "elapsed_minutes": elapsed_minutes,
        "f2_policy": "REPORT_ONLY_NOT_USED_FOR_SELECTION",
        "locked_test_used": False, "external_used": False,
    }, contract, artifacts=_hpo_evidence_artifacts(local_dir))
    atomic_write_json(local_json, payload)
    _copy_verified_hpo_evidence(local_dir, persistent_dir)
    atomic_write_json(persistent_json, payload)
    _persist_upload(
        note=f"M07 confirmation HPO C{candidate_id:02d} Fold {fold} complete",
        strict=PERSIST_STRICT,
    )
    return payload

def aggregate_candidate_results(
    candidate,
    results,
    phase,
):
    ms = [
        r["metrics"]
        for r in results
    ]

    min_dual = np.asarray(
        [
            float(m["min_dual_precision_recall"])
            for m in ms
        ],
        dtype=float,
    )
    min_recall = np.asarray(
        [
            float(m["min_class_recall"])
            for m in ms
        ],
        dtype=float,
    )
    min_precision = np.asarray(
        [
            float(m["min_class_precision"])
            for m in ms
        ],
        dtype=float,
    )
    dual_score = np.asarray(
        [
            float(m["dual_precision_recall_score"])
            for m in ms
        ],
        dtype=float,
    )

    def avg(name):
        return float(
            np.mean(
                [float(m[name]) for m in ms]
            )
        )

    # F2 is recorded for output only.
    f2_values = np.asarray(
        [
            float(m["f2"])
            for m in ms
        ],
        dtype=float,
    )

    return {
        "candidate_id": int(
            candidate["candidate_id"]
        ),
        "candidate_source": candidate["source"],
        "phase": phase,
        "n_folds": int(len(results)),
        "folds": [
            int(r["fold"])
            for r in results
        ],
        "mean_min_dual_precision_recall": float(
            np.mean(min_dual)
        ),
        "worst_fold_min_dual_precision_recall": float(
            np.min(min_dual)
        ),
        "std_min_dual_precision_recall": float(
            np.std(min_dual, ddof=0)
        ),
        "mean_min_class_recall": float(
            np.mean(min_recall)
        ),
        "mean_min_class_precision": float(
            np.mean(min_precision)
        ),
        "mean_dual_precision_recall_score": float(
            np.mean(dual_score)
        ),
        "mean_mcc": avg("mcc"),
        "mean_balanced_accuracy": avg(
            "balanced_accuracy"
        ),
        "mean_auroc": avg("auroc"),
        "mean_macro_f1": avg("macro_f1"),
        "mean_f2_report_only": float(
            np.mean(f2_values)
        ),
        "worst_f2_report_only": float(
            np.min(f2_values)
        ),
        "params": candidate_params(
            candidate
        ),
    }

def robust_rank_tuple(row):
    # IMPORTANT:
    # F2 is intentionally absent.
    return (
        float(
            row["mean_min_dual_precision_recall"]
        ),
        float(
            row[
                "worst_fold_min_dual_precision_recall"
            ]
        ),
        float(
            row["mean_min_class_recall"]
        ),
        float(
            row["mean_min_class_precision"]
        ),
        float(
            row[
                "mean_dual_precision_recall_score"
            ]
        ),
        float(row["mean_mcc"]),
        float(
            row["mean_balanced_accuracy"]
        ),
        float(row["mean_auroc"]),
        -float(
            row[
                "std_min_dual_precision_recall"
            ]
        ),
        -int(row["candidate_id"]),
    )

# ------------------------------------------------------------
# Candidate generation — deterministic.
# ------------------------------------------------------------
candidates = generate_confirmation_candidates()

candidate_table = pd.DataFrame(candidates)
candidate_table.to_csv(
    HPO_ROOT / "M07_HPO_CANDIDATES.csv",
    index=False,
)

(HPO_ROOT / "M07_HPO_CANDIDATES.json").write_text(
    json.dumps(
        json_safe(candidates),
        indent=2,
    ),
    encoding="utf-8",
)

print("=" * 110)
print("M07 FINAL CONFIRMATION HPO")
print("=" * 110)
print("Candidate generation seed:", HPO_CANDIDATE_SEED)
print("Candidates:", len(candidates))
print("Screening folds:", HPO_SCREEN_FOLDS)
print("Top-K confirmation:", HPO_TOP_K)
print("Confirmation fold:", HPO_CONFIRM_FOLD)
print("F2 selection role: NONE / REPORT-ONLY")
print("Locked Test loaded: FALSE")
print("External loaded: FALSE")

# ------------------------------------------------------------
# Stage A — all candidates on Fold 1 + Fold 2.
# ------------------------------------------------------------
all_candidate_fold_payloads = {}
screening_rows = []

for candidate in candidates:
    cid = int(candidate["candidate_id"])
    payloads = []

    for fold in HPO_SCREEN_FOLDS:
        p = run_or_restore_hpo_candidate_fold(
            candidate,
            fold,
        )
        payloads.append(p)
        all_candidate_fold_payloads[
            (cid, int(fold))
        ] = p

    screening_rows.append(
        aggregate_candidate_results(
            candidate,
            payloads,
            phase="SCREENING_FOLDS_1_2",
        )
    )

screening_ranked = sorted(
    screening_rows,
    key=robust_rank_tuple,
    reverse=True,
)

for rank, row in enumerate(
    screening_ranked,
    start=1,
):
    row["screening_rank"] = int(rank)

screening_df = pd.DataFrame(
    screening_ranked
)
screening_df.to_csv(
    HPO_ROOT / "M07_HPO_SCREENING_RANKING.csv",
    index=False,
)

top_ids = [
    int(r["candidate_id"])
    for r in screening_ranked[
        :HPO_TOP_K
    ]
]

print("\nTop screening candidates:", top_ids)

# ------------------------------------------------------------
# Stage B — top 3 on Fold 3.
# ------------------------------------------------------------
confirmation_rows = []

for candidate in candidates:
    cid = int(candidate["candidate_id"])

    if cid not in top_ids:
        continue

    payload_fold3 = run_or_restore_hpo_candidate_fold(
        candidate,
        HPO_CONFIRM_FOLD,
    )
    all_candidate_fold_payloads[
        (cid, int(HPO_CONFIRM_FOLD))
    ] = payload_fold3

    payloads = [
        all_candidate_fold_payloads[
            (cid, int(fold))
        ]
        for fold in (
            *HPO_SCREEN_FOLDS,
            HPO_CONFIRM_FOLD,
        )
    ]

    confirmation_rows.append(
        aggregate_candidate_results(
            candidate,
            payloads,
            phase="CONFIRMATION_FOLDS_1_2_3",
        )
    )

confirmation_ranked = sorted(
    confirmation_rows,
    key=robust_rank_tuple,
    reverse=True,
)

for rank, row in enumerate(
    confirmation_ranked,
    start=1,
):
    row["confirmation_rank"] = int(rank)

confirmation_df = pd.DataFrame(
    confirmation_ranked
)
confirmation_df.to_csv(
    HPO_ROOT / "M07_HPO_CONFIRMATION_RANKING.csv",
    index=False,
)

winner_row = confirmation_ranked[0]
winner_candidate_id = int(
    winner_row["candidate_id"]
)
winner_candidate = next(
    c
    for c in candidates
    if int(c["candidate_id"])
    == winner_candidate_id
)

shared_params = candidate_params(
    winner_candidate
)

# Adopt architecture HPO winner for final 5-fold.
EDGE_FILTERS = int(
    shared_params["edge_filters"]
)
EDGE_MAX_GATE = float(
    shared_params["edge_max_gate"]
)
CBAM_REDUCTION = int(
    shared_params["cbam_reduction"]
)

# ------------------------------------------------------------
# Full long table: every completed candidate×fold.
# ------------------------------------------------------------
long_rows = []

for (cid, fold), payload in sorted(
    all_candidate_fold_payloads.items()
):
    m = payload["metrics"]

    long_rows.append({
        "candidate_id": cid,
        "fold": fold,
        "candidate_source": payload[
            "candidate_source"
        ],
        **{
            f"param_{k}": v
            for k, v in payload[
                "params"
            ].items()
        },
        "min_dual_precision_recall": m[
            "min_dual_precision_recall"
        ],
        "dual_precision_recall_score": m[
            "dual_precision_recall_score"
        ],
        "min_class_recall": m[
            "min_class_recall"
        ],
        "min_class_precision": m[
            "min_class_precision"
        ],
        "precision_normal": m[
            "precision_normal"
        ],
        "recall_normal": m[
            "recall_normal"
        ],
        "precision_pneumonia": m[
            "precision_pneumonia"
        ],
        "recall_pneumonia": m[
            "recall_pneumonia"
        ],
        "balanced_accuracy": m[
            "balanced_accuracy"
        ],
        "macro_f1": m["macro_f1"],
        "mcc": m["mcc"],
        "auroc": m["auroc"],
        "selected_threshold": m[
            "threshold"
        ],
        # REPORT ONLY
        "f2_report_only": m["f2"],
    })

hpo_long_df = pd.DataFrame(
    long_rows
)
hpo_long_df.to_csv(
    HPO_ROOT / "M07_HPO_CANDIDATE_FOLD_RESULTS.csv",
    index=False,
)

recipe_core = {
    "schema": "m07.final.confirmed.recipe.v1.6",
    "experiment_contract": make_run_contract(
        "M07_CONFIRMED_HPO_RECIPE", model_id="M07", resolution=IMAGE_SIZE,
        params=shared_params, seed=HPO_CANDIDATE_SEED,
        head_epochs=HPO_HEAD_EPOCHS, finetune_epochs=HPO_FINETUNE_EPOCHS,
        extra={"candidate_fold_contracts": {
            f"C{cid:02d}_F{fold}": payload["run_fingerprint"]
            for (cid, fold), payload in sorted(all_candidate_fold_payloads.items())
        }},
    ),
    "model_id": "M07",
    "image_size": IMAGE_SIZE,
    "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
    "candidate_generation_seed": HPO_CANDIDATE_SEED,
    "hpo_training_seed_base": HPO_TRAIN_SEED_BASE,
    "candidate_count": HPO_CANDIDATES,
    "screening_folds": list(
        HPO_SCREEN_FOLDS
    ),
    "confirmation_fold": HPO_CONFIRM_FOLD,
    "top_k": HPO_TOP_K,
    "hpo_head_epochs": HPO_HEAD_EPOCHS,
    "hpo_finetune_epochs": HPO_FINETUNE_EPOCHS,
    "prior_successful_recipe_fingerprint": (
        PRIOR_SUCCESSFUL_RECIPE_FINGERPRINT
    ),
    "winner_candidate_id": winner_candidate_id,
    "winner_candidate_source": winner_candidate[
        "source"
    ],
    "shared_training_params": shared_params,
    "winner_robust_metrics": winner_row,
    "ranking_contract": [
        "mean_min_dual_precision_recall",
        "worst_fold_min_dual_precision_recall",
        "mean_min_class_recall",
        "mean_min_class_precision",
        "mean_dual_precision_recall_score",
        "mean_mcc",
        "mean_balanced_accuracy",
        "mean_auroc",
        "lower_std_min_dual_precision_recall",
    ],
    "f2_policy": (
        "REPORT_ONLY; excluded from threshold, checkpoint, "
        "early-stopping, HPO ranking and winner selection"
    ),
    "fixed_flsd53": {
        "alpha": FLSD_ALPHA,
        "threshold": FLSD_THRESHOLD,
        "hard_gamma": FLSD_HARD_GAMMA,
        "easy_gamma": FLSD_EASY_GAMMA,
    },
    "cbam_spatial_kernel": CBAM_SPATIAL_KERNEL,
    "locked_test_used": False,
    "external_used": False,
}

recipe_blob = json.dumps(
    json_safe(recipe_core),
    sort_keys=True,
    ensure_ascii=False,
).encode("utf-8")

recipe_core[
    "recipe_fingerprint_sha256"
] = hashlib.sha256(
    recipe_blob
).hexdigest()

recipe = recipe_core

if RECIPE_PATH.exists():
    previous_recipe = json.loads(RECIPE_PATH.read_text(encoding="utf-8"))
    if previous_recipe != json_safe(recipe):
        raise ValueError("FROZEN_RECIPE_CONFLICT: select a separate campaign for changed experiments")
atomic_write_json(RECIPE_PATH, recipe)

# Persist all lightweight HPO evidence.
for p in (
    HPO_ROOT / "M07_HPO_CANDIDATES.csv",
    HPO_ROOT / "M07_HPO_CANDIDATES.json",
    HPO_ROOT / "M07_HPO_SCREENING_RANKING.csv",
    HPO_ROOT / "M07_HPO_CONFIRMATION_RANKING.csv",
    HPO_ROOT / "M07_HPO_CANDIDATE_FOLD_RESULTS.csv",
    RECIPE_PATH,
):
    if p.is_file():
        shutil.copy2(
            p,
            PERSIST_ROOT / p.name,
        )

_persist_upload(
    note=(
        "M07 multi-fold Confirmation HPO complete; "
        "confirmed recipe frozen before final 5-fold/Test"
    ),
    strict=PERSIST_STRICT,
)

print("\n" + "=" * 110)
print("✅ FINAL M07 CONFIRMATION HPO COMPLETE")
print("=" * 110)
print("Winner candidate:", winner_candidate_id)
print("Winner source:", winner_candidate["source"])
print(
    "Recipe fingerprint:",
    recipe["recipe_fingerprint_sha256"],
)
print(json.dumps(shared_params, indent=2))
print("\nWinner robust metrics:")
print(
    json.dumps(
        json_safe(winner_row),
        indent=2,
    )
)
print("\n✅ F2 used for selection: FALSE")
print("✅ Locked Test used: FALSE")
print("✅ External used: FALSE")

In [ ]:
# ============================================================
# 9B) CONFIRMATION HPO FIGURES
# ============================================================
import matplotlib.pyplot as plt

HPO_FIG_ROOT = HPO_ROOT / "FIGURES"
HPO_FIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# Robust screening / confirmation comparison.
fig, ax = plt.subplots(figsize=(10, 6))

for row in screening_ranked:
    cid = int(row["candidate_id"])
    ax.scatter(
        row["mean_min_dual_precision_recall"],
        row[
            "worst_fold_min_dual_precision_recall"
        ],
        s=90,
        label=f"C{cid}",
    )
    ax.annotate(
        f"C{cid}",
        (
            row["mean_min_dual_precision_recall"],
            row[
                "worst_fold_min_dual_precision_recall"
            ],
        ),
        xytext=(5, 4),
        textcoords="offset points",
    )

ax.set_xlabel(
    "Mean min(P_N, R_N, P_P, R_P) — Fold 1+2"
)
ax.set_ylabel(
    "Worst-fold min(P_N, R_N, P_P, R_P)"
)
ax.set_title(
    "M07 Confirmation HPO — Precision/Recall Robustness Screening"
)
fig.tight_layout()
fig.savefig(
    HPO_FIG_ROOT
    / "M07_HPO_SCREENING_PRECISION_RECALL_ROBUSTNESS.png",
    dpi=180,
)
plt.close(fig)

# Final top-3 confirmation metrics.
if not confirmation_df.empty:
    display_cols = [
        "candidate_id",
        "mean_min_dual_precision_recall",
        "worst_fold_min_dual_precision_recall",
        "mean_min_class_recall",
        "mean_min_class_precision",
        "mean_dual_precision_recall_score",
        "mean_mcc",
        "mean_balanced_accuracy",
        "mean_auroc",
        "mean_f2_report_only",
    ]

    display(
        confirmation_df[
            display_cols
        ].round(4)
    )

    metric_names = [
        "mean_min_dual_precision_recall",
        "worst_fold_min_dual_precision_recall",
        "mean_min_class_recall",
        "mean_min_class_precision",
        "mean_mcc",
        "mean_balanced_accuracy",
        "mean_auroc",
    ]

    x = np.arange(
        len(confirmation_df)
    )
    width = 0.10

    fig, ax = plt.subplots(
        figsize=(13, 6)
    )

    for i, metric in enumerate(
        metric_names
    ):
        ax.bar(
            x
            + (
                i
                - (len(metric_names)-1)/2
            )
            * width,
            confirmation_df[
                metric
            ].to_numpy(dtype=float),
            width=width,
            label=metric,
        )

    ax.set_xticks(
        x,
        [
            f"C{int(v)}"
            for v in confirmation_df[
                "candidate_id"
            ]
        ],
    )
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Metric")
    ax.set_title(
        "M07 Top-3 Confirmation — Folds 1+2+3"
    )
    ax.legend(
        fontsize=8,
        ncol=2,
    )
    fig.tight_layout()
    fig.savefig(
        HPO_FIG_ROOT
        / "M07_HPO_TOP3_CONFIRMATION_METRICS.png",
        dpi=180,
    )
    plt.close(fig)

print("✅ Confirmation HPO figures:", HPO_FIG_ROOT)

In [ ]:
# ============================================================
# 10) FINAL M07 — FIVE FROZEN PATIENT-LOCKED FOLDS
# ============================================================
FINAL_ROOT = OUTPUT_ROOT / "FINAL_5FOLD"
FINAL_ROOT.mkdir(parents=True, exist_ok=True)


def _m07_final_fold_contract(fold, train_df, val_df):
    return m07_run_contract(
        "M07_FINAL_FOLD", train_df, val_df, shared_params, SEED + int(fold),
        FINAL_HEAD_EPOCHS, FINAL_FINETUNE_EPOCHS, fold_id=int(fold),
        extra={"hpo_recipe_fingerprint": recipe["recipe_fingerprint_sha256"]},
    )


def _m07_final_fold_artifacts(fold_dir):
    artifacts = m07_run_artifacts(fold_dir, keep_weights=True)
    artifacts["run_result"] = Path(fold_dir) / "run_result.json"
    return artifacts


def _read_completed_m07_fold(fold, fold_dir, train_df, val_df):
    completed_path = Path(fold_dir) / "COMPLETED.json"
    payload = json.loads(completed_path.read_text(encoding="utf-8"))
    validate_receipt(
        payload, _m07_final_fold_contract(fold, train_df, val_df),
        artifacts=_m07_final_fold_artifacts(fold_dir), allowed_statuses={"COMPLETED"},
    )
    if (payload.get("fold_id") != int(fold)
            or payload.get("params") != shared_params
            or payload.get("split_fingerprint") != EXPECTED_SPLIT_FINGERPRINT
            or payload.get("hpo_recipe_fingerprint") != recipe["recipe_fingerprint_sha256"]):
        raise ValueError("FINAL_FOLD_RECEIPT_FIELDS_MISMATCH")
    result, pred = read_verified_m07_run(
        fold_dir, train_df, val_df, shared_params, SEED + int(fold),
        FINAL_HEAD_EPOCHS, FINAL_FINETUNE_EPOCHS, keep_weights=True,
    )
    if payload.get("metrics") != result["metrics"] or payload.get("m07_fixed") != result["m07_fixed"]:
        raise ValueError("FINAL_FOLD_RESULT_MISMATCH")
    return payload, pred


fold_results = []
oof_parts = []

for fold in range(1, 6):
    fold_dir = FINAL_ROOT / f"fold_{fold}"
    completed_path = fold_dir / "COMPLETED.json"
    # Load and verify manifests even on recovery: a marker never proves data identity.
    train_df = load_fold_manifest(fold, "train")
    val_df = load_fold_manifest(fold, "val")
    if completed_path.exists():
        payload, pred = _read_completed_m07_fold(fold, fold_dir, train_df, val_df)
        print(f"\n[FOLD {fold}] exact identity, weights, history and predictions verified — reuse.")
    else:
        print("\n" + "#" * 100)
        print(f"FINAL M07 — FOLD {fold}/5")
        print("#" * 100)
        if (fold_dir / "run_result.json").exists():
            # A fully sealed fit may precede a crash before the outer marker write.
            # It is safe to finish that write only after full independent validation.
            result, pred = read_verified_m07_run(
                fold_dir, train_df, val_df, shared_params, SEED + fold,
                FINAL_HEAD_EPOCHS, FINAL_FINETUNE_EPOCHS, keep_weights=True,
            )
            print(f"[FOLD {fold}] verified finished fit; completing outer receipt.")
        else:
            result, pred = train_one_run(
                train_df=train_df, val_df=val_df, params=shared_params,
                run_dir=fold_dir, seed=SEED + fold,
                head_epochs=FINAL_HEAD_EPOCHS, finetune_epochs=FINAL_FINETUNE_EPOCHS,
                keep_weights=True,
            )
        payload = seal_receipt({
            "schema": "m07.final.fold.v1.6", "fold_id": fold, "status": "COMPLETED",
            "metrics": result["metrics"], "params": shared_params,
            "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
            "hpo_recipe_fingerprint": recipe["recipe_fingerprint_sha256"],
            "m07_fixed": result["m07_fixed"], "locked_test_used": False,
        }, _m07_final_fold_contract(fold, train_df, val_df),
            artifacts=_m07_final_fold_artifacts(fold_dir))
        atomic_write_json(completed_path, payload)
        # Verify freshly written evidence via the exact same path used on recovery.
        payload, pred = _read_completed_m07_fold(fold, fold_dir, train_df, val_df)

    fold_results.append(payload)
    pred["fold_id"] = fold
    oof_parts.append(pred)
    fm = payload["metrics"]
    fold_pred = (pred["probability_pneumonia"].to_numpy() >= fm["threshold"]).astype(int)
    print(
        f"[FOLD {fold}] Acc={fm['accuracy']:.4f} "
        f"BalAcc={fm['balanced_accuracy']:.4f} AUROC={fm['auroc']:.4f} "
        f"R_N={fm['recall_normal']:.4f} R_P={fm['recall_pneumonia']:.4f} "
        f"predN={(fold_pred==0).sum()} predP={(fold_pred==1).sum()}"
    )
    # A prior crash may have happened after the completion marker but before upload.
    # Re-snapshot verified completed folds too, so recovery cannot skip persistence.
    _snapshot_fold_to_persist(fold=fold, recipe_path=RECIPE_PATH)
    print("✅ Durable Fold snapshot:", PERSIST_ROOT / f"M07_FOLD_{fold}_RECOVERY.zip")

print("\n✅ All five M07 folds completed and verified.")


In [ ]:
# ============================================================
# 11) M07 OOF — FOLD-SPECIFIC THRESHOLDS + GLOBAL SECONDARY
# ============================================================
oof = pd.concat(oof_parts, ignore_index=True)

expected_paths = set(development_manifest["relative_path"].astype(str))
actual_paths = set(oof["relative_path"].astype(str))

assert len(oof) == len(development_manifest)
assert expected_paths == actual_paths
assert oof["relative_path"].is_unique

fold_thresholds = {
    int(payload["fold_id"]): float(payload["metrics"]["threshold"])
    for payload in fold_results
}

oof["fold_threshold"] = oof["fold_id"].map(fold_thresholds)
oof["prediction_fold_threshold"] = (
    oof["probability_pneumonia"].to_numpy()
    >= oof["fold_threshold"].to_numpy()
).astype(int)

# Primary OOF classification: each image uses only its fold-validation threshold.
oof_primary_metrics = calculate_binary_metrics(
    oof["label"].to_numpy(dtype=int),
    oof["prediction_fold_threshold"].to_numpy(dtype=int),
    score=oof["probability_pneumonia"].to_numpy(dtype=float),
)

# Secondary global threshold on all OOF predictions.
oof_global_metrics = select_balanced_threshold(
    oof["label"].to_numpy(dtype=int),
    oof["probability_pneumonia"].to_numpy(dtype=float),
)
global_threshold = float(oof_global_metrics["threshold"])

oof["prediction_global_threshold"] = (
    oof["probability_pneumonia"].to_numpy()
    >= global_threshold
).astype(int)

oof.to_csv(
    OUTPUT_ROOT / "M07_OOF_PREDICTIONS.csv",
    index=False,
)

(OUTPUT_ROOT / "M07_OOF_PRIMARY_METRICS.json").write_text(
    json.dumps(json_safe(oof_primary_metrics), indent=2),
    encoding="utf-8",
)

(OUTPUT_ROOT / "M07_OOF_GLOBAL_THRESHOLD_METRICS.json").write_text(
    json.dumps(json_safe(oof_global_metrics), indent=2),
    encoding="utf-8",
)

fold_rows = []
for payload in fold_results:
    m = payload["metrics"]
    fold_rows.append({
        "fold_id": payload["fold_id"],
        "threshold": m["threshold"],
        "accuracy": m["accuracy"],
        "balanced_accuracy": m["balanced_accuracy"],
        "precision_normal": m["precision_normal"],
        "recall_normal": m["recall_normal"],
        "precision_pneumonia": m["precision_pneumonia"],
        "recall_pneumonia": m["recall_pneumonia"],
        "macro_f1": m["macro_f1"],
        "f2": m["f2"],
        "mcc": m["mcc"],
        "auroc": m["auroc"],
        "min_dual_precision_recall": m["min_dual_precision_recall"],
    })

fold_table = pd.DataFrame(fold_rows).sort_values("fold_id")
fold_table.to_csv(
    OUTPUT_ROOT / "M07_FOLD_METRICS.csv",
    index=False,
)

print("=" * 100)
print("M07 OOF — DEVELOPMENT ONLY")
print("=" * 100)

print("Primary (fold-specific thresholds):")
for k in [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "f2",
    "mcc",
    "auroc",
    "precision_normal",
    "recall_normal",
    "precision_pneumonia",
    "recall_pneumonia",
]:
    print(f"{k:30s}: {oof_primary_metrics[k]:.6f}")

print("\nSecondary global OOF threshold:", global_threshold)
print("\nFold thresholds:")
for fold, t in sorted(fold_thresholds.items()):
    print(f"Fold {fold}: {t:.8f}")

display(fold_table)

# Persist Development-only evidence BEFORE opening the Locked Test.
_persist_lightweight_files(
    [
        OUTPUT_ROOT / "M07_OOF_PREDICTIONS.csv",
        OUTPUT_ROOT / "M07_OOF_PRIMARY_METRICS.json",
        OUTPUT_ROOT / "M07_OOF_GLOBAL_THRESHOLD_METRICS.json",
        OUTPUT_ROOT / "M07_FOLD_METRICS.csv",
        RECIPE_PATH,
    ],
    note="M07 five folds + OOF complete; Locked Test not yet opened",
)

print("✅ Five-fold + OOF state persisted before Locked Test")


# Fixed 20/80 report on raw OOF probabilities.
oof_dual_20_80, oof_dual_decision = analyze_dual_threshold(
    oof["label"].to_numpy(dtype=int),
    oof["probability_pneumonia"].to_numpy(dtype=float),
)

oof["dual20_80_decision"] = oof_dual_decision
oof.to_csv(
    OUTPUT_ROOT / "M07_OOF_PREDICTIONS.csv",
    index=False,
)

(OUTPUT_ROOT / "M07_OOF_DUAL20_80_REPORT.json").write_text(
    json.dumps(json_safe(oof_dual_20_80), indent=2),
    encoding="utf-8",
)

print("\nOOF fixed dual 20/80:")
print(" coverage:", f"{oof_dual_20_80['coverage']:.4f}")
print(" uncertain:", f"{oof_dual_20_80['uncertainty_rate']:.4f}")
print(" F2 primary OOF:", f"{oof_primary_metrics['f2']:.4f}")

_persist_lightweight_files(
    [
        OUTPUT_ROOT / "M07_OOF_DUAL20_80_REPORT.json",
        OUTPUT_ROOT / "M07_OOF_PREDICTIONS.csv",
    ],
    note="M07 OOF fixed dual-threshold 20/80 report persisted",
)

In [ ]:
# ============================================================
# 12) ONE-TIME LOCKED TEST — 614 IMAGES / 279 PATIENTS
#
# Primary aggregation was pre-registered BEFORE inference:
# threshold-normalized five-fold ensemble.
# Secondary: five-fold majority vote.
# NO TEST THRESHOLD TUNING.
# ============================================================
TEST_OUT = OUTPUT_ROOT / "LOCKED_TEST"
TEST_OUT.mkdir(parents=True, exist_ok=True)

marker = TEST_OUT / "LOCKED_TEST_EVALUATED_ONCE.json"
cached_predictions = TEST_OUT / "M07_LOCKED_TEST_PER_FOLD_PREDICTIONS.csv"

locked_test = load_locked_test_manifest()

# Re-assert final independence.
assert set(locked_test["patient_id"]).isdisjoint(
    set(development_manifest["patient_id"])
)
assert set(locked_test["sha256"]).isdisjoint(
    set(development_manifest["sha256"])
)

print("LOCKED TEST images:", len(locked_test))
print("LOCKED TEST patients:", locked_test["patient_id"].nunique())
print(locked_test["model_label"].value_counts())
print("✅ Development↔Test Patient overlap = 0")
print("✅ Development↔Test SHA overlap = 0")

def build_plain_eval_dataset(frame, batch_size):
    ds = tf.data.Dataset.from_tensor_slices((
        frame["filepath"].astype(str).to_numpy(),
        frame["label"].astype(np.float32).to_numpy(),
    ))
    ds = ds.map(
        decode_resize,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=True,
    )
    return ds.batch(
        int(batch_size),
        drop_remainder=False,
    ).prefetch(tf.data.AUTOTUNE)

def logit_np(x):
    x = np.clip(np.asarray(x, dtype=float), 1e-7, 1.0 - 1e-7)
    return np.log(x / (1.0 - x))

def locked_test_run_contract(manifest):
    checkpoint_hashes = {
        str(fold): run_file_sha256(FINAL_ROOT / f"fold_{fold}" / "best.weights.h5")
        for fold in range(1, 6)
    }
    thresholds = {str(fold): float(fold_thresholds[fold]) for fold in range(1, 6)}
    if any(not np.isfinite(t) or not 0 < t <= 1 for t in thresholds.values()):
        raise ValueError("Locked Test requires five finite validation thresholds in (0,1].")
    return make_run_contract(
        "locked_test", model_id=MODEL_ID, resolution=IMAGE_SIZE,
        params=shared_params,
        extra={
            "manifest_identity": dataframe_identity(manifest),
            "checkpoint_sha256": checkpoint_hashes,
            "recipe_fingerprint": recipe["recipe_fingerprint_sha256"],
            "fold_thresholds": thresholds,
            "primary_aggregation": "threshold_normalized_five_fold_ensemble",
            "secondary_aggregation": "majority_vote_five_fold",
            "test_threshold_tuning": False,
        },
    )


def validate_locked_test_predictions(pred_long, manifest, thresholds):
    required = {"fold_id", "row_id", "fold_threshold"}
    if not required.issubset(pred_long.columns):
        raise ValueError("Locked Test cache lacks fold and row identities or thresholds.")
    if len(pred_long) != 5 * len(manifest):
        raise ValueError("Locked Test cache does not contain five complete folds.")
    if not pred_long["fold_id"].isin(range(1, 6)).all():
        raise ValueError("Locked Test cache contains invalid fold IDs.")
    probabilities = []
    for fold in range(1, 6):
        group = pred_long.loc[pred_long["fold_id"] == fold].sort_values("row_id")
        if not np.array_equal(group["row_id"].to_numpy(), np.arange(len(manifest))):
            raise ValueError(f"Locked Test fold {fold} has missing or repeated row IDs.")
        validate_prediction_frame(group.reset_index(drop=True), manifest)
        saved_thresholds = pd.to_numeric(group["fold_threshold"], errors="raise").to_numpy()
        if not np.isfinite(saved_thresholds).all() or not np.allclose(
            saved_thresholds, float(thresholds[fold]), rtol=0, atol=1e-12
        ):
            raise ValueError(f"Locked Test fold {fold} threshold identity mismatch.")
        probabilities.append(group["probability_pneumonia"].to_numpy(dtype=float))
    return probabilities


def read_locked_test_cache(marker_path, prediction_path, contract, manifest, thresholds):
    if not marker_path.is_file() or not prediction_path.is_file():
        raise ValueError(
            "Incomplete Locked Test state: both sealed marker and predictions are required. "
            "Do not repeat test inference; inspect the saved run."
        )
    receipt = json.loads(marker_path.read_text(encoding="utf-8"))
    validate_receipt(receipt, contract, artifacts={"predictions": prediction_path},
                     allowed_statuses={"COMPLETE"})
    predictions = pd.read_csv(prediction_path, dtype={
        "relative_path": str, "patient_id": str, "model_label": str, "sha256": str,
    })
    return predictions, validate_locked_test_predictions(predictions, manifest, thresholds)


batch_size = int(shared_params["batch_size"])
test_contract = locked_test_run_contract(locked_test)

if marker.exists() or cached_predictions.exists():
    pred_long, test_probs = read_locked_test_cache(
        marker, cached_predictions, test_contract, locked_test, fold_thresholds,
    )
    print("Validated sealed Locked Test cache; no new Test inference.")
else:
    # Reserve the test evaluation before the first inference. Interrupted or
    # incomplete evaluations fail closed on restart rather than silently repeat.
    atomic_write_json(marker, seal_receipt({
        "status": "INFERENCE_STARTED",
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    }, test_contract))
    test_ds = build_plain_eval_dataset(locked_test, batch_size=batch_size)
    test_probs = []
    pred_rows = []

    for fold in range(1, 6):
        print(f"\nLocked Test inference — Fold {fold}/5")
        tf.keras.backend.clear_session()
        gc.collect()
        model, _ = build_m07(
            dropout=float(shared_params["dropout"]), seed=SEED + fold,
        )
        checkpoint = FINAL_ROOT / f"fold_{fold}" / "best.weights.h5"
        model.load_weights(checkpoint)
        p = model.predict(test_ds, verbose=0).reshape(-1)
        if len(p) != len(locked_test) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
            raise ValueError(f"Invalid prediction probabilities for Locked Test fold {fold}.")
        test_probs.append(p)
        for row_id, (prob, row) in enumerate(zip(p, locked_test.itertuples(index=False))):
            pred_rows.append({
                "row_id": row_id, "fold_id": fold,
                "relative_path": row.relative_path, "patient_id": row.patient_id,
                "model_label": row.model_label, "label": int(row.label),
                "sha256": row.sha256, "probability_pneumonia": float(prob),
                "fold_threshold": float(fold_thresholds[fold]),
            })
        del model
        tf.keras.backend.clear_session()
        gc.collect()

    # Ensure neither source images nor checkpoint bytes changed during inference.
    verified_manifest = load_locked_test_manifest()
    if contract_fingerprint(locked_test_run_contract(verified_manifest)) != contract_fingerprint(test_contract):
        raise ValueError("Locked Test inputs changed during inference; refusing to seal results.")
    pred_long = pd.DataFrame(pred_rows)
    validate_locked_test_predictions(pred_long, locked_test, fold_thresholds)
    temporary_predictions = cached_predictions.with_suffix(".csv.tmp")
    pred_long.to_csv(temporary_predictions, index=False)
    temporary_predictions.replace(cached_predictions)
    atomic_write_json(marker, seal_receipt({
        "status": "COMPLETE", "model_id": MODEL_ID,
        "test_images": int(len(locked_test)),
        "test_patients": int(locked_test["patient_id"].nunique()),
        "aggregation_primary": "threshold_normalized_five_fold_ensemble",
        "aggregation_secondary": "majority_vote_five_fold",
        "test_threshold_tuning": False,
        "fold_threshold_source": "each fold validation only",
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    }, test_contract, artifacts={"predictions": cached_predictions}))
    pred_long, test_probs = read_locked_test_cache(
        marker, cached_predictions, test_contract, locked_test, fold_thresholds,
    )

P = np.vstack(test_probs)
y_test = locked_test["label"].to_numpy(dtype=int)

# ------------------------------------------------------------
# A) Per-fold fixed-threshold test metrics
# ------------------------------------------------------------
test_fold_rows = []
votes = []

for i, fold in enumerate(range(1, 6)):
    threshold = float(fold_thresholds[fold])

    pred = (
        P[i] >= threshold
    ).astype(int)

    votes.append(pred)

    m = calculate_binary_metrics(
        y_test,
        pred,
        score=P[i],
    )

    test_fold_rows.append({
        "fold_id": fold,
        "threshold_from_validation": threshold,
        **m,
    })

test_fold_table = pd.DataFrame(test_fold_rows)

test_fold_table.to_csv(
    TEST_OUT / "M07_LOCKED_TEST_METRICS_BY_FOLD.csv",
    index=False,
)

# ------------------------------------------------------------
# B) PRIMARY: threshold-normalized ensemble
# ------------------------------------------------------------
normalized_scores = np.vstack([
    logit_np(P[i]) - logit_np(fold_thresholds[i+1])
    for i in range(5)
])

ensemble_score = normalized_scores.mean(axis=0)

primary_pred = (
    ensemble_score >= 0.0
).astype(int)

primary_metrics = calculate_binary_metrics(
    y_test,
    primary_pred,
    score=ensemble_score,
)

# ------------------------------------------------------------
# C) SECONDARY: majority vote
# ------------------------------------------------------------
votes = np.vstack(votes)

majority_pred = (
    votes.sum(axis=0) >= 3
).astype(int)

mean_probability = P.mean(axis=0)

majority_metrics = calculate_binary_metrics(
    y_test,
    majority_pred,
    score=mean_probability,
)

# ------------------------------------------------------------
# D) Save one-row-per-test-image ensemble predictions
# ------------------------------------------------------------
test_pred = locked_test[
    ["relative_path", "patient_id", "model_label", "label", "sha256"]
].copy()

for i, fold in enumerate(range(1, 6)):
    test_pred[f"probability_fold_{fold}"] = P[i]
    test_pred[f"threshold_fold_{fold}"] = fold_thresholds[fold]
    test_pred[f"vote_fold_{fold}"] = votes[i]

test_pred["mean_probability"] = mean_probability
test_pred["normalized_ensemble_score"] = ensemble_score
test_pred["prediction_primary_normalized"] = primary_pred
test_pred["prediction_primary"] = primary_pred  # canonical cross-resolution alias
test_pred["prediction_majority_vote"] = majority_pred

test_pred.to_csv(
    TEST_OUT / "M07_LOCKED_TEST_ENSEMBLE_PREDICTIONS.csv",
    index=False,
)

(TEST_OUT / "M07_LOCKED_TEST_PRIMARY_METRICS.json").write_text(
    json.dumps(json_safe(primary_metrics), indent=2),
    encoding="utf-8",
)

(TEST_OUT / "M07_LOCKED_TEST_MAJORITY_METRICS.json").write_text(
    json.dumps(json_safe(majority_metrics), indent=2),
    encoding="utf-8",
)

# ------------------------------------------------------------
# E) Train mean + OOF + Locked Test generalization table
# ------------------------------------------------------------
train_rows = []

for fold in range(1, 6):
    print(f"Train inference for generalization gap — Fold {fold}/5")

    tr = load_fold_manifest(fold, "train")
    tr_ds = build_plain_eval_dataset(
        tr,
        batch_size=batch_size,
    )

    tf.keras.backend.clear_session()
    gc.collect()

    model, _ = build_m07(
        dropout=float(shared_params["dropout"]),
        seed=SEED + fold,
    )
    model.load_weights(
        FINAL_ROOT / f"fold_{fold}" / "best.weights.h5"
    )

    p_tr = model.predict(
        tr_ds,
        verbose=0,
    ).reshape(-1)

    pred_tr = (
        p_tr >= fold_thresholds[fold]
    ).astype(int)

    m_tr = calculate_binary_metrics(
        tr["label"].to_numpy(dtype=int),
        pred_tr,
        score=p_tr,
    )

    train_rows.append({
        "fold_id": fold,
        **m_tr,
    })

    del model, tr_ds
    tf.keras.backend.clear_session()
    gc.collect()

train_metrics_by_fold = pd.DataFrame(train_rows)
train_metrics_by_fold.to_csv(
    TEST_OUT / "M07_TRAIN_METRICS_BY_FOLD.csv",
    index=False,
)

numeric_train = [
    c for c in train_metrics_by_fold.columns
    if c not in {"fold_id", "confusion_matrix"}
    and pd.api.types.is_numeric_dtype(train_metrics_by_fold[c])
]

train_mean = {
    c: float(train_metrics_by_fold[c].mean())
    for c in numeric_train
}

comparison = pd.DataFrame([
    {
        "dataset": "TRAIN_MEAN_5FOLD",
        **train_mean,
    },
    {
        "dataset": "OOF_DEVELOPMENT_FOLD_THRESHOLDS",
        **oof_primary_metrics,
    },
    {
        "dataset": "LOCKED_TEST_PRIMARY_NORMALIZED_ENSEMBLE",
        **primary_metrics,
    },
    {
        "dataset": "LOCKED_TEST_SECONDARY_MAJORITY",
        **majority_metrics,
    },
])

comparison.to_csv(
    TEST_OUT / "M07_TRAIN_OOF_LOCKED_TEST_COMPARISON.csv",
    index=False,
)

gaps = {
    "train_minus_oof_balanced_accuracy": (
        float(train_mean["balanced_accuracy"])
        - float(oof_primary_metrics["balanced_accuracy"])
    ),
    "oof_minus_test_balanced_accuracy": (
        float(oof_primary_metrics["balanced_accuracy"])
        - float(primary_metrics["balanced_accuracy"])
    ),
    "train_minus_oof_macro_f1": (
        float(train_mean["macro_f1"])
        - float(oof_primary_metrics["macro_f1"])
    ),
    "oof_minus_test_macro_f1": (
        float(oof_primary_metrics["macro_f1"])
        - float(primary_metrics["macro_f1"])
    ),
}

(TEST_OUT / "M07_GENERALIZATION_GAPS.json").write_text(
    json.dumps(gaps, indent=2),
    encoding="utf-8",
)

print("\n" + "=" * 120)
print("M07 — TRAIN / OOF / LOCKED TEST")
print("=" * 120)

display(
    comparison[
        [
            "dataset",
            "accuracy",
            "balanced_accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "f2",
            "mcc",
            "auroc",
            "precision_normal",
            "recall_normal",
            "f1_normal",
            "precision_pneumonia",
            "recall_pneumonia",
            "f1_pneumonia",
        ]
    ].round(4)
)

print("\nPRIMARY Locked Test confusion matrix:")
print(np.asarray(primary_metrics["confusion_matrix"]))

print("\nGeneralization gaps:")
for k, v in gaps.items():
    print(f"{k:42s}: {v:+.4f}")

print("\n✅ Locked Test evaluated once")
print("✅ No threshold tuning on Test")
print("✅ Primary = threshold-normalized 5-fold ensemble")
print("✅ Secondary = majority vote using fold-validation thresholds")

# Immediately persist all one-time real Locked-Test evidence.
_persist_lightweight_files(
    [
        TEST_OUT / "LOCKED_TEST_EVALUATED_ONCE.json",
        TEST_OUT / "M07_LOCKED_TEST_PER_FOLD_PREDICTIONS.csv",
        TEST_OUT / "M07_LOCKED_TEST_METRICS_BY_FOLD.csv",
        TEST_OUT / "M07_LOCKED_TEST_ENSEMBLE_PREDICTIONS.csv",
        TEST_OUT / "M07_LOCKED_TEST_PRIMARY_METRICS.json",
        TEST_OUT / "M07_LOCKED_TEST_MAJORITY_METRICS.json",
        TEST_OUT / "M07_TRAIN_METRICS_BY_FOLD.csv",
        TEST_OUT / "M07_TRAIN_OOF_LOCKED_TEST_COMPARISON.csv",
        TEST_OUT / "M07_GENERALIZATION_GAPS.json",
    ],
    note="M07 one-time real Kermany Locked Test complete",
)

print("✅ REAL LOCKED TEST RESULTS PERSISTED DURABLY")


# ------------------------------------------------------------
# F) Fixed dual-threshold 20/80 analysis — DESCRIPTIVE ONLY
#    Uses the five-fold mean raw probability, never tuned on Test.
# ------------------------------------------------------------
test_dual_20_80, test_dual_decision = analyze_dual_threshold(
    y_test,
    mean_probability,
)

test_pred["dual20_80_decision"] = test_dual_decision
test_pred.to_csv(
    TEST_OUT / "M07_LOCKED_TEST_ENSEMBLE_PREDICTIONS.csv",
    index=False,
)

(TEST_OUT / "M07_LOCKED_TEST_DUAL20_80_REPORT.json").write_text(
    json.dumps(json_safe(test_dual_20_80), indent=2),
    encoding="utf-8",
)

print("\nLOCKED TEST fixed dual 20/80:")
print(" coverage:", f"{test_dual_20_80['coverage']:.4f}")
print(" uncertain:", f"{test_dual_20_80['uncertainty_rate']:.4f}")
print(" primary F2:", f"{primary_metrics['f2']:.4f}")

_persist_lightweight_files(
    [
        TEST_OUT / "M07_LOCKED_TEST_DUAL20_80_REPORT.json",
        TEST_OUT / "M07_LOCKED_TEST_ENSEMBLE_PREDICTIONS.csv",
    ],
    note="M07 Locked Test fixed dual-threshold 20/80 report persisted",
)


In [ ]:
# ============================================================
# 12B) STATISTICAL / CALIBRATION / COVERAGE DIAGNOSTICS
# ============================================================

DIAG_ROOT = OUTPUT_ROOT / "DIAGNOSTICS"
DIAG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# A) 2000-bootstrap Locked Test CI
# ------------------------------------------------------------
def bootstrap_fixed_prediction_metrics(
    y,
    pred,
    score,
    patient_ids,
    n_boot,
    seed,
):
    boot = bootstrap_fixed_rule_by_patient(
        y=y,
        pred=pred,
        score=score,
        patient_ids=patient_ids,
        n_boot=n_boot,
        seed=seed,
    )

    ci = {}

    for column in boot.columns:
        ci[column] = {
            "point": (
                float(
                    primary_metrics[column]
                )
                if column in primary_metrics
                else None
            ),
            "ci95_low": float(
                boot[column].quantile(
                    0.025
                )
            ),
            "ci95_high": float(
                boot[column].quantile(
                    0.975
                )
            ),
            "n_bootstrap_valid": int(
                len(boot)
            ),
            "bootstrap_unit": "PATIENT_CLUSTER",
        }

    return boot, ci


test_bootstrap, test_bootstrap_ci = (
    bootstrap_fixed_prediction_metrics(
        y_test,
        primary_pred,
        ensemble_score,
        locked_test[
            "patient_id"
        ].astype(str).to_numpy(),
        n_boot=LOCKED_TEST_BOOTSTRAPS,
        seed=SEED + 9001,
    )
)

test_bootstrap.to_csv(
    TEST_OUT
    / "M07_LOCKED_TEST_BOOTSTRAP_2000.csv",
    index=False,
)

(
    TEST_OUT
    / "M07_LOCKED_TEST_BOOTSTRAP_CI95.json"
).write_text(
    json.dumps(
        json_safe(test_bootstrap_ci),
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# B) ECE + reliability diagram
# ------------------------------------------------------------
def expected_calibration_error(
    y,
    probability,
    n_bins=10,
):
    y = np.asarray(y, dtype=int)
    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        0.0,
        1.0,
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    rows = []
    ece = 0.0

    for i in range(n_bins):
        left = edges[i]
        right = edges[i + 1]

        if i == n_bins - 1:
            mask = (
                (p >= left)
                & (p <= right)
            )
        else:
            mask = (
                (p >= left)
                & (p < right)
            )

        n = int(np.sum(mask))
        if n == 0:
            rows.append({
                "bin": i,
                "left": left,
                "right": right,
                "n": 0,
                "mean_probability": np.nan,
                "observed_rate": np.nan,
                "abs_gap": np.nan,
            })
            continue

        mean_p = float(
            np.mean(p[mask])
        )
        observed = float(
            np.mean(y[mask])
        )
        gap = abs(
            mean_p - observed
        )

        ece += (
            n / len(y)
        ) * gap

        rows.append({
            "bin": i,
            "left": left,
            "right": right,
            "n": n,
            "mean_probability": mean_p,
            "observed_rate": observed,
            "abs_gap": gap,
        })

    return float(ece), pd.DataFrame(rows)

def save_reliability_plot(
    y,
    probability,
    title,
    path,
    bins_csv,
):
    ece, table = expected_calibration_error(
        y,
        probability,
        n_bins=10,
    )

    table.to_csv(
        bins_csv,
        index=False,
    )

    valid = table[
        table["n"] > 0
    ]

    fig, ax = plt.subplots(
        figsize=(6, 6)
    )
    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration",
    )
    ax.plot(
        valid["mean_probability"],
        valid["observed_rate"],
        marker="o",
        label=f"Model | ECE={ece:.4f}",
    )
    ax.set_xlabel(
        "Mean predicted P(PNEUMONIA)"
    )
    ax.set_ylabel(
        "Observed PNEUMONIA rate"
    )
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=180,
    )
    plt.close(fig)

    return ece

oof_ece = save_reliability_plot(
    oof["label"].to_numpy(dtype=int),
    oof["probability_pneumonia"].to_numpy(dtype=float),
    "M07 OOF Reliability Diagram",
    DIAG_ROOT / "M07_OOF_RELIABILITY_DIAGRAM.png",
    DIAG_ROOT / "M07_OOF_CALIBRATION_BINS.csv",
)

test_ece = save_reliability_plot(
    y_test,
    mean_probability,
    "M07 Locked Test Reliability Diagram",
    DIAG_ROOT / "M07_LOCKED_TEST_RELIABILITY_DIAGRAM.png",
    DIAG_ROOT / "M07_LOCKED_TEST_CALIBRATION_BINS.csv",
)

calibration_report = {
    "schema": "m07.calibration.report.v1",
    "fit_performed": False,
    "oof": {
        "ece_10_bins": float(oof_ece),
        "brier": float(
            brier_score_loss(
                oof["label"].to_numpy(dtype=int),
                oof["probability_pneumonia"].to_numpy(dtype=float),
            )
        ),
    },
    "locked_test": {
        "ece_10_bins": float(test_ece),
        "brier": float(
            brier_score_loss(
                y_test,
                mean_probability,
            )
        ),
    },
}

(DIAG_ROOT / "M07_CALIBRATION_REPORT.json").write_text(
    json.dumps(
        json_safe(calibration_report),
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# C) Risk-Coverage
#     Retain samples from highest model confidence downward.
#     Primary predictions are not changed.
# ------------------------------------------------------------
def risk_coverage_table(
    y,
    pred,
    confidence,
    min_coverage=0.10,
):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    confidence = np.asarray(
        confidence,
        dtype=float,
    )

    order = np.argsort(
        -confidence
    )
    y = y[order]
    pred = pred[order]
    confidence = confidence[order]

    rows = []

    for coverage in np.linspace(
        min_coverage,
        1.0,
        19,
    ):
        n = max(
            1,
            int(
                round(
                    coverage * len(y)
                )
            ),
        )

        yy = y[:n]
        pp = pred[:n]

        rows.append({
            "coverage": float(
                n / len(y)
            ),
            "n_retained": int(n),
            "risk_error_rate": float(
                1.0
                - accuracy_score(
                    yy,
                    pp,
                )
            ),
            "accuracy": float(
                accuracy_score(
                    yy,
                    pp,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    yy,
                    pp,
                )
            ) if len(np.unique(yy)) > 1 else np.nan,
            "recall_pneumonia": float(
                recall_score(
                    yy,
                    pp,
                    pos_label=1,
                    zero_division=0,
                )
            ),
            "recall_normal": float(
                recall_score(
                    yy,
                    pp,
                    pos_label=0,
                    zero_division=0,
                )
            ),
        })

    return pd.DataFrame(rows)

# OOF confidence relative to each fold-selected decision boundary.
oof_confidence = np.abs(
    logit_np(
        oof["probability_pneumonia"].to_numpy(dtype=float)
    )
    - np.asarray([
        logit_np(
            fold_thresholds[
                int(fold)
            ]
        )
        for fold in oof[
            "fold_id"
        ].to_numpy(dtype=int)
    ])
)

oof_rc = risk_coverage_table(
    oof["label"].to_numpy(dtype=int),
    oof["prediction_fold_threshold"].to_numpy(dtype=int),
    oof_confidence,
)

test_rc = risk_coverage_table(
    y_test,
    primary_pred,
    np.abs(
        ensemble_score
    ),
)

oof_rc.to_csv(
    DIAG_ROOT / "M07_OOF_RISK_COVERAGE.csv",
    index=False,
)
test_rc.to_csv(
    DIAG_ROOT / "M07_LOCKED_TEST_RISK_COVERAGE.csv",
    index=False,
)

for name, table in (
    ("OOF", oof_rc),
    ("LOCKED_TEST", test_rc),
):
    fig, ax = plt.subplots(
        figsize=(7, 5)
    )
    ax.plot(
        table["coverage"],
        table["risk_error_rate"],
        marker="o",
    )
    ax.set_xlabel("Coverage")
    ax.set_ylabel("Risk = 1 - Accuracy")
    ax.set_title(
        f"M07 {name} Risk–Coverage"
    )
    fig.tight_layout()
    fig.savefig(
        DIAG_ROOT
        / f"M07_{name}_RISK_COVERAGE.png",
        dpi=180,
    )
    plt.close(fig)

# ------------------------------------------------------------
# D) Decision curve — descriptive only, no tuning.
# ------------------------------------------------------------
def decision_curve_table(
    y,
    probability,
    thresholds=None,
):
    y = np.asarray(y, dtype=int)
    p = np.asarray(
        probability,
        dtype=float,
    )

    if thresholds is None:
        thresholds = np.linspace(
            0.05,
            0.95,
            19,
        )

    prevalence = float(
        np.mean(y)
    )
    rows = []

    for threshold in thresholds:
        pred = (
            p >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y,
            pred,
            labels=[0, 1],
        ).ravel()

        odds = (
            threshold
            / (1.0 - threshold)
        )

        nb_model = (
            tp / len(y)
            - fp / len(y) * odds
        )

        # Treat-all:
        # TP prevalence; FP 1-prevalence.
        nb_all = (
            prevalence
            - (1.0 - prevalence)
            * odds
        )

        rows.append({
            "threshold": float(
                threshold
            ),
            "net_benefit_model": float(
                nb_model
            ),
            "net_benefit_treat_all": float(
                nb_all
            ),
            "net_benefit_treat_none": 0.0,
        })

    return pd.DataFrame(rows)

oof_dc = decision_curve_table(
    oof["label"].to_numpy(dtype=int),
    oof["probability_pneumonia"].to_numpy(dtype=float),
)

test_dc = decision_curve_table(
    y_test,
    mean_probability,
)

oof_dc.to_csv(
    DIAG_ROOT / "M07_OOF_DECISION_CURVE.csv",
    index=False,
)
test_dc.to_csv(
    DIAG_ROOT / "M07_LOCKED_TEST_DECISION_CURVE.csv",
    index=False,
)

for name, table in (
    ("OOF", oof_dc),
    ("LOCKED_TEST", test_dc),
):
    fig, ax = plt.subplots(
        figsize=(7, 5)
    )
    ax.plot(
        table["threshold"],
        table[
            "net_benefit_model"
        ],
        label="Model",
    )
    ax.plot(
        table["threshold"],
        table[
            "net_benefit_treat_all"
        ],
        label="Treat All",
    )
    ax.plot(
        table["threshold"],
        table[
            "net_benefit_treat_none"
        ],
        label="Treat None",
    )
    ax.set_xlabel(
        "Decision threshold"
    )
    ax.set_ylabel(
        "Net benefit"
    )
    ax.set_title(
        f"M07 {name} Decision Curve"
    )
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        DIAG_ROOT
        / f"M07_{name}_DECISION_CURVE.png",
        dpi=180,
    )
    plt.close(fig)

# ------------------------------------------------------------
# E) Per-fold Test diversity / correlation
# ------------------------------------------------------------
corr = pd.DataFrame(
    P.T,
    columns=[
        f"fold_{i}"
        for i in range(1, 6)
    ],
).corr()

corr.to_csv(
    DIAG_ROOT
    / "M07_LOCKED_TEST_FOLD_PROBABILITY_CORRELATION.csv"
)

disagreement_rows = []

for i in range(5):
    for j in range(
        i + 1,
        5,
    ):
        pred_i = (
            P[i]
            >= fold_thresholds[
                i + 1
            ]
        ).astype(int)
        pred_j = (
            P[j]
            >= fold_thresholds[
                j + 1
            ]
        ).astype(int)

        disagreement_rows.append({
            "fold_a": i + 1,
            "fold_b": j + 1,
            "disagreement_rate": float(
                np.mean(
                    pred_i
                    != pred_j
                )
            ),
            "probability_correlation": float(
                np.corrcoef(
                    P[i],
                    P[j],
                )[0, 1]
            ),
        })

pd.DataFrame(
    disagreement_rows
).to_csv(
    DIAG_ROOT
    / "M07_LOCKED_TEST_FOLD_DIVERSITY.csv",
    index=False,
)

print("✅ Locked Test bootstrap:", LOCKED_TEST_BOOTSTRAPS)
print("✅ Calibration/Brier/ECE complete")
print("✅ Risk–Coverage complete")
print("✅ Decision Curve complete")
print("✅ Fold diversity complete")

In [ ]:
# ============================================================
# 13) COMPLETE FIGURE SUITE
#     Final-version style outputs + new 20/80 figures
# ============================================================
import matplotlib.pyplot as plt

FIG_ROOT = OUTPUT_ROOT / "FIGURES"
FIG_ROOT.mkdir(parents=True, exist_ok=True)

def history_global_epochs(h):
    """Preserve recorded Keras epochs, including gaps after an early-stopped head."""
    if "epoch" not in h.columns:
        return np.arange(1, len(h) + 1)
    epochs = pd.to_numeric(h["epoch"], errors="raise").to_numpy(dtype=float)
    if (
        not np.isfinite(epochs).all()
        or np.any(epochs < 1)
        or np.any(epochs != np.floor(epochs))
        or np.any(np.diff(epochs) <= 0)
    ):
        raise ValueError("Training history contains invalid or unordered epoch identities")
    return epochs.astype(int)


def save_confusion_figure(cm, title, path):
    cm = np.asarray(cm, dtype=int)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(cm)
    ax.set_xticks([0, 1], ["NORMAL", "PNEUMONIA"])
    ax.set_yticks([0, 1], ["NORMAL", "PNEUMONIA"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)

def save_roc_figure(y_true, score, title, path):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    fpr, tpr, _ = roc_curve(y_true, score)
    auc = roc_auc_score(y_true, score)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f"AUROC={auc:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)

def save_pr_figure(y_true, score, title, path):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    precision, recall, _ = precision_recall_curve(y_true, score)
    ap = average_precision_score(y_true, score)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recall, precision, label=f"AP={ap:.4f}")
    ax.set_xlabel("Recall — PNEUMONIA")
    ax.set_ylabel("Precision — PNEUMONIA")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)

def save_dual_probability_figure(
    y_true,
    probability,
    title,
    path,
    low=DUAL_THRESHOLD_LOW,
    high=DUAL_THRESHOLD_HIGH,
):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(
        probability[y_true == 0],
        bins=25,
        alpha=0.55,
        label="True NORMAL",
    )
    ax.hist(
        probability[y_true == 1],
        bins=25,
        alpha=0.55,
        label="True PNEUMONIA",
    )
    ax.axvline(low, linestyle="--", label=f"Low={low:.2f}")
    ax.axvline(high, linestyle="--", label=f"High={high:.2f}")
    ax.set_xlabel("P(PNEUMONIA)")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)

# ------------------------------------------------------------
# A) Per-fold validation outputs exactly in final-version spirit
# ------------------------------------------------------------
for fold in range(1, 6):
    fold_dir = FINAL_ROOT / f"fold_{fold}"
    pred_path = fold_dir / "validation_predictions.csv"
    history_path = fold_dir / "history.csv"

    if pred_path.is_file():
        v = pd.read_csv(pred_path)
        yv = v["label"].to_numpy(dtype=int)
        pv = v["probability_pneumonia"].to_numpy(dtype=float)
        tv = float(fold_thresholds[fold])
        predv = (pv >= tv).astype(int)

        fold_metric = calculate_binary_metrics(yv, predv, score=pv)
        save_confusion_figure(
            fold_metric["confusion_matrix"],
            f"M07 Fold {fold} Validation",
            fold_dir / "validation_confusion_matrix.png",
        )
        save_roc_figure(
            yv,
            pv,
            f"M07 Fold {fold} Validation ROC",
            fold_dir / "validation_roc_curve.png",
        )
        save_pr_figure(
            yv,
            pv,
            f"M07 Fold {fold} Validation Precision-Recall",
            fold_dir / "validation_pr_curve.png",
        )

        fold_dual, fold_dual_decision = analyze_dual_threshold(yv, pv)
        (fold_dir / "validation_dual20_80_report.json").write_text(
            json.dumps(json_safe(fold_dual), indent=2),
            encoding="utf-8",
        )
        save_dual_probability_figure(
            yv,
            pv,
            f"M07 Fold {fold} Validation — Fixed 20/80",
            fold_dir / "validation_dual20_80_probability.png",
        )

    if history_path.is_file():
        h = pd.read_csv(history_path).reset_index(drop=True)
        h["global_epoch"] = history_global_epochs(h)

        for metric in ("loss", "auc_pr", "auc_roc", "recall"):
            if metric not in h.columns:
                continue

            fig, ax = plt.subplots(figsize=(7, 5))
            ax.plot(
                h["global_epoch"],
                h[metric],
                label=f"train_{metric}",
            )

            val_metric = f"val_{metric}"
            if val_metric in h.columns:
                ax.plot(
                    h["global_epoch"],
                    h[val_metric],
                    label=val_metric,
                )

            ax.set_xlabel("Global epoch")
            ax.set_ylabel(metric)
            ax.set_title(f"M07 Fold {fold} — {metric}")
            ax.legend()
            fig.tight_layout()
            fig.savefig(
                fold_dir / f"history_{metric}.png",
                dpi=180,
            )
            plt.close(fig)

# ------------------------------------------------------------
# B) OOF outputs
# ------------------------------------------------------------
y_oof = oof["label"].to_numpy(dtype=int)
p_oof = oof["probability_pneumonia"].to_numpy(dtype=float)
pred_oof = oof["prediction_fold_threshold"].to_numpy(dtype=int)

save_confusion_figure(
    confusion_matrix(y_oof, pred_oof, labels=[0, 1]),
    "M07 OOF Confusion Matrix — Fold thresholds",
    FIG_ROOT / "M07_OOF_CONFUSION_MATRIX.png",
)
save_roc_figure(
    y_oof,
    p_oof,
    "M07 OOF ROC — Development",
    FIG_ROOT / "M07_OOF_ROC_CURVE.png",
)
save_pr_figure(
    y_oof,
    p_oof,
    "M07 OOF Precision-Recall — Development",
    FIG_ROOT / "M07_OOF_PR_CURVE.png",
)
save_dual_probability_figure(
    y_oof,
    p_oof,
    "M07 OOF — Fixed 20/80 probability bands",
    FIG_ROOT / "M07_OOF_DUAL20_80_PROBABILITY.png",
)

# ------------------------------------------------------------
# C) Locked Test outputs
# ------------------------------------------------------------
save_confusion_figure(
    primary_metrics["confusion_matrix"],
    "M07 Locked Test — Primary normalized ensemble",
    FIG_ROOT / "M07_LOCKED_TEST_CONFUSION_MATRIX.png",
)
save_roc_figure(
    y_test,
    ensemble_score,
    "M07 Locked Test ROC — Primary ensemble score",
    FIG_ROOT / "M07_LOCKED_TEST_ROC_CURVE.png",
)
save_pr_figure(
    y_test,
    ensemble_score,
    "M07 Locked Test Precision-Recall — Primary ensemble score",
    FIG_ROOT / "M07_LOCKED_TEST_PR_CURVE.png",
)
save_dual_probability_figure(
    y_test,
    mean_probability,
    "M07 Locked Test — Fixed mean-probability 20/80 bands",
    FIG_ROOT / "M07_LOCKED_TEST_DUAL20_80_PROBABILITY.png",
)

# Per-fold Test metrics.
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(1, 6)
for metric in ("balanced_accuracy", "macro_f1", "f2", "auroc"):
    if metric in test_fold_table.columns:
        ax.plot(
            x,
            test_fold_table[metric].to_numpy(dtype=float),
            marker="o",
            label=metric,
        )
ax.set_xticks(x)
ax.set_xlabel("Fold")
ax.set_ylabel("Metric")
ax.set_title("M07 Locked Test — Per-fold Metrics")
ax.legend()
fig.tight_layout()
fig.savefig(
    FIG_ROOT / "M07_LOCKED_TEST_PER_FOLD_METRICS.png",
    dpi=180,
)
plt.close(fig)

# Train / OOF / Test.
compare_names = ["Train mean", "OOF", "Locked Test"]
compare_rows = [train_mean, oof_primary_metrics, primary_metrics]
fig, ax = plt.subplots(figsize=(10, 5))
base = np.arange(len(compare_names))
width = 0.19
for k, metric in enumerate(
    ("balanced_accuracy", "macro_f1", "f2", "auroc")
):
    vals = [float(r[metric]) for r in compare_rows]
    ax.bar(
        base + (k - 1.5) * width,
        vals,
        width=width,
        label=metric,
    )
ax.set_xticks(base, compare_names)
ax.set_ylim(0.0, 1.05)
ax.set_ylabel("Metric")
ax.set_title("M07 Generalization — Train vs OOF vs Locked Test")
ax.legend()
fig.tight_layout()
fig.savefig(
    FIG_ROOT / "M07_TRAIN_OOF_LOCKED_TEST_COMPARISON.png",
    dpi=180,
)
plt.close(fig)

# Five-fold combined histories.
for metric in ("loss", "auc_pr", "auc_roc", "recall"):
    fig, ax = plt.subplots(figsize=(8, 5))
    plotted = False

    for fold in range(1, 6):
        history_path = FINAL_ROOT / f"fold_{fold}" / "history.csv"
        if not history_path.is_file():
            continue

        h = pd.read_csv(history_path).reset_index(drop=True)
        if metric not in h.columns:
            continue

        h["global_epoch"] = history_global_epochs(h)
        ax.plot(
            h["global_epoch"],
            h[metric],
            label=f"Fold {fold}",
        )
        plotted = True

    if plotted:
        ax.set_xlabel("Global epoch")
        ax.set_ylabel(metric)
        ax.set_title(f"M07 Five-Fold History — {metric}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            FIG_ROOT / f"M07_HISTORY_{metric.upper()}.png",
            dpi=180,
        )

    plt.close(fig)

print("✅ Full figure suite generated:", FIG_ROOT)

# ============================================================
# D) ENHANCED LOSS / TRAINING DIAGNOSTICS
# ============================================================
LOSS_ROOT = FIG_ROOT / "LOSS_DIAGNOSTICS"
LOSS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

loss_summary_rows = []
val_loss_long = []

def _best_epoch_from_checkpoint(h, checkpoint_path):
    """Return the saved epoch; incomplete history cannot reconstruct its rank."""
    checkpoint_path = Path(checkpoint_path)
    state_path = checkpoint_path.with_suffix(checkpoint_path.suffix + ".state.json")
    if not checkpoint_path.is_file() or not state_path.is_file():
        return None
    state = json.loads(state_path.read_text(encoding="utf-8"))
    raw_epoch = state.get("best_epoch")
    if isinstance(raw_epoch, bool) or not isinstance(raw_epoch, int) or raw_epoch < 1:
        raise ValueError(f"Invalid checkpoint best_epoch in {state_path}")
    if raw_epoch not in set(h["global_epoch"].astype(int)):
        raise ValueError(f"Checkpoint epoch is absent from training history: {state_path}")
    return raw_epoch

for fold in range(1, 6):
    history_path = (
        FINAL_ROOT
        / f"fold_{fold}"
        / "history.csv"
    )

    if not history_path.is_file():
        continue

    h = pd.read_csv(
        history_path
    ).reset_index(drop=True)

    h["global_epoch"] = history_global_epochs(h)

    best_epoch = _best_epoch_from_checkpoint(
        h, FINAL_ROOT / f"fold_{fold}" / "best.weights.h5"
    )

    # Per-fold Loss: train + validation.
    if (
        "loss" in h.columns
        and "val_loss" in h.columns
    ):
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
        ax.plot(
            h["global_epoch"],
            h["loss"],
            label="Train loss",
        )
        ax.plot(
            h["global_epoch"],
            h["val_loss"],
            label="Validation loss",
        )
        ax.axvline(
            FINAL_HEAD_EPOCHS
            + 0.5,
            linestyle="--",
            label="Fine-tuning begins",
        )

        if best_epoch is not None:
            ax.axvline(
                best_epoch,
                linestyle=":",
                label=f"Best epoch={best_epoch}",
            )

        ax.set_xlabel("Global epoch")
        ax.set_ylabel("FLSD-53 loss")
        ax.set_title(
            f"M07 Fold {fold} — Train vs Validation Loss"
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            LOSS_ROOT
            / f"fold_{fold}_TRAIN_VALIDATION_LOSS.png",
            dpi=180,
        )
        plt.close(fig)

        val_loss_long.extend([
            {
                "fold": fold,
                "epoch": int(e),
                "val_loss": float(v),
            }
            for e, v in zip(
                h["global_epoch"],
                h["val_loss"],
            )
            if pd.notna(v)
        ])

        best_row = (
            h[
                h["global_epoch"]
                == best_epoch
            ].iloc[0]
            if best_epoch is not None
            else pd.Series(dtype=float)
        )

        loss_summary_rows.append({
            "fold": fold,
            "best_epoch_from_checkpoint_state": best_epoch,
            "best_epoch_source": "checkpoint_state" if best_epoch is not None else "unavailable",
            "train_loss_at_best": float(
                best_row["loss"]
            ) if pd.notna(
                best_row.get("loss")
            ) else np.nan,
            "validation_loss_at_best": float(
                best_row["val_loss"]
            ) if pd.notna(
                best_row.get("val_loss")
            ) else np.nan,
            "final_epoch": int(
                h["global_epoch"].iloc[-1]
            ),
            "final_train_loss": float(
                h["loss"].iloc[-1]
            ),
            "final_validation_loss": float(
                h["val_loss"].iloc[-1]
            ),
            "final_train_val_loss_gap": float(
                h["val_loss"].iloc[-1]
                - h["loss"].iloc[-1]
            ),
        })

    # More history figures requested.
    history_specs = [
        (
            "precision",
            "val_precision",
            "Precision",
        ),
        (
            "recall",
            "val_recall",
            "Recall",
        ),
        (
            "auc_roc",
            "val_auc_roc",
            "AUROC",
        ),
        (
            "auc_pr",
            "val_auc_pr",
            "AUPRC",
        ),
    ]

    for train_col, val_col, title_metric in history_specs:
        if train_col not in h.columns:
            continue

        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
        ax.plot(
            h["global_epoch"],
            h[train_col],
            label=f"Train {title_metric}",
        )

        if val_col in h.columns:
            ax.plot(
                h["global_epoch"],
                h[val_col],
                label=f"Validation {title_metric}",
            )

        ax.axvline(
            FINAL_HEAD_EPOCHS
            + 0.5,
            linestyle="--",
            label="Fine-tuning begins",
        )
        if best_epoch is not None:
            ax.axvline(
                best_epoch,
                linestyle=":",
                label="Best epoch",
            )

        ax.set_xlabel("Global epoch")
        ax.set_ylabel(title_metric)
        ax.set_title(
            f"M07 Fold {fold} — {title_metric}"
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            LOSS_ROOT
            / f"fold_{fold}_{title_metric.upper()}_HISTORY.png",
            dpi=180,
        )
        plt.close(fig)

    # Validation P/R balance score.
    if (
        "val_min_dual_precision_recall_selected"
        in h.columns
    ):
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
        ax.plot(
            h["global_epoch"],
            h[
                "val_min_dual_precision_recall_selected"
            ],
            label="Validation min dual P/R",
        )
        if (
            "val_dual_precision_recall_score"
            in h.columns
        ):
            ax.plot(
                h["global_epoch"],
                h[
                    "val_dual_precision_recall_score"
                ],
                label="Validation dual P/R score",
            )
        ax.axvline(
            FINAL_HEAD_EPOCHS
            + 0.5,
            linestyle="--",
            label="Fine-tuning begins",
        )
        if best_epoch is not None:
            ax.axvline(
                best_epoch,
                linestyle=":",
                label="Best epoch",
            )
        ax.set_xlabel("Global epoch")
        ax.set_ylabel("Score")
        ax.set_ylim(0.0, 1.02)
        ax.set_title(
            f"M07 Fold {fold} — Precision/Recall Balance"
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            LOSS_ROOT
            / f"fold_{fold}_MIN_DUAL_PRECISION_RECALL.png",
            dpi=180,
        )
        plt.close(fig)

    if (
        "val_selected_threshold"
        in h.columns
    ):
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
        ax.plot(
            h["global_epoch"],
            h[
                "val_selected_threshold"
            ],
            marker="o",
        )
        ax.axvline(
            FINAL_HEAD_EPOCHS
            + 0.5,
            linestyle="--",
        )
        ax.set_xlabel("Global epoch")
        ax.set_ylabel(
            "Validation-selected threshold"
        )
        ax.set_title(
            f"M07 Fold {fold} — Threshold Evolution"
        )
        fig.tight_layout()
        fig.savefig(
            LOSS_ROOT
            / f"fold_{fold}_SELECTED_THRESHOLD.png",
            dpi=180,
        )
        plt.close(fig)

    if "learning_rate" in h.columns:
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
        ax.plot(
            h["global_epoch"],
            h["learning_rate"],
        )
        ax.axvline(
            FINAL_HEAD_EPOCHS
            + 0.5,
            linestyle="--",
        )
        ax.set_xlabel("Global epoch")
        ax.set_ylabel("Learning rate")
        ax.set_yscale("log")
        ax.set_title(
            f"M07 Fold {fold} — Learning Rate"
        )
        fig.tight_layout()
        fig.savefig(
            LOSS_ROOT
            / f"fold_{fold}_LEARNING_RATE.png",
            dpi=180,
        )
        plt.close(fig)

loss_summary = pd.DataFrame(
    loss_summary_rows
)
loss_summary.to_csv(
    LOSS_ROOT / "M07_LOSS_SUMMARY.csv",
    index=False,
)

val_loss_df = pd.DataFrame(
    val_loss_long
)

if not val_loss_df.empty:
    # All validation loss traces.
    fig, ax = plt.subplots(
        figsize=(9, 5)
    )
    for fold, group in val_loss_df.groupby(
        "fold"
    ):
        ax.plot(
            group["epoch"],
            group["val_loss"],
            label=f"Fold {fold}",
        )
    ax.set_xlabel("Global epoch")
    ax.set_ylabel("Validation FLSD-53 loss")
    ax.set_title(
        "M07 Five-Fold Validation Loss"
    )
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        LOSS_ROOT
        / "M07_5FOLD_VALIDATION_LOSS.png",
        dpi=180,
    )
    plt.close(fig)

    # Mean ± SD across available folds at each epoch.
    agg = (
        val_loss_df.groupby(
            "epoch"
        )["val_loss"]
        .agg(
            ["mean", "std", "count"]
        )
        .reset_index()
    )
    agg["std"] = agg[
        "std"
    ].fillna(0.0)
    agg.to_csv(
        LOSS_ROOT
        / "M07_MEAN_SD_VALIDATION_LOSS.csv",
        index=False,
    )

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )
    ax.plot(
        agg["epoch"],
        agg["mean"],
        label="Mean Validation Loss",
    )
    ax.fill_between(
        agg["epoch"],
        agg["mean"] - agg["std"],
        agg["mean"] + agg["std"],
        alpha=0.20,
        label="±1 SD",
    )
    ax.axvline(
        FINAL_HEAD_EPOCHS
        + 0.5,
        linestyle="--",
        label="Fine-tuning begins",
    )
    ax.set_xlabel("Global epoch")
    ax.set_ylabel("Validation FLSD-53 loss")
    ax.set_title(
        "M07 Mean ± SD Validation Loss Across Folds"
    )
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        LOSS_ROOT
        / "M07_MEAN_SD_VALIDATION_LOSS.png",
        dpi=180,
    )
    plt.close(fig)

print("✅ Enhanced Loss/training diagnostics:", LOSS_ROOT)

In [ ]:
# ============================================================
# 14) VALIDATION XAI
#     Grad-CAM / Grad-CAM++ approximation / IG magnitude / Occlusion
#
# XAI sample selection uses Fold-1 validation only.
# Locked Test is NOT used to select XAI examples.
# ============================================================

XAI_ROOT = OUTPUT_ROOT / "XAI_VALIDATION"
XAI_ROOT.mkdir(parents=True, exist_ok=True)

xai_summary = {
    "enabled": bool(RUN_VALIDATION_XAI),
    "status": "SKIPPED",
    "methods": list(XAI_METHODS),
    "gradcampp_is_approximation": True,
    "ig_display_is_magnitude": True,
    "interpretation": "Descriptive; signed IG and completeness diagnostics accompany magnitude maps.",
    "selection_source": "Fold-1 validation only",
    "locked_test_used_for_sample_selection": False,
}

def normalize_map(values):
    a = np.array(values, dtype=np.float32, copy=True)
    a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
    a -= a.min()
    maximum = a.max()
    if maximum > 1e-8:
        a /= maximum
    return a

def load_model_input(path):
    image, _ = decode_resize(
        tf.constant(str(path)),
        tf.constant(0.0, dtype=tf.float32),
    )
    return image.numpy().astype(np.float32)

def target_score(probability, target_class):
    probability = tf.reshape(
        tf.cast(probability, tf.float32),
        [-1],
    )
    return (
        probability
        if int(target_class) == 1
        else 1.0 - probability
    )

def build_grad_model(model):
    feature_layer = model.get_layer("aez_spatial_features")
    return tf.keras.Model(
        inputs=model.inputs,
        outputs=[feature_layer.output, model.output],
    )

def gradcam_map(model, image, target_class):
    grad_model = build_grad_model(model)
    x = tf.convert_to_tensor(image[None, ...], dtype=tf.float32)

    with tf.GradientTape() as tape:
        features, predictions = grad_model(x, training=False)
        score = target_score(predictions, target_class)[0]

    gradients = tape.gradient(score, features)
    weights = tf.reduce_mean(
        gradients,
        axis=(1, 2),
        keepdims=True,
    )
    cam = tf.reduce_sum(weights * features, axis=-1)[0]
    cam = tf.nn.relu(cam)
    cam = tf.image.resize(
        cam[..., None],
        image.shape[:2],
        method="bilinear",
    )[..., 0]
    return normalize_map(cam.numpy())

def gradcampp_map(model, image, target_class):
    """Legacy identifier: first-gradient-power approximation, not exact Grad-CAM++.

    Squared/cubed first derivatives are not the true second/third derivatives
    of this nonlinear head. Treat this map as a descriptive approximation.
    """
    grad_model = build_grad_model(model)
    x = tf.convert_to_tensor(image[None, ...], dtype=tf.float32)

    with tf.GradientTape() as tape:
        features, predictions = grad_model(x, training=False)
        score = target_score(predictions, target_class)[0]

    gradients = tf.cast(
        tape.gradient(score, features),
        tf.float32,
    )
    features = tf.cast(features, tf.float32)

    grad2 = tf.square(gradients)
    grad3 = grad2 * gradients

    denominator = 2.0 * grad2 + tf.reduce_sum(
        features * grad3,
        axis=(1, 2),
        keepdims=True,
    )
    denominator = tf.where(
        tf.abs(denominator) > 1e-8,
        denominator,
        tf.ones_like(denominator),
    )

    alpha = grad2 / denominator
    weights = tf.reduce_sum(
        alpha * tf.nn.relu(gradients),
        axis=(1, 2),
        keepdims=True,
    )
    cam = tf.reduce_sum(weights * features, axis=-1)[0]
    cam = tf.nn.relu(cam)
    cam = tf.image.resize(
        cam[..., None],
        image.shape[:2],
        method="bilinear",
    )[..., 0]
    return normalize_map(cam.numpy())

def summarize_signed_attribution(signed_attribution, input_score, baseline_score):
    """Keep attribution signs and diagnose completeness before display scaling."""
    signed = np.asarray(signed_attribution, dtype=np.float32)
    if signed.ndim != 3 or not np.isfinite(signed).all():
        raise ValueError("Expected finite H x W x C signed attributions")
    input_score = float(input_score)
    baseline_score = float(baseline_score)
    if not np.isfinite([input_score, baseline_score]).all():
        raise ValueError("Non-finite attribution endpoint scores")
    score_delta = input_score - baseline_score
    attribution_sum = float(np.sum(signed, dtype=np.float64))
    residual = score_delta - attribution_sum
    # Display magnitude removes signs; raw tensor remains available separately.
    magnitude = np.sum(np.abs(signed), axis=-1)
    return normalize_map(magnitude), {
        "input_target_score": input_score,
        "baseline_target_score": baseline_score,
        "target_score_delta": score_delta,
        "signed_attribution_sum": attribution_sum,
        "completeness_residual": residual,
        "relative_completeness_residual": abs(residual) / max(abs(score_delta), 1e-8),
        "visualization": "per-image normalized absolute attribution magnitude",
        "visualization_preserves_sign": False,
        "completeness_interpretation": (
            "Diagnostic only; finite integration error and detached normalization "
            "extrema can violate completeness. No completeness pass is asserted."
        ),
    }


def integrated_gradients_map(
    model,
    image,
    target_class,
    steps=32,
    return_details=False,
):
    if int(steps) != steps or int(steps) < 1:
        raise ValueError("Integrated Gradients steps must be a positive integer")
    steps = int(steps)
    image_tensor = tf.convert_to_tensor(image, dtype=tf.float32)
    baseline_value = tf.reduce_mean(image_tensor)
    baseline = tf.ones_like(image_tensor) * baseline_value
    alphas = tf.linspace(0.0, 1.0, steps + 1)
    interpolated = (
        baseline[None, ...]
        + alphas[:, None, None, None] * (image_tensor - baseline)[None, ...]
    )
    gradients = []
    for start in range(0, steps + 1, 8):
        batch = interpolated[start:start + 8]
        with tf.GradientTape() as tape:
            tape.watch(batch)
            predictions = model(batch, training=False)
            scores = target_score(predictions, target_class)
        gradient = tape.gradient(scores, batch)
        if gradient is None:
            raise ValueError("Integrated Gradients target is disconnected from input")
        gradients.append(gradient)
    gradients = tf.concat(gradients, axis=0)
    average_gradients = tf.reduce_mean((gradients[:-1] + gradients[1:]) / 2.0, axis=0)
    signed = ((image_tensor - baseline) * average_gradients).numpy()
    endpoints = model(tf.stack([baseline, image_tensor]), training=False)
    endpoint_scores = target_score(endpoints, target_class).numpy()
    saliency, details = summarize_signed_attribution(
        signed, endpoint_scores[1], endpoint_scores[0]
    )
    details.update({
        "integration_steps": steps,
        "baseline": "constant image equal to the input image mean",
        "baseline_value": float(baseline_value.numpy()),
        "target_class": int(target_class),
    })
    if return_details:
        return saliency, signed, details
    return saliency


def occlusion_map(
    model,
    image,
    target_class,
    patch=XAI_OCCLUSION_PATCH,
    stride=XAI_OCCLUSION_STRIDE,
):
    height, width = image.shape[:2]
    baseline_value = float(np.mean(image))

    original_probability = float(
        model.predict(
            image[None, ...],
            verbose=0,
        ).reshape(-1)[0]
    )
    original_score = (
        original_probability
        if int(target_class) == 1
        else 1.0 - original_probability
    )

    variants = []
    locations = []

    for y1 in range(0, height, stride):
        for x1 in range(0, width, stride):
            y2 = min(y1 + patch, height)
            x2 = min(x1 + patch, width)

            variant = image.copy()
            variant[y1:y2, x1:x2, :] = baseline_value
            variants.append(variant)
            locations.append((y1, y2, x1, x2))

    predictions = model.predict(
        np.asarray(variants, dtype=np.float32),
        batch_size=16,
        verbose=0,
    ).reshape(-1)

    scores = (
        predictions
        if int(target_class) == 1
        else 1.0 - predictions
    )
    drops = np.maximum(
        original_score - scores,
        0.0,
    )

    heat = np.zeros(
        (height, width),
        dtype=np.float32,
    )
    counts = np.zeros(
        (height, width),
        dtype=np.float32,
    )

    for drop, (y1, y2, x1, x2) in zip(
        drops,
        locations,
    ):
        heat[y1:y2, x1:x2] += float(drop)
        counts[y1:y2, x1:x2] += 1.0

    heat /= np.maximum(counts, 1.0)
    return normalize_map(heat)

def save_xai_visual(
    image,
    saliency,
    path,
    title,
):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(
        np.clip(image / 255.0, 0, 1)
    )
    axes[0].axis("off")
    axes[0].set_title("Model input")

    axes[1].imshow(
        saliency,
        cmap="jet",
    )
    axes[1].axis("off")
    axes[1].set_title("Saliency")

    axes[2].imshow(
        np.clip(image / 255.0, 0, 1)
    )
    axes[2].imshow(
        saliency,
        cmap="jet",
        alpha=0.42,
    )
    axes[2].axis("off")
    axes[2].set_title("Overlay")

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def xai_sample_id(relative_path):
    """Avoid silently overwriting maps for different directories with equal stems."""
    relative_path = str(relative_path)
    suffix = hashlib.sha256(relative_path.encode("utf-8")).hexdigest()[:12]
    return f"{Path(relative_path).stem}-{suffix}"


def xai_method_metadata(method):
    labels = {
        "gradcam": "Grad-CAM",
        "gradcampp": "Grad-CAM++ approximation",
        "integrated_gradients": "Integrated Gradients magnitude",
        "occlusion": "Occlusion score-drop map",
    }
    if method not in labels:
        raise ValueError(f"Unsupported XAI method: {method}")
    return {
        "method": str(method),
        "method_display_name": labels[method],
        "is_gradcampp_approximation": method == "gradcampp",
        "method_limitation": (
            "First-gradient powers substitute for higher derivatives; not validated exact Grad-CAM++."
            if method == "gradcampp" else
            "Autodiff may include surrogate gradients from detached extrema; inspect signed data and completeness residual."
            if method == "integrated_gradients" else
            "Descriptive visualization; not validated lesion localization or causal evidence."
        ),
    }


def compute_xai_map(model, image, target_class, method, artifact_prefix=None):
    """Compute one map with honest method metadata and optional raw IG artifacts."""
    metadata = xai_method_metadata(method)
    metadata["target_class"] = int(target_class)
    if method == "integrated_gradients":
        saliency, signed, details = integrated_gradients_map(
            model, image, target_class, steps=32, return_details=True
        )
        metadata.update(details)
        if artifact_prefix is not None:
            signed_path = Path(str(artifact_prefix) + "_signed.npz")
            np.savez_compressed(signed_path, signed_attribution=signed)
            metadata["signed_attribution_path"] = str(signed_path)
    else:
        fn = {"gradcam": gradcam_map, "gradcampp": gradcampp_map, "occlusion": occlusion_map}[method]
        saliency = fn(model, image, target_class)
    if artifact_prefix is not None:
        metadata_path = Path(str(artifact_prefix) + "_metadata.json")
        metadata_path.write_text(json.dumps(json_safe(metadata), indent=2), encoding="utf-8")
    return saliency, metadata


if RUN_VALIDATION_XAI:
    # ------------------------------------------------------------
    # Balanced diagnostic sample selection from Fold-1 Validation.
    # Prefer TP/TN/FP/FN close to threshold.
    # ------------------------------------------------------------
    fold1_pred = pd.read_csv(
        FINAL_ROOT / "fold_1" / "validation_predictions.csv"
    )
    fold1_threshold = float(fold_thresholds[1])

    fold1_pred["prediction"] = (
        fold1_pred["probability_pneumonia"].to_numpy(dtype=float)
        >= fold1_threshold
    ).astype(int)

    fold1_pred["category"] = np.select(
        [
            (fold1_pred["label"] == 1)
            & (fold1_pred["prediction"] == 1),
            (fold1_pred["label"] == 0)
            & (fold1_pred["prediction"] == 0),
            (fold1_pred["label"] == 0)
            & (fold1_pred["prediction"] == 1),
            (fold1_pred["label"] == 1)
            & (fold1_pred["prediction"] == 0),
        ],
        ["TP", "TN", "FP", "FN"],
        default="OTHER",
    )

    fold1_pred["distance_to_threshold"] = np.abs(
        fold1_pred["probability_pneumonia"]
        - fold1_threshold
    )

    parts = []
    each = max(1, XAI_SAMPLES // 4)

    for category in ("TP", "TN", "FP", "FN"):
        g = fold1_pred[
            fold1_pred["category"] == category
        ]
        if not g.empty:
            parts.append(
                g.sort_values(
                    "distance_to_threshold"
                ).head(each)
            )

    selected = (
        pd.concat(parts, ignore_index=True)
        if parts
        else fold1_pred.sort_values(
            "distance_to_threshold"
        ).head(XAI_SAMPLES)
    )

    if len(selected) < XAI_SAMPLES:
        used = set(
            selected["relative_path"].astype(str)
        )
        extra = (
            fold1_pred[
                ~fold1_pred["relative_path"]
                .astype(str)
                .isin(used)
            ]
            .sort_values("distance_to_threshold")
            .head(XAI_SAMPLES - len(selected))
        )
        selected = pd.concat(
            [selected, extra],
            ignore_index=True,
        )

    selected = selected.head(XAI_SAMPLES)
    selected.to_csv(
        XAI_ROOT / "selected_samples.csv",
        index=False,
    )

    # Rebuild the exact Fold-1 model.
    tf.keras.backend.clear_session()
    gc.collect()

    xai_model, _ = build_m07(
        dropout=float(shared_params["dropout"]),
        seed=SEED + 1,
    )
    xai_model.load_weights(
        FINAL_ROOT / "fold_1" / "best.weights.h5"
    )

    xai_rows = []

    for _, row in selected.iterrows():
        relative_path = str(row["relative_path"])
        filepath = DATA_ROOT / relative_path
        image = load_model_input(filepath)

        probability = float(
            xai_model.predict(
                image[None, ...],
                verbose=0,
            ).reshape(-1)[0]
        )
        target_class = int(
            probability >= fold1_threshold
        )

        sample_id = xai_sample_id(relative_path)
        sample_dir = XAI_ROOT / sample_id
        sample_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        for method in XAI_METHODS:
            started = time.time()

            saliency, attribution_metadata = compute_xai_map(
                xai_model, image, target_class, method,
                artifact_prefix=sample_dir / method,
            )

            np.save(
                sample_dir / f"{method}.npy",
                saliency,
            )

            visual = sample_dir / f"{method}.png"
            save_xai_visual(
                image,
                saliency,
                visual,
                (
                    f"M07 {attribution_metadata['method_display_name']} | "
                    f"{row['category']} | "
                    f"true={int(row['label'])} | "
                    f"p={probability:.4f} | explained_class={target_class}"
                ),
            )

            xai_rows.append({
                "sample_id": sample_id,
                "relative_path": relative_path,
                "true_label": int(row["label"]),
                "category": str(row["category"]),
                "probability_pneumonia": probability,
                "fold_threshold": fold1_threshold,
                "target_class": target_class,
                **attribution_metadata,
                "runtime_seconds": float(
                    time.time() - started
                ),
                "visual_path": str(visual),
            })

    xai_table = pd.DataFrame(xai_rows)
    xai_table.to_csv(
        XAI_ROOT / "xai_results.csv",
        index=False,
    )

    xai_summary.update({
        "status": "COMPLETE",
        "n_selected": int(len(selected)),
        "n_visuals": int(len(xai_table)),
    })

    (XAI_ROOT / "xai_summary.json").write_text(
        json.dumps(
            json_safe(xai_summary),
            indent=2,
        ),
        encoding="utf-8",
    )

    del xai_model
    tf.keras.backend.clear_session()
    gc.collect()

    print("✅ XAI complete:", XAI_METHODS)
    print("Validation samples:", len(selected))
    print("PNG visuals:", len(xai_table))
else:
    (XAI_ROOT / "xai_summary.json").write_text(
        json.dumps(
            json_safe(xai_summary),
            indent=2,
        ),
        encoding="utf-8",
    )

In [ ]:
# ============================================================
# 14B) M07 MODEL-INTERNAL VISUALS
#      EdgeBlock + CBAM
# ============================================================

INTERNAL_ROOT = OUTPUT_ROOT / "MODEL_INTERNAL_VISUALS"
EDGE_VIS_ROOT = INTERNAL_ROOT / "EDGE_BLOCK"
CBAM_VIS_ROOT = INTERNAL_ROOT / "CBAM"

EDGE_VIS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
CBAM_VIS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if globals().get("RUN_MODEL_INTERNAL_VISUALS", RUN_VALIDATION_XAI):
    # Use pre-selected Fold-1 Validation XAI cases if available.
    selected_path = (
        XAI_ROOT
        / "selected_samples.csv"
    )

    if RUN_VALIDATION_XAI and selected_path.is_file():
        internal_samples = pd.read_csv(
            selected_path
        ).head(
            MODEL_INTERNAL_VISUAL_SAMPLES
        )
    else:
        internal_samples = (
            pd.read_csv(
                FINAL_ROOT
                / "fold_1"
                / "validation_predictions.csv"
            )
            .head(
                MODEL_INTERNAL_VISUAL_SAMPLES
            )
        )

    tf.keras.backend.clear_session()
    gc.collect()

    internal_model, _ = build_m07(
        dropout=float(
            shared_params["dropout"]
        ),
        seed=SEED + 1,
    )
    internal_model.load_weights(
        FINAL_ROOT
        / "fold_1"
        / "best.weights.h5"
    )

    edge_layer = internal_model.get_layer(
        "edge_block"
    )
    cbam_layer = internal_model.get_layer(
        "cbam"
    )

    pre_cbam_model = tf.keras.Model(
        internal_model.inputs,
        cbam_layer.input,
    )
    post_cbam_model = tf.keras.Model(
        internal_model.inputs,
        cbam_layer.output,
    )

    def edge_internal_maps(
        image,
    ):
        x = tf.convert_to_tensor(
            image[None, ...],
            dtype=tf.float32,
        )
        gray = tf.image.rgb_to_grayscale(
            x
        ) / 255.0

        sobel = tf.image.sobel_edges(
            gray
        )
        sobel_y = tf.abs(
            sobel[..., 0]
        )
        sobel_x = tf.abs(
            sobel[..., 1]
        )

        lap_kernel = tf.constant(
            [
                [0.0, 1.0, 0.0],
                [1.0, -4.0, 1.0],
                [0.0, 1.0, 0.0],
            ],
            dtype=tf.float32,
        )
        lap_kernel = tf.reshape(
            lap_kernel,
            [3, 3, 1, 1],
        )

        lap = tf.abs(
            tf.nn.conv2d(
                gray,
                lap_kernel,
                strides=1,
                padding="SAME",
            )
        )

        edge_input = tf.concat([
            edge_layer._normalize_per_image(
                sobel_x
            ),
            edge_layer._normalize_per_image(
                sobel_y
            ),
            edge_layer._normalize_per_image(
                lap
            ),
        ], axis=-1)

        edge_delta = (
            tf.cast(
                edge_layer.conv2(
                    edge_layer.conv1(
                        edge_input
                    )
                ),
                tf.float32,
            )
            * 255.0
        )

        gate = (
            tf.nn.sigmoid(
                tf.cast(
                    edge_layer.raw_gate,
                    tf.float32,
                )
            )
            * edge_layer.max_gate
        )

        output = tf.clip_by_value(
            x + gate * edge_delta,
            0.0,
            255.0,
        )

        return {
            "sobel_x": normalize_map(
                sobel_x[0, ..., 0].numpy()
            ),
            "sobel_y": normalize_map(
                sobel_y[0, ..., 0].numpy()
            ),
            "laplacian": normalize_map(
                lap[0, ..., 0].numpy()
            ),
            "edge_delta": normalize_map(
                tf.reduce_mean(
                    tf.abs(
                        edge_delta[0]
                    ),
                    axis=-1,
                ).numpy()
            ),
            "gate": float(gate.numpy()),
            "gated_output": output[
                0
            ].numpy(),
        }

    def cbam_internal_maps(
        image,
    ):
        x = tf.convert_to_tensor(
            image[None, ...],
            dtype=tf.float32,
        )

        before = tf.cast(
            pre_cbam_model(
                x,
                training=False,
            ),
            tf.float32,
        )

        avg_pool = tf.reduce_mean(
            before,
            axis=[1, 2],
        )
        max_pool = tf.reduce_max(
            before,
            axis=[1, 2],
        )

        channel_attention = tf.nn.sigmoid(
            cbam_layer.fc2(
                cbam_layer.fc1(
                    avg_pool
                )
            )
            + cbam_layer.fc2(
                cbam_layer.fc1(
                    max_pool
                )
            )
        )

        after_channel = (
            before
            * channel_attention[
                :, None, None, :
            ]
        )

        spatial_avg = tf.reduce_mean(
            after_channel,
            axis=-1,
            keepdims=True,
        )
        spatial_max = tf.reduce_max(
            after_channel,
            axis=-1,
            keepdims=True,
        )

        spatial_attention = cbam_layer.spatial(
            tf.concat(
                [
                    spatial_avg,
                    spatial_max,
                ],
                axis=-1,
            )
        )

        after = tf.cast(
            post_cbam_model(
                x,
                training=False,
            ),
            tf.float32,
        )

        before_energy = tf.reduce_mean(
            tf.abs(before),
            axis=-1,
        )[0]

        after_energy = tf.reduce_mean(
            tf.abs(after),
            axis=-1,
        )[0]

        return {
            "channel_attention": channel_attention[
                0
            ].numpy(),
            "spatial_attention": tf.image.resize(
                spatial_attention,
                image.shape[:2],
            )[0, ..., 0].numpy(),
            "before_energy": tf.image.resize(
                before_energy[
                    ..., None
                ],
                image.shape[:2],
            )[..., 0].numpy(),
            "after_energy": tf.image.resize(
                after_energy[
                    ..., None
                ],
                image.shape[:2],
            )[..., 0].numpy(),
        }

    internal_rows = []

    for _, row in internal_samples.iterrows():
        relative_path = str(
            row["relative_path"]
        )
        filepath = DATA_ROOT / relative_path
        image = load_model_input(
            filepath
        )
        sample_id = xai_sample_id(relative_path)

        edge_maps = edge_internal_maps(
            image
        )
        cbam_maps = cbam_internal_maps(
            image
        )

        # Edge montage.
        fig, axes = plt.subplots(
            2,
            3,
            figsize=(13, 8),
        )
        axes = axes.ravel()

        axes[0].imshow(
            np.clip(
                image / 255.0,
                0,
                1,
            )
        )
        axes[0].set_title("Original")

        axes[1].imshow(
            edge_maps["sobel_x"],
            cmap="gray",
        )
        axes[1].set_title("Sobel X")

        axes[2].imshow(
            edge_maps["sobel_y"],
            cmap="gray",
        )
        axes[2].set_title("Sobel Y")

        axes[3].imshow(
            edge_maps["laplacian"],
            cmap="gray",
        )
        axes[3].set_title("Laplacian")

        axes[4].imshow(
            edge_maps["edge_delta"],
            cmap="jet",
        )
        axes[4].set_title(
            f"Edge delta | gate={edge_maps['gate']:.4f}"
        )

        axes[5].imshow(
            np.clip(
                edge_maps["gated_output"]
                / 255.0,
                0,
                1,
            )
        )
        axes[5].set_title("Gated RGB output")

        for ax in axes:
            ax.axis("off")

        fig.suptitle(
            f"M07 EdgeBlock — {sample_id}"
        )
        fig.tight_layout()
        edge_path = (
            EDGE_VIS_ROOT
            / f"{sample_id}_EDGE_BLOCK.png"
        )
        fig.savefig(
            edge_path,
            dpi=160,
        )
        plt.close(fig)

        # CBAM montage.
        fig, axes = plt.subplots(
            2,
            2,
            figsize=(10, 9),
        )
        axes = axes.ravel()

        axes[0].plot(
            np.sort(
                cbam_maps[
                    "channel_attention"
                ]
            )
        )
        axes[0].set_title(
            "Channel attention — sorted"
        )
        axes[0].set_xlabel(
            "Channel rank"
        )
        axes[0].set_ylabel(
            "Attention"
        )

        axes[1].imshow(
            cbam_maps[
                "spatial_attention"
            ],
            cmap="jet",
        )
        axes[1].set_title(
            "Spatial attention"
        )
        axes[1].axis("off")

        axes[2].imshow(
            normalize_map(
                cbam_maps[
                    "before_energy"
                ]
            ),
            cmap="magma",
        )
        axes[2].set_title(
            "Before CBAM (individually normalized)"
        )
        axes[2].axis("off")

        axes[3].imshow(
            normalize_map(
                cbam_maps[
                    "after_energy"
                ]
            ),
            cmap="magma",
        )
        axes[3].set_title(
            "After CBAM (individually normalized)"
        )
        axes[3].axis("off")

        fig.suptitle(
            f"M07 CBAM — {sample_id}"
        )
        fig.tight_layout()
        cbam_path = (
            CBAM_VIS_ROOT
            / f"{sample_id}_CBAM.png"
        )
        fig.savefig(
            cbam_path,
            dpi=160,
        )
        plt.close(fig)

        internal_rows.append({
            "sample_id": sample_id,
            "relative_path": relative_path,
            "edge_gate": edge_maps[
                "gate"
            ],
            "edge_visual": str(
                edge_path
            ),
            "cbam_visual": str(
                cbam_path
            ),
            "cbam_channel_attention_mean": float(
                np.mean(
                    cbam_maps[
                        "channel_attention"
                    ]
                )
            ),
            "cbam_channel_attention_std": float(
                np.std(
                    cbam_maps[
                        "channel_attention"
                    ]
                )
            ),
            "energy_visualization_limitation": "Per-map normalization removes scale; compare patterns only.",
            "cbam_spatial_attention_mean": float(
                np.mean(
                    cbam_maps[
                        "spatial_attention"
                    ]
                )
            ),
        })

    pd.DataFrame(
        internal_rows
    ).to_csv(
        INTERNAL_ROOT
        / "M07_MODEL_INTERNAL_VISUALS_INDEX.csv",
        index=False,
    )

    del internal_model
    tf.keras.backend.clear_session()
    gc.collect()

    print("✅ EdgeBlock visuals:", EDGE_VIS_ROOT)
    print("✅ CBAM visuals:", CBAM_VIS_ROOT)
else:
    print("Model-internal visuals skipped by configuration.")


In [ ]:
# ============================================================
# 14C) LOCKED-TEST MISCLASSIFICATION CASEBOOK
#      Post-hoc descriptive analysis only.
#      Does NOT modify model/thresholds.
# ============================================================

CASEBOOK_ROOT = OUTPUT_ROOT / "CASEBOOK"
FP_ROOT = CASEBOOK_ROOT / "FALSE_POSITIVES"
FN_ROOT = CASEBOOK_ROOT / "FALSE_NEGATIVES"

FP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
FN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

def select_representative_fold(per_fold_centered, ensemble_centered, ensemble_class):
    """Select nearest fold only among decisions agreeing with the ensemble."""
    centered = np.asarray(per_fold_centered, dtype=float)
    ensemble_centered = float(ensemble_centered)
    if centered.ndim != 1 or not len(centered) or not np.isfinite(centered).all():
        raise ValueError("Representative fold requires a finite nonempty score vector")
    if not np.isfinite(ensemble_centered) or int(ensemble_class) not in (0, 1):
        raise ValueError("Invalid ensemble decision")
    ensemble_class = int(ensemble_class)
    if ensemble_class != int(ensemble_centered >= 0.0):
        raise ValueError("Ensemble score and recorded primary decision disagree")
    agreeing = np.flatnonzero((centered >= 0.0).astype(int) == ensemble_class)
    if not len(agreeing):
        raise ValueError("No fold agrees with the ensemble decision")
    distance = np.abs(centered[agreeing] - ensemble_centered)
    return int(agreeing[int(np.argmin(distance))]) + 1


CASEBOOK_XAI_LIMITATION = (
    "Representative fold explanation, not ensemble attribution; the attributed "
    "class agrees with the ensemble. Descriptive post-hoc cases only."
)

if globals().get("RUN_TEST_CASEBOOK", RUN_VALIDATION_XAI):
    casebook = test_pred.copy()

    casebook["true_label"] = y_test
    casebook["primary_prediction"] = primary_pred
    casebook["primary_score"] = ensemble_score

    casebook["error_type"] = np.select(
        [
            (
                casebook["true_label"] == 0
            )
            & (
                casebook[
                    "primary_prediction"
                ] == 1
            ),
            (
                casebook["true_label"] == 1
            )
            & (
                casebook[
                    "primary_prediction"
                ] == 0
            ),
        ],
        ["FP", "FN"],
        default="CORRECT",
    )

    # Confidence in wrong primary classification.
    casebook[
        "wrong_decision_confidence"
    ] = np.abs(
        casebook["primary_score"]
    )

    selected_fp = (
        casebook[
            casebook["error_type"]
            == "FP"
        ]
        .sort_values(
            "wrong_decision_confidence",
            ascending=False,
        )
        .head(
            CASEBOOK_PER_ERROR_CLASS
        )
    )

    selected_fn = (
        casebook[
            casebook["error_type"]
            == "FN"
        ]
        .sort_values(
            "wrong_decision_confidence",
            ascending=False,
        )
        .head(
            CASEBOOK_PER_ERROR_CLASS
        )
    )

    selected_cases = pd.concat(
        [
            selected_fp,
            selected_fn,
        ],
        ignore_index=True,
    )

    selected_cases.to_csv(
        CASEBOOK_ROOT
        / "M07_CASEBOOK_SELECTED_ERRORS.csv",
        index=False,
    )

    case_rows = []

    # Require agreement with the ensemble class, then choose the nearest fold.
    # This remains representative single-model XAI, not an ensemble attribution.
    for _, row in selected_cases.iterrows():
        relative_path = str(
            row["relative_path"]
        )
        image = load_model_input(
            DATA_ROOT / relative_path
        )

        matching_rows = np.flatnonzero(
            locked_test["relative_path"].astype(str).to_numpy() == relative_path
        )
        if len(matching_rows) != 1:
            raise ValueError(f"Casebook sample identity is ambiguous: {relative_path}")
        row_idx = int(matching_rows[0])

        per_fold_centered = np.asarray([
            logit_np(
                P[i, row_idx]
            )
            - logit_np(
                fold_thresholds[
                    i + 1
                ]
            )
            for i in range(len(P))
        ])

        representative_fold = select_representative_fold(
            per_fold_centered,
            ensemble_score[row_idx],
            int(row["primary_prediction"]),
        )

        tf.keras.backend.clear_session()
        gc.collect()

        case_model, _ = build_m07(
            dropout=float(
                shared_params[
                    "dropout"
                ]
            ),
            seed=SEED
            + representative_fold,
        )
        case_model.load_weights(
            FINAL_ROOT
            / f"fold_{representative_fold}"
            / "best.weights.h5"
        )

        representative_probability = float(
            case_model.predict(
                image[None, ...],
                verbose=0,
            ).reshape(-1)[0]
        )

        representative_prediction = int(
            representative_probability >= fold_thresholds[representative_fold]
        )
        target_class = int(row["primary_prediction"])
        representative_agrees_with_ensemble = representative_prediction == target_class
        if not representative_agrees_with_ensemble:
            raise ValueError("Reloaded representative model no longer agrees with the ensemble")
        cached_probability = float(P[representative_fold - 1, row_idx])
        if not np.isclose(representative_probability, cached_probability, rtol=1e-5, atol=5e-6):
            raise ValueError("Reloaded representative probability differs from locked-test evidence")

        out_root = (
            FP_ROOT
            if row["error_type"] == "FP"
            else FN_ROOT
        )
        sample_id = xai_sample_id(relative_path)
        sample_dir = (
            out_root / sample_id
        )
        sample_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        maps = {}
        map_metadata = {}
        for method in ("gradcam", "gradcampp", "integrated_gradients", "occlusion"):
            maps[method], map_metadata[method] = compute_xai_map(
                case_model, image, target_class, method,
                artifact_prefix=sample_dir / method,
            )
            np.save(sample_dir / f"{method}.npy", maps[method])

        # Individual images.
        for method, saliency in maps.items():
            save_xai_visual(
                image,
                saliency,
                sample_dir
                / f"{method}.png",
                (
                    f"M07 {map_metadata[method]['method_display_name']} | {row['error_type']} | "
                    f"true={int(row['true_label'])} | "
                    f"ensemble_score={float(row['primary_score']):.4f} | "
                    f"meanP={float(row['mean_probability']):.4f} | "
                    f"representative_fold={representative_fold} | explained_class={target_class} | "
                    "agrees_with_ensemble=True\nRepresentative fold only; not ensemble attribution"
                ),
            )

        # Compact casebook montage.
        fig, axes = plt.subplots(
            1,
            5,
            figsize=(19, 4),
        )

        axes[0].imshow(
            np.clip(
                image / 255.0,
                0,
                1,
            )
        )
        axes[0].set_title(
            (
                f"Original\n"
                f"{row['error_type']} | "
                f"meanP={float(row['mean_probability']):.3f}"
            )
        )
        axes[0].axis("off")

        for ax, (
            method,
            saliency,
        ) in zip(
            axes[1:],
            maps.items(),
        ):
            ax.imshow(
                np.clip(
                    image / 255.0,
                    0,
                    1,
                )
            )
            ax.imshow(
                saliency,
                cmap="jet",
                alpha=0.42,
            )
            ax.set_title(map_metadata[method]["method_display_name"])
            ax.axis("off")

        fig.suptitle(
            (
                f"M07 Locked-Test {row['error_type']} | "
                f"True={int(row['true_label'])} | "
                f"Pred={int(row['primary_prediction'])} | "
                f"RepFold={representative_fold} | explained_class={target_class} | agreement=True\n"
                "Representative fold only; not ensemble attribution"
            )
        )
        fig.tight_layout()
        montage_path = (
            sample_dir
            / "CASEBOOK_MONTAGE.png"
        )
        fig.savefig(
            montage_path,
            dpi=160,
        )
        plt.close(fig)

        case_rows.append({
            "relative_path": relative_path,
            "error_type": row[
                "error_type"
            ],
            "true_label": int(
                row["true_label"]
            ),
            "primary_prediction": int(
                row[
                    "primary_prediction"
                ]
            ),
            "normalized_ensemble_score": float(
                row["primary_score"]
            ),
            "mean_probability": float(
                row[
                    "mean_probability"
                ]
            ),
            "wrong_decision_confidence": float(
                row[
                    "wrong_decision_confidence"
                ]
            ),
            "representative_fold_for_xai": representative_fold,
            "representative_fold_probability": representative_probability,
            "explained_target_class": target_class,
            "representative_prediction": representative_prediction,
            "representative_agrees_with_ensemble": representative_agrees_with_ensemble,
            "xai_limitation": CASEBOOK_XAI_LIMITATION,
            "gradcampp_is_approximation": True,
            "ig_visualization_is_magnitude": True,
            "ig_completeness_residual": map_metadata["integrated_gradients"]["completeness_residual"],
            "ig_signed_attribution_path": map_metadata["integrated_gradients"].get("signed_attribution_path"),
            "montage_path": str(
                montage_path
            ),
        })

        del case_model
        tf.keras.backend.clear_session()
        gc.collect()

    pd.DataFrame(
        case_rows
    ).to_csv(
        CASEBOOK_ROOT
        / "M07_CASEBOOK_INDEX.csv",
        index=False,
    )

    print(
        "✅ Casebook FP:",
        len(selected_fp),
        "| FN:",
        len(selected_fn),
    )
else:
    print("Locked-test casebook skipped by configuration.")


In [ ]:
# ============================================================
# 15) EXTERNAL VALIDATION — FROZEN MODEL, ZERO TUNING
#
# Optional:
#   M07_EXTERNAL_MANIFEST=/path/to/external.csv
# columns:
#   filepath,label,patient_id (real, non-missing grouping IDs required)
#
# Default:
#   NIH ChestX-ray14 sample cross-domain SENTINEL.
# ============================================================

EXT_ROOT = OUTPUT_ROOT / "EXTERNAL_VALIDATION"
EXT_ROOT.mkdir(parents=True, exist_ok=True)

external_reports = []

def sha256_external_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def bootstrap_external_fixed_rule(
    y,
    pred,
    score,
    patient_ids,
    n_boot=EXTERNAL_BOOTSTRAPS,
    seed=SEED,
):
    boot = bootstrap_fixed_rule_by_patient(
        y=y,
        pred=pred,
        score=score,
        patient_ids=patient_ids,
        n_boot=n_boot,
        seed=seed,
    )

    ci = {}

    for column in boot.columns:
        ci[column] = {
            "ci95_low": float(
                boot[column].quantile(
                    0.025
                )
            ),
            "ci95_high": float(
                boot[column].quantile(
                    0.975
                )
            ),
            "bootstrap_n": int(
                len(boot)
            ),
            "bootstrap_unit": "PATIENT_CLUSTER",
        }

    return ci, boot

def validate_external_manifest(cohort_name, frame):
    """Validate an explicit patient-grouped cohort without silent exclusions."""
    frame = frame.copy().reset_index(drop=True)
    required = {"filepath", "label", "patient_id"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{cohort_name}: missing required columns {sorted(missing)}; "
                         "patient IDs cannot be invented for patient-cluster intervals.")
    if frame.empty or frame[list(required)].isna().any().any():
        raise ValueError(f"{cohort_name}: empty cohort or missing path, label, or patient ID.")
    frame["patient_id"] = frame["patient_id"].astype(str).str.strip()
    invalid_ids = frame["patient_id"].str.casefold().isin(
        {"", "nan", "none", "null", "<na>", "unknown", "n/a"}
    )
    if invalid_ids.any():
        raise ValueError(f"{cohort_name}: invalid or unknown patient grouping IDs.")
    labels = pd.to_numeric(frame["label"], errors="raise")
    if not labels.isin([0, 1]).all():
        raise ValueError(f"{cohort_name}: every label must be exactly 0 or 1.")
    frame["label"] = labels.astype(int)
    if frame["label"].nunique() != 2:
        raise ValueError(f"{cohort_name}: external cohort must contain both classes.")
    frame["filepath"] = frame["filepath"].astype(str).map(
        lambda value: str(Path(value).expanduser().resolve())
    )
    if frame["filepath"].duplicated().any():
        raise ValueError(f"{cohort_name}: duplicate image paths are not allowed.")
    missing_files = [value for value in frame["filepath"] if not Path(value).is_file()]
    if missing_files:
        raise FileNotFoundError(f"{cohort_name}: missing images: {missing_files[:5]}")
    external_sha = [sha256_external_file(Path(value)) for value in frame["filepath"]]
    if "sha256" in frame.columns:
        if frame["sha256"].isna().any() or frame["sha256"].astype(str).tolist() != external_sha:
            raise ValueError(f"{cohort_name}: current image bytes differ from declared SHA256.")
    frame["sha256"] = external_sha
    duplicate_sha = frame["sha256"].duplicated(keep=False)
    if duplicate_sha.any():
        conflicting = frame.loc[duplicate_sha].groupby("sha256")["label"].nunique().gt(1).any()
        detail = "conflicting labels for identical images" if conflicting else "duplicate image bytes"
        raise ValueError(f"{cohort_name}: {detail}; resolve cohort duplicates before evaluation.")
    kermany_sha = set(development_manifest["sha256"].astype(str))
    kermany_sha.update(locked_test["sha256"].astype(str))
    overlap = set(external_sha) & kermany_sha
    if overlap:
        raise ValueError(f"{cohort_name}: exact SHA overlap with Kermany = {len(overlap)}")
    print(f"✅ {cohort_name}: byte hashes verified; internal duplicates and Kermany overlap = 0")
    return frame


def evaluate_external_frame(cohort_name, frame, cohort_kind):
    frame = validate_external_manifest(cohort_name, frame)
    external_checkpoint_hashes = {
        str(fold): run_file_sha256(FINAL_ROOT / f"fold_{fold}" / "best.weights.h5")
        for fold in range(1, 6)
    }
    external_contract = make_run_contract(
        "external_validation", model_id=MODEL_ID, resolution=IMAGE_SIZE,
        params=shared_params,
        extra={
            "cohort": cohort_name, "cohort_kind": cohort_kind,
            "manifest_identity": dataframe_identity(frame),
            "checkpoint_sha256": external_checkpoint_hashes,
            "fold_thresholds": {str(fold): float(fold_thresholds[fold]) for fold in range(1, 6)},
            "recipe_fingerprint": recipe["recipe_fingerprint_sha256"],
            "bootstrap_unit": "PATIENT_CLUSTER", "bootstrap_n": EXTERNAL_BOOTSTRAPS,
            "primary_aggregation": "threshold_normalized_five_fold_ensemble",
        },
    )
    ext_ds = build_plain_eval_dataset(
        frame,
        batch_size=int(shared_params["batch_size"]),
    )

    fold_probs = []

    for fold in range(1, 6):
        print(
            f"{cohort_name}: Fold {fold}/5 inference"
        )

        tf.keras.backend.clear_session()
        gc.collect()

        model, _ = build_m07(
            dropout=float(shared_params["dropout"]),
            seed=SEED + fold,
        )
        model.load_weights(
            FINAL_ROOT
            / f"fold_{fold}"
            / "best.weights.h5"
        )

        probabilities = model.predict(ext_ds, verbose=0).reshape(-1)
        if len(probabilities) != len(frame) or not np.isfinite(probabilities).all() or (
            (probabilities < 0) | (probabilities > 1)
        ).any():
            raise ValueError(f"{cohort_name}: invalid probabilities from fold {fold}.")
        fold_probs.append(probabilities)

        del model
        tf.keras.backend.clear_session()
        gc.collect()

    P_ext = np.vstack(fold_probs)
    # Freeze auditable provenance for this evaluation, including current bytes.
    validate_external_manifest(cohort_name, frame)
    for fold in range(1, 6):
        checkpoint = FINAL_ROOT / f"fold_{fold}" / "best.weights.h5"
        if run_file_sha256(checkpoint) != external_checkpoint_hashes[str(fold)]:
            raise ValueError(f"{cohort_name}: checkpoint changed during external inference.")

    normalized_ext = np.vstack([
        logit_np(P_ext[i])
        - logit_np(fold_thresholds[i + 1])
        for i in range(5)
    ])

    ensemble_score_ext = normalized_ext.mean(
        axis=0
    )
    pred_ext = (
        ensemble_score_ext >= 0.0
    ).astype(int)
    mean_probability_ext = P_ext.mean(axis=0)
    y_ext = frame["label"].to_numpy(dtype=int)

    metrics_ext = calculate_binary_metrics(
        y_ext,
        pred_ext,
        score=ensemble_score_ext,
    )

    dual_ext, dual_decision_ext = analyze_dual_threshold(
        y_ext,
        mean_probability_ext,
    )

    cohort_dir = EXT_ROOT / cohort_name
    cohort_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    predictions = frame.copy()
    for i, fold in enumerate(range(1, 6)):
        predictions[
            f"probability_fold_{fold}"
        ] = P_ext[i]
        predictions[
            f"threshold_fold_{fold}"
        ] = fold_thresholds[fold]

    predictions[
        "mean_probability"
    ] = mean_probability_ext
    predictions[
        "normalized_ensemble_score"
    ] = ensemble_score_ext
    predictions[
        "prediction_primary"
    ] = pred_ext
    predictions[
        "dual20_80_decision"
    ] = dual_decision_ext

    predictions.to_csv(
        cohort_dir / "external_predictions.csv",
        index=False,
    )

    (cohort_dir / "external_metrics.json").write_text(
        json.dumps(
            json_safe(metrics_ext),
            indent=2,
        ),
        encoding="utf-8",
    )

    (cohort_dir / "external_dual20_80_report.json").write_text(
        json.dumps(
            json_safe(dual_ext),
            indent=2,
        ),
        encoding="utf-8",
    )

    ci, boot = bootstrap_external_fixed_rule(
        y_ext,
        pred_ext,
        ensemble_score_ext,
        frame["patient_id"].astype(str).to_numpy(),
    )
    boot.to_csv(
        cohort_dir / "bootstrap_samples.csv",
        index=False,
    )

    for metric_name in ci:
        if metric_name in metrics_ext:
            ci[metric_name]["point"] = float(
                metrics_ext[metric_name]
            )

    (cohort_dir / "bootstrap_ci95.json").write_text(
        json.dumps(
            json_safe(ci),
            indent=2,
        ),
        encoding="utf-8",
    )

    save_confusion_figure(
        metrics_ext["confusion_matrix"],
        f"M07 External — {cohort_name}",
        cohort_dir / "external_confusion_matrix.png",
    )
    save_roc_figure(
        y_ext,
        ensemble_score_ext,
        f"M07 External ROC — {cohort_name}",
        cohort_dir / "external_roc_curve.png",
    )
    save_pr_figure(
        y_ext,
        ensemble_score_ext,
        f"M07 External Precision-Recall — {cohort_name}",
        cohort_dir / "external_pr_curve.png",
    )
    save_dual_probability_figure(
        y_ext,
        mean_probability_ext,
        f"M07 External — {cohort_name} — Fixed 20/80",
        cohort_dir / "external_dual20_80_probability.png",
    )

    report = {
        "cohort": cohort_name,
        "kind": cohort_kind,
        "n_images": int(len(frame)),
        "n_patients": int(
            frame["patient_id"].nunique()
        ),
        "class_counts": {
            str(k): int(v)
            for k, v in frame[
                "label"
            ].value_counts().sort_index().to_dict().items()
        },
        "external_kermany_sha_overlap": 0,
        "internal_duplicate_image_sha_count": 0,
        "bootstrap_unit": "PATIENT_CLUSTER",
        "patient_identity_requirement": "EXPLICIT_NONMISSING_SOURCE_GROUPING_IDS",
        "patient_group_source": (
            "NIH_OFFICIAL_PATIENT_ID" if cohort_kind == "CROSS_DOMAIN_SENTINEL_ONLY"
            else "USER_SUPPLIED_PATIENT_IDS_NOT_INDEPENDENTLY_VERIFIED"
        ),
        "threshold_tuning": False,
        "calibration_fitting": False,
        "ensemble_weight_fitting": False,
        "primary_aggregation": (
            "threshold-normalized 5-fold ensemble "
            "using frozen Kermany validation thresholds"
        ),
        "metrics": metrics_ext,
        "dual20_80": dual_ext,
        "bootstrap_ci95": ci,
    }

    report = seal_receipt({"status": "COMPLETE", **json_safe(report)}, external_contract,
                          artifacts={"predictions": cohort_dir / "external_predictions.csv"})
    atomic_write_json(cohort_dir / "external_report.json", report)

    external_reports.append(report)

    print("\n" + "=" * 90)
    print("EXTERNAL:", cohort_name)
    print("=" * 90)
    print(
        "BalAcc:",
        f"{metrics_ext['balanced_accuracy']:.4f}",
    )
    print(
        "MacroF1:",
        f"{metrics_ext['macro_f1']:.4f}",
    )
    print(
        "F2:",
        f"{metrics_ext['f2']:.4f}",
    )
    print(
        "AUROC:",
        f"{metrics_ext['auroc']:.4f}",
    )
    print(
        "Dual20/80 coverage:",
        f"{dual_ext['coverage']:.4f}",
    )

    return report

# ------------------------------------------------------------
# A) Optional full/user-supplied external validation
# ------------------------------------------------------------
if CUSTOM_EXTERNAL_MANIFEST:
    custom_path = Path(
        CUSTOM_EXTERNAL_MANIFEST
    )

    if not custom_path.is_file():
        raise FileNotFoundError(
            f"M07_EXTERNAL_MANIFEST not found: {custom_path}"
        )

    custom_frame = pd.read_csv(custom_path, dtype={"patient_id": str, "sha256": str})
    evaluate_external_frame(
        "custom_external",
        custom_frame,
        "USER_SUPPLIED_EXTERNAL",
    )

# ------------------------------------------------------------
# B) NIH sample: cross-domain SENTINEL only
# ------------------------------------------------------------
nih_status = {
    "requested": bool(RUN_NIH_SENTINEL),
    "status": "SKIPPED",
    "classification": "CROSS_DOMAIN_SENTINEL_NOT_FINAL_PEDIATRIC_EXTERNAL",
}

if RUN_NIH_SENTINEL:
    try:
        import importlib.util

        if importlib.util.find_spec(
            "kagglehub"
        ) is None:
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "kagglehub",
            ])

        import kagglehub

        def locate_nih_sample():
            # Search attached inputs first.
            for base in (
                Path("/kaggle/input"),
                Path("/kaggle/working"),
            ):
                if not base.exists():
                    continue

                for csv_path in base.rglob("*.csv"):
                    try:
                        head = pd.read_csv(
                            csv_path,
                            nrows=3,
                        )
                    except Exception:
                        continue

                    if {
                        "Image Index",
                        "Finding Labels",
                    }.issubset(
                        head.columns
                    ):
                        return (
                            csv_path.parent,
                            csv_path,
                        )

            # Fallback public sample download.
            dl = Path(
                kagglehub.dataset_download(
                    "nih-chest-xrays/sample"
                )
            )

            for csv_path in dl.rglob("*.csv"):
                try:
                    head = pd.read_csv(
                        csv_path,
                        nrows=3,
                    )
                except Exception:
                    continue

                if {
                    "Image Index",
                    "Finding Labels",
                }.issubset(
                    head.columns
                ):
                    return (
                        csv_path.parent,
                        csv_path,
                    )

            raise FileNotFoundError(
                "NIH sample labels CSV was not found."
            )

        nih_root, nih_csv = locate_nih_sample()
        nih_raw = pd.read_csv(nih_csv)

        # Filename -> one actual file.
        image_index = {}
        for pth in nih_root.rglob("*"):
            if (
                pth.is_file()
                and pth.suffix.lower()
                in {".png", ".jpg", ".jpeg", ".bmp"}
            ):
                image_index.setdefault(
                    pth.name,
                    pth,
                )

        patient_col = "Patient ID"
        if patient_col not in nih_raw.columns:
            raise ValueError("NIH sentinel requires official Patient ID metadata; "
                             "image filenames cannot substitute for patient grouping IDs.")

        rows = []
        for _, row in nih_raw.iterrows():
            filename = str(
                row["Image Index"]
            )
            path = image_index.get(filename)

            if path is None:
                continue

            labels = str(
                row["Finding Labels"]
            ).split("|")

            if "Pneumonia" in labels:
                label = 1
            elif (
                len(labels) == 1
                and labels[0].strip()
                == "No Finding"
            ):
                label = 0
            else:
                continue

            if pd.isna(row[patient_col]) or str(row[patient_col]).strip().casefold() in {
                "", "nan", "none", "null", "unknown", "<na>", "n/a"
            }:
                raise ValueError(f"NIH sentinel has a missing patient ID for {filename}.")
            patient_id = str(row[patient_col]).strip()

            rows.append({
                "filepath": str(path),
                "label": label,
                "patient_id": f"NIH_{patient_id}",
                "patient_group_source": "NIH_OFFICIAL_PATIENT_ID",
                "filename": filename,
            })

        nih = pd.DataFrame(rows)

        if (
            nih.empty
            or nih["label"].nunique() != 2
        ):
            raise RuntimeError(
                "NIH sample did not provide both classes."
            )

        # One image per patient.
        nih = (
            nih.sort_values(
                ["patient_id", "filename"]
            )
            .drop_duplicates(
                "patient_id",
                keep="first",
            )
            .reset_index(drop=True)
        )

        pos = nih[
            nih["label"] == 1
        ].copy()
        neg = nih[
            nih["label"] == 0
        ].copy()

        n = min(
            len(pos),
            len(neg),
        )

        if n < 10:
            raise RuntimeError(
                f"NIH sentinel too small: {n} per class"
            )

        pos = pos.sample(
            n=n,
            random_state=SEED,
        )
        neg = neg.sample(
            n=n,
            random_state=SEED,
        )

        sentinel = (
            pd.concat(
                [pos, neg],
                ignore_index=True,
            )
            .sample(
                frac=1.0,
                random_state=SEED,
            )
            .reset_index(drop=True)
        )

        sentinel = validate_external_manifest("nih_sample_sentinel", sentinel)
        sentinel.to_csv(
            EXT_ROOT / "NIH_SENTINEL_MANIFEST.csv",
            index=False,
        )

        fingerprint_lines = [
            (
                f"{r.patient_id}\t"
                f"{r.filename}\t"
                f"{int(r.label)}"
            )
            for r in sentinel.sort_values(
                ["patient_id", "filename"]
            ).itertuples()
        ]

        sentinel_fingerprint = hashlib.sha256(
            "\n".join(
                fingerprint_lines
            ).encode("utf-8")
        ).hexdigest()

        nih_status.update({
            "status": "READY",
            "images": int(len(sentinel)),
            "patients": int(
                sentinel["patient_id"].nunique()
            ),
            "per_class": int(n),
            "fingerprint": sentinel_fingerprint,
            "labels_csv": str(nih_csv),
        })

        report = evaluate_external_frame(
            "nih_sample_sentinel",
            sentinel,
            "CROSS_DOMAIN_SENTINEL_ONLY",
        )

        nih_status["status"] = "COMPLETE"
        nih_status["metrics"] = report[
            "metrics"
        ]
        nih_status["dual20_80"] = report[
            "dual20_80"
        ]

    except Exception as exc:
        nih_status.update({
            "status": "UNAVAILABLE_OR_FAILED",
            "error": repr(exc),
        })
        print(
            "⚠️ NIH sentinel unavailable/failed:",
            repr(exc),
        )

(EXT_ROOT / "NIH_SENTINEL_STATUS.json").write_text(
    json.dumps(
        json_safe(nih_status),
        indent=2,
    ),
    encoding="utf-8",
)

external_master = {
    "schema": "m07.external.validation.v1",
    "model_id": "M07",
    "prior_successful_recipe_fingerprint": PRIOR_SUCCESSFUL_RECIPE_FINGERPRINT,
    "confirmed_recipe_fingerprint": recipe["recipe_fingerprint_sha256"],
    "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
    "external_tuning": False,
    "reports": external_reports,
    "nih_sentinel": nih_status,
    "manual_gate_required": True,
}

(EXT_ROOT / "M07_EXTERNAL_MASTER_REPORT.json").write_text(
    json.dumps(
        json_safe(external_master),
        indent=2,
    ),
    encoding="utf-8",
)

print("=" * 100)
print("EXTERNAL STAGE COMPLETE")
print("=" * 100)
print(
    "Completed cohorts:",
    [r["cohort"] for r in external_reports],
)
print(
    "NIH sentinel status:",
    nih_status["status"],
)
print("✅ No external tuning performed")

# ------------------------------------------------------------
# C) External generalization summary + diagnostic figures
# ------------------------------------------------------------
external_summary_rows = []

for report in external_reports:
    cohort = report["cohort"]
    cohort_dir = (
        EXT_ROOT / cohort
    )
    pred_path = (
        cohort_dir
        / "external_predictions.csv"
    )

    if not pred_path.is_file():
        continue

    ext_pred = pd.read_csv(
        pred_path
    )
    y_ext = ext_pred[
        "label"
    ].to_numpy(dtype=int)
    p_ext = ext_pred[
        "mean_probability"
    ].to_numpy(dtype=float)
    score_ext = ext_pred[
        "normalized_ensemble_score"
    ].to_numpy(dtype=float)
    pred_ext = ext_pred[
        "prediction_primary"
    ].to_numpy(dtype=int)

    ext_ece = save_reliability_plot(
        y_ext,
        p_ext,
        f"M07 External Reliability — {cohort}",
        cohort_dir
        / "external_reliability_diagram.png",
        cohort_dir
        / "external_calibration_bins.csv",
    )

    ext_cal = {
        "ece_10_bins": float(
            ext_ece
        ),
        "brier": float(
            brier_score_loss(
                y_ext,
                p_ext,
            )
        ),
        "calibration_fit_performed": False,
    }
    (
        cohort_dir
        / "external_calibration_report.json"
    ).write_text(
        json.dumps(
            json_safe(ext_cal),
            indent=2,
        ),
        encoding="utf-8",
    )

    ext_rc = risk_coverage_table(
        y_ext,
        pred_ext,
        np.abs(
            score_ext
        ),
    )
    ext_rc.to_csv(
        cohort_dir
        / "external_risk_coverage.csv",
        index=False,
    )

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )
    ax.plot(
        ext_rc["coverage"],
        ext_rc[
            "risk_error_rate"
        ],
        marker="o",
    )
    ax.set_xlabel("Coverage")
    ax.set_ylabel(
        "Risk = 1 - Accuracy"
    )
    ax.set_title(
        f"M07 External Risk–Coverage — {cohort}"
    )
    fig.tight_layout()
    fig.savefig(
        cohort_dir
        / "external_risk_coverage.png",
        dpi=180,
    )
    plt.close(fig)

    metrics_ext = report[
        "metrics"
    ]

    external_summary_rows.append({
        "dataset": f"EXTERNAL:{cohort}",
        "kind": report["kind"],
        "n_images": report["n_images"],
        "n_patients": report["n_patients"],
        "balanced_accuracy": metrics_ext[
            "balanced_accuracy"
        ],
        "macro_precision": metrics_ext[
            "macro_precision"
        ],
        "macro_recall": metrics_ext[
            "macro_recall"
        ],
        "macro_f1": metrics_ext[
            "macro_f1"
        ],
        "mcc": metrics_ext[
            "mcc"
        ],
        "auroc": metrics_ext[
            "auroc"
        ],
        "precision_normal": metrics_ext[
            "precision_normal"
        ],
        "recall_normal": metrics_ext[
            "recall_normal"
        ],
        "precision_pneumonia": metrics_ext[
            "precision_pneumonia"
        ],
        "recall_pneumonia": metrics_ext[
            "recall_pneumonia"
        ],
        # report-only
        "f2": metrics_ext[
            "f2"
        ],
        "dual20_80_coverage": report[
            "dual20_80"
        ]["coverage"],
        "dual20_80_uncertainty": report[
            "dual20_80"
        ]["uncertainty_rate"],
        "ece": ext_cal[
            "ece_10_bins"
        ],
        "brier": ext_cal[
            "brier"
        ],
    })

external_summary_df = pd.DataFrame(
    external_summary_rows
)

if not external_summary_df.empty:
    external_summary_df.to_csv(
        EXT_ROOT
        / "M07_EXTERNAL_GENERALIZATION_SUMMARY.csv",
        index=False,
    )

    baseline_row = {
        "dataset": "LOCKED_TEST",
        "kind": "INTERNAL_LOCKED_TEST",
        "n_images": int(
            len(y_test)
        ),
        "n_patients": int(
            locked_test[
                "patient_id"
            ].nunique()
        ),
        "balanced_accuracy": primary_metrics[
            "balanced_accuracy"
        ],
        "macro_precision": primary_metrics[
            "macro_precision"
        ],
        "macro_recall": primary_metrics[
            "macro_recall"
        ],
        "macro_f1": primary_metrics[
            "macro_f1"
        ],
        "mcc": primary_metrics[
            "mcc"
        ],
        "auroc": primary_metrics[
            "auroc"
        ],
        "precision_normal": primary_metrics[
            "precision_normal"
        ],
        "recall_normal": primary_metrics[
            "recall_normal"
        ],
        "precision_pneumonia": primary_metrics[
            "precision_pneumonia"
        ],
        "recall_pneumonia": primary_metrics[
            "recall_pneumonia"
        ],
        "f2": primary_metrics[
            "f2"
        ],
        "dual20_80_coverage": test_dual_20_80[
            "coverage"
        ],
        "dual20_80_uncertainty": test_dual_20_80[
            "uncertainty_rate"
        ],
        "ece": calibration_report[
            "locked_test"
        ]["ece_10_bins"],
        "brier": calibration_report[
            "locked_test"
        ]["brier"],
    }

    combined_ext = pd.concat(
        [
            pd.DataFrame(
                [baseline_row]
            ),
            external_summary_df,
        ],
        ignore_index=True,
    )

    combined_ext.to_csv(
        EXT_ROOT
        / "M07_TEST_EXTERNAL_COMPARISON.csv",
        index=False,
    )

print("✅ External calibration/risk/generalization diagnostics complete")


In [ ]:
# ============================================================
# 16) FINAL M07 REPLICATION GATE REPORT + EVIDENCE PACKAGE
# ============================================================

remaining_model_matrix = {
    "status": "MANUALLY_UNLOCKED" if UNLOCK_REMAINING_MODELS else "LOCKED_PENDING_MANUAL_APPROVAL",
    "auto_run": bool(UNLOCK_REMAINING_MODELS),
    "models_after_gate": [
        "M01", "M02", "M03", "M04", "M05", "M06",
        "M08", "M09", "M10", "M11", "M12",
    ],
    "model_definitions": {
        "M01": "ConvNeXt-Tiny + BCE",
        "M02": "ConvNeXt-Tiny + weighted BCE",
        "M03": "ConvNeXt-Tiny + static focal",
        "M04": "ConvNeXt-Tiny + FLSD-53",
        "M05": "M04 + CBAM",
        "M06": "M04 + EdgeBlock",
        "M08": "M07 + MixUp",
        "M09": "M07 + CutMix",
        "M10": "M07 + MixUp + CutMix",
        "M11": "M10 + EMA",
        "M12": "M10 + SWA",
    },
    "planned_resolutions_after_approval": [
        224, 320, 384
    ],
    "shared_scientific_contract": {
        "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
        "patient_locked": True,
        "test_selection_allowed": False,
        "external_selection_allowed": False,
        "balanced_batch_policy": True,
        "flsd_alpha_when_balanced": 0.50,
        "f2_positive_class": "PNEUMONIA",
        "f2_beta": 2,
        "dual_threshold_low": DUAL_THRESHOLD_LOW,
        "dual_threshold_high": DUAL_THRESHOLD_HIGH,
        "dual_threshold_tuning": False,
    },
}

gate_report = {
    "schema": "m07.final.confirmation_hpo.gate.v1",
    "status": "M07_R224_ANCHOR_COMPLETE_PENDING_MULTIRES_GATE",
    "manual_decision_required": True,
    "allow_remaining_models": False,
    "model_id": MODEL_ID,
    "model_description": MODEL_DESCRIPTION,
    "image_size": IMAGE_SIZE,
    "gate_resolutions_planned": list(M07_GATE_RESOLUTIONS),
    "note": "R224 anchor complete; R320/R384 gate training continues below before Phase-2 can unlock.",
    "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
    "prior_successful_recipe_fingerprint": PRIOR_SUCCESSFUL_RECIPE_FINGERPRINT,
    "confirmed_recipe_fingerprint": recipe[
        "recipe_fingerprint_sha256"
    ],
    "shared_training_params": shared_params,
    "m07_fixed": {
        "edge_filters": EDGE_FILTERS,
        "edge_max_gate": EDGE_MAX_GATE,
        "cbam_reduction": CBAM_REDUCTION,
        "cbam_spatial_kernel": CBAM_SPATIAL_KERNEL,
        "flsd_alpha": FLSD_ALPHA,
        "flsd_threshold": FLSD_THRESHOLD,
        "flsd_hard_gamma": FLSD_HARD_GAMMA,
        "flsd_easy_gamma": FLSD_EASY_GAMMA,
    },
    "metrics": {
        "oof_primary": oof_primary_metrics,
        "locked_test_primary": primary_metrics,
        "locked_test_majority": majority_metrics,
    },
    "f2": {
        "definition": "sklearn fbeta_score beta=2, positive class PNEUMONIA",
        "oof": float(oof_primary_metrics["f2"]),
        "locked_test_primary": float(
            primary_metrics["f2"]
        ),
        "locked_test_majority": float(
            majority_metrics["f2"]
        ),
    },
    "dual20_80": {
        "policy": {
            "normal_if_probability_lte": DUAL_THRESHOLD_LOW,
            "uncertain_if_between": [
                DUAL_THRESHOLD_LOW,
                DUAL_THRESHOLD_HIGH,
            ],
            "pneumonia_if_probability_gte": DUAL_THRESHOLD_HIGH,
            "tuned": False,
        },
        "oof": oof_dual_20_80,
        "locked_test": test_dual_20_80,
    },
    "external_validation": external_master,
    "xai": xai_summary,
    "test_images": int(len(locked_test)),
    "test_patients": int(
        locked_test["patient_id"].nunique()
    ),
    "test_threshold_tuning": False,
    "external_tuning": False,
    "remaining_models": remaining_model_matrix,
}

(OUTPUT_ROOT / "M07_REPLICATION_GATE_REPORT.json").write_text(
    json.dumps(
        json_safe(gate_report),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

(OUTPUT_ROOT / "REMAINING_MODELS_LOCKED_PLAN.json").write_text(
    json.dumps(
        json_safe(remaining_model_matrix),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# Compact tabular report for quick inspection
# ------------------------------------------------------------
quick_rows = [
    {
        "dataset": "OOF",
        "balanced_accuracy": oof_primary_metrics["balanced_accuracy"],
        "macro_f1": oof_primary_metrics["macro_f1"],
        "f2": oof_primary_metrics["f2"],
        "auroc": oof_primary_metrics["auroc"],
        "recall_normal": oof_primary_metrics["recall_normal"],
        "recall_pneumonia": oof_primary_metrics["recall_pneumonia"],
        "dual20_80_coverage": oof_dual_20_80["coverage"],
        "dual20_80_uncertainty": oof_dual_20_80["uncertainty_rate"],
    },
    {
        "dataset": "LOCKED_TEST",
        "balanced_accuracy": primary_metrics["balanced_accuracy"],
        "macro_f1": primary_metrics["macro_f1"],
        "f2": primary_metrics["f2"],
        "auroc": primary_metrics["auroc"],
        "recall_normal": primary_metrics["recall_normal"],
        "recall_pneumonia": primary_metrics["recall_pneumonia"],
        "dual20_80_coverage": test_dual_20_80["coverage"],
        "dual20_80_uncertainty": test_dual_20_80["uncertainty_rate"],
    },
]

for ext_report in external_reports:
    quick_rows.append({
        "dataset": f"EXTERNAL:{ext_report['cohort']}",
        "balanced_accuracy": ext_report["metrics"]["balanced_accuracy"],
        "macro_f1": ext_report["metrics"]["macro_f1"],
        "f2": ext_report["metrics"]["f2"],
        "auroc": ext_report["metrics"]["auroc"],
        "recall_normal": ext_report["metrics"]["recall_normal"],
        "recall_pneumonia": ext_report["metrics"]["recall_pneumonia"],
        "dual20_80_coverage": ext_report["dual20_80"]["coverage"],
        "dual20_80_uncertainty": ext_report["dual20_80"]["uncertainty_rate"],
    })

quick_report = pd.DataFrame(quick_rows)
quick_report.to_csv(
    OUTPUT_ROOT / "M07_GATE_QUICK_RESULTS.csv",
    index=False,
)

print("=" * 120)
print("M07 REPLICATION GATE — QUICK RESULTS")
print("=" * 120)
display(quick_report.round(4))

# ------------------------------------------------------------
# Evidence ZIP — all report/figure/XAI/external evidence;
# deliberately exclude 5 large model weights because fold recovery
# ZIPs are already persisted separately.
# ------------------------------------------------------------
evidence_zip = WORK / "M07_REPLICATION_GATE_EVIDENCE_V17.zip"
if evidence_zip.exists():
    evidence_zip.unlink()

with zipfile.ZipFile(
    evidence_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    for p in OUTPUT_ROOT.rglob("*"):
        if not p.is_file():
            continue

        if p.name == "best.weights.h5":
            continue

        if p.name.endswith(".weights.h5"):
            continue

        z.write(
            p,
            arcname=str(
                p.relative_to(OUTPUT_ROOT)
            ),
        )

print("\n✅ Gate report:", OUTPUT_ROOT / "M07_REPLICATION_GATE_REPORT.json")
print("✅ Quick results:", OUTPUT_ROOT / "M07_GATE_QUICK_RESULTS.csv")
print("✅ Evidence ZIP:", evidence_zip)
print("✅ Remaining-model manual unlock:", bool(UNLOCK_REMAINING_MODELS))
print("✅ R224 anchor complete; R320/R384 M07 Gate training continues below before Phase-2.")

_persist_lightweight_files(
    [
        OUTPUT_ROOT / "M07_REPLICATION_GATE_REPORT.json",
        OUTPUT_ROOT / "M07_GATE_QUICK_RESULTS.csv",
        OUTPUT_ROOT / "REMAINING_MODELS_LOCKED_PLAN.json",
        evidence_zip,
    ],
    note="M07 replication gate complete; remaining models locked pending manual approval",
)

print("Phase-2 remains locked until the M07 multi-resolution gate receipt is sealed.")

# ============================================================
# 16B) FINAL MODEL CARD + HASH MANIFEST + COMPLETE EVIDENCE INDEX
# ============================================================

def sha256_file_final(
    path,
    chunk=1024 * 1024,
):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

checkpoint_hashes = {}

for fold in range(1, 6):
    ck = (
        FINAL_ROOT
        / f"fold_{fold}"
        / "best.weights.h5"
    )
    if ck.is_file():
        checkpoint_hashes[
            f"fold_{fold}"
        ] = {
            "path": str(ck),
            "sha256": sha256_file_final(
                ck
            ),
            "bytes": int(
                ck.stat().st_size
            ),
        }

important_files = [
    RECIPE_PATH,
    HPO_ROOT
    / "M07_HPO_CANDIDATE_FOLD_RESULTS.csv",
    HPO_ROOT
    / "M07_HPO_SCREENING_RANKING.csv",
    HPO_ROOT
    / "M07_HPO_CONFIRMATION_RANKING.csv",
    OUTPUT_ROOT
    / "M07_OOF_PREDICTIONS.csv",
    OUTPUT_ROOT
    / "M07_OOF_PRIMARY_METRICS.json",
    TEST_OUT
    / "M07_LOCKED_TEST_ENSEMBLE_PREDICTIONS.csv",
    TEST_OUT
    / "M07_LOCKED_TEST_PRIMARY_METRICS.json",
    TEST_OUT
    / "M07_LOCKED_TEST_BOOTSTRAP_CI95.json",
    DIAG_ROOT
    / "M07_CALIBRATION_REPORT.json",
    LOSS_ROOT
    / "M07_LOSS_SUMMARY.csv",
    CASEBOOK_ROOT
    / "M07_CASEBOOK_INDEX.csv",
    INTERNAL_ROOT
    / "M07_MODEL_INTERNAL_VISUALS_INDEX.csv",
    EXT_ROOT
    / "M07_EXTERNAL_MASTER_REPORT.json",
]

artifact_hashes = {}

for p in important_files:
    p = Path(p)
    if p.is_file():
        artifact_hashes[
            str(
                p.relative_to(
                    OUTPUT_ROOT
                )
            )
        ] = sha256_file_final(
            p
        )

model_card = {
    "schema": "m07.model.card.v1",
    "model_id": "M07",
    "architecture": (
        "Input → augmentation → EdgeBlock → "
        "ConvNeXt-Tiny ImageNet → CBAM → "
        "GAP → LayerNorm → Dropout → Sigmoid"
    ),
    "image_size": IMAGE_SIZE,
    "loss": {
        "name": "FLSD-53 binary focal loss",
        "alpha": FLSD_ALPHA,
        "threshold": FLSD_THRESHOLD,
        "hard_gamma": FLSD_HARD_GAMMA,
        "easy_gamma": FLSD_EASY_GAMMA,
        "alpha_reason": (
            "alpha=0.5 because exact balanced batches are active"
        ),
    },
    "augmentation": {
        "rotation": 0.025,
        "translation": 0.03,
        "zoom": [-0.05, 0.05],
        "contrast": 0.08,
    },
    "preprocessing": {
        "resize": "tf.image.resize_with_pad",
        "aspect_ratio_preserved": True,
        "input_pixel_contract": "0..255",
        "convnext_preprocessing_inside_model": True,
    },
    "confirmed_recipe": recipe,
    "dataset": {
        "source": "Kermany/Mooney Kaggle pediatric chest X-ray package",
        "raw_canonical_images": 5856,
        "clean_images": 5823,
        "clean_patients": 2789,
        "development_images": int(
            len(development_manifest)
        ),
        "development_patients": int(
            development_manifest[
                "patient_id"
            ].nunique()
        ),
        "locked_test_images": int(
            len(locked_test)
        ),
        "locked_test_patients": int(
            locked_test[
                "patient_id"
            ].nunique()
        ),
        "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
        "development_test_patient_overlap": 0,
        "development_test_exact_sha_overlap": 0,
        "exact_duplicate_policy": (
            "Exact duplicates removed/quarantined before frozen split"
        ),
        "near_duplicate_policy": (
            "Audit only; pHash candidates exhaustively SSIM-checked; "
            "no strong SSIM>=0.95 evidence"
        ),
    },
    "threshold_policy": {
        "validation": (
            "Per-fold threshold selected on Validation with "
            "precision/recall-first balanced rank"
        ),
        "locked_test_primary": (
            "threshold-normalized five-fold ensemble"
        ),
        "locked_test_secondary": (
            "majority vote using each fold's Validation threshold"
        ),
        "locked_test_threshold_tuning": False,
    },
    "f2_policy": {
        "beta": 2,
        "positive_class": "PNEUMONIA",
        "role": "REPORT_ONLY",
        "used_for_selection": False,
    },
    "dual_threshold_policy": {
        "low": DUAL_THRESHOLD_LOW,
        "high": DUAL_THRESHOLD_HIGH,
        "role": "REPORT_ONLY",
        "tuned": False,
    },
    "external_policy": {
        "adaptation": False,
        "threshold_tuning": False,
        "calibration_fitting": False,
        "weight_update": False,
        "ensemble_weight_fitting": False,
        "nih_fallback_label": (
            "CROSS_DOMAIN_SENTINEL_NOT_FINAL_PEDIATRIC_EXTERNAL"
        ),
    },
    "runtime": {
        "tensorflow": tf.__version__,
        "sklearn": sklearn.__version__,
        "mixed_precision_policy": str(
            mixed_precision.global_policy()
        ),
        "gpu_devices": [
            str(d)
            for d in tf.config.list_physical_devices(
                "GPU"
            )
        ],
        "seed_base": SEED,
        "hpo_candidate_seed": HPO_CANDIDATE_SEED,
    },
    "checkpoint_hashes": checkpoint_hashes,
    "artifact_hashes": artifact_hashes,
    "known_limitations": [
        (
            "OOF is a Development estimate, not the final independent generalization estimate: "
            "Confirmation HPO uses Development Folds 1–3 and final OOF reuses Development. "
            "The independent Locked Test is the primary final internal generalization evidence."
        ),
        (
            "OOF classification metrics also use thresholds selected on each corresponding "
            "Validation Fold; AUROC/AUPRC are threshold-independent."
        ),
        (
            "NIH sample, when used, is a cross-domain sentinel and is not "
            "a final independent pediatric external validation cohort."
        ),
        (
            "External validation is inference-only; no adaptation is permitted."
        ),
        (
            "Seed-stability full repeated 5-fold training is deferred to the "
            "later dedicated repeated-seed study; Phase-2 currently uses one seed per fold."
        ),
    ],
}

(OUTPUT_ROOT / "M07_MODEL_CARD.json").write_text(
    json.dumps(
        json_safe(model_card),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

model_card_md = f"""# M07 Model Card

- Model: M07 — ConvNeXt-Tiny + EdgeBlock + CBAM + FLSD-53
- Resolution: {IMAGE_SIZE}×{IMAGE_SIZE}
- Split fingerprint: `{EXPECTED_SPLIT_FINGERPRINT}`
- Confirmed recipe fingerprint: `{recipe["recipe_fingerprint_sha256"]}`
- Development: {len(development_manifest)} images / {development_manifest["patient_id"].nunique()} patients
- Locked Test: {len(locked_test)} images / {locked_test["patient_id"].nunique()} patients
- Locked-Test threshold tuning: **No**
- External adaptation/tuning: **No**
- F2: **report-only**, beta=2, positive=PNEUMONIA
- Fixed dual threshold: <=0.20 NORMAL / 0.20–0.80 UNCERTAIN / >=0.80 PNEUMONIA
- Near-duplicate policy: audited before freeze; no strong SSIM>=0.95 evidence
- Remaining models: **LOCKED pending manual review of M07**
"""

(OUTPUT_ROOT / "M07_MODEL_CARD.md").write_text(
    model_card_md,
    encoding="utf-8",
)

# ------------------------------------------------------------
# One very compact final decision table.
# ------------------------------------------------------------
final_rows = [
    {
        "dataset": "OOF",
        "balanced_accuracy": oof_primary_metrics[
            "balanced_accuracy"
        ],
        "precision_normal": oof_primary_metrics[
            "precision_normal"
        ],
        "recall_normal": oof_primary_metrics[
            "recall_normal"
        ],
        "precision_pneumonia": oof_primary_metrics[
            "precision_pneumonia"
        ],
        "recall_pneumonia": oof_primary_metrics[
            "recall_pneumonia"
        ],
        "macro_f1": oof_primary_metrics[
            "macro_f1"
        ],
        "mcc": oof_primary_metrics[
            "mcc"
        ],
        "auroc": oof_primary_metrics[
            "auroc"
        ],
        "f2_report_only": oof_primary_metrics[
            "f2"
        ],
        "dual20_80_coverage": oof_dual_20_80[
            "coverage"
        ],
        "dual20_80_uncertainty": oof_dual_20_80[
            "uncertainty_rate"
        ],
        "ece": calibration_report[
            "oof"
        ]["ece_10_bins"],
        "brier": calibration_report[
            "oof"
        ]["brier"],
    },
    {
        "dataset": "LOCKED_TEST",
        "balanced_accuracy": primary_metrics[
            "balanced_accuracy"
        ],
        "precision_normal": primary_metrics[
            "precision_normal"
        ],
        "recall_normal": primary_metrics[
            "recall_normal"
        ],
        "precision_pneumonia": primary_metrics[
            "precision_pneumonia"
        ],
        "recall_pneumonia": primary_metrics[
            "recall_pneumonia"
        ],
        "macro_f1": primary_metrics[
            "macro_f1"
        ],
        "mcc": primary_metrics[
            "mcc"
        ],
        "auroc": primary_metrics[
            "auroc"
        ],
        "f2_report_only": primary_metrics[
            "f2"
        ],
        "dual20_80_coverage": test_dual_20_80[
            "coverage"
        ],
        "dual20_80_uncertainty": test_dual_20_80[
            "uncertainty_rate"
        ],
        "ece": calibration_report[
            "locked_test"
        ]["ece_10_bins"],
        "brier": calibration_report[
            "locked_test"
        ]["brier"],
    },
]

for ext_report in external_reports:
    cohort = ext_report[
        "cohort"
    ]
    cal_path = (
        EXT_ROOT
        / cohort
        / "external_calibration_report.json"
    )
    ext_cal = (
        json.loads(
            cal_path.read_text(
                encoding="utf-8"
            )
        )
        if cal_path.is_file()
        else {
            "ece_10_bins": None,
            "brier": None,
        }
    )

    em = ext_report["metrics"]
    ed = ext_report["dual20_80"]

    final_rows.append({
        "dataset": f"EXTERNAL:{cohort}",
        "balanced_accuracy": em[
            "balanced_accuracy"
        ],
        "precision_normal": em[
            "precision_normal"
        ],
        "recall_normal": em[
            "recall_normal"
        ],
        "precision_pneumonia": em[
            "precision_pneumonia"
        ],
        "recall_pneumonia": em[
            "recall_pneumonia"
        ],
        "macro_f1": em[
            "macro_f1"
        ],
        "mcc": em["mcc"],
        "auroc": em["auroc"],
        "f2_report_only": em["f2"],
        "dual20_80_coverage": ed[
            "coverage"
        ],
        "dual20_80_uncertainty": ed[
            "uncertainty_rate"
        ],
        "ece": ext_cal[
            "ece_10_bins"
        ],
        "brier": ext_cal[
            "brier"
        ],
    })

final_quick = pd.DataFrame(
    final_rows
)

final_quick.to_csv(
    OUTPUT_ROOT
    / "M07_FINAL_QUICK_RESULTS.csv",
    index=False,
)

display(
    final_quick.round(4)
)

# ------------------------------------------------------------
# Evidence index.
# ------------------------------------------------------------
evidence_rows = []

for p in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if not p.is_file():
        continue

    # Do not hash huge weights again here.
    if p.name.endswith(
        ".weights.h5"
    ):
        continue

    evidence_rows.append({
        "relative_path": str(
            p.relative_to(
                OUTPUT_ROOT
            )
        ),
        "bytes": int(
            p.stat().st_size
        ),
    })

evidence_index = pd.DataFrame(
    evidence_rows
)
evidence_index.to_csv(
    OUTPUT_ROOT
    / "M07_EVIDENCE_INDEX.csv",
    index=False,
)

# Seal the M07 matrix source only after every required report exists.
def m07_matrix_contract():
    return make_run_contract(
        "m07_matrix_source", model_id="M07", resolution=IMAGE_SIZE,
        params=shared_params, head_epochs=FINAL_HEAD_EPOCHS,
        finetune_epochs=FINAL_FINETUNE_EPOCHS,
        extra={"recipe_fingerprint": recipe["recipe_fingerprint_sha256"],
               "locked_test_identity": dataframe_identity(locked_test)},
    )

def m07_matrix_artifacts():
    paths = {name: OUTPUT_ROOT / name for name in (
        "M07_FINAL_QUICK_RESULTS.csv", "M07_MODEL_CARD.json",
        "M07_REPLICATION_GATE_REPORT.json", "M07_OOF_PREDICTIONS.csv",
    )}
    paths.update({f"fold_{fold}_weights": FINAL_ROOT / f"fold_{fold}" / "best.weights.h5"
                  for fold in range(1, 6)})
    paths["locked_test_predictions"] = TEST_OUT / "M07_LOCKED_TEST_ENSEMBLE_PREDICTIONS.csv"
    return paths

atomic_write_json(
    OUTPUT_ROOT / "M07_MATRIX_SOURCE_RECEIPT.json",
    seal_receipt({"status": "COMPLETE", "model_id": "M07", "resolution": IMAGE_SIZE},
                 m07_matrix_contract(), m07_matrix_artifacts()),
)

# Rebuild final evidence ZIP after all outputs exist.
final_evidence_zip = (
    WORK
    / "M07_FINAL_EVIDENCE_PACKAGE_V17.zip"
)

if final_evidence_zip.exists():
    final_evidence_zip.unlink()

with zipfile.ZipFile(
    final_evidence_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    for p in OUTPUT_ROOT.rglob("*"):
        if not p.is_file():
            continue
        if p.name.endswith(
            ".weights.h5"
        ):
            continue
        z.write(
            p,
            arcname=str(
                p.relative_to(
                    OUTPUT_ROOT
                )
            ),
        )

print("✅ Model Card JSON:", OUTPUT_ROOT / "M07_MODEL_CARD.json")
print("✅ Model Card MD:", OUTPUT_ROOT / "M07_MODEL_CARD.md")
print("✅ Final quick results:", OUTPUT_ROOT / "M07_FINAL_QUICK_RESULTS.csv")
print("✅ Evidence index:", OUTPUT_ROOT / "M07_EVIDENCE_INDEX.csv")
print("✅ Final evidence ZIP:", final_evidence_zip)
print("🔒 Phase-2 code for M01–M06 and M08–M12 is embedded below but remains locked pending manual M07 review.")

_persist_lightweight_files(
    [
        OUTPUT_ROOT / "M07_MODEL_CARD.json",
        OUTPUT_ROOT / "M07_MODEL_CARD.md",
        OUTPUT_ROOT / "M07_FINAL_QUICK_RESULTS.csv",
        OUTPUT_ROOT / "M07_EVIDENCE_INDEX.csv",
        final_evidence_zip,
    ],
    note=(
        "M07 FINAL Confirmation-HPO Gate complete; "
        "model card/evidence persisted; remaining models locked"
    ),
)

In [ ]:
# ============================================================
# 17) PHASE-2 CANONICAL MODEL REGISTRY + GENERIC ENGINE
# ============================================================

from dataclasses import dataclass, asdict
import inspect

@dataclass(frozen=True)
class CanonicalModelSpec:
    model_id: str
    loss_name: str
    use_cbam: bool = False
    use_edge: bool = False
    batch_policy: str = "none"   # none/mixup/cutmix/mixup_cutmix
    use_ema: bool = False
    use_swa: bool = False
    description: str = ""

CANONICAL_MODELS = {
    "M01": CanonicalModelSpec(
        "M01", "bce",
        description="ConvNeXt-Tiny + BCE",
    ),
    "M02": CanonicalModelSpec(
        "M02", "weighted_bce",
        description="ConvNeXt-Tiny + weighted BCE",
    ),
    "M03": CanonicalModelSpec(
        "M03", "static_focal",
        description="ConvNeXt-Tiny + static focal",
    ),
    "M04": CanonicalModelSpec(
        "M04", "dynamic_focal",
        description="ConvNeXt-Tiny + FLSD-53",
    ),
    "M05": CanonicalModelSpec(
        "M05", "dynamic_focal",
        use_cbam=True,
        description="M04 + CBAM",
    ),
    "M06": CanonicalModelSpec(
        "M06", "dynamic_focal",
        use_edge=True,
        description="M04 + EdgeBlock",
    ),
    "M07": CanonicalModelSpec(
        "M07", "dynamic_focal",
        use_cbam=True,
        use_edge=True,
        description="M04 + EdgeBlock + CBAM",
    ),
    "M08": CanonicalModelSpec(
        "M08", "dynamic_focal",
        use_cbam=True,
        use_edge=True,
        batch_policy="mixup",
        description="M07 + MixUp",
    ),
    "M09": CanonicalModelSpec(
        "M09", "dynamic_focal",
        use_cbam=True,
        use_edge=True,
        batch_policy="cutmix",
        description="M07 + CutMix",
    ),
    "M10": CanonicalModelSpec(
        "M10", "dynamic_focal",
        use_cbam=True,
        use_edge=True,
        batch_policy="mixup_cutmix",
        description="M07 + stochastic MixUp/CutMix",
    ),
    "M11": CanonicalModelSpec(
        "M11", "dynamic_focal",
        use_cbam=True,
        use_edge=True,
        batch_policy="mixup_cutmix",
        use_ema=True,
        description="M10 + EMA",
    ),
    "M12": CanonicalModelSpec(
        "M12", "dynamic_focal",
        use_cbam=True,
        use_edge=True,
        batch_policy="mixup_cutmix",
        use_swa=True,
        description="M10 + SWA",
    ),
}

assert tuple(
    m for m in CANONICAL_MODELS if m != "M07"
) == PHASE2_MODEL_ORDER

@tf.keras.utils.register_keras_serializable(package="PneumoniaAI")
class BinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(
        self,
        alpha,
        gamma=2.0,
        name="binary_focal_loss",
    ):
        super().__init__(name=name)
        self.alpha = float(alpha)
        self.gamma = float(gamma)

    def call(self, y_true, y_pred):
        y_pred = tf.cast(y_pred, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        y_true = tf.reshape(
            y_true,
            tf.shape(y_pred),
        )
        y_pred = tf.clip_by_value(
            y_pred,
            1e-6,
            1.0 - 1e-6,
        )
        p_t = (
            y_true * y_pred
            + (1.0 - y_true)
            * (1.0 - y_pred)
        )
        alpha_t = (
            y_true * self.alpha
            + (1.0 - y_true)
            * (1.0 - self.alpha)
        )
        focal = tf.pow(
            1.0 - p_t,
            self.gamma,
        )
        bce = tf.keras.backend.binary_crossentropy(
            y_true,
            y_pred,
        )
        return tf.reduce_mean(
            alpha_t * focal * bce,
            axis=-1,
        )

    def get_config(self):
        return {
            **super().get_config(),
            "alpha": self.alpha,
            "gamma": self.gamma,
        }

@tf.function(reduce_retracing=True)
def _phase2_beta(size, concentration):
    g1 = tf.random.gamma(
        [size],
        concentration,
    )
    g2 = tf.random.gamma(
        [size],
        concentration,
    )
    return g1 / (
        g1 + g2 + 1e-7
    )

@tf.function(reduce_retracing=True)
def _phase2_mixup(images, labels, alpha):
    batch = tf.shape(images)[0]
    order = tf.random.shuffle(
        tf.range(batch)
    )
    lam = _phase2_beta(
        batch,
        alpha,
    )
    image_lam = tf.reshape(
        lam,
        [batch, 1, 1, 1],
    )
    label_lam = tf.reshape(
        lam,
        [batch],
    )
    return (
        images * image_lam
        + tf.gather(
            images,
            order,
        ) * (1.0 - image_lam),
        labels * label_lam
        + tf.gather(
            labels,
            order,
        ) * (1.0 - label_lam),
    )

@tf.function(reduce_retracing=True)
def _phase2_cutmix(images, labels, alpha):
    batch = tf.shape(images)[0]
    height = tf.shape(images)[1]
    width = tf.shape(images)[2]
    order = tf.random.shuffle(
        tf.range(batch)
    )

    lam = _phase2_beta(
        tf.constant(1),
        alpha,
    )[0]
    cut_ratio = tf.sqrt(
        1.0 - lam
    )
    cut_w = tf.cast(
        tf.cast(
            width,
            tf.float32,
        ) * cut_ratio,
        tf.int32,
    )
    cut_h = tf.cast(
        tf.cast(
            height,
            tf.float32,
        ) * cut_ratio,
        tf.int32,
    )

    center_x = tf.random.uniform(
        [],
        0,
        width,
        dtype=tf.int32,
    )
    center_y = tf.random.uniform(
        [],
        0,
        height,
        dtype=tf.int32,
    )

    x1 = tf.clip_by_value(
        center_x - cut_w // 2,
        0,
        width,
    )
    x2 = tf.clip_by_value(
        center_x + cut_w // 2,
        0,
        width,
    )
    y1 = tf.clip_by_value(
        center_y - cut_h // 2,
        0,
        height,
    )
    y2 = tf.clip_by_value(
        center_y + cut_h // 2,
        0,
        height,
    )

    yr = tf.range(
        height
    )[:, None]
    xr = tf.range(
        width
    )[None, :]
    patch = (
        (yr >= y1)
        & (yr < y2)
        & (xr >= x1)
        & (xr < x2)
    )
    patch = tf.cast(
        patch[
            None,
            :,
            :,
            None,
        ],
        images.dtype,
    )

    shuffled = tf.gather(
        images,
        order,
    )
    mixed_images = (
        images * (1.0 - patch)
        + shuffled * patch
    )

    patch_area = tf.cast(
        (x2 - x1)
        * (y2 - y1),
        tf.float32,
    )
    total_area = tf.cast(
        width * height,
        tf.float32,
    )
    adjusted_lam = (
        1.0
        - patch_area
        / tf.maximum(
            total_area,
            1.0,
        )
    )

    mixed_labels = (
        labels * adjusted_lam
        + tf.gather(
            labels,
            order,
        )
        * (1.0 - adjusted_lam)
    )
    return mixed_images, mixed_labels

def phase2_apply_batch_policy(
    images,
    labels,
    policy,
):
    if policy == "mixup":
        return _phase2_mixup(
            images,
            labels,
            PHASE2_MIX_ALPHA,
        )
    if policy == "cutmix":
        return _phase2_cutmix(
            images,
            labels,
            PHASE2_MIX_ALPHA,
        )
    if policy == "mixup_cutmix":
        return tf.cond(
            tf.random.uniform([])
            < 0.5,
            lambda: _phase2_mixup(
                images,
                labels,
                PHASE2_MIX_ALPHA,
            ),
            lambda: _phase2_cutmix(
                images,
                labels,
                PHASE2_MIX_ALPHA,
            ),
        )
    return images, labels

def phase2_decode_resize(
    path,
    label,
    resolution,
):
    data = tf.io.read_file(
        path
    )
    image = tf.io.decode_image(
        data,
        channels=3,
        expand_animations=False,
    )
    image.set_shape(
        [None, None, 3]
    )
    image = tf.cast(
        image,
        tf.float32,
    )
    image = tf.image.resize_with_pad(
        image,
        target_height=int(
            resolution
        ),
        target_width=int(
            resolution
        ),
        method="bilinear",
        antialias=True,
    )
    image = tf.clip_by_value(
        image,
        0.0,
        255.0,
    )
    return image, tf.cast(
        label,
        tf.float32,
    )

def phase2_batch_plan(resolution):
    resolution = int(resolution)
    effective = int(shared_params["batch_size"])
    ceiling = PHASE2_MICROBATCH_MAX_BY_RESOLUTION.get(resolution)
    if ceiling is None or ceiling >= effective:
        return {"microbatch_size": effective, "gradient_accumulation_steps": 1,
                "effective_optimizer_batch_size": effective}
    divisors = [d for d in range(min(int(ceiling), effective), 1, -1) if effective % d == 0]
    if not divisors:
        raise RuntimeError(f"No exact microbatch divisor for effective batch={effective}, resolution={resolution}")
    micro = int(divisors[0])
    accum = int(effective // micro)
    return {"microbatch_size": micro, "gradient_accumulation_steps": accum,
            "effective_optimizer_batch_size": micro * accum}


def phase2_batch_size(resolution):
    return int(phase2_batch_plan(resolution)["microbatch_size"])


def phase2_accumulation_steps(resolution):
    return int(phase2_batch_plan(resolution)["gradient_accumulation_steps"])


def phase2_seed(model_id, resolution, fold):
    # M07 resolution sensitivity uses the SAME fold seed at 224/320/384 so
    # the intended factor is resolution rather than a deliberately changed seed.
    if str(model_id) == "M07":
        return int(SEED + int(fold))
    return int(SEED + int(fold) + int(resolution) * 10 + int(str(model_id)[1:]) * 1000)


def phase2_balanced_steps(
    frame,
    batch_size,
):
    counts = frame[
        "label"
    ].value_counts().to_dict()
    n0 = batch_size // 2
    n1 = batch_size - n0
    return max(
        1,
        int(
            math.ceil(
                counts[0]
                / max(
                    n0,
                    1,
                )
            )
        ),
        int(
            math.ceil(
                counts[1]
                / max(
                    n1,
                    1,
                )
            )
        ),
    )


def phase2_training_schedule(
    frame,
    resolution,
    balanced_batches,
):
    """Preserve optimizer-update count and forbid partial accumulation groups."""
    effective_batch = int(shared_params["batch_size"])
    plan = phase2_batch_plan(resolution)
    accumulation = int(plan["gradient_accumulation_steps"])

    if balanced_batches:
        optimizer_updates = int(
            phase2_balanced_steps(
                frame,
                effective_batch,
            )
        )
    else:
        optimizer_updates = int(
            math.ceil(
                len(frame) / float(effective_batch)
            )
        )

    micro_steps = int(
        optimizer_updates * accumulation
    )

    if micro_steps % accumulation != 0:
        raise RuntimeError(
            "Partial gradient-accumulation group at epoch boundary."
        )

    return {
        **plan,
        "optimizer_updates_per_epoch": optimizer_updates,
        "micro_steps_per_epoch": micro_steps,
        "partial_accumulation_steps": 0,
    }



def phase2_build_dataset(
    frame,
    resolution,
    batch_size,
    training,
    seed,
    batch_policy="none",
    balanced_batches=True,
):
    options = tf.data.Options()
    options.experimental_deterministic = True

    if (
        training
        and balanced_batches
    ):
        normal = frame[
            frame["label"] == 0
        ]
        pneumonia = frame[
            frame["label"] == 1
        ]

        n0 = batch_size // 2
        n1 = batch_size - n0

        if (
            n0 < 1
            or n1 < 1
        ):
            raise ValueError(
                "Balanced batches require batch size >=2"
            )

        def stream(
            part,
            class_batch,
            class_seed,
        ):
            ds = (
                tf.data.Dataset
                .from_tensor_slices((
                    part[
                        "filepath"
                    ].astype(str).to_numpy(),
                    part[
                        "label"
                    ].astype(
                        np.float32
                    ).to_numpy(),
                ))
                .with_options(
                    options
                )
            )
            ds = ds.shuffle(
                len(part),
                seed=class_seed,
                reshuffle_each_iteration=True,
            ).repeat()
            ds = ds.map(
                lambda p, y: phase2_decode_resize(
                    p,
                    y,
                    resolution,
                ),
                num_parallel_calls=AUTOTUNE,
                deterministic=True,
            )
            return ds.batch(
                class_batch,
                drop_remainder=True,
            )

        ds0 = stream(
            normal,
            n0,
            seed + 17,
        )
        ds1 = stream(
            pneumonia,
            n1,
            seed + 31,
        )

        ds = tf.data.Dataset.zip(
            (ds0, ds1)
        )

        def merge(a, b):
            x0, y0 = a
            x1, y1 = b
            x = tf.concat(
                [x0, x1],
                axis=0,
            )
            y = tf.concat(
                [y0, y1],
                axis=0,
            )
            order = tf.random.shuffle(
                tf.range(
                    tf.shape(y)[0]
                ),
                seed=seed,
            )
            x = tf.gather(
                x,
                order,
            )
            y = tf.gather(
                y,
                order,
            )
            if batch_policy != "none":
                x, y = phase2_apply_batch_policy(
                    x,
                    y,
                    batch_policy,
                )
            return x, y

        return ds.map(
            merge,
            num_parallel_calls=1,
            deterministic=True,
        ).prefetch(
            AUTOTUNE
        )

    ds = (
        tf.data.Dataset
        .from_tensor_slices((
            frame[
                "filepath"
            ].astype(str).to_numpy(),
            frame[
                "label"
            ].astype(
                np.float32
            ).to_numpy(),
        ))
        .with_options(
            options
        )
    )

    if training:
        ds = ds.shuffle(
            len(frame),
            seed=seed,
            reshuffle_each_iteration=True,
        ).repeat()

    ds = ds.map(
        lambda p, y: phase2_decode_resize(
            p,
            y,
            resolution,
        ),
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )

    drop_remainder = bool(
        training
        and batch_policy != "none"
    )
    ds = ds.batch(
        batch_size,
        drop_remainder=drop_remainder,
    )

    if (
        training
        and batch_policy != "none"
    ):
        ds = ds.map(
            lambda x, y: phase2_apply_batch_policy(
                x,
                y,
                batch_policy,
            ),
            num_parallel_calls=1,
            deterministic=True,
        )

    return ds.prefetch(
        AUTOTUNE
    )

def phase2_class_balance(
    train_df,
):
    counts = train_df[
        "label"
    ].value_counts().to_dict()
    negative = int(
        counts.get(
            0,
            0,
        )
    )
    positive = int(
        counts.get(
            1,
            0,
        )
    )
    total = (
        negative
        + positive
    )
    if (
        negative == 0
        or positive == 0
    ):
        raise RuntimeError(
            "Both classes are required."
        )

    return {
        "negative": negative,
        "positive": positive,
        "total": total,
        "positive_alpha_original": float(
            negative / total
        ),
        "class_weight": {
            0: float(
                total
                / (
                    2.0
                    * negative
                )
            ),
            1: float(
                total
                / (
                    2.0
                    * positive
                )
            ),
        },
    }

def phase2_build_model(
    spec,
    resolution,
    dropout,
    seed,
):
    inputs = tf.keras.Input(
        shape=(
            int(resolution),
            int(resolution),
            3,
        ),
        dtype=tf.float32,
        name="image",
    )

    x = augmentation_block(
        int(seed)
    )(inputs)

    if spec.use_edge:
        x = EdgeBlock(
            filters=int(
                shared_params.get(
                    "edge_filters",
                    8,
                )
            ),
            max_gate=float(
                shared_params.get(
                    "edge_max_gate",
                    0.15,
                )
            ),
            name="edge_block",
        )(x)

    kwargs = dict(
        include_top=False,
        weights="imagenet",
        input_shape=(
            int(resolution),
            int(resolution),
            3,
        ),
        pooling=None,
    )
    signature = inspect.signature(
        tf.keras.applications.ConvNeXtTiny
    )
    if (
        "include_preprocessing"
        in signature.parameters
    ):
        kwargs[
            "include_preprocessing"
        ] = True

    backbone = (
        tf.keras.applications
        .ConvNeXtTiny(
            **kwargs
        )
    )
    x = backbone(x)

    if spec.use_cbam:
        x = CBAM(
            reduction=int(
                shared_params.get(
                    "cbam_reduction",
                    8,
                )
            ),
            spatial_kernel=CBAM_SPATIAL_KERNEL,
            name="cbam",
        )(x)

    x = tf.keras.layers.Activation(
        "linear",
        name="aez_spatial_features",
    )(x)

    x = (
        tf.keras.layers
        .GlobalAveragePooling2D(
            name="global_pool"
        )(x)
    )
    x = (
        tf.keras.layers
        .LayerNormalization(
            epsilon=1e-6,
            dtype="float32",
            name="head_norm",
        )(x)
    )
    x = tf.keras.layers.Dropout(
        float(dropout),
        name="head_dropout",
    )(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="probability",
    )(x)

    return (
        tf.keras.Model(
            inputs,
            outputs,
            name=spec.model_id,
        ),
        backbone,
    )

def phase2_loss_and_weights(
    spec,
    train_df,
):
    balance = phase2_class_balance(
        train_df
    )

    # M02 is the canonical weighted-BCE natural-sampling ablation.
    if (
        spec.loss_name
        == "weighted_bce"
    ):
        return (
            tf.keras.losses
            .BinaryCrossentropy(),
            balance[
                "class_weight"
            ],
            {
                "balanced_batches": False,
                "positive_alpha": None,
                "alpha_policy": "NOT_APPLICABLE",
            },
        )

    if spec.loss_name == "bce":
        return (
            tf.keras.losses
            .BinaryCrossentropy(),
            None,
            {
                "balanced_batches": True,
                "positive_alpha": None,
                "alpha_policy": "NOT_APPLICABLE",
            },
        )

    if (
        spec.loss_name
        == "static_focal"
    ):
        alpha = float(
            balance[
                "positive_alpha_original"
            ]
        )
        return (
            BinaryFocalLoss(
                alpha=alpha,
                gamma=PHASE2_STATIC_FOCAL_GAMMA,
            ),
            None,
            {
                "balanced_batches": True,
                "positive_alpha": alpha,
                "alpha_policy": (
                    "CANONICAL_STATIC_FOCAL_PREVALENCE_ALPHA"
                ),
            },
        )

    if (
        spec.loss_name
        == "dynamic_focal"
    ):
        # Confirmed correction from M07:
        # balanced sampling => neutral focal alpha.
        alpha = 0.50
        return (
            FLSD53BinaryFocalLoss(
                alpha=alpha,
                threshold=FLSD_THRESHOLD,
                hard_gamma=FLSD_HARD_GAMMA,
                easy_gamma=FLSD_EASY_GAMMA,
            ),
            None,
            {
                "balanced_batches": True,
                "positive_alpha": alpha,
                "alpha_policy": (
                    "NEUTRAL_0.5_WHEN_BALANCED"
                ),
            },
        )

    raise ValueError(
        spec.loss_name
    )

def phase2_build_optimizer(
    lr,
    weight_decay,
    use_ema,
    resolution,
):
    signature = inspect.signature(tf.keras.optimizers.AdamW)
    kwargs = {
        "learning_rate": float(lr),
        "weight_decay": float(weight_decay),
        "global_clipnorm": 1.0,
    }
    accumulation = phase2_accumulation_steps(resolution)
    if accumulation > 1:
        if "gradient_accumulation_steps" not in signature.parameters:
            raise RuntimeError(
                "Keras AdamW lacks gradient_accumulation_steps; high-resolution "
                "training is blocked rather than silently changing effective batch size."
            )
        kwargs["gradient_accumulation_steps"] = int(accumulation)
    if "use_ema" in signature.parameters:
        kwargs["use_ema"] = bool(use_ema)
        if use_ema:
            kwargs["ema_momentum"] = float(PHASE2_EMA_MOMENTUM)
    elif use_ema:
        raise RuntimeError("Current Keras does not support AdamW(use_ema=True), required by M11.")
    return tf.keras.optimizers.AdamW(**kwargs)


def phase2_compile(
    model,
    loss,
    lr,
    weight_decay,
    resolution,
    use_ema=False,
):
    model.compile(
        optimizer=phase2_build_optimizer(
            lr,
            weight_decay,
            use_ema,
            resolution,
        ),
        loss=loss,
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.Precision(
                name="precision"
            ),
            tf.keras.metrics.Recall(
                name="recall"
            ),
            tf.keras.metrics.AUC(
                curve="ROC",
                name="auc_roc",
            ),
            tf.keras.metrics.AUC(
                curve="PR",
                name="auc_pr",
            ),
        ],
    )

class Phase2BalancedCheckpoint(
    BalancedCheckpoint
):
    def __init__(
        self,
        *args,
        start_from_global_epoch=0,
        **kwargs,
    ):
        super().__init__(
            *args,
            **kwargs,
        )
        self.start_from_global_epoch = int(
            start_from_global_epoch
        )

    def on_epoch_end(
        self,
        epoch,
        logs=None,
    ):
        # Always compute/log validation metrics.
        logs = (
            logs
            if logs is not None
            else {}
        )
        p = self.model.predict(
            self.val_ds,
            verbose=0,
        ).reshape(-1)
        metrics = select_balanced_threshold(
            self.y_val,
            p,
        )
        rank = threshold_rank(
            metrics
        )

        logs[
            "val_min_dual_precision_recall_selected"
        ] = metrics[
            "min_dual_precision_recall"
        ]
        logs[
            "val_dual_precision_recall_score"
        ] = metrics[
            "dual_precision_recall_score"
        ]
        logs[
            "val_selected_threshold"
        ] = metrics[
            "threshold"
        ]
        logs[
            "val_recall_normal_selected"
        ] = metrics[
            "recall_normal"
        ]
        logs[
            "val_recall_pneumonia_selected"
        ] = metrics[
            "recall_pneumonia"
        ]
        logs[
            "val_precision_normal_selected"
        ] = metrics[
            "precision_normal"
        ]
        logs[
            "val_precision_pneumonia_selected"
        ] = metrics[
            "precision_pneumonia"
        ]

        if (
            self.best_rank is None
            or rank > self.best_rank
        ):
            self.best_rank = rank
            self.best_metrics = metrics
            self.best_epoch = (
                int(epoch) + 1
            )
            self.wait = 0
            self.checkpoint_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            self.model.save_weights(
                self.checkpoint_path
            )
            self._save_state()
            print(
                f"[best] epoch={epoch+1} "
                f"minPR={metrics['min_dual_precision_recall']:.4f} "
                f"P_N={metrics['precision_normal']:.4f} "
                f"R_N={metrics['recall_normal']:.4f} "
                f"P_P={metrics['precision_pneumonia']:.4f} "
                f"R_P={metrics['recall_pneumonia']:.4f}"
            )
            return

        if (
            int(epoch) + 1
            <= self.start_from_global_epoch
        ):
            self.wait = 0
            self._save_state()
            return

        self.wait += 1
        self._save_state()

        if (
            self.patience >= 0
            and self.wait
            > self.patience
        ):
            print(
                f"[early-stop] best_epoch={self.best_epoch} "
                f"best_minPR={self.best_metrics['min_dual_precision_recall']:.4f}"
            )
            self.model.stop_training = True

class Phase2SWA(
    tf.keras.callbacks.Callback
):
    def __init__(
        self,
        phase_start_epoch,
        start_local_epoch,
    ):
        super().__init__()
        self.phase_start_epoch = int(
            phase_start_epoch
        )
        self.start_local_epoch = int(
            start_local_epoch
        )
        self.n_models = 0
        self.average_weights = None

    def on_epoch_end(
        self,
        epoch,
        logs=None,
    ):
        local_epoch = (
            int(epoch)
            - self.phase_start_epoch
            + 1
        )
        if (
            local_epoch
            < self.start_local_epoch
        ):
            return

        current = [
            np.asarray(w).copy()
            for w
            in self.model.get_weights()
        ]

        if (
            self.average_weights
            is None
        ):
            self.average_weights = current
            self.n_models = 1
            return

        self.n_models += 1
        factor = (
            1.0
            / self.n_models
        )
        for i, w in enumerate(
            current
        ):
            self.average_weights[
                i
            ] += (
                w
                - self.average_weights[i]
            ) * factor

    def on_train_end(
        self,
        logs=None,
    ):
        if (
            self.average_weights
            is None
        ):
            raise RuntimeError(
                "M12 SWA did not collect any epoch snapshots."
            )
        self.model.set_weights(
            self.average_weights
        )
        print(
            f"[SWA] applied average of {self.n_models} snapshots"
        )

def phase2_callbacks(
    spec,
    run_dir,
    val_ds,
    y_val,
    checkpoint,
    patience,
    schedule,
    initial_lr,
    total_epochs,
    warmup_epochs,
    min_lr,
    phase_start_epoch,
    allow_early_stop,
    swa_start_local=None,
):
    callbacks = [
        tf.keras.callbacks
        .TerminateOnNaN()
    ]

    # For EMA, validation/checkpointing must see EMA weights.
    if spec.use_ema:
        if not hasattr(
            tf.keras.callbacks,
            "SwapEMAWeights",
        ):
            raise RuntimeError(
                "M11 requires SwapEMAWeights."
            )
        callbacks.append(
            tf.keras.callbacks
            .SwapEMAWeights(
                swap_on_epoch=True
            )
        )

    start_from = 0
    if (
        spec.use_swa
        and swa_start_local
        is not None
    ):
        start_from = (
            int(phase_start_epoch)
            + int(swa_start_local)
            - 1
        )

    monitor = Phase2BalancedCheckpoint(
        val_ds=val_ds,
        y_val=y_val,
        checkpoint_path=checkpoint,
        patience=(
            int(patience)
            if allow_early_stop
            else 10**9
        ),
        start_from_global_epoch=start_from,
    )
    callbacks.append(
        monitor
    )

    if schedule == "cosine":
        cosine = WarmupCosine(
            initial_lr=initial_lr,
            total_epochs=total_epochs,
            warmup_epochs=warmup_epochs,
            min_lr=min_lr,
        )
        cosine.phase_start_epoch = int(
            phase_start_epoch
        )
        callbacks.append(
            cosine
        )
    elif schedule == "plateau":
        callbacks.append(
            tf.keras.callbacks
            .ReduceLROnPlateau(
                monitor=(
                    "val_min_dual_precision_recall_selected"
                ),
                mode="max",
                factor=0.5,
                patience=2,
                min_lr=float(
                    min_lr
                ),
                verbose=1,
            )
        )
    else:
        raise ValueError(
            schedule
        )

    if (
        spec.use_swa
        and swa_start_local
        is not None
    ):
        # Last callback: after monitor's on_train_end restores
        # the validation-selected checkpoint, SWA installs its
        # averaged weights, preserving canonical M12 identity.
        callbacks.append(
            Phase2SWA(
                phase_start_epoch=phase_start_epoch,
                start_local_epoch=swa_start_local,
            )
        )

    return (
        callbacks,
        monitor,
    )

def phase2_history_frame(
    history,
    stage,
):
    if history is None:
        return pd.DataFrame()
    df = pd.DataFrame(
        history.history
    )
    if not df.empty:
        df.insert(
            0,
            "epoch",
            np.arange(
                history.epoch[0] + 1,
                history.epoch[-1] + 2,
            ),
        )
        df.insert(
            1,
            "stage",
            stage,
        )
    return df

def phase2_train_one_fold(
    model_id,
    resolution,
    fold,
    run_dir,
):
    spec = CANONICAL_MODELS[
        model_id
    ]
    run_dir = Path(
        run_dir
    )
    if (run_dir / "COMPLETED.json").exists():
        raise ValueError("Existing Phase-2 completion must be validated/reused, never overwritten")
    run_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    tf.keras.backend.clear_session()
    gc.collect()

    seed = phase2_seed(model_id, resolution, fold)
    tf.keras.utils.set_random_seed(
        seed
    )

    train_df = load_fold_manifest(
        int(fold),
        "train",
    )
    val_df = load_fold_manifest(
        int(fold),
        "val",
    )

    batch_size = phase2_batch_size(
        resolution
    )

    loss, class_weight, balance_contract = (
        phase2_loss_and_weights(
            spec,
            train_df,
        )
    )
    balanced_batches = bool(
        balance_contract[
            "balanced_batches"
        ]
    )

    train_ds = phase2_build_dataset(
        train_df,
        resolution,
        batch_size,
        True,
        seed,
        batch_policy=spec.batch_policy,
        balanced_batches=balanced_batches,
    )
    val_ds = phase2_build_dataset(
        val_df,
        resolution,
        batch_size,
        False,
        seed,
        batch_policy="none",
        balanced_batches=False,
    )

    training_schedule = phase2_training_schedule(
        train_df,
        resolution,
        balanced_batches=balanced_batches,
    )
    steps = int(
        training_schedule[
            "micro_steps_per_epoch"
        ]
    )

    checkpoint = (
        run_dir
        / "best.weights.h5"
    )
    state = checkpoint.with_suffix(
        checkpoint.suffix
        + ".state.json"
    )
    for p in (
        checkpoint,
        state,
    ):
        if p.exists():
            p.unlink()

    model, backbone = phase2_build_model(
        spec,
        resolution,
        dropout=float(
            shared_params[
                "dropout"
            ]
        ),
        seed=seed,
    )

    # -----------------------
    # Head stage
    # -----------------------
    backbone.trainable = False
    phase2_compile(
        model,
        loss,
        lr=float(
            shared_params[
                "head_lr"
            ]
        ),
        weight_decay=float(
            shared_params[
                "weight_decay"
            ]
        ),
        resolution=resolution,
        use_ema=False,
    )

    head_callbacks, _ = phase2_callbacks(
        spec=CanonicalModelSpec(
            **{
                **asdict(spec),
                "use_ema": False,
                "use_swa": False,
            }
        ),
        run_dir=run_dir,
        val_ds=val_ds,
        y_val=val_df[
            "label"
        ].to_numpy(dtype=int),
        checkpoint=checkpoint,
        patience=999,
        schedule=shared_params[
            "lr_schedule"
        ],
        initial_lr=float(
            shared_params[
                "head_lr"
            ]
        ),
        total_epochs=PHASE2_FINAL_HEAD_EPOCHS,
        warmup_epochs=int(
            shared_params[
                "head_warmup_epochs"
            ]
        ),
        min_lr=float(
            shared_params[
                "min_lr"
            ]
        ),
        phase_start_epoch=0,
        allow_early_stop=False,
    )

    h1 = model.fit(
        train_ds,
        validation_data=val_ds,
        steps_per_epoch=steps,
        epochs=PHASE2_FINAL_HEAD_EPOCHS,
        callbacks=head_callbacks,
        class_weight=class_weight,
        verbose=1,
    )

    if state.is_file():
        try:
            payload = json.loads(
                state.read_text(
                    encoding="utf-8"
                )
            )
            payload[
                "wait"
            ] = 0
            state.write_text(
                json.dumps(
                    payload,
                    indent=2,
                ),
                encoding="utf-8",
            )
        except Exception:
            pass

    # -----------------------
    # Fine-tuning stage
    # -----------------------
    backbone.trainable = True
    phase2_compile(
        model,
        loss,
        lr=float(
            shared_params[
                "finetune_lr"
            ]
        ),
        weight_decay=float(
            shared_params[
                "weight_decay"
            ]
        ),
        resolution=resolution,
        use_ema=spec.use_ema,
    )

    effective_swa_start = min(
        max(
            1,
            PHASE2_SWA_START,
        ),
        PHASE2_FINAL_FINETUNE_EPOCHS,
    )

    ft_callbacks, ft_monitor = (
        phase2_callbacks(
            spec=spec,
            run_dir=run_dir,
            val_ds=val_ds,
            y_val=val_df[
                "label"
            ].to_numpy(dtype=int),
            checkpoint=checkpoint,
            patience=int(
                shared_params[
                    "patience"
                ]
            ),
            schedule=shared_params[
                "lr_schedule"
            ],
            initial_lr=float(
                shared_params[
                    "finetune_lr"
                ]
            ),
            total_epochs=PHASE2_FINAL_FINETUNE_EPOCHS,
            warmup_epochs=int(
                shared_params[
                    "finetune_warmup_epochs"
                ]
            ),
            min_lr=float(
                shared_params[
                    "min_lr"
                ]
            ),
            phase_start_epoch=PHASE2_FINAL_HEAD_EPOCHS,
            allow_early_stop=True,
            swa_start_local=(
                effective_swa_start
                if spec.use_swa
                else None
            ),
        )
    )

    h2 = model.fit(
        train_ds,
        validation_data=val_ds,
        steps_per_epoch=steps,
        initial_epoch=PHASE2_FINAL_HEAD_EPOCHS,
        epochs=(
            PHASE2_FINAL_HEAD_EPOCHS
            + PHASE2_FINAL_FINETUNE_EPOCHS
        ),
        callbacks=ft_callbacks,
        class_weight=class_weight,
        verbose=1,
    )

    hist = pd.concat([
        phase2_history_frame(h1, "head"), phase2_history_frame(h2, "finetune"),
    ], ignore_index=True)
    fine_count = int((hist.get("stage", pd.Series(dtype=str)) == "finetune").sum())
    training_evidence = {
        "head_epochs_observed": int((hist.get("stage", pd.Series(dtype=str)) == "head").sum()),
        "finetune_epochs_observed": fine_count,
        "finetune_stop_reason": (
            "VALIDATION_EARLY_STOPPING" if fine_count < PHASE2_FINAL_FINETUNE_EPOCHS
            else "EPOCH_BUDGET_REACHED"
        ),
        "finetune_monitor_wait": int(ft_monitor.wait),
        "finetune_monitor_patience": int(ft_monitor.patience),
        "model_stop_training": bool(model.stop_training),
        "swa_snapshots": int(sum(
            cb.n_models for cb in ft_callbacks if isinstance(cb, Phase2SWA)
        )),
    }
    phase2_validate_history(hist, model_id, training_evidence)

    # Standard and EMA use validation-selected checkpoint.
    # M12 final weights are SWA weights installed by last callback.
    if (
        not spec.use_swa
        and checkpoint.is_file()
    ):
        model.load_weights(
            checkpoint
        )

    probability = np.asarray(model.predict(
        val_ds,
        verbose=0,
    ).reshape(-1), dtype=np.float64)

    y_val = val_df[
        "label"
    ].to_numpy(dtype=int)
    metrics = select_balanced_threshold(
        y_val,
        probability,
    )

    pred_df = phase2_save_validation_predictions(
        val_df, probability, run_dir / "validation_predictions.csv",
    )
    hist.to_csv(run_dir / "history.csv", index=False, float_format="%.17g")
    phase2_validate_history(
        pd.read_csv(run_dir / "history.csv", float_precision="round_trip"),
        model_id, training_evidence,
    )

    # Train metrics using frozen selected threshold.
    train_eval_ds = phase2_build_dataset(
        train_df,
        resolution,
        batch_size,
        False,
        seed,
        batch_policy="none",
        balanced_batches=False,
    )
    train_probability = model.predict(
        train_eval_ds,
        verbose=0,
    ).reshape(-1)
    train_metrics = calculate_metrics(
        train_df[
            "label"
        ].to_numpy(dtype=int),
        train_probability,
        float(
            metrics[
                "threshold"
            ]
        ),
    )

    result = {
        "schema": "pneumonia.phase2.fold.v1.7",
        "training_evidence": training_evidence,
        "status": "COMPLETED",
        "model_id": model_id,
        "description": spec.description,
        "resolution": int(
            resolution
        ),
        "fold_id": int(fold),
        "seed": int(seed),
        "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
        "confirmed_m07_recipe_fingerprint": recipe[
            "recipe_fingerprint_sha256"
        ],
        "shared_training_params": shared_params,
        "model_spec": asdict(
            spec
        ),
        "fixed_identity": {
            "static_focal_gamma": PHASE2_STATIC_FOCAL_GAMMA,
            "flsd_alpha_balanced": 0.50,
            "flsd_threshold": FLSD_THRESHOLD,
            "flsd_hard_gamma": FLSD_HARD_GAMMA,
            "flsd_easy_gamma": FLSD_EASY_GAMMA,
            "mix_alpha": PHASE2_MIX_ALPHA,
            "ema_momentum": PHASE2_EMA_MOMENTUM,
            "swa_start": PHASE2_SWA_START,
            "edge_filters": int(
                shared_params.get(
                    "edge_filters",
                    8,
                )
            ) if spec.use_edge else None,
            "edge_max_gate": float(
                shared_params.get(
                    "edge_max_gate",
                    0.15,
                )
            ) if spec.use_edge else None,
            "cbam_reduction": int(
                shared_params.get(
                    "cbam_reduction",
                    8,
                )
            ) if spec.use_cbam else None,
            "cbam_spatial_kernel": (
                CBAM_SPATIAL_KERNEL
                if spec.use_cbam
                else None
            ),
        },
        "balance_contract": balance_contract,
        "optimizer_update_contract": training_schedule,
        "batch_plan": phase2_batch_plan(resolution),
        "validation_metrics": json_safe(
            metrics
        ),
        "train_metrics": json_safe(
            train_metrics
        ),
        "f2_role": "REPORT_ONLY",
        "locked_test_used_for_training": False,
        "external_used_for_training": False,
    }

    # Completion receipt is written last, after weights and predictions exist.
    model.save_weights(run_dir / "final_selected.weights.h5")
    validate_prediction_frame(pred_df, val_df)
    result = seal_receipt(
        json_safe(result), phase2_contract(model_id, resolution, fold=fold),
        phase2_fold_artifacts(run_dir),
    )
    atomic_write_json(run_dir / "COMPLETED.json", result)

    del (
        model,
        backbone,
        train_ds,
        val_ds,
        train_eval_ds,
    )
    tf.keras.backend.clear_session()
    gc.collect()

    return result, pred_df

print("✅ Phase-2 canonical M01–M12 engine ready")
print("Model IDs:", list(CANONICAL_MODELS))

In [ ]:
# ============================================================
# 18) PHASE-2 REPORTING / PERSISTENCE / EVIDENCE HELPERS
# ============================================================

# Receipt identities bind the scientific recipe, model, data and output hashes.
PHASE2_RESTORE_VERIFIED = set()
PHASE2_REMOTE_FOLD_RECEIPTS = {}
PHASE2_REMOTE_FINAL_RECEIPTS = {}

def phase2_contract(model_id, resolution, fold=None, stage="phase2_fold"):
    fixed = {
        "model_spec": asdict(CANONICAL_MODELS[model_id]),
        "campaign_role": "M07_GATE_MULTIRES" if model_id == "M07" else "PHASE2",
        "batch_plan": phase2_batch_plan(resolution),
        "recipe_fingerprint": recipe["recipe_fingerprint_sha256"],
        "static_focal_gamma": PHASE2_STATIC_FOCAL_GAMMA,
        "flsd": [FLSD_THRESHOLD, FLSD_HARD_GAMMA, FLSD_EASY_GAMMA],
        "mix_alpha": PHASE2_MIX_ALPHA, "ema_momentum": PHASE2_EMA_MOMENTUM,
        "swa_start": PHASE2_SWA_START, "cbam_spatial_kernel": CBAM_SPATIAL_KERNEL,
    }
    fold_ids = [int(fold)] if fold is not None else range(1, 6)
    fixed["fold_data"] = {
        str(f): {part: dataframe_identity(load_fold_manifest(f, part)) for part in ("train", "val")}
        for f in fold_ids
    }
    fixed["optimizer_update_schedule"] = {
        str(f): phase2_training_schedule(
            load_fold_manifest(f, "train"),
            resolution,
            balanced_batches=(
                CANONICAL_MODELS[model_id].loss_name != "weighted_bce"
            ),
        )
        for f in fold_ids
    }
    if stage == "phase2_report":
        fixed["reporting"] = {
            "xai": bool(PHASE2_RUN_XAI), "xai_samples": int(PHASE2_XAI_SAMPLES),
            "external": bool(PHASE2_RUN_EXTERNAL), "bootstraps": int(PHASE2_BOOTSTRAPS),
            "bootstrap_unit": "PATIENT_CLUSTER",
        }
        fixed["locked_test_identity"] = dataframe_identity(load_locked_test_manifest())
        sentinel = OUTPUT_ROOT / "EXTERNAL_VALIDATION" / "NIH_SENTINEL_MANIFEST.csv"
        fixed["sentinel_manifest_sha256"] = run_file_sha256(sentinel) if PHASE2_RUN_EXTERNAL and sentinel.is_file() else None
        custom_external_value = str(
            globals().get("CUSTOM_EXTERNAL_MANIFEST", "") or ""
        ).strip()
        custom_external = Path(custom_external_value) if custom_external_value else None
        fixed["custom_external_manifest_sha256"] = (
            run_file_sha256(custom_external)
            if PHASE2_RUN_EXTERNAL and custom_external is not None and custom_external.is_file()
            else None
        )
    return make_run_contract(
        stage, model_id=model_id, resolution=int(resolution), params=shared_params,
        fold_id=int(fold) if fold is not None else None,
        seed=phase2_seed(model_id, resolution, fold) if fold is not None else SEED,
        head_epochs=PHASE2_FINAL_HEAD_EPOCHS, finetune_epochs=PHASE2_FINAL_FINETUNE_EPOCHS,
        extra=fixed,
    )


def phase2_save_validation_predictions(val_df, probability, path):
    """Persist exactly the float64 representation used for metric calculation.

    TensorFlow float32 probabilities are promoted BEFORE CSV formatting. Verify
    a round-trip before publishing COMPLETED; never relax metric tolerances.
    """
    probability = np.asarray(probability, dtype=np.float64)
    if probability.ndim != 1 or len(probability) != len(val_df):
        raise ValueError("Phase-2 validation probability shape mismatch")
    pred = val_df[["relative_path", "patient_id", "model_label", "label", "sha256"]].copy()
    pred["probability_pneumonia"] = probability
    validate_prediction_frame(pred, val_df)
    pred.to_csv(path, index=False, float_format="%.17g")
    restored = pd.read_csv(path, float_precision="round_trip", dtype={
        c: str for c in ("relative_path", "patient_id", "model_label", "sha256")
    })
    validate_prediction_frame(restored, val_df)
    if not np.array_equal(restored["probability_pneumonia"].to_numpy(dtype=np.float64), probability):
        raise ValueError("Phase-2 probability serialization is not lossless")
    return restored


def phase2_validate_history(history, model_id, evidence):
    """Require both completed stages and a verifiable reason for a shortened fit.

    Evidence records callback counters; it is not independent proof that GPU
    training occurred. Validation is run both before sealing and on every reuse.
    """
    validate_m07_history(history, PHASE2_FINAL_HEAD_EPOCHS, PHASE2_FINAL_FINETUNE_EPOCHS)
    spec = CANONICAL_MODELS[model_id]
    head_count = int((history["stage"] == "head").sum())
    fine_count = int((history["stage"] == "finetune").sum())
    if not isinstance(evidence, dict):
        raise ValueError("PHASE2_TRAINING_EVIDENCE_MISSING")
    def integer_field(name):
        value = evidence.get(name)
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or int(value) < 0:
            raise ValueError(f"PHASE2_TRAINING_EVIDENCE_INVALID: {name}")
        return int(value)
    if integer_field("head_epochs_observed") != head_count or integer_field("finetune_epochs_observed") != fine_count:
        raise ValueError("PHASE2_TRAINING_EVIDENCE_EPOCH_COUNT_MISMATCH")
    wait = integer_field("finetune_monitor_wait")
    patience = integer_field("finetune_monitor_patience")
    if patience != int(shared_params["patience"]):
        raise ValueError("PHASE2_EARLY_STOP_PATIENCE_MISMATCH")
    shortened = fine_count < int(PHASE2_FINAL_FINETUNE_EPOCHS)
    expected_reason = "VALIDATION_EARLY_STOPPING" if shortened else "EPOCH_BUDGET_REACHED"
    if evidence.get("finetune_stop_reason") != expected_reason:
        raise ValueError("PHASE2_FINETUNE_STOP_REASON_MISMATCH")
    if shortened and (evidence.get("model_stop_training") is not True or wait <= patience):
        raise ValueError("PHASE2_SHORTENED_FIT_WITHOUT_EARLY_STOP_EVIDENCE")
    snapshots = integer_field("swa_snapshots")
    if spec.use_swa:
        start = min(max(1, int(PHASE2_SWA_START)), int(PHASE2_FINAL_FINETUNE_EPOCHS))
        if fine_count < start or snapshots != fine_count - start + 1:
            raise ValueError("PHASE2_SWA_HISTORY_OR_SNAPSHOT_COUNT_INVALID")
    elif snapshots != 0:
        raise ValueError("PHASE2_UNEXPECTED_SWA_SNAPSHOTS")
    return history


def phase2_fold_artifacts(fold_dir):
    return {name: Path(fold_dir) / name for name in (
        "final_selected.weights.h5", "validation_predictions.csv", "history.csv",
    )}

def phase2_validate_fold(model_id, resolution, fold, fold_dir):
    fold_dir = Path(fold_dir)
    receipt = json.loads((fold_dir / "COMPLETED.json").read_text(encoding="utf-8"))
    validate_receipt(receipt, phase2_contract(model_id, resolution, fold=fold),
                     phase2_fold_artifacts(fold_dir), allowed_statuses={"COMPLETED"})
    if (receipt.get("model_id"), receipt.get("resolution"), receipt.get("fold_id")) != (model_id, int(resolution), int(fold)):
        raise ValueError("Phase-2 fold receipt identity mismatch.")
    try:
        history = pd.read_csv(fold_dir / "history.csv", float_precision="round_trip")
    except (pd.errors.EmptyDataError, pd.errors.ParserError) as exc:
        raise ValueError("PHASE2_TRAINING_HISTORY_UNREADABLE") from exc
    phase2_validate_history(history, model_id, receipt.get("training_evidence"))
    pred = pd.read_csv(fold_dir / "validation_predictions.csv", float_precision="round_trip", dtype={
        "relative_path": str, "patient_id": str, "sha256": str, "model_label": str,
    })
    val = load_fold_manifest(int(fold), "val")
    validate_prediction_frame(pred, val)
    expected = select_balanced_threshold(val["label"].to_numpy(dtype=int), pred["probability_pneumonia"].to_numpy(dtype=float))
    reported = receipt.get("validation_metrics", {})
    if set(reported) != set(expected):
        raise ValueError("Phase-2 saved validation metric fields differ from recomputation.")
    for name, value in expected.items():
        try:
            agrees = np.allclose(np.asarray(reported[name], dtype=float), np.asarray(value, dtype=float),
                                 rtol=1e-8, atol=1e-10, equal_nan=False)
        except (TypeError, ValueError):
            agrees = False
        if not agrees:
            raise ValueError(f"Phase-2 saved validation metric mismatch: {name}")
    return receipt, pred

def phase2_required_report_paths(out):
    names = [
        "QUICK_RESULTS.csv", "OOF_PREDICTIONS.csv", "OOF_METRICS.json", "OOF_DUAL20_80.json",
        "TRAIN_OOF_TEST_COMPARISON.csv", "GENERALIZATION_GAPS.json",
        "LOCKED_TEST/LOCKED_TEST_PREDICTIONS.csv", "LOCKED_TEST/LOCKED_TEST_PRIMARY_METRICS.json",
        "LOCKED_TEST/LOCKED_TEST_BOOTSTRAP.csv", "LOCKED_TEST/LOCKED_TEST_BOOTSTRAP_CI95.json",
        "LOCKED_TEST/INFERENCE_RECEIPT.json", "LOCKED_TEST/INFERENCE_CACHE.npz",
        "DIAGNOSTICS/CALIBRATION_REPORT.json",
    ]
    if PHASE2_RUN_EXTERNAL and str(
        globals().get("CUSTOM_EXTERNAL_MANIFEST", "") or ""
    ).strip():
        names.extend([
            "EXTERNAL/CUSTOM_EXTERNAL/PREDICTIONS.csv",
            "EXTERNAL/CUSTOM_EXTERNAL/METRICS.json",
            "EXTERNAL/CUSTOM_EXTERNAL/DUAL20_80.json",
            "EXTERNAL/CUSTOM_EXTERNAL/BOOTSTRAP.csv",
            "EXTERNAL/CUSTOM_EXTERNAL/BOOTSTRAP_CI95.json",
        ])
    return {name: Path(out) / name for name in names}

def phase2_validate_report(model_id, resolution, out=None):
    out = Path(out) if out is not None else phase2_output_root(model_id, resolution)
    payload = json.loads((out / "FINAL_REPORT.json").read_text(encoding="utf-8"))
    keys = payload.get("artifact_sha256", {})
    required = set(phase2_required_report_paths(out))
    if not isinstance(keys, dict) or not required.issubset(keys):
        raise ValueError("Phase-2 final report lacks required evidence hashes.")
    paths = {}
    for name in keys:
        path = (out / name).resolve()
        if path == out.resolve() or out.resolve() not in path.parents:
            raise ValueError("Unsafe Phase-2 report artifact path.")
        paths[name] = path
    validate_receipt(payload, phase2_contract(model_id, resolution, stage="phase2_report"),
                     paths, allowed_statuses={"COMPLETE"})
    if (payload.get("model_id"), payload.get("resolution")) != (model_id, int(resolution)):
        raise ValueError("Phase-2 report identity mismatch.")
    for f in range(1, 6):
        phase2_validate_fold(model_id, resolution, f, out / "FOLDS" / f"fold_{f}")
    quick = pd.read_csv(out / "QUICK_RESULTS.csv")
    if quick.empty or set(quick["model_id"]) != {model_id} or set(quick["resolution"]) != {int(resolution)}:
        raise ValueError("Phase-2 quick-results model/resolution mismatch.")
    if not {"OOF", "LOCKED_TEST"}.issubset(set(quick["dataset"])) or quick["dataset"].duplicated().any():
        raise ValueError("Phase-2 quick-results datasets incomplete or duplicated.")
    return payload, quick

def phase2_locked_test_bundle(model_id, resolution, frame, thresholds, out):
    root = Path(out) / "LOCKED_TEST"
    root.mkdir(parents=True, exist_ok=True)
    marker = root / "INFERENCE_RECEIPT.json"
    cache = root / "INFERENCE_CACHE.npz"
    weights = {f"fold_{f}_weights": Path(out) / "FOLDS" / f"fold_{f}" / "final_selected.weights.h5" for f in range(1, 6)}
    contract = make_run_contract(
        "phase2_locked_test", model_id=model_id, resolution=int(resolution), params=shared_params,
        extra={"training_contract": phase2_contract(model_id, resolution, stage="phase2_campaign"),
               "test_identity": dataframe_identity(frame), "thresholds": {str(k): float(v) for k, v in thresholds.items()},
               "weight_sha256": {k: run_file_sha256(v) for k, v in weights.items()}},
    )
    if marker.exists() or cache.exists():
        if not marker.is_file() or not cache.is_file():
            raise ValueError("Incomplete Phase-2 locked-test inference. Explicit recovery required; no automatic repeat.")
        payload = json.loads(marker.read_text(encoding="utf-8"))
        validate_receipt(payload, contract, {"cache": cache, **weights}, allowed_statuses={"COMPLETE"})
        with np.load(cache, allow_pickle=False) as loaded:
            P = loaded["P"].copy()
    else:
        atomic_write_json(marker, seal_receipt({"status": "STARTED"}, contract))
        bundle = phase2_predict_five_folds(model_id, resolution, frame, thresholds)
        P = np.asarray(bundle["P"], dtype=float)
        if P.shape != (5, len(frame)) or not np.isfinite(P).all() or ((P < 0) | (P > 1)).any():
            raise ValueError("Invalid Phase-2 locked-test probabilities.")
        np.savez_compressed(cache, P=P)
        atomic_write_json(marker, seal_receipt({"status": "COMPLETE"}, contract, {"cache": cache, **weights}))
    if P.shape != (5, len(frame)) or not np.isfinite(P).all() or ((P < 0) | (P > 1)).any():
        raise ValueError("Invalid Phase-2 locked-test cache dimensions/probabilities.")
    score = np.vstack([logit_np(P[f - 1]) - logit_np(float(thresholds[f])) for f in range(1, 6)]).mean(axis=0)
    majority = (np.vstack([P[f - 1] >= thresholds[f] for f in range(1, 6)]).sum(axis=0) >= 3).astype(int)
    return {"P": P, "score": score, "mean_probability": P.mean(axis=0),
            "primary_pred": (score >= 0).astype(int), "majority_pred": majority}

def phase2_seal_persistence(model_id, resolution):
    state = phase2_state_root(model_id, resolution)
    artifacts = {p.name: p for p in state.glob("*.zip")}
    atomic_write_json(state / "CAMPAIGN_STATE.json", seal_receipt(
        {"schema": "phase2.state.v2", "status": "COMPLETE", "model_id": model_id,
         "resolution": int(resolution), "updated_utc": pd.Timestamp.utcnow().isoformat()},
        phase2_contract(model_id, resolution, stage="phase2_campaign"), artifacts,
    ))

def phase2_slug(model_id, resolution):
    if str(model_id) == "M07":
        return f"m07-gate-r{int(resolution)}-state-v1-7"
    return f"pneumonia-{model_id.lower()}-r{int(resolution)}-state-v1-7"


def phase2_handle(
    model_id,
    resolution,
):
    return (
        f"{PHASE2_PERSIST_OWNER}/"
        f"{phase2_slug(model_id, resolution)}"
    )

def phase2_state_root(model_id, resolution):
    if str(model_id) == "M07":
        root = M07_GATE_MULTIRES_ROOT / "STATE" / f"R{int(resolution)}"
    else:
        root = PHASE2_CAMPAIGN_ROOT / "STATE" / model_id / f"R{int(resolution)}"
    root.mkdir(parents=True, exist_ok=True)
    return root


def phase2_output_root(model_id, resolution):
    if str(model_id) == "M07":
        root = M07_GATE_MULTIRES_ROOT / "RUNS" / f"R{int(resolution)}"
    else:
        root = PHASE2_CAMPAIGN_ROOT / "RUNS" / model_id / f"R{int(resolution)}"
    root.mkdir(parents=True, exist_ok=True)
    return root


def phase2_upload(
    model_id,
    resolution,
    note,
):
    root = phase2_state_root(
        model_id,
        resolution,
    )
    if (model_id, int(resolution)) not in PHASE2_RESTORE_VERIFIED:
        raise RuntimeError("Phase-2 upload blocked until remote reconciliation succeeds.")
    kagglehub.dataset_upload(
        phase2_handle(
            model_id,
            resolution,
        ),
        str(root),
        version_notes=str(note),
    )

def phase2_restore(model_id, resolution):
    identity = (model_id, int(resolution))
    PHASE2_RESTORE_VERIFIED.discard(identity)
    PHASE2_REMOTE_FINAL_RECEIPTS.pop(identity, None)
    for fold in range(1, 6):
        PHASE2_REMOTE_FOLD_RECEIPTS.pop((*identity, fold), None)
    out = phase2_output_root(model_id, resolution)
    state = phase2_state_root(model_id, resolution)
    temp = WORK / "_PHASE2_RESTORE_V17" / model_id / f"R{int(resolution)}"
    if temp.exists():
        shutil.rmtree(temp)
    temp.mkdir(parents=True, exist_ok=True)
    try:
        kagglehub.dataset_download(phase2_handle(model_id, resolution), output_dir=str(temp), force_download=True)
    except Exception as exc:
        if _is_explicit_http_not_found(exc):
            PHASE2_RESTORE_VERIFIED.add(identity)
            print(f"[{model_id} R{resolution}] first run: remote state returned HTTP404.")
            return 0
        raise RuntimeError(f"{model_id} R{resolution}: restore unavailable; refusing blind continuation/upload.") from exc
    marker = temp / "CAMPAIGN_STATE.json"
    if not marker.is_file():
        raise ValueError("Phase-2 remote state lacks its sealed campaign manifest.")
    payload = json.loads(marker.read_text(encoding="utf-8"))
    archives = {p.name: p for p in temp.glob("*.zip")}
    if set(archives) != set(payload.get("artifact_sha256", {})) or not archives:
        raise ValueError("Phase-2 archive inventory mismatch.")
    if any(not re.fullmatch(r"(?:FOLD_[1-5]_RECOVERY|FINAL_EVIDENCE)\.zip", name) for name in archives):
        raise ValueError("Unexpected Phase-2 archive name.")
    validate_receipt(payload, phase2_contract(model_id, resolution, stage="phase2_campaign"),
                     archives, allowed_statuses={"COMPLETE"})
    staged = temp / "_validated_output"
    staged.mkdir()
    for name, archive in sorted(archives.items()):
        safe_extract_zip(archive, staged)
    completed = []
    for fold in range(1, 6):
        fold_dir = staged / "FOLDS" / f"fold_{fold}"
        if fold_dir.exists():
            phase2_validate_fold(model_id, resolution, fold, fold_dir)
            completed.append(fold)
    if (staged / "FINAL_REPORT.json").exists():
        phase2_validate_report(model_id, resolution, staged)
    if not completed:
        raise ValueError("Phase-2 remote snapshot contains no validated completed fold.")
    # All receipts validate before touching active outputs. Never overwrite divergent local evidence.
    for source in staged.rglob("*"):
        if source.is_file():
            target = out / source.relative_to(staged)
            if target.exists() and run_file_sha256(target) != run_file_sha256(source):
                raise ValueError(f"Phase-2 local/remote evidence conflict at {target.name}; explicit reconciliation required.")
    for source in staged.rglob("*"):
        if source.is_file():
            target = out / source.relative_to(staged)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
    for source in [marker, *archives.values()]:
        shutil.copy2(source, state / source.name)
    PHASE2_RESTORE_VERIFIED.add(identity)
    for fold in completed:
        PHASE2_REMOTE_FOLD_RECEIPTS[(*identity, fold)] = run_file_sha256(staged / "FOLDS" / f"fold_{fold}" / "COMPLETED.json")
    if (staged / "FINAL_REPORT.json").is_file():
        PHASE2_REMOTE_FINAL_RECEIPTS[identity] = run_file_sha256(staged / "FINAL_REPORT.json")
    return len(completed)

def phase2_persist_fold(
    model_id,
    resolution,
    fold,
):
    out = phase2_output_root(
        model_id,
        resolution,
    )
    state = phase2_state_root(
        model_id,
        resolution,
    )
    fold_dir = (
        out
        / "FOLDS"
        / f"fold_{int(fold)}"
    )

    phase2_validate_fold(model_id, resolution, fold, fold_dir)
    zpath = state / f"FOLD_{int(fold)}_RECOVERY.zip"

    if zpath.exists():
        zpath.unlink()

    with zipfile.ZipFile(
        zpath,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as z:
        for p in fold_dir.rglob("*"):
            if p.is_file():
                z.write(
                    p,
                    arcname=str(
                        Path("FOLDS")
                        / f"fold_{int(fold)}"
                        / p.relative_to(
                            fold_dir
                        )
                    ),
                )

    phase2_seal_persistence(model_id, resolution)

    phase2_upload(
        model_id,
        resolution,
        note=(
            f"{model_id} R{resolution} completed fold {fold}; "
            f"validated fold receipt and archive sealed"
        ),
    )

    # This is recorded only after upload returns successfully.
    PHASE2_REMOTE_FOLD_RECEIPTS[(model_id, int(resolution), int(fold))] = run_file_sha256(fold_dir / "COMPLETED.json")


def phase2_ensure_fold_persisted(model_id, resolution, fold, fold_dir):
    phase2_validate_fold(model_id, resolution, fold, fold_dir)
    identity = (model_id, int(resolution))
    if identity not in PHASE2_RESTORE_VERIFIED:
        raise RuntimeError("Reconcile Phase-2 remote state before a completed-fold skip.")
    receipt_hash = run_file_sha256(Path(fold_dir) / "COMPLETED.json")
    if PHASE2_REMOTE_FOLD_RECEIPTS.get((*identity, int(fold))) == receipt_hash:
        return False
    phase2_persist_fold(model_id, resolution, fold)
    return True


def phase2_ensure_final_persisted(model_id, resolution, out=None):
    out = Path(out) if out is not None else phase2_output_root(model_id, resolution)
    phase2_validate_report(model_id, resolution, out)
    identity = (model_id, int(resolution))
    if identity not in PHASE2_RESTORE_VERIFIED:
        raise RuntimeError("Reconcile Phase-2 remote state before a final-report skip.")
    receipt_hash = run_file_sha256(out / "FINAL_REPORT.json")
    if PHASE2_REMOTE_FINAL_RECEIPTS.get(identity) == receipt_hash:
        return False
    state = phase2_state_root(model_id, resolution)
    final_archive = state / "FINAL_EVIDENCE.zip"
    with zipfile.ZipFile(final_archive, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(out.rglob("*")):
            if path.is_file():
                archive.write(path, arcname=path.relative_to(out).as_posix())
    phase2_seal_persistence(model_id, resolution)
    phase2_upload(model_id, resolution, note=f"{model_id} R{resolution} validated final evidence complete")
    # Failed uploads leave these entries absent, so the next run retries persistence.
    PHASE2_REMOTE_FINAL_RECEIPTS[identity] = receipt_hash
    for fold in range(1, 6):
        PHASE2_REMOTE_FOLD_RECEIPTS[(*identity, fold)] = run_file_sha256(out / "FOLDS" / f"fold_{fold}" / "COMPLETED.json")
    return True

def phase2_save_basic_figures(
    root,
    y,
    probability,
    prediction,
    title_prefix,
):
    root = Path(root)
    root.mkdir(
        parents=True,
        exist_ok=True,
    )

    metrics = calculate_binary_metrics(
        y,
        prediction,
        score=probability,
    )

    save_confusion_figure(
        metrics[
            "confusion_matrix"
        ],
        f"{title_prefix} Confusion Matrix",
        root
        / "confusion_matrix.png",
    )
    save_roc_figure(
        y,
        probability,
        f"{title_prefix} ROC",
        root
        / "roc_curve.png",
    )
    save_pr_figure(
        y,
        probability,
        f"{title_prefix} Precision–Recall",
        root
        / "pr_curve.png",
    )
    save_dual_probability_figure(
        y,
        probability,
        f"{title_prefix} — Fixed 20/80",
        root
        / "dual20_80_probability.png",
    )
    return metrics

def phase2_save_fold_history_figures(
    fold_dir,
    head_epochs,
):
    fold_dir = Path(
        fold_dir
    )
    path = (
        fold_dir
        / "history.csv"
    )
    if not path.is_file():
        return

    h = pd.read_csv(
        path
    ).reset_index(
        drop=True
    )
    h[
        "global_epoch"
    ] = np.arange(
        1,
        len(h) + 1,
    )

    best_epoch = None
    state_files = list(
        fold_dir.glob(
            "*.state.json"
        )
    )
    for s in state_files:
        try:
            payload = json.loads(
                s.read_text(
                    encoding="utf-8"
                )
            )
            if payload.get(
                "best_epoch"
            ):
                best_epoch = int(
                    payload[
                        "best_epoch"
                    ]
                )
                break
        except Exception:
            pass

    specs = [
        (
            "loss",
            "val_loss",
            "Loss",
        ),
        (
            "precision",
            "val_precision",
            "Precision",
        ),
        (
            "recall",
            "val_recall",
            "Recall",
        ),
        (
            "auc_roc",
            "val_auc_roc",
            "AUROC",
        ),
        (
            "auc_pr",
            "val_auc_pr",
            "AUPRC",
        ),
    ]

    for train_col, val_col, title in specs:
        if train_col not in h.columns:
            continue
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
        ax.plot(
            h["global_epoch"],
            h[train_col],
            label=f"Train {title}",
        )
        if val_col in h.columns:
            ax.plot(
                h["global_epoch"],
                h[val_col],
                label=f"Validation {title}",
            )
        ax.axvline(
            int(head_epochs) + 0.5,
            linestyle="--",
            label="Fine-tuning begins",
        )
        if best_epoch:
            ax.axvline(
                best_epoch,
                linestyle=":",
                label=f"Best epoch {best_epoch}",
            )
        ax.set_xlabel(
            "Global epoch"
        )
        ax.set_ylabel(
            title
        )
        ax.set_title(
            f"{fold_dir.parent.parent.name} "
            f"{fold_dir.parent.name} "
            f"{fold_dir.name} — {title}"
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            fold_dir
            / f"history_{title.lower()}.png",
            dpi=180,
        )
        plt.close(fig)

    if (
        "val_min_dual_precision_recall_selected"
        in h.columns
    ):
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
        ax.plot(
            h[
                "global_epoch"
            ],
            h[
                "val_min_dual_precision_recall_selected"
            ],
            label="Validation min-dual P/R",
        )
        if (
            "val_dual_precision_recall_score"
            in h.columns
        ):
            ax.plot(
                h[
                    "global_epoch"
                ],
                h[
                    "val_dual_precision_recall_score"
                ],
                label="Validation dual P/R score",
            )
        ax.axvline(
            int(head_epochs) + 0.5,
            linestyle="--",
        )
        ax.set_ylim(
            0,
            1.02,
        )
        ax.set_title(
            "Precision/Recall Balance"
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            fold_dir
            / "history_min_dual_precision_recall.png",
            dpi=180,
        )
        plt.close(fig)

    if (
        "val_selected_threshold"
        in h.columns
    ):
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
        ax.plot(
            h[
                "global_epoch"
            ],
            h[
                "val_selected_threshold"
            ],
            marker="o",
        )
        ax.axvline(
            int(head_epochs) + 0.5,
            linestyle="--",
        )
        ax.set_title(
            "Validation-selected Threshold"
        )
        fig.tight_layout()
        fig.savefig(
            fold_dir
            / "history_selected_threshold.png",
            dpi=180,
        )
        plt.close(fig)

    if (
        "learning_rate"
        in h.columns
    ):
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
        ax.plot(
            h[
                "global_epoch"
            ],
            h[
                "learning_rate"
            ],
        )
        ax.set_yscale(
            "log"
        )
        ax.axvline(
            int(head_epochs) + 0.5,
            linestyle="--",
        )
        ax.set_title(
            "Learning Rate"
        )
        fig.tight_layout()
        fig.savefig(
            fold_dir
            / "history_learning_rate.png",
            dpi=180,
        )
        plt.close(fig)

def phase2_predict_five_folds(
    model_id,
    resolution,
    frame,
    fold_thresholds_local,
):
    spec = CANONICAL_MODELS[
        model_id
    ]
    batch_size = phase2_batch_size(
        resolution
    )
    ds = phase2_build_dataset(
        frame,
        resolution,
        batch_size,
        False,
        SEED,
        batch_policy="none",
        balanced_batches=False,
    )

    probs = []

    for fold in range(
        1,
        6,
    ):
        tf.keras.backend.clear_session()
        gc.collect()

        model, _ = phase2_build_model(
            spec,
            resolution,
            dropout=float(
                shared_params[
                    "dropout"
                ]
            ),
            seed=(
                SEED
                + fold
                + int(
                    resolution
                ) * 10
                + int(
                    model_id[1:]
                ) * 1000
            ),
        )

        model.load_weights(
            phase2_output_root(
                model_id,
                resolution,
            )
            / "FOLDS"
            / f"fold_{fold}"
            / "final_selected.weights.h5"
        )

        probs.append(
            model.predict(
                ds,
                verbose=0,
            ).reshape(-1)
        )

        del model
        tf.keras.backend.clear_session()
        gc.collect()

    P = np.vstack(
        probs
    )

    centered = np.vstack([
        logit_np(
            P[i]
        )
        - logit_np(
            float(
                fold_thresholds_local[
                    i + 1
                ]
            )
        )
        for i in range(5)
    ])

    score = centered.mean(
        axis=0
    )
    primary_pred = (
        score >= 0.0
    ).astype(int)
    mean_probability = P.mean(
        axis=0
    )

    votes = np.vstack([
        (
            P[i]
            >= float(
                fold_thresholds_local[
                    i + 1
                ]
            )
        ).astype(int)
        for i in range(5)
    ])
    majority = (
        votes.sum(
            axis=0
        )
        >= 3
    ).astype(int)

    return {
        "P": P,
        "score": score,
        "mean_probability": mean_probability,
        "primary_pred": primary_pred,
        "majority_pred": majority,
    }

def phase2_bootstrap_ci(y, pred, score, point_metrics, patient_ids):
    patient_ids = np.asarray(patient_ids, dtype=object)
    if len(patient_ids) != len(y) or pd.isna(patient_ids).any() or any(not str(x).strip() for x in patient_ids):
        raise ValueError("Phase-2 confidence intervals require complete real patient IDs.")
    boot = bootstrap_fixed_rule_by_patient(
        y=y, pred=pred, score=score, patient_ids=patient_ids,
        n_boot=PHASE2_BOOTSTRAPS, seed=SEED + 991,
    ).rename(columns={"f2": "f2_report_only"})
    if boot.empty:
        raise ValueError("No valid patient-cluster bootstrap replicate; report cannot claim a confidence interval.")
    ci = {}
    for c in boot.columns:
        point_key = "f2" if c == "f2_report_only" else c
        ci[c] = {
            "point": float(point_metrics[point_key]) if point_key in point_metrics else None,
            "ci95_low": float(boot[c].quantile(0.025)),
            "ci95_high": float(boot[c].quantile(0.975)),
            "n_bootstrap_valid": int(len(boot)), "bootstrap_unit": "PATIENT_CLUSTER",
            "n_patients": int(pd.Series(patient_ids).nunique()),
        }
    return boot, ci


In [ ]:
# ============================================================
# 19) RUN ONE PHASE-2 MODEL × RESOLUTION
# ============================================================

def run_phase2_model_resolution(
    model_id,
    resolution,
):
    spec = CANONICAL_MODELS[
        model_id
    ]
    out = phase2_output_root(
        model_id,
        resolution,
    )
    folds_root = (
        out
        / "FOLDS"
    )
    folds_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    execution_role = "M07_GATE_MULTIRES" if model_id == "M07" else "PHASE2"
    print("\n" + "#" * 118)
    print(
        f"{execution_role} | {model_id} | R{resolution} | {spec.description}"
    )
    print("#" * 118)

    if (model_id, int(resolution)) not in PHASE2_RESTORE_VERIFIED:
        restored = phase2_restore(model_id, resolution)
        print("Restored Fold snapshots:", restored)

    if (out / "FINAL_REPORT.json").exists():
        restored_report, _ = phase2_validate_report(model_id, resolution, out)
        phase2_ensure_final_persisted(model_id, resolution, out)
        return restored_report

    fold_results = []
    oof_parts = []
    thresholds = {}

    # --------------------------------------------------------
    # Five final folds
    # --------------------------------------------------------
    for fold in range(
        1,
        6,
    ):
        fold_dir = (
            folds_root
            / f"fold_{fold}"
        )
        completed = (
            fold_dir
            / "COMPLETED.json"
        )
        pred_path = (
            fold_dir
            / "validation_predictions.csv"
        )

        if completed.exists() or pred_path.exists() or (fold_dir / "final_selected.weights.h5").exists():
            payload, pred = phase2_validate_fold(model_id, resolution, fold, fold_dir)
            phase2_ensure_fold_persisted(model_id, resolution, fold, fold_dir)
            print(f"[{model_id} R{resolution} F{fold}] validated completion and persistence — SKIP")
        else:
            payload, pred = (
                phase2_train_one_fold(
                    model_id,
                    resolution,
                    fold,
                    fold_dir,
                )
            )
            phase2_save_fold_history_figures(
                fold_dir,
                PHASE2_FINAL_HEAD_EPOCHS,
            )
            phase2_persist_fold(
                model_id,
                resolution,
                fold,
            )

        fold_results.append(
            payload
        )
        pred[
            "fold_id"
        ] = int(fold)
        oof_parts.append(
            pred
        )
        thresholds[
            fold
        ] = float(
            payload[
                "validation_metrics"
            ][
                "threshold"
            ]
        )

    # --------------------------------------------------------
    # OOF
    # --------------------------------------------------------
    oof = pd.concat(
        oof_parts,
        ignore_index=True,
    )
    if (
        len(oof)
        != len(
            development_manifest
        )
    ):
        raise RuntimeError(
            f"{model_id} R{resolution}: OOF size mismatch."
        )

    y_oof = oof[
        "label"
    ].to_numpy(dtype=int)
    p_oof = oof[
        "probability_pneumonia"
    ].to_numpy(dtype=float)
    pred_oof = np.asarray([
        int(
            p
            >= thresholds[
                int(f)
            ]
        )
        for p, f
        in zip(
            p_oof,
            oof[
                "fold_id"
            ].to_numpy(dtype=int),
        )
    ])

    oof[
        "prediction_fold_threshold"
    ] = pred_oof

    oof_metrics = calculate_binary_metrics(
        y_oof,
        pred_oof,
        score=p_oof,
    )

    oof_dual, oof_dual_decision = (
        analyze_dual_threshold(
            y_oof,
            p_oof,
        )
    )
    oof[
        "dual20_80_decision"
    ] = oof_dual_decision

    oof.to_csv(
        out
        / "OOF_PREDICTIONS.csv",
        index=False,
    )
    (
        out
        / "OOF_METRICS.json"
    ).write_text(
        json.dumps(
            json_safe(
                oof_metrics
            ),
            indent=2,
        ),
        encoding="utf-8",
    )
    (
        out
        / "OOF_DUAL20_80.json"
    ).write_text(
        json.dumps(
            json_safe(
                oof_dual
            ),
            indent=2,
        ),
        encoding="utf-8",
    )

    oof_fig_root = (
        out
        / "FIGURES"
        / "OOF"
    )
    phase2_save_basic_figures(
        oof_fig_root,
        y_oof,
        p_oof,
        pred_oof,
        f"{model_id} R{resolution} OOF",
    )

    # --------------------------------------------------------
    # One-time predeclared Locked-Test REPORTING
    # Never used for model selection/tuning.
    # --------------------------------------------------------
    test_frame = (
        load_locked_test_manifest()
    )

    dev_patients = set(
        development_manifest[
            "patient_id"
        ].astype(str)
    )
    test_patients = set(
        test_frame[
            "patient_id"
        ].astype(str)
    )
    dev_sha = set(
        development_manifest[
            "sha256"
        ].astype(str)
    )
    test_sha = set(
        test_frame[
            "sha256"
        ].astype(str)
    )

    if (
        dev_patients
        & test_patients
    ):
        raise RuntimeError(
            f"{model_id} R{resolution}: patient leakage before Test."
        )
    if dev_sha & test_sha:
        raise RuntimeError(
            f"{model_id} R{resolution}: SHA leakage before Test."
        )

    test_bundle = phase2_locked_test_bundle(
        model_id, resolution, test_frame, thresholds, out,
    )

    y_test_local = test_frame[
        "label"
    ].to_numpy(dtype=int)

    primary_metrics_local = (
        calculate_binary_metrics(
            y_test_local,
            test_bundle[
                "primary_pred"
            ],
            score=test_bundle[
                "score"
            ],
        )
    )
    majority_metrics_local = (
        calculate_binary_metrics(
            y_test_local,
            test_bundle[
                "majority_pred"
            ],
            score=test_bundle[
                "mean_probability"
            ],
        )
    )

    test_dual, test_dual_decision = (
        analyze_dual_threshold(
            y_test_local,
            test_bundle[
                "mean_probability"
            ],
        )
    )

    test_root = (
        out
        / "LOCKED_TEST"
    )
    test_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    test_pred = test_frame[
        [
            "relative_path",
            "patient_id",
            "model_label",
            "label",
            "sha256",
        ]
    ].copy()

    for i in range(5):
        test_pred[
            f"probability_fold_{i+1}"
        ] = test_bundle[
            "P"
        ][i]
        test_pred[
            f"threshold_fold_{i+1}"
        ] = thresholds[
            i + 1
        ]

    test_pred[
        "mean_probability"
    ] = test_bundle[
        "mean_probability"
    ]
    test_pred[
        "normalized_ensemble_score"
    ] = test_bundle[
        "score"
    ]
    test_pred[
        "prediction_primary"
    ] = test_bundle[
        "primary_pred"
    ]
    test_pred[
        "prediction_majority"
    ] = test_bundle[
        "majority_pred"
    ]
    test_pred[
        "dual20_80_decision"
    ] = test_dual_decision

    test_pred.to_csv(
        test_root
        / "LOCKED_TEST_PREDICTIONS.csv",
        index=False,
    )

    (
        test_root
        / "LOCKED_TEST_PRIMARY_METRICS.json"
    ).write_text(
        json.dumps(
            json_safe(
                primary_metrics_local
            ),
            indent=2,
        ),
        encoding="utf-8",
    )
    (
        test_root
        / "LOCKED_TEST_MAJORITY_METRICS.json"
    ).write_text(
        json.dumps(
            json_safe(
                majority_metrics_local
            ),
            indent=2,
        ),
        encoding="utf-8",
    )
    (
        test_root
        / "LOCKED_TEST_DUAL20_80.json"
    ).write_text(
        json.dumps(
            json_safe(
                test_dual
            ),
            indent=2,
        ),
        encoding="utf-8",
    )

    # Bootstrap.
    boot, boot_ci = phase2_bootstrap_ci(
        y_test_local,
        test_bundle[
            "primary_pred"
        ],
        test_bundle[
            "score"
        ],
        primary_metrics_local,
        patient_ids=test_frame["patient_id"].astype(str).to_numpy(),
    )
    boot.to_csv(
        test_root
        / "LOCKED_TEST_BOOTSTRAP.csv",
        index=False,
    )
    (
        test_root
        / "LOCKED_TEST_BOOTSTRAP_CI95.json"
    ).write_text(
        json.dumps(
            json_safe(
                boot_ci
            ),
            indent=2,
        ),
        encoding="utf-8",
    )

    test_fig_root = (
        out
        / "FIGURES"
        / "LOCKED_TEST"
    )
    phase2_save_basic_figures(
        test_fig_root,
        y_test_local,
        test_bundle[
            "mean_probability"
        ],
        test_bundle[
            "primary_pred"
        ],
        f"{model_id} R{resolution} Locked Test",
    )

    # --------------------------------------------------------
    # Calibration / reliability
    # --------------------------------------------------------
    diag_root = (
        out
        / "DIAGNOSTICS"
    )
    diag_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    oof_ece_local = save_reliability_plot(
        y_oof,
        p_oof,
        f"{model_id} R{resolution} OOF Reliability",
        diag_root
        / "OOF_RELIABILITY.png",
        diag_root
        / "OOF_CALIBRATION_BINS.csv",
    )
    test_ece_local = save_reliability_plot(
        y_test_local,
        test_bundle[
            "mean_probability"
        ],
        f"{model_id} R{resolution} Test Reliability",
        diag_root
        / "TEST_RELIABILITY.png",
        diag_root
        / "TEST_CALIBRATION_BINS.csv",
    )

    calibration = {
        "fit_performed": False,
        "oof_ece": float(
            oof_ece_local
        ),
        "oof_brier": float(
            brier_score_loss(
                y_oof,
                p_oof,
            )
        ),
        "test_ece": float(
            test_ece_local
        ),
        "test_brier": float(
            brier_score_loss(
                y_test_local,
                test_bundle[
                    "mean_probability"
                ],
            )
        ),
    }
    (
        diag_root
        / "CALIBRATION_REPORT.json"
    ).write_text(
        json.dumps(
            json_safe(
                calibration
            ),
            indent=2,
        ),
        encoding="utf-8",
    )

    # --------------------------------------------------------
    # Risk-Coverage
    # --------------------------------------------------------
    test_rc = risk_coverage_table(
        y_test_local,
        test_bundle[
            "primary_pred"
        ],
        np.abs(
            test_bundle[
                "score"
            ]
        ),
    )
    test_rc.to_csv(
        diag_root
        / "TEST_RISK_COVERAGE.csv",
        index=False,
    )
    fig, ax = plt.subplots(
        figsize=(7, 5)
    )
    ax.plot(
        test_rc[
            "coverage"
        ],
        test_rc[
            "risk_error_rate"
        ],
        marker="o",
    )
    ax.set_xlabel(
        "Coverage"
    )
    ax.set_ylabel(
        "Risk = 1 - Accuracy"
    )
    ax.set_title(
        f"{model_id} R{resolution} Risk–Coverage"
    )
    fig.tight_layout()
    fig.savefig(
        diag_root
        / "TEST_RISK_COVERAGE.png",
        dpi=180,
    )
    plt.close(fig)

    # --------------------------------------------------------
    # Decision Curve
    # --------------------------------------------------------
    test_dc = decision_curve_table(
        y_test_local,
        test_bundle[
            "mean_probability"
        ],
    )
    test_dc.to_csv(
        diag_root
        / "TEST_DECISION_CURVE.csv",
        index=False,
    )
    fig, ax = plt.subplots(
        figsize=(7, 5)
    )
    ax.plot(
        test_dc[
            "threshold"
        ],
        test_dc[
            "net_benefit_model"
        ],
        label="Model",
    )
    ax.plot(
        test_dc[
            "threshold"
        ],
        test_dc[
            "net_benefit_treat_all"
        ],
        label="Treat All",
    )
    ax.plot(
        test_dc[
            "threshold"
        ],
        test_dc[
            "net_benefit_treat_none"
        ],
        label="Treat None",
    )
    ax.set_xlabel(
        "Decision threshold"
    )
    ax.set_ylabel(
        "Net benefit"
    )
    ax.set_title(
        f"{model_id} R{resolution} Decision Curve"
    )
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        diag_root
        / "TEST_DECISION_CURVE.png",
        dpi=180,
    )
    plt.close(fig)

    # --------------------------------------------------------
    # Five-fold Loss summary.
    # --------------------------------------------------------
    loss_rows = []
    val_loss_long = []

    for fold in range(
        1,
        6,
    ):
        fold_dir = (
            folds_root
            / f"fold_{fold}"
        )
        phase2_save_fold_history_figures(
            fold_dir,
            PHASE2_FINAL_HEAD_EPOCHS,
        )
        hp = (
            fold_dir
            / "history.csv"
        )
        if not hp.is_file():
            continue
        h = pd.read_csv(
            hp
        ).reset_index(
            drop=True
        )
        h[
            "global_epoch"
        ] = np.arange(
            1,
            len(h) + 1,
        )
        for e, v in zip(
            h[
                "global_epoch"
            ],
            h[
                "val_loss"
            ],
        ):
            if pd.notna(v):
                val_loss_long.append({
                    "fold": fold,
                    "epoch": int(e),
                    "val_loss": float(v),
                })

        completed = json.loads(
            (
                fold_dir
                / "COMPLETED.json"
            ).read_text(
                encoding="utf-8"
            )
        )
        vm = completed[
            "validation_metrics"
        ]
        tm = completed[
            "train_metrics"
        ]
        loss_rows.append({
            "fold": fold,
            "best_threshold": vm[
                "threshold"
            ],
            "train_balanced_accuracy": tm[
                "balanced_accuracy"
            ],
            "validation_balanced_accuracy": vm[
                "balanced_accuracy"
            ],
            "train_macro_f1": tm[
                "macro_f1"
            ],
            "validation_macro_f1": vm[
                "macro_f1"
            ],
            "train_f2_report_only": tm[
                "f2"
            ],
            "validation_f2_report_only": vm[
                "f2"
            ],
            "final_train_loss": float(
                h[
                    "loss"
                ].iloc[-1]
            ),
            "final_validation_loss": float(
                h[
                    "val_loss"
                ].iloc[-1]
            ),
        })

    loss_root = (
        out
        / "FIGURES"
        / "LOSS"
    )
    loss_root.mkdir(
        parents=True,
        exist_ok=True,
    )
    pd.DataFrame(
        loss_rows
    ).to_csv(
        loss_root
        / "LOSS_SUMMARY.csv",
        index=False,
    )

    vlf = pd.DataFrame(
        val_loss_long
    )
    if not vlf.empty:
        fig, ax = plt.subplots(
            figsize=(9, 5)
        )
        for fold, g in vlf.groupby(
            "fold"
        ):
            ax.plot(
                g[
                    "epoch"
                ],
                g[
                    "val_loss"
                ],
                label=f"Fold {fold}",
            )
        ax.set_title(
            f"{model_id} R{resolution} Five-Fold Validation Loss"
        )
        ax.set_xlabel(
            "Epoch"
        )
        ax.set_ylabel(
            "Loss"
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            loss_root
            / "5FOLD_VALIDATION_LOSS.png",
            dpi=180,
        )
        plt.close(fig)

        agg = (
            vlf.groupby(
                "epoch"
            )[
                "val_loss"
            ]
            .agg(
                ["mean", "std"]
            )
            .reset_index()
        )
        agg[
            "std"
        ] = agg[
            "std"
        ].fillna(
            0.0
        )
        agg.to_csv(
            loss_root
            / "MEAN_SD_VALIDATION_LOSS.csv",
            index=False,
        )

        fig, ax = plt.subplots(
            figsize=(9, 5)
        )
        ax.plot(
            agg[
                "epoch"
            ],
            agg[
                "mean"
            ],
            label="Mean Validation Loss",
        )
        ax.fill_between(
            agg[
                "epoch"
            ],
            agg[
                "mean"
            ]
            - agg[
                "std"
            ],
            agg[
                "mean"
            ]
            + agg[
                "std"
            ],
            alpha=0.20,
            label="±1 SD",
        )
        ax.set_title(
            f"{model_id} R{resolution} Mean±SD Validation Loss"
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            loss_root
            / "MEAN_SD_VALIDATION_LOSS.png",
            dpi=180,
        )
        plt.close(fig)

    # --------------------------------------------------------
    # Train / OOF / Test generalization summary.
    # --------------------------------------------------------
    train_mean = {}
    metric_keys = [
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "mcc",
        "auroc",
        "precision_normal",
        "recall_normal",
        "f1_normal",
        "precision_pneumonia",
        "recall_pneumonia",
        "f1_pneumonia",
        "f2",
    ]

    for key in metric_keys:
        values = [
            float(
                fr[
                    "train_metrics"
                ][key]
            )
            for fr
            in fold_results
            if key
            in fr[
                "train_metrics"
            ]
        ]
        if values:
            train_mean[
                key
            ] = float(
                np.mean(
                    values
                )
            )

    generalization_rows = []
    for name, metrics in (
        (
            "TRAIN_MEAN",
            train_mean,
        ),
        (
            "OOF",
            oof_metrics,
        ),
        (
            "LOCKED_TEST",
            primary_metrics_local,
        ),
    ):
        generalization_rows.append({
            "dataset": name,
            **{
                k: metrics.get(
                    k
                )
                for k in metric_keys
            },
        })

    gen_df = pd.DataFrame(
        generalization_rows
    )
    gen_df.to_csv(
        out
        / "TRAIN_OOF_TEST_COMPARISON.csv",
        index=False,
    )

    gaps = {
        "train_to_oof_balanced_accuracy": (
            train_mean.get(
                "balanced_accuracy",
                np.nan,
            )
            - oof_metrics[
                "balanced_accuracy"
            ]
        ),
        "oof_to_test_balanced_accuracy": (
            oof_metrics[
                "balanced_accuracy"
            ]
            - primary_metrics_local[
                "balanced_accuracy"
            ]
        ),
        "train_to_oof_macro_f1": (
            train_mean.get(
                "macro_f1",
                np.nan,
            )
            - oof_metrics[
                "macro_f1"
            ]
        ),
        "oof_to_test_macro_f1": (
            oof_metrics[
                "macro_f1"
            ]
            - primary_metrics_local[
                "macro_f1"
            ]
        ),
        "f2_role": "REPORT_ONLY",
    }
    (
        out
        / "GENERALIZATION_GAPS.json"
    ).write_text(
        json.dumps(
            json_safe(
                gaps
            ),
            indent=2,
        ),
        encoding="utf-8",
    )

    # --------------------------------------------------------
    # Optional generic XAI.
    # --------------------------------------------------------
    xai_report = {
        "enabled": bool(
            PHASE2_RUN_XAI
        ),
        "status": "SKIPPED",
    }

    if PHASE2_RUN_XAI:
        xai_root = (
            out
            / "XAI"
        )
        xai_root.mkdir(
            parents=True,
            exist_ok=True,
        )

        fold1_pred = pd.read_csv(
            folds_root
            / "fold_1"
            / "validation_predictions.csv"
        )
        t1 = thresholds[
            1
        ]
        fold1_pred[
            "pred"
        ] = (
            fold1_pred[
                "probability_pneumonia"
            ]
            >= t1
        ).astype(int)
        fold1_pred[
            "distance"
        ] = np.abs(
            fold1_pred[
                "probability_pneumonia"
            ]
            - t1
        )
        selected = (
            fold1_pred.sort_values(
                "distance"
            )
            .head(
                PHASE2_XAI_SAMPLES
            )
        )
        selected.to_csv(
            xai_root
            / "selected_samples.csv",
            index=False,
        )

        tf.keras.backend.clear_session()
        gc.collect()
        xai_model, _ = phase2_build_model(
            spec,
            resolution,
            dropout=float(
                shared_params[
                    "dropout"
                ]
            ),
            seed=(
                SEED
                + 1
                + int(
                    resolution
                ) * 10
                + int(
                    model_id[1:]
                ) * 1000
            ),
        )
        xai_model.load_weights(
            folds_root
            / "fold_1"
            / "final_selected.weights.h5"
        )

        xai_rows = []
        for _, row in selected.iterrows():
            rel = str(
                row[
                    "relative_path"
                ]
            )
            image, _ = phase2_decode_resize(
                tf.constant(
                    str(
                        DATA_ROOT
                        / rel
                    )
                ),
                tf.constant(
                    0.0,
                    dtype=tf.float32,
                ),
                resolution,
            )
            image = image.numpy().astype(
                np.float32
            )
            prob = float(
                xai_model.predict(
                    image[
                        None,
                        ...
                    ],
                    verbose=0,
                ).reshape(-1)[0]
            )
            target = int(
                prob >= t1
            )

            # Include full-path identity to avoid basename collisions across patients.
            sample_id = Path(rel).stem + "_" + hashlib.sha256(rel.encode("utf-8")).hexdigest()[:12]
            sample_root = (
                xai_root
                / sample_id
            )
            sample_root.mkdir(
                parents=True,
                exist_ok=True,
            )

            for method in ("gradcam", "gradcampp", "integrated_gradients", "occlusion"):
                saliency, method_meta = compute_xai_map(
                    xai_model, image, target, method,
                    artifact_prefix=sample_root / method,
                )
                path = (
                    sample_root
                    / f"{method}.png"
                )
                save_xai_visual(
                    image,
                    saliency,
                    path,
                    (
                        f"{model_id} R{resolution} {method_meta['method_display_name']} | "
                        f"true={int(row['label'])} | p={prob:.4f}"
                    ),
                )
                xai_rows.append({
                    "sample_id": sample_id,
                    "relative_path": rel,
                    "method": method,
                    "method_display_name": method_meta["method_display_name"],
                    "is_gradcampp_approximation": method_meta.get("is_gradcampp_approximation", False),
                    "explanation_scope": "FOLD_1_OOF_MODEL_PREDICTION",
                    "target_class": target,
                    "probability": prob,
                    "true_label": int(
                        row[
                            "label"
                        ]
                    ),
                    "path": str(
                        path
                    ),
                })

        pd.DataFrame(
            xai_rows
        ).to_csv(
            xai_root
            / "XAI_INDEX.csv",
            index=False,
        )
        xai_report = {
            "enabled": True,
            "status": "COMPLETE",
            "samples": int(
                len(selected)
            ),
            "visuals": int(
                len(xai_rows)
            ),
            "methods": [xai_method_metadata(method) for method in
                        ("gradcam", "gradcampp", "integrated_gradients", "occlusion")],
            "explanation_scope": "FOLD_1_OOF_MODEL_PREDICTION; these are not five-fold ensemble explanations",
        }

        del xai_model
        tf.keras.backend.clear_session()
        gc.collect()

    # --------------------------------------------------------
    # Model-specific Edge / CBAM outputs.
    # Generic lightweight activation summaries.
    # --------------------------------------------------------
    internal_report = {
        "status": "ARCHITECTURE_FLAGS_ONLY",
        "limitation": "Phase-2 does not reproduce M07's full internal-activation report or class casebook.",
        "edge": bool(
            spec.use_edge
        ),
        "cbam": bool(
            spec.use_cbam
        ),
    }

    # --------------------------------------------------------
    # External — use exact M07-frozen sentinel manifest if available.
    # No adaptation.
    # --------------------------------------------------------
    external_report = {
        "enabled": bool(
            PHASE2_RUN_EXTERNAL
        ),
        "status": "SKIPPED",
        "reason": "DISABLED_BY_CONFIG" if not PHASE2_RUN_EXTERNAL else "NIH_SENTINEL_MANIFEST_UNAVAILABLE",
        "scope": "NIH_SENTINEL_ONLY; user-supplied external cohorts are evaluated in M07 only",
    }

    sentinel_path = (
        OUTPUT_ROOT
        / "EXTERNAL_VALIDATION"
        / "NIH_SENTINEL_MANIFEST.csv"
    )

    if (
        PHASE2_RUN_EXTERNAL
        and sentinel_path.is_file()
    ):
        ext = validate_external_manifest(
            "PHASE2_NIH_SENTINEL", pd.read_csv(sentinel_path, dtype={"patient_id": str, "sha256": str}),
        )

        ext_bundle = (
            phase2_predict_five_folds(
                model_id,
                resolution,
                ext,
                thresholds,
            )
        )
        y_ext = ext[
            "label"
        ].to_numpy(dtype=int)
        ext_metrics = (
            calculate_binary_metrics(
                y_ext,
                ext_bundle[
                    "primary_pred"
                ],
                score=ext_bundle[
                    "score"
                ],
            )
        )
        ext_dual, ext_decision = (
            analyze_dual_threshold(
                y_ext,
                ext_bundle[
                    "mean_probability"
                ],
            )
        )

        ext_root = (
            out
            / "EXTERNAL"
            / "NIH_SAMPLE_SENTINEL"
        )
        ext_root.mkdir(
            parents=True,
            exist_ok=True,
        )

        ext_pred = ext.copy()
        ext_pred[
            "mean_probability"
        ] = ext_bundle[
            "mean_probability"
        ]
        ext_pred[
            "normalized_ensemble_score"
        ] = ext_bundle[
            "score"
        ]
        ext_pred[
            "prediction_primary"
        ] = ext_bundle[
            "primary_pred"
        ]
        ext_pred[
            "dual20_80_decision"
        ] = ext_decision
        ext_pred.to_csv(
            ext_root
            / "PREDICTIONS.csv",
            index=False,
        )

        (
            ext_root
            / "METRICS.json"
        ).write_text(
            json.dumps(
                json_safe(
                    ext_metrics
                ),
                indent=2,
            ),
            encoding="utf-8",
        )
        (
            ext_root
            / "DUAL20_80.json"
        ).write_text(
            json.dumps(
                json_safe(
                    ext_dual
                ),
                indent=2,
            ),
            encoding="utf-8",
        )

        phase2_save_basic_figures(
            ext_root
            / "FIGURES",
            y_ext,
            ext_bundle[
                "mean_probability"
            ],
            ext_bundle[
                "primary_pred"
            ],
            f"{model_id} R{resolution} NIH Sentinel",
        )

        ext_boot, ext_ci = phase2_bootstrap_ci(
            y_ext, ext_bundle["primary_pred"], ext_bundle["score"], ext_metrics,
            patient_ids=ext["patient_id"].astype(str).to_numpy(),
        )
        ext_boot.to_csv(ext_root / "BOOTSTRAP.csv", index=False)
        atomic_write_json(ext_root / "BOOTSTRAP_CI95.json", json_safe(ext_ci))
        external_report = {
            "enabled": True,
            "scope": "NIH_SENTINEL_ONLY; custom external cohorts are not part of Phase-2",
            "bootstrap_ci95": ext_ci,
            "bootstrap_unit": "PATIENT_CLUSTER",
            "manifest_identity": dataframe_identity(ext),
            "status": "COMPLETE",
            "kind": (
                "CROSS_DOMAIN_SENTINEL_NOT_FINAL_PEDIATRIC_EXTERNAL"
            ),
            "metrics": ext_metrics,
            "dual20_80": ext_dual,
            "adaptation": False,
            "threshold_tuning": False,
        }


    # --------------------------------------------------------
    # Optional user-supplied external cohort at EVERY model/resolution.
    # This includes M07 R320/R384 inside the Gate.
    # --------------------------------------------------------
    custom_external_report = {
        "status": "NOT_REQUESTED",
        "requested": bool(CUSTOM_EXTERNAL_MANIFEST),
    }

    if PHASE2_RUN_EXTERNAL and CUSTOM_EXTERNAL_MANIFEST:
        custom_path = Path(CUSTOM_EXTERNAL_MANIFEST)
        if not custom_path.is_file():
            raise FileNotFoundError(
                f"Custom external manifest not found: {custom_path}"
            )

        custom_ext = validate_external_manifest(
            f"{model_id}_R{resolution}_CUSTOM_EXTERNAL",
            pd.read_csv(
                custom_path,
                dtype={"patient_id": str, "sha256": str},
            ),
        )

        custom_bundle = phase2_predict_five_folds(
            model_id,
            resolution,
            custom_ext,
            thresholds,
        )

        y_custom = custom_ext["label"].to_numpy(dtype=int)
        custom_metrics = calculate_binary_metrics(
            y_custom,
            custom_bundle["primary_pred"],
            score=custom_bundle["score"],
        )
        custom_dual, custom_decision = analyze_dual_threshold(
            y_custom,
            custom_bundle["mean_probability"],
        )

        custom_root = out / "EXTERNAL" / "CUSTOM_EXTERNAL"
        custom_root.mkdir(parents=True, exist_ok=True)

        custom_pred = custom_ext.copy()
        custom_pred["mean_probability"] = custom_bundle["mean_probability"]
        custom_pred["normalized_ensemble_score"] = custom_bundle["score"]
        custom_pred["prediction_primary"] = custom_bundle["primary_pred"]
        custom_pred["dual20_80_decision"] = custom_decision
        custom_pred.to_csv(
            custom_root / "PREDICTIONS.csv",
            index=False,
        )

        atomic_write_json(
            custom_root / "METRICS.json",
            json_safe(custom_metrics),
        )
        atomic_write_json(
            custom_root / "DUAL20_80.json",
            json_safe(custom_dual),
        )

        custom_boot, custom_ci = phase2_bootstrap_ci(
            y_custom,
            custom_bundle["primary_pred"],
            custom_bundle["score"],
            custom_metrics,
            patient_ids=custom_ext["patient_id"].astype(str).to_numpy(),
        )
        custom_boot.to_csv(
            custom_root / "BOOTSTRAP.csv",
            index=False,
        )
        atomic_write_json(
            custom_root / "BOOTSTRAP_CI95.json",
            json_safe(custom_ci),
        )

        phase2_save_basic_figures(
            custom_root / "FIGURES",
            y_custom,
            custom_bundle["mean_probability"],
            custom_bundle["primary_pred"],
            f"{model_id} R{resolution} Custom External",
        )

        custom_external_report = {
            "status": "COMPLETE",
            "kind": "USER_SUPPLIED_EXTERNAL",
            "manifest_identity": dataframe_identity(custom_ext),
            "bootstrap_ci95": custom_ci,
            "bootstrap_unit": "PATIENT_CLUSTER",
            "metrics": custom_metrics,
            "dual20_80": custom_dual,
            "adaptation": False,
            "threshold_tuning": False,
        }

    external_report["custom_external"] = custom_external_report

    # --------------------------------------------------------
    # Final model card + quick summary
    # --------------------------------------------------------
    summary = {
        "schema": (
            "m07.gate.multires.model_resolution.v1.6"
            if model_id == "M07"
            else "pneumonia.phase2.model_resolution.v1.6"
        ),
        "execution_role": execution_role,
        "status": "COMPLETE",
        "model_id": model_id,
        "description": spec.description,
        "resolution": int(
            resolution
        ),
        "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
        "confirmed_m07_recipe_fingerprint": recipe[
            "recipe_fingerprint_sha256"
        ],
        "shared_training_params": shared_params,
        "model_spec": asdict(
            spec
        ),
        "f2_policy": {
            "beta": 2,
            "positive": "PNEUMONIA",
            "role": "REPORT_ONLY",
            "selection_use": False,
        },
        "dual20_80_policy": {
            "low": DUAL_THRESHOLD_LOW,
            "high": DUAL_THRESHOLD_HIGH,
            "role": "REPORT_ONLY",
            "tuned": False,
        },
        "oof": oof_metrics,
        "locked_test_primary": primary_metrics_local,
        "locked_test_majority": majority_metrics_local,
        "locked_test_bootstrap_ci95": boot_ci,
        "oof_dual20_80": oof_dual,
        "test_dual20_80": test_dual,
        "calibration": calibration,
        "generalization_gaps": gaps,
        "xai": xai_report,
        "model_internals": internal_report,
        "external": external_report,
        "locked_test_used_for_selection": False,
        "external_used_for_selection": False,
        "known_limitations": [
            "OOF reuses validation-selected checkpoints and thresholds; it is development evidence.",
            "M01/M02 change both batch sampling and loss weighting; M03/M04 change alpha and gamma.",
            "The shared recipe was optimized on M07, not independently on each comparator.",
            "One seed per fold; repeated-seed stability is deferred. Paired patient-bootstrap and Holm-adjusted comparisons are produced by the campaign statistics stage.",
            "Custom external (when configured) and NIH sentinel are evaluated across resolutions; internal diagnostics and casebooks remain reduced in Phase-2.",
        ],
    }

    quick = pd.DataFrame([
        {
            "dataset": "OOF",
            "model_id": model_id,
            "resolution": resolution,
            "balanced_accuracy": oof_metrics[
                "balanced_accuracy"
            ],
            "precision_normal": oof_metrics[
                "precision_normal"
            ],
            "recall_normal": oof_metrics[
                "recall_normal"
            ],
            "precision_pneumonia": oof_metrics[
                "precision_pneumonia"
            ],
            "recall_pneumonia": oof_metrics[
                "recall_pneumonia"
            ],
            "macro_f1": oof_metrics[
                "macro_f1"
            ],
            "mcc": oof_metrics[
                "mcc"
            ],
            "auroc": oof_metrics[
                "auroc"
            ],
            "f2_report_only": oof_metrics[
                "f2"
            ],
            "dual20_80_coverage": oof_dual[
                "coverage"
            ],
        },
        {
            "dataset": "LOCKED_TEST",
            "model_id": model_id,
            "resolution": resolution,
            "balanced_accuracy": primary_metrics_local[
                "balanced_accuracy"
            ],
            "precision_normal": primary_metrics_local[
                "precision_normal"
            ],
            "recall_normal": primary_metrics_local[
                "recall_normal"
            ],
            "precision_pneumonia": primary_metrics_local[
                "precision_pneumonia"
            ],
            "recall_pneumonia": primary_metrics_local[
                "recall_pneumonia"
            ],
            "macro_f1": primary_metrics_local[
                "macro_f1"
            ],
            "mcc": primary_metrics_local[
                "mcc"
            ],
            "auroc": primary_metrics_local[
                "auroc"
            ],
            "f2_report_only": primary_metrics_local[
                "f2"
            ],
            "dual20_80_coverage": test_dual[
                "coverage"
            ],
        },
    ])

    if (
        external_report[
            "status"
        ] == "COMPLETE"
    ):
        em = external_report[
            "metrics"
        ]
        ed = external_report[
            "dual20_80"
        ]
        quick = pd.concat([
            quick,
            pd.DataFrame([
                {
                    "dataset": "EXTERNAL_NIH_SENTINEL",
                    "model_id": model_id,
                    "resolution": resolution,
                    "balanced_accuracy": em[
                        "balanced_accuracy"
                    ],
                    "precision_normal": em[
                        "precision_normal"
                    ],
                    "recall_normal": em[
                        "recall_normal"
                    ],
                    "precision_pneumonia": em[
                        "precision_pneumonia"
                    ],
                    "recall_pneumonia": em[
                        "recall_pneumonia"
                    ],
                    "macro_f1": em[
                        "macro_f1"
                    ],
                    "mcc": em[
                        "mcc"
                    ],
                    "auroc": em[
                        "auroc"
                    ],
                    "f2_report_only": em[
                        "f2"
                    ],
                    "dual20_80_coverage": ed[
                        "coverage"
                    ],
                }
            ]),
        ], ignore_index=True)


    if custom_external_report["status"] == "COMPLETE":
        cem = custom_external_report["metrics"]
        ced = custom_external_report["dual20_80"]
        quick = pd.concat([
            quick,
            pd.DataFrame([
                {
                    "dataset": "EXTERNAL_CUSTOM",
                    "model_id": model_id,
                    "resolution": resolution,
                    "balanced_accuracy": cem["balanced_accuracy"],
                    "precision_normal": cem["precision_normal"],
                    "recall_normal": cem["recall_normal"],
                    "precision_pneumonia": cem["precision_pneumonia"],
                    "recall_pneumonia": cem["recall_pneumonia"],
                    "macro_f1": cem["macro_f1"],
                    "mcc": cem["mcc"],
                    "auroc": cem["auroc"],
                    "f2_report_only": cem["f2"],
                    "dual20_80_coverage": ced["coverage"],
                }
            ]),
        ], ignore_index=True)

    quick.to_csv(
        out
        / "QUICK_RESULTS.csv",
        index=False,
    )

    # Publish completion only after every requested artifact has been generated.
    report_artifacts = phase2_required_report_paths(out)
    report_artifacts.update({p.relative_to(out).as_posix(): p for p in out.rglob("*")
                             if p.is_file() and p.name != "FINAL_REPORT.json"})
    summary = seal_receipt(json_safe(summary), phase2_contract(model_id, resolution, stage="phase2_report"), report_artifacts)
    atomic_write_json(out / "FINAL_REPORT.json", summary)
    phase2_validate_report(model_id, resolution, out)

    # Retryable/idempotent durable publication of the verified final evidence.
    phase2_ensure_final_persisted(model_id, resolution, out)

    print(
        f"✅ {model_id} R{resolution} COMPLETE"
    )
    display(
        quick.round(
            4
        )
    )

    return summary

print("✅ Phase-2 per-model/resolution campaign function ready")

In [ ]:
# ============================================================
# 19B) M07 MULTI-RESOLUTION GATE + PAIRED PATIENT STATISTICS
#      HPO remains frozen from R224. R320/R384 reuse the exact confirmed recipe.
#      M07 is completed here and is NEVER retrained in Phase-2.
# ============================================================

M07_MULTIRES_STATS_ROOT = OUTPUT_ROOT / "MULTIRES_STATISTICS"
M07_MULTIRES_STATS_ROOT.mkdir(parents=True, exist_ok=True)
M07_MULTIRES_GATE_RECEIPT_PATH = OUTPUT_ROOT / "M07_MULTIRES_GATE_RECEIPT.json"


def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    if p.ndim != 1 or len(p) == 0 or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Holm adjustment requires finite p-values in [0,1].")
    order = np.argsort(p, kind="mergesort")
    adjusted = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        candidate = min(1.0, (m - rank) * p[idx])
        running = max(running, candidate)
        adjusted[idx] = running
    return adjusted


def _paired_metric_vector(y, pred, score):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    score = np.asarray(score, dtype=float)
    if len(y) == 0 or not (len(y) == len(pred) == len(score)):
        raise ValueError("Metric vector arrays must have equal nonzero length.")
    if len(np.unique(y)) < 2:
        raise ValueError("Metric vector requires both classes.")
    tn = int(np.sum((y == 0) & (pred == 0)))
    fp = int(np.sum((y == 0) & (pred == 1)))
    fn = int(np.sum((y == 1) & (pred == 0)))
    tp = int(np.sum((y == 1) & (pred == 1)))
    def _safe(num, den):
        return float(num / den) if den else 0.0
    r0 = _safe(tn, tn + fp)
    r1 = _safe(tp, tp + fn)
    p0 = _safe(tn, tn + fn)
    p1 = _safe(tp, tp + fp)
    f10 = _safe(2.0 * p0 * r0, p0 + r0)
    f11 = _safe(2.0 * p1 * r1, p1 + r1)
    denom = math.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    mcc = float((tp * tn - fp * fn) / denom) if denom else 0.0
    auroc = float(roc_auc_score(y, score))
    return {
        "balanced_accuracy": float((r0 + r1) / 2.0),
        "macro_f1": float((f10 + f11) / 2.0),
        "mcc": mcc,
        "auroc": auroc,
        "recall_normal": r0,
        "recall_pneumonia": r1,
    }


def _paired_metric_value(y, pred, score, metric):
    metrics = _paired_metric_vector(y, pred, score)
    if metric not in metrics:
        raise KeyError(metric)
    return float(metrics[metric])


def paired_patient_bootstrap_metrics(
    frame,
    pred_reference,
    score_reference,
    pred_candidate,
    score_candidate,
    metrics,
    n_boot=PAIRED_BOOTSTRAPS,
    seed=PAIRED_BOOTSTRAP_SEED,
):
    required = {"patient_id", "label"}
    if not required.issubset(frame.columns):
        raise ValueError("Paired bootstrap requires patient_id and label.")
    if frame["patient_id"].isna().any():
        raise ValueError("Paired bootstrap requires non-missing patient IDs.")
    patient_ids = frame["patient_id"].astype(str).to_numpy()
    if pd.Series(patient_ids).str.strip().eq("").any():
        raise ValueError("Paired bootstrap requires non-empty patient IDs.")
    y = frame["label"].to_numpy(dtype=int)
    pred_reference = np.asarray(pred_reference, dtype=int)
    pred_candidate = np.asarray(pred_candidate, dtype=int)
    score_reference = np.asarray(score_reference, dtype=float)
    score_candidate = np.asarray(score_candidate, dtype=float)
    if not (len(y) == len(pred_reference) == len(pred_candidate) ==
            len(score_reference) == len(score_candidate) == len(patient_ids)):
        raise ValueError("Paired arrays must have identical length.")
    metrics = tuple(metrics)
    unknown = set(metrics) - set(PAIRED_SECONDARY_METRICS) - {PAIRED_PRIMARY_METRIC}
    if unknown:
        raise ValueError(f"Unknown paired metrics: {sorted(unknown)}")
    point_ref = _paired_metric_vector(y, pred_reference, score_reference)
    point_cand = _paired_metric_vector(y, pred_candidate, score_candidate)
    delta_store = {metric: [] for metric in metrics}
    rng = np.random.default_rng(int(seed))
    unique_patients = np.unique(patient_ids)
    # Preserve np.unique ordering and rng.choice calls EXACTLY as V1.6.
    patient_rows = {}
    for row_index, patient_id in enumerate(patient_ids):
        patient_rows.setdefault(patient_id, []).append(row_index)
    patient_rows = {pid: np.asarray(rows, dtype=np.intp) for pid, rows in patient_rows.items()}
    valid = 0
    for _ in range(int(n_boot)):
        sampled = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        idx = np.concatenate([patient_rows[pid] for pid in sampled])
        yy = y[idx]
        if len(np.unique(yy)) < 2:
            continue
        ref_vec = _paired_metric_vector(yy, pred_reference[idx], score_reference[idx])
        cand_vec = _paired_metric_vector(yy, pred_candidate[idx], score_candidate[idx])
        for metric in metrics:
            delta_store[metric].append(cand_vec[metric] - ref_vec[metric])
        valid += 1
    if valid < max(100, int(n_boot) // 2):
        raise RuntimeError("Too few valid paired patient-bootstrap replicates.")
    rows = []
    for metric in metrics:
        deltas = np.asarray(delta_store[metric], dtype=float)
        left = (np.sum(deltas <= 0.0) + 1.0) / (len(deltas) + 1.0)
        right = (np.sum(deltas >= 0.0) + 1.0) / (len(deltas) + 1.0)
        rows.append({
            "metric": metric,
            "reference": float(point_ref[metric]),
            "candidate": float(point_cand[metric]),
            "delta_candidate_minus_reference": float(point_cand[metric] - point_ref[metric]),
            "ci95_low": float(np.quantile(deltas, 0.025)),
            "ci95_high": float(np.quantile(deltas, 0.975)),
            "p_raw_two_sided": float(min(1.0, 2.0 * min(left, right))),
            "bootstrap_unit": "PATIENT_CLUSTER_PAIRED",
            "n_bootstrap_valid": int(len(deltas)),
            "n_patients": int(len(unique_patients)),
        })
    return rows


def paired_patient_bootstrap_difference(
    frame,
    pred_reference,
    score_reference,
    pred_candidate,
    score_candidate,
    metric,
    n_boot=PAIRED_BOOTSTRAPS,
    seed=PAIRED_BOOTSTRAP_SEED,
):
    return paired_patient_bootstrap_metrics(
        frame, pred_reference, score_reference, pred_candidate, score_candidate,
        metrics=(metric,), n_boot=n_boot, seed=seed,
    )[0]


def _prediction_identity_columns(frame):
    return frame[["relative_path", "patient_id", "label", "sha256"]].astype({
        "relative_path": str, "patient_id": str, "label": int, "sha256": str,
    }).reset_index(drop=True)


def _m07_gate_prediction_frame(resolution, dataset):
    resolution = int(resolution)
    dataset = str(dataset).upper()
    if resolution == 224:
        if dataset == "OOF":
            path = OUTPUT_ROOT / "M07_OOF_PREDICTIONS.csv"
        elif dataset == "LOCKED_TEST":
            path = TEST_OUT / "M07_LOCKED_TEST_ENSEMBLE_PREDICTIONS.csv"
        else:
            raise ValueError(dataset)
    else:
        root = phase2_output_root("M07", resolution)
        if dataset == "OOF":
            path = root / "OOF_PREDICTIONS.csv"
        elif dataset == "LOCKED_TEST":
            path = root / "LOCKED_TEST" / "LOCKED_TEST_PREDICTIONS.csv"
        else:
            raise ValueError(dataset)
    frame = pd.read_csv(path, float_precision="round_trip", dtype={"patient_id": str, "sha256": str})
    required = {"relative_path", "patient_id", "label", "sha256"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Prediction file lacks identity columns: {path}")
    if dataset == "OOF":
        required_pred = {"prediction_fold_threshold", "probability_pneumonia"}
    else:
        # R224 historically emitted prediction_primary_normalized; R320/R384
        # emit prediction_primary. Canonicalize and cross-check aliases.
        if "prediction_primary" not in frame.columns:
            if "prediction_primary_normalized" not in frame.columns:
                raise ValueError(f"Prediction file lacks primary prediction column: {path}")
            frame["prediction_primary"] = pd.to_numeric(
                frame["prediction_primary_normalized"],
                errors="raise",
            ).astype(int)
        elif "prediction_primary_normalized" in frame.columns:
            primary = pd.to_numeric(
                frame["prediction_primary"],
                errors="raise",
            ).astype(int)
            normalized = pd.to_numeric(
                frame["prediction_primary_normalized"],
                errors="raise",
            ).astype(int)
            if not np.array_equal(primary.to_numpy(), normalized.to_numpy()):
                raise ValueError(f"Primary prediction alias mismatch: {path}")
            frame["prediction_primary"] = primary
        required_pred = {"prediction_primary", "normalized_ensemble_score"}
    if not required_pred.issubset(frame.columns):
        raise ValueError(f"Prediction file lacks prediction columns: {path}")
    return frame


def _aligned_m07_resolution_pair(reference_resolution, candidate_resolution, dataset):
    a = _m07_gate_prediction_frame(reference_resolution, dataset)
    b = _m07_gate_prediction_frame(candidate_resolution, dataset)
    ida = _prediction_identity_columns(a)
    idb = _prediction_identity_columns(b)
    if not ida.equals(idb):
        raise ValueError(
            f"M07 resolution prediction identity mismatch: {reference_resolution} vs {candidate_resolution} {dataset}"
        )
    if str(dataset).upper() == "OOF":
        return ida, a["prediction_fold_threshold"].to_numpy(dtype=int), \
            a["probability_pneumonia"].to_numpy(dtype=float), \
            b["prediction_fold_threshold"].to_numpy(dtype=int), \
            b["probability_pneumonia"].to_numpy(dtype=float)
    return ida, a["prediction_primary"].to_numpy(dtype=int), \
        a["normalized_ensemble_score"].to_numpy(dtype=float), \
        b["prediction_primary"].to_numpy(dtype=int), \
        b["normalized_ensemble_score"].to_numpy(dtype=float)


def _apply_holm_columns(table, grouping_columns):
    table = table.copy()
    table["p_holm"] = np.nan
    if grouping_columns:
        grouped = table.groupby(grouping_columns, dropna=False, sort=False)
        for _, idx in grouped.groups.items():
            idx = list(idx)
            table.loc[idx, "p_holm"] = holm_adjust(
                table.loc[idx, "p_raw_two_sided"].to_numpy(dtype=float)
            )
    else:
        table["p_holm"] = holm_adjust(table["p_raw_two_sided"].to_numpy(dtype=float))
    table["reject_holm_0_05"] = table["p_holm"] < PAIRED_ALPHA
    return table


def m07_highres_accumulation_preflight(resolution):
    """Exercise head AND fine-tune, one full accumulation group per stage.

    Synthetic test doubles can test the control logic on CPU. This function
    itself uses the real training backend when the notebook is run on Kaggle.
    A failed or interrupted retry must never leave a stale PASS marker.
    """
    resolution = int(resolution)
    if resolution not in (320, 384):
        raise ValueError("High-resolution preflight supports R320/R384 only")
    receipt_path = M07_MULTIRES_STATS_ROOT / f"M07_R{resolution}_ACCUMULATION_PREFLIGHT.json"
    receipt = {
        "schema": "m07.highres.accumulation.preflight.v1.7",
        "status": "STARTED", "resolution": resolution, "stages": [],
    }
    atomic_write_json(receipt_path, receipt)
    model = backbone = ds = images = labels = result = None
    try:
        plan = phase2_batch_plan(resolution)
        accumulation = int(plan["gradient_accumulation_steps"])
        microbatch = int(plan["microbatch_size"])
        if accumulation < 1 or microbatch < 1:
            raise RuntimeError("Invalid high-resolution accumulation plan")
        train_df = load_fold_manifest(1, "train")
        training_schedule = phase2_training_schedule(train_df, resolution, balanced_batches=True)
        if (int(training_schedule["micro_steps_per_epoch"]) < accumulation
                or int(training_schedule["micro_steps_per_epoch"]) % accumulation != 0
                or int(training_schedule["partial_accumulation_steps"]) != 0):
            raise RuntimeError("High-resolution preflight found a partial accumulation group.")
        tf.keras.backend.clear_session()
        gc.collect()
        ds = phase2_build_dataset(
            train_df, resolution, microbatch, True, SEED + 9100 + resolution,
            batch_policy="none", balanced_batches=True,
        )
        images, labels = next(iter(ds.take(1)))
        spec = CANONICAL_MODELS["M07"]
        model, backbone = phase2_build_model(
            spec, resolution, dropout=float(shared_params["dropout"]), seed=SEED + 1,
        )
        loss, _, _ = phase2_loss_and_weights(spec, train_df)
        for stage, trainable, lr_name in (
            ("head", False, "head_lr"), ("finetune", True, "finetune_lr"),
        ):
            backbone.trainable = trainable
            phase2_compile(
                model, loss, lr=float(shared_params[lr_name]),
                weight_decay=float(shared_params["weight_decay"]),
                resolution=resolution, use_ema=False,
            )
            before = int(model.optimizer.iterations.numpy())
            for microstep in range(accumulation):
                result = model.train_on_batch(images, labels, return_dict=True)
                if not isinstance(result, dict) or "loss" not in result:
                    raise RuntimeError(f"Missing loss in {stage} accumulation preflight")
                for metric, value in result.items():
                    try:
                        numeric = np.asarray(value, dtype=np.float64)
                    except (TypeError, ValueError) as exc:
                        raise RuntimeError(f"Non-numeric {stage} preflight metric: {metric}") from exc
                    if numeric.size == 0 or not np.isfinite(numeric).all():
                        raise RuntimeError(f"Non-finite value in {stage} accumulation preflight: {metric}")
                observed = int(model.optimizer.iterations.numpy()) - before
                expected = (microstep + 1) // accumulation
                if observed != expected:
                    raise RuntimeError(
                        f"Gradient accumulation runtime mismatch at R{resolution} {stage}: "
                        f"expected {expected} update(s) after {microstep + 1} microsteps, observed {observed}."
                    )
            receipt["stages"].append({
                "stage": stage, "backbone_trainable": trainable,
                "observed_optimizer_updates_for_one_group": observed,
                "metrics_finite": True,
            })
        receipt.update({
            "status": "PASS",
            "microbatch_size": microbatch,
            "gradient_accumulation_steps": accumulation,
            "effective_optimizer_batch_size": int(plan["effective_optimizer_batch_size"]),
            "optimizer_updates_per_epoch": int(training_schedule["optimizer_updates_per_epoch"]),
            "micro_steps_per_epoch": int(training_schedule["micro_steps_per_epoch"]),
            "partial_accumulation_steps": int(training_schedule["partial_accumulation_steps"]),
            "observed_optimizer_updates_for_one_group": 1,
            "validated_training_stages": ["head", "finetune"],
        })
        atomic_write_json(receipt_path, receipt)
        print(f"R{resolution} head + finetune accumulation preflight PASS")
        return receipt
    except Exception as exc:
        receipt.update(status="FAILED", error_type=type(exc).__name__, error=str(exc))
        atomic_write_json(receipt_path, receipt)
        raise
    finally:
        model = backbone = ds = images = labels = result = None
        tf.keras.backend.clear_session()
        gc.collect()


def run_m07_gate_resolution(resolution):
    resolution = int(resolution)
    if resolution not in M07_GATE_ADDITIONAL_RESOLUTIONS:
        raise ValueError(f"Unexpected M07 Gate resolution: {resolution}")
    return run_phase2_model_resolution("M07", resolution)


# 1) Complete M07 at R320/R384 using the exact frozen R224 recipe.
# The generic engine is reused, but routing/persistence identifies these runs as M07_GATE_MULTIRES.
for _resolution in M07_GATE_ADDITIONAL_RESOLUTIONS:
    print("\n" + "=" * 118)
    print(f"M07 GATE RESOLUTION R{_resolution} — full 5-fold training/evaluation")
    print("Confirmed recipe is frozen from R224; no high-resolution HPO or Test tuning.")
    print("=" * 118)
    m07_highres_accumulation_preflight(int(_resolution))
    run_m07_gate_resolution(int(_resolution))

# 2) Paired resolution-sensitivity statistics on the same patients/images.
_resolution_pairs = ((224, 320), (224, 384), (320, 384))
_metrics = (PAIRED_PRIMARY_METRIC,) + tuple(PAIRED_SECONDARY_METRICS)
_rows = []
for _dataset in ("OOF", "LOCKED_TEST"):
    for _reference, _candidate in _resolution_pairs:
        _frame, _pr, _sr, _pc, _sc = _aligned_m07_resolution_pair(
            _reference, _candidate, _dataset
        )
        _pair_results = paired_patient_bootstrap_metrics(
            _frame, _pr, _sr, _pc, _sc, metrics=_metrics,
            n_boot=PAIRED_BOOTSTRAPS,
            seed=PAIRED_BOOTSTRAP_SEED + _reference * 10 + _candidate,
        )
        for _result in _pair_results:
            _rows.append({
                "dataset": _dataset,
                "reference_resolution": int(_reference),
                "candidate_resolution": int(_candidate),
                **_result,
            })

m07_resolution_paired = pd.DataFrame(_rows)
# Holm family 1: each metric across the three predeclared resolution contrasts, separately for OOF/Test.
m07_resolution_paired = _apply_holm_columns(
    m07_resolution_paired, ["dataset", "metric"]
).rename(columns={"p_holm": "p_holm_within_metric_family",
                  "reject_holm_0_05": "reject_holm_within_metric_family_0_05"})
# Stronger global family across every reported resolution contrast/metric, separately for OOF/Test.
m07_resolution_paired["p_holm_all_reported_metrics"] = np.nan
for _dataset, _idx in m07_resolution_paired.groupby("dataset").groups.items():
    _idx = list(_idx)
    m07_resolution_paired.loc[_idx, "p_holm_all_reported_metrics"] = holm_adjust(
        m07_resolution_paired.loc[_idx, "p_raw_two_sided"].to_numpy(dtype=float)
    )
m07_resolution_paired["reject_holm_all_reported_0_05"] = (
    m07_resolution_paired["p_holm_all_reported_metrics"] < PAIRED_ALPHA
)
m07_resolution_paired.to_csv(
    M07_MULTIRES_STATS_ROOT / "M07_RESOLUTION_PAIRED_PATIENT_BOOTSTRAP_HOLM.csv", index=False
)

# 3) Compact cross-resolution results table. No resolution is selected from Locked Test.
_quick_parts = []
_q224 = pd.read_csv(OUTPUT_ROOT / "M07_FINAL_QUICK_RESULTS.csv")
_q224["model_id"] = "M07"
_q224["resolution"] = 224
_quick_parts.append(_q224)
for _resolution in M07_GATE_ADDITIONAL_RESOLUTIONS:
    _q = pd.read_csv(phase2_output_root("M07", _resolution) / "QUICK_RESULTS.csv")
    _quick_parts.append(_q)
m07_multires_quick = pd.concat(_quick_parts, ignore_index=True, sort=False)
m07_multires_quick.to_csv(
    M07_MULTIRES_STATS_ROOT / "M07_MULTIRES_QUICK_RESULTS.csv", index=False
)

# Optional custom-external multi-resolution summary — report-only.
M07_CUSTOM_EXTERNAL_MULTIRES_PATH = (
    M07_MULTIRES_STATS_ROOT / "M07_CUSTOM_EXTERNAL_MULTIRES_RESULTS.csv"
)
if CUSTOM_EXTERNAL_MANIFEST:
    _custom_rows = []

    _r224_metrics_path = (
        OUTPUT_ROOT
        / "EXTERNAL_VALIDATION"
        / "custom_external"
        / "external_metrics.json"
    )
    if not _r224_metrics_path.is_file():
        raise FileNotFoundError(
            "Custom external was requested but R224 external metrics are missing."
        )
    _m224 = json.loads(_r224_metrics_path.read_text(encoding="utf-8"))
    _custom_rows.append({
        "resolution": 224,
        **{
            key: _m224.get(key)
            for key in (
                "balanced_accuracy", "macro_f1", "mcc", "auroc",
                "precision_normal", "recall_normal",
                "precision_pneumonia", "recall_pneumonia", "f2",
            )
        },
    })

    for _resolution in M07_GATE_ADDITIONAL_RESOLUTIONS:
        _metrics_path = (
            phase2_output_root("M07", _resolution)
            / "EXTERNAL"
            / "CUSTOM_EXTERNAL"
            / "METRICS.json"
        )
        if not _metrics_path.is_file():
            raise FileNotFoundError(
                f"Custom external was requested but R{_resolution} metrics are missing."
            )
        _metrics = json.loads(_metrics_path.read_text(encoding="utf-8"))
        _custom_rows.append({
            "resolution": int(_resolution),
            **{
                key: _metrics.get(key)
                for key in (
                    "balanced_accuracy", "macro_f1", "mcc", "auroc",
                    "precision_normal", "recall_normal",
                    "precision_pneumonia", "recall_pneumonia", "f2",
                )
            },
        })

    pd.DataFrame(_custom_rows).sort_values(
        "resolution"
    ).to_csv(
        M07_CUSTOM_EXTERNAL_MULTIRES_PATH,
        index=False,
    )

# Development-only resolution ordering: descriptive/predeclared sensitivity rank, NOT Locked-Test selection.
_oof_rank = m07_multires_quick[m07_multires_quick["dataset"] == "OOF"].copy()
_oof_rank = _oof_rank.sort_values(
    ["balanced_accuracy", "macro_f1", "mcc", "auroc", "resolution"],
    ascending=[False, False, False, False, True],
).reset_index(drop=True)
_oof_rank.insert(0, "development_rank", np.arange(1, len(_oof_rank) + 1))
_oof_rank.to_csv(M07_MULTIRES_STATS_ROOT / "M07_RESOLUTION_DEVELOPMENT_RANK.csv", index=False)

# 4) Seal a single multi-resolution Gate receipt required by Phase-2.
def m07_multires_gate_contract():
    return make_run_contract(
        "m07_multires_gate", model_id="M07", resolution=0, params=shared_params,
        head_epochs=FINAL_HEAD_EPOCHS, finetune_epochs=FINAL_FINETUNE_EPOCHS,
        extra={
            "gate_resolutions": list(M07_GATE_RESOLUTIONS),
            "hpo_resolution": 224,
            "hpo_recipe_fingerprint": recipe["recipe_fingerprint_sha256"],
            "resolution_selection_policy": "DEVELOPMENT_OOF_ONLY; locked test is confirmatory and never tunes resolution",
            "paired_bootstraps": PAIRED_BOOTSTRAPS,
            "holm_alpha": PAIRED_ALPHA,
            "multiplicity_policy": PAIRED_MULTIPLICITY_POLICY,
            "primary_metric": PAIRED_PRIMARY_METRIC,
        },
    )


def m07_multires_gate_artifacts():
    paths = {
        "r224_matrix_receipt": OUTPUT_ROOT / "M07_MATRIX_SOURCE_RECEIPT.json",
        "multires_quick": M07_MULTIRES_STATS_ROOT / "M07_MULTIRES_QUICK_RESULTS.csv",
        "resolution_paired_stats": M07_MULTIRES_STATS_ROOT / "M07_RESOLUTION_PAIRED_PATIENT_BOOTSTRAP_HOLM.csv",
        "development_resolution_rank": M07_MULTIRES_STATS_ROOT / "M07_RESOLUTION_DEVELOPMENT_RANK.csv",
    }
    for _resolution in M07_GATE_ADDITIONAL_RESOLUTIONS:
        _root = phase2_output_root("M07", _resolution)
        paths[f"r{_resolution}_accumulation_preflight"] = (
            M07_MULTIRES_STATS_ROOT
            / f"M07_R{_resolution}_ACCUMULATION_PREFLIGHT.json"
        )
        paths[f"r{_resolution}_final_report"] = _root / "FINAL_REPORT.json"
        paths[f"r{_resolution}_quick"] = _root / "QUICK_RESULTS.csv"
        paths[f"r{_resolution}_oof"] = _root / "OOF_PREDICTIONS.csv"
        paths[f"r{_resolution}_locked_test"] = _root / "LOCKED_TEST" / "LOCKED_TEST_PREDICTIONS.csv"
    if CUSTOM_EXTERNAL_MANIFEST:
        paths["custom_external_multires"] = M07_CUSTOM_EXTERNAL_MULTIRES_PATH
    return paths


_m07_multires_receipt = seal_receipt(
    {
        "status": "COMPLETE",
        "model_id": "M07",
        "gate_resolutions": list(M07_GATE_RESOLUTIONS),
        "hpo_resolution": 224,
        "phase2_m07_retraining_allowed": False,
        "locked_test_used_for_resolution_selection": False,
        "paired_statistics": "PATIENT_CLUSTER_PAIRED_BOOTSTRAP_WITH_HOLM",
    },
    m07_multires_gate_contract(),
    m07_multires_gate_artifacts(),
)
atomic_write_json(M07_MULTIRES_GATE_RECEIPT_PATH, _m07_multires_receipt)
validate_receipt(
    _m07_multires_receipt, m07_multires_gate_contract(), m07_multires_gate_artifacts(),
    allowed_statuses={"COMPLETE"},
)

atomic_write_json(
    M07_MULTIRES_STATS_ROOT / "M07_MULTIRES_GATE_REPORT.json",
    {
        "schema": "m07.multires.gate.report.v1.6",
        "status": "COMPLETE",
        "resolutions": list(M07_GATE_RESOLUTIONS),
        "hpo_resolution": 224,
        "confirmed_recipe_fingerprint": recipe["recipe_fingerprint_sha256"],
        "development_resolution_rank": _oof_rank.to_dict(orient="records"),
        "paired_bootstraps": PAIRED_BOOTSTRAPS,
        "primary_metric": PAIRED_PRIMARY_METRIC,
        "holm_alpha": PAIRED_ALPHA,
        "locked_test_role": "CONFIRMATORY_REPORT_ONLY_NO_RESOLUTION_TUNING",
        "phase2_m07_retraining_allowed": False,
    },
)
_persist_lightweight_files(
    [
        M07_MULTIRES_GATE_RECEIPT_PATH,
        M07_MULTIRES_STATS_ROOT / "M07_MULTIRES_GATE_REPORT.json",
        M07_MULTIRES_STATS_ROOT / "M07_MULTIRES_QUICK_RESULTS.csv",
        M07_MULTIRES_STATS_ROOT / "M07_RESOLUTION_DEVELOPMENT_RANK.csv",
        M07_MULTIRES_STATS_ROOT / "M07_RESOLUTION_PAIRED_PATIENT_BOOTSTRAP_HOLM.csv",
        *(
            [M07_CUSTOM_EXTERNAL_MULTIRES_PATH]
            if CUSTOM_EXTERNAL_MANIFEST
            else []
        ),
    ],
    note="M07 multi-resolution Gate 224/320/384 sealed with paired statistics",
)

print("\n✅ M07 MULTI-RESOLUTION GATE COMPLETE: 224 / 320 / 384")
print("✅ Confirmation HPO occurred only at R224; frozen recipe reused at R320/R384")
print("✅ Paired patient bootstrap + Holm resolution analysis complete")
print("✅ Locked Test was NOT used to select/tune resolution")
print("🔒 M07 is now finished and MUST NOT be retrained in Phase-2")
display(m07_multires_quick.round(4))

In [ ]:
# ============================================================
# 20) MASTER PHASE-2 ORCHESTRATOR
# ============================================================

def _campaign_prediction_frame(model_id, resolution, dataset):
    resolution = int(resolution)
    dataset = str(dataset).upper()
    if model_id == "M07":
        return _m07_gate_prediction_frame(resolution, dataset)
    root = phase2_output_root(model_id, resolution)
    path = (root / "OOF_PREDICTIONS.csv") if dataset == "OOF" else (root / "LOCKED_TEST" / "LOCKED_TEST_PREDICTIONS.csv")
    frame = pd.read_csv(path, float_precision="round_trip", dtype={"patient_id": str, "sha256": str})
    return frame

def _aligned_campaign_pair(reference_model, candidate_model, resolution, dataset):
    a = _campaign_prediction_frame(reference_model, resolution, dataset)
    b = _campaign_prediction_frame(candidate_model, resolution, dataset)
    ida = _prediction_identity_columns(a)
    idb = _prediction_identity_columns(b)
    if not ida.equals(idb):
        raise ValueError(
            f"Campaign paired identity mismatch: {reference_model} vs {candidate_model} R{resolution} {dataset}"
        )
    if dataset == "OOF":
        return (
            ida,
            a["prediction_fold_threshold"].to_numpy(dtype=int),
            a["probability_pneumonia"].to_numpy(dtype=float),
            b["prediction_fold_threshold"].to_numpy(dtype=int),
            b["probability_pneumonia"].to_numpy(dtype=float),
        )
    return (
        ida,
        a["prediction_primary"].to_numpy(dtype=int),
        a["normalized_ensemble_score"].to_numpy(dtype=float),
        b["prediction_primary"].to_numpy(dtype=int),
        b["normalized_ensemble_score"].to_numpy(dtype=float),
    )

def build_campaign_paired_statistics():
    _comparison_rows = []
    _comparison_metrics = (PAIRED_PRIMARY_METRIC,) + tuple(PAIRED_SECONDARY_METRICS)
    for _dataset in ("OOF", "LOCKED_TEST"):
        for _resolution in PHASE2_RESOLUTIONS:
            for _reference, _candidate, _contrast in PREDECLARED_MODEL_COMPARISONS:
                _frame, _pr, _sr, _pc, _sc = _aligned_campaign_pair(
                    _reference, _candidate, int(_resolution), _dataset
                )
                _pair_results = paired_patient_bootstrap_metrics(
                    _frame, _pr, _sr, _pc, _sc, metrics=_comparison_metrics,
                    n_boot=PAIRED_BOOTSTRAPS,
                    seed=PAIRED_BOOTSTRAP_SEED + int(_resolution) + sum(map(ord, _reference + _candidate)),
                )
                for _res in _pair_results:
                    _comparison_rows.append({
                        "dataset": _dataset,
                        "resolution": int(_resolution),
                        "reference_model": _reference,
                        "candidate_model": _candidate,
                        "contrast": _contrast,
                        **_res,
                    })

    campaign_paired = pd.DataFrame(_comparison_rows)
    campaign_paired["p_holm_primary_within_resolution"] = np.nan
    _primary_mask = campaign_paired["metric"] == PAIRED_PRIMARY_METRIC
    for (_dataset, _resolution), _idx in campaign_paired[_primary_mask].groupby(["dataset", "resolution"]).groups.items():
        _idx = list(_idx)
        campaign_paired.loc[_idx, "p_holm_primary_within_resolution"] = holm_adjust(
            campaign_paired.loc[_idx, "p_raw_two_sided"].to_numpy(dtype=float)
        )
    campaign_paired["p_holm_primary_global"] = np.nan
    for _dataset, _idx in campaign_paired[_primary_mask].groupby("dataset").groups.items():
        _idx = list(_idx)
        campaign_paired.loc[_idx, "p_holm_primary_global"] = holm_adjust(
            campaign_paired.loc[_idx, "p_raw_two_sided"].to_numpy(dtype=float)
        )
    campaign_paired["reject_holm_primary_global_0_05"] = (
        campaign_paired["p_holm_primary_global"] < PAIRED_ALPHA
    ).fillna(False)
    # Adjust each endpoint in its own predeclared family. This avoids
    # an unusable 216-test omnibus family with insufficient empirical
    # p-value resolution at the configured bootstrap budget.
    campaign_paired["p_holm_metric_global"] = np.nan
    for (_dataset, _metric), _idx in campaign_paired.groupby(
        ["dataset", "metric"]
    ).groups.items():
        _idx = list(_idx)
        campaign_paired.loc[_idx, "p_holm_metric_global"] = holm_adjust(
            campaign_paired.loc[_idx, "p_raw_two_sided"].to_numpy(dtype=float)
        )
    campaign_paired["reject_holm_metric_global_0_05"] = (
        campaign_paired["p_holm_metric_global"] < PAIRED_ALPHA
    )
    return campaign_paired

def master_statistics_policy():
    return {
        "schema": "pneumonia.master.paired.statistics.v1.7",
        "primary_metric": PAIRED_PRIMARY_METRIC,
        "secondary_metrics": list(PAIRED_SECONDARY_METRICS),
        "paired_bootstraps": PAIRED_BOOTSTRAPS,
        "paired_bootstrap_seed": PAIRED_BOOTSTRAP_SEED,
        "bootstrap_unit": "PATIENT_CLUSTER_PAIRED",
        "multiple_comparison_correction": "HOLM",
        "multiplicity_policy": PAIRED_MULTIPLICITY_POLICY,
        "predeclared_comparisons": [list(x) for x in PREDECLARED_MODEL_COMPARISONS],
        "locked_test_role": "CONFIRMATORY_REPORT_ONLY_NO_MODEL_OR_THRESHOLD_TUNING",
        "p_value_method": "APPROXIMATE_TWO_SIDED_BOOTSTRAP_TAIL_NOT_EXACT_PERMUTATION",
        "estimator_changed_from_v1_6": False,
    }


def validate_campaign_statistical_table(table):
    metrics = (PAIRED_PRIMARY_METRIC,) + tuple(PAIRED_SECONDARY_METRICS)
    key_cols = ["dataset", "resolution", "reference_model", "candidate_model", "contrast", "metric"]
    required = set(key_cols) | {
        "reference", "candidate", "delta_candidate_minus_reference", "ci95_low", "ci95_high",
        "p_raw_two_sided", "p_holm_metric_global", "p_holm_primary_global",
        "p_holm_primary_within_resolution", "n_bootstrap_valid", "n_patients", "bootstrap_unit",
    }
    if table.empty or not required.issubset(table.columns):
        raise ValueError("Campaign statistics are empty or missing required fields")
    expected = {
        (dataset, int(resolution), ref, cand, contrast, metric)
        for dataset in ("OOF", "LOCKED_TEST") for resolution in PHASE2_RESOLUTIONS
        for ref, cand, contrast in PREDECLARED_MODEL_COMPARISONS for metric in metrics
    }
    actual = set(table[key_cols].itertuples(index=False, name=None))
    if table.duplicated(key_cols).any() or actual != expected or len(table) != len(expected):
        raise ValueError("Campaign statistics do not cover the exact predeclared comparison matrix")
    for col in ("reference", "candidate", "delta_candidate_minus_reference", "ci95_low", "ci95_high",
                "p_raw_two_sided", "p_holm_metric_global", "n_bootstrap_valid", "n_patients"):
        values = pd.to_numeric(table[col], errors="raise").to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError(f"Non-finite campaign statistical value: {col}")
    for col in ("p_raw_two_sided", "p_holm_metric_global"):
        if not table[col].between(0, 1).all():
            raise ValueError(f"Campaign p-values out of range: {col}")
    if not (table["ci95_low"] <= table["ci95_high"]).all():
        raise ValueError("Reversed campaign confidence interval")
    if not (table["bootstrap_unit"] == "PATIENT_CLUSTER_PAIRED").all():
        raise ValueError("Unexpected campaign bootstrap unit")
    for col, lower, upper in (
        ("n_bootstrap_valid", max(100, int(PAIRED_BOOTSTRAPS) // 2), int(PAIRED_BOOTSTRAPS)),
        ("n_patients", 2, np.inf),
    ):
        values = table[col].to_numpy(dtype=float)
        if not ((values >= lower) & (values <= upper) & (values == np.floor(values))).all():
            raise ValueError(f"Invalid campaign count: {col}")
    primary = table["metric"] == PAIRED_PRIMARY_METRIC
    for col in ("p_holm_primary_global", "p_holm_primary_within_resolution"):
        values = table.loc[primary, col].to_numpy(dtype=float)
        if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
            raise ValueError(f"Invalid primary Holm p-values: {col}")
    return table


def finalize_master_statistics(matrix_report, matrix_contract, matrix_artifacts):
    """Report-only: no restore, training, prediction or hyperparameter selection.

    The master status is pending BEFORE statistics, and COMPLETE is published
    atomically only after checking and hashing both final statistical outputs.
    A failed stage keeps an explicit, restartable non-complete receipt.
    """
    root = PHASE2_CAMPAIGN_ROOT
    marker = root / "MASTER_CAMPAIGN_REPORT.json"
    artifacts = dict(matrix_artifacts)
    input_hashes = {name: run_file_sha256(path) for name, path in artifacts.items()}
    report = dict(matrix_report, schema="pneumonia.master.m01_m12.v1.7",
                  status="TRAINING_COMPLETE_STATS_PENDING", statistics_status="PENDING")
    report.pop("statistics_error", None)
    for key in ("run_contract", "run_fingerprint", "artifact_sha256", "receipt_sha256"):
        report.pop(key, None)
    atomic_write_json(marker, seal_receipt(report, matrix_contract, artifacts))
    try:
        table = validate_campaign_statistical_table(build_campaign_paired_statistics())
        stats_path = root / "MASTER_PAIRED_PATIENT_BOOTSTRAP_HOLM.csv"
        table.to_csv(stats_path, index=False, float_format="%.17g")
        validate_campaign_statistical_table(pd.read_csv(stats_path, float_precision="round_trip"))
        policy_path = root / "MASTER_STATISTICAL_COMPARISON_POLICY.json"
        atomic_write_json(policy_path, master_statistics_policy())
        if input_hashes != {name: run_file_sha256(path) for name, path in artifacts.items()}:
            raise ValueError("Campaign evidence changed during statistical reporting")
        artifacts.update(paired_statistics=stats_path, statistical_policy=policy_path)
        report.update(status="PHASE2_COMPLETE", statistics_status="COMPLETE",
                      statistical_rows=int(len(table)),
                      statistics_runtime=runtime_environment())
        final = seal_receipt(report, matrix_contract, artifacts)
        validate_receipt(final, matrix_contract, artifacts, allowed_statuses={"PHASE2_COMPLETE"})
        atomic_write_json(marker, final)
        return final
    except Exception as exc:
        report.update(status="TRAINING_COMPLETE_STATS_FAILED", statistics_status="FAILED",
                      statistics_error={"type": type(exc).__name__, "message": str(exc)})
        # Preserve initial input hashes; never re-attest mutated inputs on failure.
        failed = seal_receipt(report, matrix_contract)
        failed.pop("receipt_sha256", None)
        failed["artifact_sha256"] = input_hashes
        failed["receipt_sha256"] = contract_fingerprint(failed)
        atomic_write_json(marker, failed)
        raise


MASTER_PHASE2_PLAN = [
    (model_id, int(resolution))
    for resolution in PHASE2_RESOLUTIONS
    for model_id in PHASE2_MODEL_ORDER
]

if any(model_id == "M07" for model_id, _ in MASTER_PHASE2_PLAN):
    raise RuntimeError("M07 must never appear in Phase-2; it is completed in the multi-resolution Gate.")
if set(PHASE2_RESOLUTIONS) != set(M07_GATE_RESOLUTIONS):
    raise RuntimeError("Phase-2 resolution plan must match the predeclared M07 Gate resolutions.")

master_plan_df = pd.DataFrame(
    MASTER_PHASE2_PLAN,
    columns=[
        "model_id",
        "resolution",
    ],
)
master_plan_df.insert(
    0,
    "run_order",
    np.arange(
        1,
        len(
            master_plan_df
        ) + 1,
    ),
)

master_plan_df.to_csv(
    PHASE2_CAMPAIGN_ROOT
    / "MASTER_PHASE2_PLAN.csv",
    index=False,
)

print("=" * 110)
print("MASTER CAMPAIGN PHASE-2 PLAN")
print("=" * 110)
display(
    master_plan_df
)

if not UNLOCK_REMAINING_MODELS:
    lock_report = {
        "schema": "pneumonia.master.gate.v1",
        "status": "LOCKED_AFTER_M07_MULTIRES_GATE",
        "unlock_flag": False,
        "m07_gate_report": str(
            M07_MULTIRES_GATE_RECEIPT_PATH
        ),
        "phase2_plan": master_plan_df.to_dict(
            orient="records"
        ),
        "instruction": (
            "Review M07 execution integrity against the predeclared protocol. "
            "Only after manual approval set UNLOCK_REMAINING_MODELS=True "
            "and Run All again. Do not tune model, recipe, or thresholds using locked-test or external results."
        ),
    }
    (
        PHASE2_CAMPAIGN_ROOT
        / "MASTER_LOCKED_AFTER_M07.json"
    ).write_text(
        json.dumps(
            json_safe(
                lock_report
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("\n🔒 PHASE-2 IS LOCKED.")
    print(
        "M01–M06 and M08–M12 are planned; M07 is permanently excluded from Phase-2."
    )
    print(
        "After manual M07 approval: set UNLOCK_REMAINING_MODELS=True and Run All."
    )

else:
    _gate_payload = json.loads(M07_MULTIRES_GATE_RECEIPT_PATH.read_text(encoding="utf-8"))
    validate_receipt(
        _gate_payload, m07_multires_gate_contract(), m07_multires_gate_artifacts(),
        allowed_statuses={"COMPLETE"},
    )
    if not _gate_payload.get("phase2_m07_retraining_allowed") is False:
        raise RuntimeError("M07 multi-resolution Gate receipt does not prohibit Phase-2 retraining.")

    gate_source_path = M07_MULTIRES_GATE_RECEIPT_PATH
    gate_source = _gate_payload

    print("\n🔓 PHASE-2 UNLOCKED BY EXPLICIT USER FLAG")
    print(
        "The M07 Gate state will be restored; remaining campaign now runs."
    )

    phase2_reports = []

    for (
        model_id,
        resolution,
    ) in MASTER_PHASE2_PLAN:
        report_path = (
            phase2_output_root(
                model_id,
                resolution,
            )
            / "FINAL_REPORT.json"
        )

        # Reconcile before deciding whether a local completion may be skipped.
        if not MASTER_REPORT_ONLY:
            phase2_restore(model_id, resolution)
        if report_path.exists():
            existing, _ = phase2_validate_report(model_id, resolution)
            if not MASTER_REPORT_ONLY:
                phase2_ensure_final_persisted(model_id, resolution)
            print(f"[MASTER SKIP] {model_id} R{resolution} validated complete")
            phase2_reports.append(existing)
            continue

        if MASTER_REPORT_ONLY:
            raise ValueError(f"Report-only requires an existing valid local report: {model_id} R{resolution}")
        report = run_phase2_model_resolution(
            model_id,
            resolution,
        )
        phase2_reports.append(
            report
        )

    # --------------------------------------------------------
    # Aggregate matrix report
    # --------------------------------------------------------
    # Read only current-plan runs after validating their complete evidence receipts.
    quick_parts = []
    validated_plan = []
    for model_id, resolution in MASTER_PHASE2_PLAN:
        validated, quick = phase2_validate_report(model_id, resolution)
        quick_parts.append(quick)
        validated_plan.append({"model_id": model_id, "resolution": int(resolution),
                               "report_fingerprint": validated["run_fingerprint"]})

    # M07 was completed before Phase-2 at all three resolutions. Add its sealed Gate rows
    # to the final matrix without retraining it.
    _gate_payload = json.loads(M07_MULTIRES_GATE_RECEIPT_PATH.read_text(encoding="utf-8"))
    validate_receipt(
        _gate_payload, m07_multires_gate_contract(), m07_multires_gate_artifacts(),
        allowed_statuses={"COMPLETE"},
    )
    _m07_gate_quick = pd.read_csv(M07_MULTIRES_STATS_ROOT / "M07_MULTIRES_QUICK_RESULTS.csv")
    if set(_m07_gate_quick["resolution"].astype(int)) != set(M07_GATE_RESOLUTIONS):
        raise ValueError("M07 multi-resolution matrix rows are incomplete.")
    quick_parts.append(_m07_gate_quick)

    matrix = (
        pd.concat(
            quick_parts,
            ignore_index=True,
            sort=False,
        )
        if quick_parts
        else pd.DataFrame()
    )

    matrix.to_csv(
        PHASE2_CAMPAIGN_ROOT
        / "MASTER_M01_M12_RESULTS_MATRIX.csv",
        index=False,
    )

    matrix_report = {
        "schema": "pneumonia.master.m01_m12.v1",
        "status": "TRAINING_COMPLETE_STATS_PENDING",
        "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
        "confirmed_m07_recipe_fingerprint": recipe[
            "recipe_fingerprint_sha256"
        ],
        "resolutions": list(
            PHASE2_RESOLUTIONS
        ),
        "models": sorted({"M07", *(m for m, _ in MASTER_PHASE2_PLAN)}),
        "validated_current_plan": validated_plan,
        "completion_scope": "CONFIGURED_PLAN_ONLY",
        "repeated_seed_stability": "NOT_IMPLEMENTED",
        "m07_gate_resolution": 224,
        "m07_multires_gate_preceded_phase2": True,
        "m07_gate_resolutions": list(M07_GATE_RESOLUTIONS),
        "m07_retrained_in_phase2": False,
        "f2_role": "REPORT_ONLY",
        "dual20_80_role": "REPORT_ONLY",
        "locked_test_used_for_selection": False,
        "external_used_for_selection": False,
        "results_rows": int(
            len(matrix)
        ),
    }

    matrix_contract = make_run_contract(
        "phase2_matrix", model_id="M01_M12_CONFIGURED_PLAN", resolution=224, params=shared_params,
        extra={"validated_plan": validated_plan, "gate_fingerprint": gate_source["run_fingerprint"]},
    )
    matrix_artifacts = {
        "matrix": PHASE2_CAMPAIGN_ROOT / "MASTER_M01_M12_RESULTS_MATRIX.csv",
        "plan": PHASE2_CAMPAIGN_ROOT / "MASTER_PHASE2_PLAN.csv",
        "m07_gate_receipt": gate_source_path,
    }
    matrix_artifacts.update({f"{model_id}_R{resolution}_report": phase2_output_root(model_id, resolution) / "FINAL_REPORT.json"
                             for model_id, resolution in MASTER_PHASE2_PLAN})

    # --------------------------------------------------------
    # Predeclared paired patient-bootstrap + Holm comparison stage.
    # Locked-Test statistics are confirmatory/report-only and MUST NOT feed tuning.
    # --------------------------------------------------------
    finalize_master_statistics(matrix_report, matrix_contract, matrix_artifacts)

    print("\n" + "=" * 110)
    print("MASTER M01–M12 CAMPAIGN COMPLETE")
    print("=" * 110)
    display(
        matrix.round(
            4
        )
    )